# Pillar-0 feature extraction + classification — Indian hospital matched scans **and** RSNA-ICH

**One switch, two datasets.** `DATA_SOURCE` in the config cell selects:

* `"india"` (Part A) — your matched government-hospital scans: day-zips on Drive, labels from
  matched radiology reports.
* `"rsna"` (Part B) — RSNA 2019 Intracranial Hemorrhage Detection: a split archive on Drive,
  expert per-slice labels aggregated to series level.

There is also a **Part C**: one binary hemorrhage model trained on RSNA alone, on the Indian scans
alone, and on both pooled, with all three scored on the same held-out patients from each dataset.

Only the manifest-building cells differ. Preprocessing, the Pillar-0 call, the feature store, the
QC checks and the whole classifier sweep are **the same code for both**, which is what makes a
difference in results between them a difference in the *data* rather than in the pipeline. The
RSNA section has its own header further down; everything below describes Part A.

---


**What this notebook is.** A merge of two earlier notebooks, rebuilt around what the matched-scans
CSV actually tells us:

* `pillar0_headct_zipfolder_colab (14).ipynb` — verified Pillar-0 loading, the 11-channel
  windowing / 256x256x128 preprocessing recipe, and the model-calling convention.
* `matched_scans_feature_extraction_and_eval_v6.ipynb` — the matched-series selection, zip-based
  volume assembly, resumability pattern and the classifier sweep.

Everything runs on the **matched** scans only (the series that were successfully paired to a
radiology report), extracts **Pillar-0 embeddings**, saves them, and then trains classifiers
against the report-derived finding labels.

---

## The single biggest change: the manifest is now *deterministic*

The v6 notebook found each wanted series by opening every day-zip and `dcmread`-ing **every entry's
header** — hundreds of gigabytes of zip members decoded just to learn where things are.

That is not necessary. In `matched_scans_combined_*.csv`, `SourceFolder` is literally

```
MM/DD/<StudyInstanceUID>/<SeriesInstanceUID>
```

Verified directly on all 42,193 rows of `matched_scans_combined_slim.csv`:

* every `SourceFolder` has exactly 4 components,
* component 3 == `StudyInstanceUID` for **100%** of rows,
* component 4 == `SeriesInstanceUID` for **100%** of rows.

And on Drive the archives are laid out as `.../2025/<MM>/<DD>.zip` (confirmed by listing the Drive
folder: `2025/01/01.zip … 2025/01/28.zip`, `2025/05/01.zip …`, each ~7–28 GB).

So the day-zip holding any matched series is **computable, not discoverable**. Cell 9 builds the
whole manifest from the CSV in seconds. Zip contents are then only inspected through
`ZipFile.namelist()` (the central directory) — no DICOM header decoding just to locate things —
and the located series is verified by reading exactly **one** header before any pixel work.

## What gets extracted, and with which checkpoint

`Pillar0-HeadCT` is a *head CT* model. Of the matched data:

| slice of the data | series | studies |
|---|---|---|
| all matched rows | 42,193 | 6,124 |
| `Modality == CT` | 29,805 | 6,105 |
| `BodyPartExamined` contains HEAD | 13,134 | 3,222 |
| … and `NumSlices >= 20` (drops scouts/dose reports) | 9,852 | 3,205 |

A typical head study here holds three real reconstructions —
`Brain Plain 5.00 Hr40 S3`, `Brain Plain 2.00 Hr40 S3`, `Brain Plain 1.00 Hr60 S3` — plus a
`Topogram 0.70 Tr20 sag` scout. Cell 8 scores them and, by default, keeps **one primary series per
study**: brain-ish description, soft-tissue kernel preferred over bone kernel, then most slices.
That is ~3,205 volumes. Set `PROCESS_ALL_QUALIFYING_SERIES = True` to do all ~9,852 instead.

`Pillar0-ChestCT` and `Pillar0-AbdomenCT` are also registered (cell 15) so the chest/abdomen half of
the matched data can be run later with the right checkpoint — but head is the default, because
running a head model over chest CTs produces embeddings that mean nothing.

## Resumability — every stage, not just extraction

| stage | checkpoint | what re-running does |
|---|---|---|
| load 800 MB CSV | `matched_table.parquet` | reloads the cached parquet in seconds |
| manifest | `manifest.csv` | rebuilt only if the CSV or config changed (hash-keyed) |
| zip member location | `zip_index/<mm>_<dd>.json` | skips zips already indexed |
| local zip copy | `.part` + size check | resumes/reuses; never half-copies |
| embedding extraction | `extraction_progress.jsonl` | only `status=="ok"` counts as done; failures retry |
| whole-zip completion | `zips_done.json` | a fully-done zip is never reopened |
| feature table | `features_<model>.parquet` + `.npz` | rebuilt incrementally |
| classifier sweep | `classifier_results.jsonl` | per (cohort, feature set, model) records |

Extraction also honours `MAX_SERIES_PER_RUN` / `MAX_SECONDS_PER_RUN` and traps `KeyboardInterrupt`,
so a Colab session can end cleanly at any point and the next one picks up exactly where it stopped.

## Things that were checked against primary sources, not assumed

* **11 CT windows**: `rve`'s `ANATOMICAL_WINDOWS["CT"]` has 10 entries + `minmax` = 11. Read from
  the actual `vision_engine/utils/windowing_utils.py` source.
* **Pad value**: RVE's `_center_crop_pad_3d` uses `pad_value = float(volume.min())`. Read from
  `vision_engine/processing/ct_processor.py`. This notebook does the same.
* **Resample default pixel value**: RVE uses `SetDefaultPixelValue(0)`. Notebook 14 used
  `volume.min()`. Matched to RVE here; the difference is immaterial (the output grid covers the
  input extent, so out-of-bounds sampling is at most a one-voxel border).
* **Honest discrepancy**: RVE's own `_crop_pad_volume` **crops** Z to 128 but never **pads** it, and
  `_center_crop_pad_3d` only touches H and W. The paper (arXiv 2511.17803, Appendix A.5) says
  head CTs were "cropped or padded to 256 x 256 x 128". This notebook pads Z to 128, following the
  paper, because the model's patch grid depends on it. Flagged so you know it is a choice.
* **Modality key**: not hardcoded any more. Cell 15 reads the loaded model's own modality-keyed
  `ModuleDict`s and picks the key from the model itself, falling back to `"brain_ct"` (the value
  two real tracebacks confirmed) only if discovery finds nothing.

## A real bug carried over from v6, fixed here

v6's hemorrhage cohort matched finding columns by keyword — `"subdural"`, `"subarachnoid"`,
`"intraventricular"`. **This dataset names those columns `b_sdh`, `b_sah`, `b_ivh`, `b_edh`,
`b_ich`.** None of them contain the keywords. Only `b_hemorrhagic_contusion` would have matched, so
`ANY_HEMORRHAGE` silently dropped ~145 SDH + 133 SAH + 144 ICH + EDH + IVH positives. Cell 20 now
matches abbreviations as whole tokens as well, and prints every column it matched so you can check
it on the first run.

In [ ]:
# 0. Repository and paths -- every location is read from config.yaml (see README), none is typed here.
import os
import sys


def _find_repo():
    for c in (os.environ.get("QCT3D_REPO", ""), os.getcwd(), os.path.dirname(os.getcwd()),
              "/content/qct3d-saturation"):
        if c and os.path.isfile(os.path.join(c, "src", "qct3d", "config.py")):
            return os.path.abspath(c)
    raise RuntimeError("Cannot find the repository. Clone it first (README, 'Running on Colab') "
                       "or set the environment variable QCT3D_REPO to its folder.")


REPO_ROOT = _find_repo()
if os.path.join(REPO_ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
from qct3d.config import load_config  # noqa: E402

P = load_config()

In [ ]:
# 1. Google Drive and credentials -- the ONLY cell that needs you at the keyboard
#
# Everything interactive is deliberately front-loaded into this one cell so that the rest of the
# notebook can run unattended. Two things happen here:
#
#   1. Drive is mounted. On the first run of a session this pops an authorisation dialog; click
#      through it and you are done for the night.
#   2. The Hugging Face token is read from Colab Secrets.
#
# `notebook_login()` is deliberately NOT used. It renders a widget and blocks until someone types
# into it, which means an unattended "Run all" stops dead here and you come back in the morning to
# a notebook that has done nothing. Reading the token from Secrets is the unattended-safe form.
#
# NOTE ON ORDERING: this cell runs BEFORE the installs on purpose. It imports nothing from the
# packages the next cell pins (transformers, huggingface_hub), so pip is free to change them
# without leaving a stale module loaded in this kernel.
import os
import sys

# ---- Drive ------------------------------------------------------------------------------------
try:
    from google.colab import drive
    drive.mount(P["drive_mount"])
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("not running in Colab -- skipping the Drive mount. Point the paths in the config cell "
          "at wherever the data actually is.")

if IN_COLAB:
    _mydrive = P["drive_root"]
    if not os.path.isdir(_mydrive):
        raise RuntimeError(
            f"{_mydrive} does not exist after mounting. The mount did not complete -- re-run this "
            f"cell and finish the authorisation dialog before starting a long run."
        )
    print(f"Drive mounted: {_mydrive}")

# ---- Hugging Face token -----------------------------------------------------------------------
# Add it under the key icon in the left sidebar: name HF_TOKEN, value your token, and turn on
# "Notebook access" for this notebook. The Pillar-0 checkpoints are gated, so the token also needs
# access granted at https://huggingface.co/YalaLab/Pillar0-HeadCT -- which is a separate thing from
# having a valid token, and is checked properly in the preflight cell.
HF_TOKEN = None
_token_source = None

if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        _token_source = "Colab Secrets (HF_TOKEN)"
    except Exception as e:
        _cls = type(e).__name__
        if "NotebookAccess" in _cls:
            print("HF_TOKEN exists in Secrets but this notebook is not authorised to read it.")
            print("  Fix: key icon in the left sidebar -> HF_TOKEN -> turn ON 'Notebook access'.")
        elif "SecretNotFound" in _cls:
            print("No secret named HF_TOKEN found.")
            print("  Fix: key icon in the left sidebar -> Add new secret -> name it exactly")
            print("       HF_TOKEN, paste your token, and enable 'Notebook access'.")
        else:
            print(f"could not read Colab Secrets ({_cls}: {e})")

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if HF_TOKEN:
        _token_source = "environment variable"

if not HF_TOKEN:
    raise RuntimeError(
        "No Hugging Face token available.\n"
        "\n"
        "This is raised NOW, on purpose, rather than letting the run get as far as downloading the\n"
        "model and failing there. Add the secret and re-run this cell:\n"
        "  1. key icon in the Colab sidebar -> Add new secret\n"
        "  2. name:  HF_TOKEN     value: your token from https://huggingface.co/settings/tokens\n"
        "  3. enable 'Notebook access' for this notebook\n"
        "and make sure that account has been GRANTED access at\n"
        "  https://huggingface.co/YalaLab/Pillar0-HeadCT"
    )

# Exported so every library picks it up without a second login step, and so the extraction loop
# can re-authenticate silently if a session-level token ever expires mid-run.
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print(f"HF token loaded from {_token_source} "
      f"({HF_TOKEN[:4]}...{HF_TOKEN[-4:]}, {len(HF_TOKEN)} chars) -- validated in the preflight cell")

---
## What changed from the run you just did, and why

Four lines. Everything else is byte-identical to the Kadapa notebook.

| line | was | now | why |
|---|---|---|---|
| `PILLAR0_INPLANE_MODE` (cell 4) | `"crop"` | `"fit_head"` | the crop threw away a **median 29.4% of the head** — 100% of series lost more than 2%, 10th percentile kept only 62.8%. What it cut is the lateral convexity, where SDH and EDH live. |
| `INDIA_OUTPUT_DIR` (cell 4) | `..._output` | `..._output_fithead` | fresh dir, so the crop run stays intact as the comparison arm |
| `MIN_POSITIVES` (cell 4) | `30` | `20` | at 30, `b_sdh` (25 positives) and `b_sah` (24) were dropped before evaluation |
| `REQUIRE_STUDYTYPE_AGREEMENT` (cell 21) | `False` | `True` | 123 scans (10.2%) carried a label from a report about a **different body part** — forced negatives with wrong labels |

**On the zips you haven't uploaded yet:** start this run now. Extraction is resumable per
`(series, model)`, so when you later drop months 02, 03/23–31, 04 and 05/17–31 into
`CT_ROOT`, you re-run **cell 9** (the manifest cache key includes `CT_ROOT`, so it re-resolves)
and then cell 17 — it picks up only the ~379 new studies and leaves the finished ones alone.

**Runtime:** ~1,200 series at the rate you just measured (13 s/series end to end) is about
4.5 hours. `fit_head` adds a bounding-box pass, so budget 5.

---

---
## 1b. Build the Nellore CT-only Pillar-0 input  *(new cell — run once)*

Everything below this cell is the Kadapa notebook, unchanged except three lines in cell 7
(`DATA_SOURCE`, `CSV_PATH`, `INDIA_OUTPUT_DIR`).

This cell assembles the table those three lines point at:

| piece | where it comes from |
|---|---|
| **part 2** series rows | `nellore_matched_scans_pillar0_ready.csv` in Drive — **read only, never modified** |
| **part 1** series rows | `part1_slim.csv` in Drive, filtered to the 2,103 hand-matched study UIDs |
| **part 1** report block | `nellore_2025_wide_final.csv` in Drive, joined on `study_id` |
| **the part-1 map** | 2,261 hand-checked `(study_id → StudyInstanceUID)` pairs, embedded in this cell as gzip+base64 so nothing can go missing |

**MRI and SR are dropped.** The part-2 half carried 15,463 MR and 3,106 SR series: they came
along because that merge joined on `PatientID`, so every series belonging to a matched patient
was pulled in, including the MRI from the same visit. The reports are all CT reports, so an MR
study with a CT report attached is a false label. Pillar-0 filters `Modality == "CT"` anyway —
this just removes them from the file itself. Part 1 was matched CT-only from the start.

Output: `nellore_matched_scans_pillar0_ready_v2_CT.csv` in the folder `external_input_dir` (config.yaml).
Also writes `nellore_ALL_matched_pairs_combined.csv` there for provenance.

Runtime: plain CPU, ~3–6 min, peak RAM well under 2 GB. Safe to re-run — it skips if the
output already exists and passes its checks.

---

In [ ]:
# 1b. BUILD THE NELLORE CT-ONLY PILLAR-0 INPUT  [run once; idempotent]
# ---------------------------------------------------------------------------------------------
# Reads part 2's ready CSV (unmodified), adds part 1's hand-matched studies, drops MR/SR,
# and writes the single table cell 7's CSV_PATH points at.
# ---------------------------------------------------------------------------------------------
import os, io, gc, glob, gzip, base64, json, shutil
import pandas as pd, numpy as np

INPUT_DIR = P["external_input_dir"]
os.makedirs(INPUT_DIR, exist_ok=True)
OUT_V2    = P["external_matched_csv"]
OUT_PAIRS = os.path.join(INPUT_DIR, "nellore_ALL_matched_pairs_combined.csv")

# Optional fallback: Google Drive file IDs of the raw inputs, read from the private
# config.yaml (external_drive_ids). Used only if the mounted path cannot be found.
DRIVE_IDS = dict(P.get("external_drive_ids") or {})

def locate(name):
    """Mounted-path first (free), Drive API by file id second (robust)."""
    direct = os.path.join(INPUT_DIR, name)
    if os.path.exists(direct):
        return direct
    hits = glob.glob(os.path.join(P["drive_root"], "**", name), recursive=True)
    if hits:
        hits.sort(key=lambda p: -os.path.getsize(p))
        return hits[0]
    local = "/content/" + name
    if os.path.exists(local) and os.path.getsize(local) > 1000:
        return local
    fid = DRIVE_IDS.get(name)
    if not fid:
        raise FileNotFoundError(name)
    print(f"  {name}: not on the mounted path - downloading by Drive id ...")
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()
    svc = build("drive", "v3", cache_discovery=False)
    with open(local, "wb") as fh:
        dl = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid),
                                 chunksize=64 * 1024 * 1024)
        done = False
        while not done:
            st, done = dl.next_chunk()
            print(f"    {int(st.progress()*100)}%", end="\r")
    print(f"\n    -> {local} ({os.path.getsize(local)/1e6:.1f} MB)")
    return local

# --- the 2,261 hand-checked part-1 pairs, read from a private file ---------------------------
# This table (study_id, StudyInstanceUID, scan_id, report_date, track) identifies patients'
# studies, so it is NOT distributed. In the original notebook it was embedded here; it is now
# read from the private path external_part1_map_csv in config.yaml.
if not os.path.exists(P["external_part1_map_csv"]):
    raise FileNotFoundError(
        f"{P['external_part1_map_csv']} not found. This study map is part of the external cohort's "
        f"protected data and is available only under the institutional data-sharing agreement.")
p1map = pd.read_csv(P["external_part1_map_csv"], dtype=str)
assert len(p1map) == 2261, len(p1map)
print(f"part-1 map: {len(p1map):,} pairs | "
      f"{p1map.study_id.nunique():,} reports | {p1map.StudyInstanceUID.nunique():,} studies")

if os.path.exists(OUT_V2) and os.path.getsize(OUT_V2) > 10e6:
    _chk = pd.read_csv(OUT_V2, dtype=str, usecols=["Modality", "StudyInstanceUID"])
    if set(_chk.Modality.dropna().unique()) == {"CT"}:
        print(f"\n{OUT_V2}\nalready built: {len(_chk):,} CT series, "
              f"{_chk.StudyInstanceUID.nunique():,} studies. Skipping rebuild.")
        del _chk; gc.collect()
        BUILD = False
    else:
        BUILD = True
else:
    BUILD = True

if BUILD:
    # ---- schema (the 208 columns part 2's file already uses) -----------------------------
    READY = locate("nellore_matched_scans_pillar0_ready.csv")
    COLS  = list(pd.read_csv(READY, dtype=str, nrows=1).columns)
    assert len(COLS) == 208, len(COLS)
    META, REP = COLS[:40], COLS[40:]
    MATCH_COLS = ["Hospital", "MatchMethod", "MatchedReportName", "MatchedReportAge",
                  "MatchedReportSex", "MatchedReportDate", "MatchedStudyType"]

    # ---- part 2: read, keep CT only, leave the source file untouched ----------------------
    print("\n[1/4] part 2 - reading and filtering to CT ...")
    keep = []
    for ch in pd.read_csv(READY, dtype=str, chunksize=100_000):
        keep.append(ch[ch["Modality"] == "CT"])
    p2 = pd.concat(keep, ignore_index=True); del keep; gc.collect()
    p2 = p2.reindex(columns=COLS)
    print(f"      {len(p2):,} CT series | {p2.StudyInstanceUID.nunique():,} studies")

    # ---- part 1: pull just the matched studies out of the 241 MB slim file ----------------
    print("\n[2/4] part 1 - streaming part1_slim.csv for the matched studies ...")
    SLIM  = locate("part1_slim.csv")
    want  = set(p1map["StudyInstanceUID"])
    grab, scanned = [], 0
    for ch in pd.read_csv(SLIM, dtype=str, chunksize=200_000, on_bad_lines="skip"):
        scanned += len(ch)
        sel = ch[ch["StudyInstanceUID"].isin(want) & (ch["Modality"] == "CT")]
        if len(sel):
            grab.append(sel)
        del ch
    p1s = pd.concat(grab, ignore_index=True); del grab; gc.collect()
    print(f"      scanned {scanned:,} rows -> {len(p1s):,} CT series | "
          f"{p1s.StudyInstanceUID.nunique():,}/{len(want):,} studies found")

    # ---- part 1: attach the report block, exactly the way merge_core did ------------------
    print("\n[3/4] part 1 - attaching reports ...")
    RPT = locate("nellore_2025_wide_final.csv")
    rpt = pd.read_csv(RPT, dtype=str)
    rpt = rpt.drop_duplicates("study_id", keep="first")     # 6 study_ids are duplicated at source
    rpt = rpt.reindex(columns=REP)

    link = p1map[["StudyInstanceUID", "study_id"]].drop_duplicates("StudyInstanceUID")
    link = link.merge(rpt[["study_id", "patient_name", "age", "sex",
                           "report_date", "study_type", "institution"]],
                      on="study_id", how="left")
    link["Hospital"]          = link["institution"].fillna("GGH NELLORE")
    link["MatchMethod"]       = "MATCH_HIGH | part1 hand-verified (name+date)"
    link["MatchedReportName"] = link["patient_name"]
    link["MatchedReportAge"]  = link["age"]
    link["MatchedReportSex"]  = link["sex"]
    link["MatchedReportDate"] = link["report_date"]
    link["MatchedStudyType"]  = link["study_type"]
    link = link[["StudyInstanceUID", "study_id"] + MATCH_COLS]

    p1 = p1s.merge(link, on="StudyInstanceUID", how="left")
    p1 = p1.merge(rpt, on="study_id", how="left", suffixes=("", "_rpt"))
    p1 = p1.reindex(columns=COLS)
    print(f"      {len(p1):,} rows in the 208-column schema | "
          f"report attached on {p1.study_id.notna().mean()*100:.1f}% of rows")

    # ---- stack, verify, write ------------------------------------------------------------
    print("\n[4/4] stacking and verifying ...")
    v2 = pd.concat([p2, p1], ignore_index=True)

    assert list(v2.columns) == COLS, "column drift"
    assert set(v2["Modality"].dropna().unique()) == {"CT"}, v2["Modality"].unique()
    dup = v2.duplicated("SeriesInstanceUID").sum()
    if dup:
        print(f"      note: {dup} duplicate SeriesInstanceUID rows - dropping")
        v2 = v2.drop_duplicates("SeriesInstanceUID", keep="first")
    for c in ("SourceFolder", "SeriesInstanceUID", "StudyInstanceUID", "NumSlices"):
        n = v2[c].isna().sum()
        assert n == 0, f"{c} has {n} nulls"
    ok = v2["SourceFolder"].str.count("/").eq(3).mean()
    assert ok > 0.99, f"SourceFolder contract only {ok:.3%}"

    v2.to_csv(OUT_V2, index=False)
    print(f"\n  -> {OUT_V2}")
    print(f"     {len(v2):,} CT series | {v2.StudyInstanceUID.nunique():,} studies | "
          f"{v2.PatientID.nunique():,} patients | {os.path.getsize(OUT_V2)/1e6:.1f} MB")

    # provenance: the full 4,659-pair table, rebuilt from the two halves
    try:
        pr2 = pd.read_csv(locate("nellore_matched_pairs_high_confidence_2398.csv"), dtype=str)
        pr2["track"] = "part2"
        allp = pd.concat([pr2, p1map], ignore_index=True)
        allp.to_csv(OUT_PAIRS, index=False)
        print(f"  -> {OUT_PAIRS}  ({len(allp):,} pairs)")
    except Exception as e:
        print(f"  (provenance pairs file skipped: {e})")

    del p2, p1, p1s, rpt; gc.collect()

# ---- what cell 7 will see -----------------------------------------------------------------
_v = pd.read_csv(OUT_V2, dtype=str, usecols=["Modality", "BodyPartExamined", "StudyInstanceUID",
                                             "PatientID", "NumSlices", "SeriesDescription"])
print("\n" + "=" * 78)
print("INPUT READY FOR PILLAR-0")
print("=" * 78)
print(f"  rows (CT series) : {len(_v):,}")
print(f"  studies          : {_v.StudyInstanceUID.nunique():,}")
print(f"  patients         : {_v.PatientID.nunique():,}")
print("  by body part     :", _v.BodyPartExamined.value_counts().head(6).to_dict())
_h = _v[_v.BodyPartExamined.astype(str).str.upper().eq("HEAD")]
print(f"  HEAD series      : {len(_h):,}  ({_h.StudyInstanceUID.nunique():,} studies)")
print("\n  cell 7 CSV_PATH is already set to this file. Run the notebook straight through.")
del _v, _h; gc.collect()


In [ ]:
# 2. Installs -- every version the notebook later CHECKS is pinned here
#
# This cell exists in this form because of a real failure: an earlier version checked that
# transformers was v4 and then raised when Colab supplied v5 -- but the install line never pinned
# it. Checking a condition that nothing establishes is not a safety net, it is a trap that fires on
# somebody else's schedule. So the rule now is: anything cell 2b asserts, this cell pins.
#
# What each pin is for:
#   transformers<5         Pillar-0's shipped modelling code targets the v4 API. v4 is the version
#                          this project's real runs were verified on (750/750 weight keys matched);
#                          v5 is simply unverified. Colab currently ships 5.x, so without this pin
#                          the notebook cannot run at all.
#   huggingface_hub<1.0    transformers 4.x requires it, and hub 1.x is what Colab ships alongside
#                          transformers 5. Pinned explicitly so the resolution is deterministic
#                          rather than a side effect.
#   pydicom>=3.0           `pydicom.pixels.pixel_array(ds, decoding_plugin=...)` is a 3.x API. On
#                          2.x it silently falls back to pydicom's default decoder, which may route
#                          through Pillow -- and pydicom's own docs warn Pillow "performs a number
#                          of undesirable operations on the decoded images which are not always
#                          reversible". Not acceptable when HU values are the entire signal.
#   pandas>=2.1            `DataFrame.map` (the replacement for `applymap`) is used in the Part C
#                          tables.
#   scikit-learn>=1.3      StratifiedGroupKFold, HistGradientBoosting(class_weight=...).
#   pylibjpeg/gdcm/...     pixel-data decompression plugins. Without them a compressed transfer
#                          syntax raises on every slice; in an earlier real run that took out 100%
#                          of extraction.
#
# numpy and pillow are deliberately NOT installed: Colab already has them, and upgrading a package
# that is already imported into a running kernel leaves a stale module behind -- exactly the kind
# of failure that only shows up three hours into an overnight run.
import sys

_already = [m for m in ("transformers", "huggingface_hub", "tokenizers", "pydicom")
            if m in sys.modules]
if _already:
    print(f"WARNING: {_already} are already imported in this kernel. pip can change them on disk "
          f"but cannot replace what is already loaded, so a restart may be required -- cell 2b "
          f"will say so explicitly.")

# The install is a Python list driven through `sys.executable -m pip`, not a shell magic with
# backslash continuations. Three reasons, all of which matter more at the top of an unattended run
# than the extra lines cost: the exit status is actually checked (a magic's failure is a printed
# message and nothing else), the version list is a data structure the test suite reads and compares
# against cell 2b's requirements, and it behaves identically outside Colab.
import subprocess
import sys

PINS = [
    "transformers>=4.40,<5",
    "huggingface_hub>=0.30,<1.0",
    "pydicom>=3.0",
    "pandas>=2.1",
    "scikit-learn>=1.3",
    "SimpleITK>=2.3",
]
EXTRAS = [
    # ---- added: our three 3D backbones are MONAI models -----------------------------
    # These MUST be installed here, in the cell that runs before anything imports numpy,
    # torch or the DICOM codecs -- not later. Installing MONAI half way down a notebook
    # that has already imported compiled extensions leaves the new versions on disk and
    # the old ones in memory, and the kernel dies on the next compiled import with no
    # traceback at all: execution simply stops. That is what happened on the previous run,
    # where every cell through the preprocessing self-test printed correctly and the
    # backbone cell was never entered.
    #
    #   monai>=1.4,<2   SwinUNETR's constructor changed in 1.4; Phase A was built on >=1.4
    #   einops          SwinUNETR's attention blocks import it at module import time
    "monai>=1.4,<2", "einops",
    "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "pyjpegls", "python-gdcm",
    "safetensors", "einops", "tqdm", "pyarrow", "matplotlib",
]

try:
    _smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=60)
    print(_smi.stdout or "(nvidia-smi produced no output)")
except Exception:
    print("(no nvidia-smi -- the config cell explains what running without a GPU costs)")

print("installing:", " ".join(PINS + EXTRAS))
_res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS, *EXTRAS],
                       capture_output=True, text=True)
if _res.stdout.strip():
    print(_res.stdout.strip()[-2000:])
if _res.returncode != 0:
    print(_res.stderr.strip()[-4000:])
    raise RuntimeError(
        f"pip exited {_res.returncode}. Nothing below this works with the wrong versions "
        f"installed, so it stops here rather than letting cell 2b explain it a minute later."
    )
if _res.stderr.strip():
    # pip writes dependency-resolution complaints about packages this notebook never imports to
    # stderr even on success. Shown, but not treated as failure.
    print("\npip notes (non-fatal):")
    print(_res.stderr.strip()[-1500:])

print("\ninstall finished -- cell 2b verifies what actually landed on disk.")

In [ ]:
# 2b. Version contract -- what is on disk, what is loaded, and whether they agree
#
# Two distinct failure modes, which need distinguishing because the fixes differ:
#
#   PIN NOT SATISFIED   pip did not install what cell 2 asked for (a resolution conflict, or the
#                       cell was skipped). Re-run cell 2.
#   RESTART REQUIRED    the right version is on disk, but this kernel already has an older/newer
#                       one loaded from before the install. pip cannot replace a module that is
#                       already in memory. Runtime > Restart session, then Run all again.
#
# The second one is the reason this is a separate cell from cell 2: inside a single cell the newly
# installed version would not be visible anyway.
import importlib
import importlib.metadata as _md

REQUIRED = {
    # dist name        (module,            min,       max_exclusive)
    "transformers":    ("transformers",    (4, 40),   (5, 0)),
    "huggingface_hub": ("huggingface_hub", (0, 30),   (1, 0)),
    "pydicom":         ("pydicom",         (3, 0),    None),
    "pandas":          ("pandas",          (2, 1),    None),
    "scikit-learn":    ("sklearn",         (1, 3),    None),
    "SimpleITK":       ("SimpleITK",       (2, 3),    None),
}


def _parse(v):
    out = []
    for part in str(v).split(".")[:3]:
        num = "".join(ch for ch in part if ch.isdigit())
        out.append(int(num) if num else 0)
    while len(out) < 3:
        out.append(0)
    return tuple(out)


problems, restarts = [], []
print(f"{'package':<18}{'on disk':<14}{'imported':<14}status")
print("-" * 66)
for dist, (mod_name, lo, hi) in REQUIRED.items():
    try:
        disk = _md.version(dist)
    except Exception:
        disk = None
    try:
        mod = importlib.import_module(mod_name)
        loaded = getattr(mod, "__version__", None)
    except Exception as e:
        mod, loaded = None, f"IMPORT FAILED ({type(e).__name__})"

    status = "ok"
    if disk is None:
        status = "NOT INSTALLED"
        problems.append(f"{dist} is not installed -- re-run cell 2")
    else:
        dv = _parse(disk)
        if dv < lo + (0,) * (3 - len(lo)):
            status = f"TOO OLD (need >= {'.'.join(map(str, lo))})"
            problems.append(f"{dist} {disk} on disk, need >= {'.'.join(map(str, lo))}")
        elif hi is not None and dv >= hi + (0,) * (3 - len(hi)):
            status = f"TOO NEW (need < {'.'.join(map(str, hi))})"
            problems.append(f"{dist} {disk} on disk, need < {'.'.join(map(str, hi))}")
        elif isinstance(loaded, str) and not loaded.startswith("IMPORT FAILED"):
            if _parse(loaded)[:2] != dv[:2]:
                status = "RESTART REQUIRED"
                restarts.append(f"{dist}: {loaded} loaded, {disk} on disk")
    print(f"{dist:<18}{str(disk):<14}{str(loaded):<14}{status}")

if restarts:
    raise RuntimeError(
        "A different version is loaded in this kernel than is installed on disk:\n  "
        + "\n  ".join(restarts)
        + "\n\npip cannot replace a module that is already imported.\n"
          "Fix: Runtime > Restart session, then Run all again. Nothing is lost -- every stage of\n"
          "this notebook is resumable, so a restart costs only the cells above this one."
    )
if problems:
    raise RuntimeError(
        "The installed versions do not match what this notebook requires:\n  "
        + "\n  ".join(problems)
        + "\n\nRe-run cell 2. If a pin genuinely cannot be satisfied, do NOT just delete the check "
          "-- it is there because the notebook uses an API that only exists in that range."
    )
print("\nall version pins satisfied, nothing stale in memory.")

In [ ]:
# 2c. Install rve (rad-vision-engine) at a PINNED commit, and verify the window contract
#
# Pinned to a commit, not to the branch. The entire 11-channel input contract depends on
# `get_available_windows("CT")` returning exactly these 11 names in exactly this order -- if
# upstream ever adds a twelfth window, an unpinned install would silently change the model's input
# channel count on some future night. The assertion below would catch it, but only after the
# install had already changed under you; pinning means it cannot happen in the first place.
#
# Same reasoning as the transformers pin in cell 2: anything this notebook asserts, it also fixes.
RVE_COMMIT = "20adeb873021c864e9410fae8898b0c8da309769"   # YalaLab/rave, 2025-12-09

import subprocess
import sys

_spec = f"git+https://github.com/YalaLab/rave.git@{RVE_COMMIT}"
print(f"installing {_spec}")
_r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", _spec],
                     capture_output=True, text=True)
if _r.returncode != 0:
    print(_r.stderr.strip()[-3000:])
    raise RuntimeError(
        f"pip exited {_r.returncode} installing rve. Without it there is no windowing, so this "
        f"stops here rather than continuing with whatever version happened to be present."
    )

import rve

_WINDOWS_CT = list(rve.get_available_windows("CT"))
_EXPECTED_WINDOWS = [
    "lung", "mediastinum", "abdomen", "liver", "bone", "brain",
    "subdural", "stroke", "temporal_bone", "soft_tissue", "minmax",
]
print(f"rve {getattr(rve, '__version__', '?')} @ {RVE_COMMIT[:8]}")
print(f"CT windows ({len(_WINDOWS_CT)}): {_WINDOWS_CT}")
if _WINDOWS_CT != _EXPECTED_WINDOWS:
    raise RuntimeError(
        f"rve's CT window list is not what this notebook was written against.\n"
        f"  got:      {_WINDOWS_CT}\n"
        f"  expected: {_EXPECTED_WINDOWS}\n"
        f"Pillar-0's CT checkpoints take an 11-channel multi-windowed volume in this exact order. "
        f"Since the install is pinned to a commit, seeing this means the pin did not take -- check "
        f"that the pip line above actually succeeded."
    )

# rve.apply_windowing takes a (D,H,W) array and must return (11,D,H,W) in [0,1]. Checked on a tiny
# array rather than assumed, because it is the one contract that would otherwise fail silently in
# the middle of the night as a shape error 40 minutes into extraction.
import numpy as _np

_probe = (_np.random.default_rng(0).standard_normal((3, 8, 8)).astype(_np.float32) * 300.0) - 400.0
_w = rve.apply_windowing(_probe, _WINDOWS_CT, "CT")
if tuple(_w.shape) != (11, 3, 8, 8):
    raise RuntimeError(f"rve.apply_windowing returned {tuple(_w.shape)}, expected (11, 3, 8, 8)")
if not (_w.min() >= -1e-6 and _w.max() <= 1 + 1e-6):
    raise RuntimeError(f"rve.apply_windowing returned values outside [0,1]: "
                        f"[{float(_w.min())}, {float(_w.max())}]")
print(f"windowing contract verified: (3,8,8) -> {tuple(_w.shape)} in "
      f"[{float(_w.min()):.3f}, {float(_w.max()):.3f}]")

In [ ]:
# 2d. Make every read from a Drive-mounted zip survive a transient mount failure
# ==============================================================================================
#
# Why this exists
# ---------------
# A real overnight run produced hundreds of lines like
#     located 9/9 wanted series in 09/05 (70,053 zip entries scanned); 9 FAILED VERIFICATION
#       ...: read failed (BadZipFile); ...: read failed (BadZipFile)
#
# The archive is fine. `namelist()` succeeded -- every wanted series was located, which means the
# zip's central directory (at the very end of the file) read correctly. What failed was the seek
# back into the middle of the file to read one member: Drive returned bytes that are not a zip
# local header, so Python raised BadZipFile.
#
# Two things in your own log prove this is a transient READ failure and not damaged data:
#   * 08/31 located 11 series and failed 9 -- two members in the SAME archive read fine.
#   * 09/16, 09/17 and 09/18 are 100k+-entry archives that read perfectly a few minutes later.
#
# ZIP_ACCESS_MODE is "stream", so every slice is a random-access read into a multi-GB file on the
# Drive FUSE mount. Under sustained load those reads intermittently return garbage. The extraction
# loop treats one bad read as one bad member and moves on -- correct for a genuinely corrupt file,
# wrong for a mount that would have answered correctly 300 ms later.
#
# What this patch does
# --------------------
# Wraps zipfile.ZipFile.open() so a read that fails with BadZipFile/OSError is retried with
# backoff, re-opening the archive handle in between (a stale handle is the usual culprit). This
# covers BOTH the verification in cell 12 and the per-slice reads in cell 13 -- one wrapper, no
# surgery on your notebook's cells.
#
# Nothing here writes to your output folder or invalidates anything already extracted.
# ==============================================================================================
import io
import os
import time
import zipfile

if not getattr(zipfile.ZipFile.open, "_drive_retry_patched", False):

    _ZIP_OPEN_ORIG = zipfile.ZipFile.open

    ZIP_READ_ATTEMPTS = 4          # total tries per member
    ZIP_READ_BACKOFF = (0.4, 1.2, 3.0)   # seconds before retries 2, 3, 4

    ZIP_RETRY_STATS = {"calls": 0, "retried": 0, "recovered": 0, "gave_up": 0}

    def _reopen_zipfile(zf):
        """Rebuild this ZipFile's file handle in place, so a stale/poisoned handle is discarded.

        Only attempted for a ZipFile that owns a real path. If anything about the reopen looks
        unusual the original handle is left exactly as it was -- a failed refresh must never be
        worse than no refresh.
        """
        name = getattr(zf, "filename", None)
        if not name or not isinstance(name, str) or not os.path.exists(name):
            return False
        fp = getattr(zf, "fp", None)
        if fp is None or not hasattr(fp, "name"):
            return False
        # Never swap the handle while another member is still being read from this archive.
        # ZipFile.open() hands each reader a _SharedFile bound to the fp that was current when
        # it was created; closing that fp underneath a live reader turns an unrelated, healthy
        # read into a ValueError on a closed file. _fileRefCnt is 1 when only the ZipFile itself
        # holds the handle. If the attribute is missing on some Python build, decline to reopen
        # -- a retry without the refresh is still worth having.
        if getattr(zf, "_fileRefCnt", None) != 1:
            return False
        if getattr(zf, "_writing", False):
            return False
        try:
            new_fp = io.open(name, "rb")
        except Exception:
            return False
        try:
            fp.close()
        except Exception:
            pass
        zf.fp = new_fp
        return True

    def _patched_open(self, name, mode="r", pwd=None, force_zip64=False):
        ZIP_RETRY_STATS["calls"] += 1
        last = None
        for attempt in range(ZIP_READ_ATTEMPTS):
            try:
                fh = _ZIP_OPEN_ORIG(self, name, mode, pwd, force_zip64=force_zip64)
                if attempt:
                    ZIP_RETRY_STATS["recovered"] += 1
                return fh
            except (zipfile.BadZipFile, OSError) as e:
                # A write-mode open, or a name that is genuinely not in the archive, is not
                # something a retry can fix -- surface it immediately.
                if mode != "r" or isinstance(e, KeyError):
                    raise
                last = e
                if attempt == 0:
                    ZIP_RETRY_STATS["retried"] += 1
                if attempt == ZIP_READ_ATTEMPTS - 1:
                    break
                time.sleep(ZIP_READ_BACKOFF[min(attempt, len(ZIP_READ_BACKOFF) - 1)])
                if mode == "r":
                    _reopen_zipfile(self)
        ZIP_RETRY_STATS["gave_up"] += 1
        raise last

    _patched_open._drive_retry_patched = True
    zipfile.ZipFile.open = _patched_open
    print("zipfile.ZipFile.open patched: "
          f"{ZIP_READ_ATTEMPTS} attempts, backoff {ZIP_READ_BACKOFF}, handle reopened between tries")
else:
    print("retry patch already active -- not applied twice")


# ---- pydicom: the supported off switch, not a filter on the symptom ---------------------------
#
# RSNA de-identified every UID to the literal string "ID_<9 hex>", which is not a legal DICOM UI,
# so pydicom complains about essentially every file. Cell B2 touches three UIDs per member, so it
# is three messages each -- 2.3 million for a full pass. A real run reached 100,000 members with
# 918 KB of them already in the cell output.
#
# THE FIRST FIX WAS THE WRONG LEVER. pydicom.misc.warn_and_log is
#
#     LOGGER.warning(msg)                  <- the logging module
#     warnings.warn(msg, category, ...)    <- the warnings module
#
# so warnings.filterwarnings() silenced exactly half. Adding a logging filter silenced the other
# half, and both are still treating the symptom: the message is *generated* either way, formatted
# either way, and only then discarded.
#
# pydicom's own supported switch turns the VALIDATION off, above the warning:
#
#     validate_value():
#         if validation_mode == config.IGNORE:
#             return                       <- returns before the validator, before warn_and_log
#
# It is a documented API (pydicom.config.settings.reading_validation_mode, added in 2.3; the
# constants are pydicom.config.IGNORE / WARN / RAISE). Measured on this exact workload it is also
# about 21% faster end to end than the noisy default -- 470 vs 595 microseconds per file, which
# over 752,803 files is ~6 minutes instead of ~7.5, plus whatever the terminal costs.
#
# The message filters below are KEPT as belt and braces, because reading_validation_mode is a
# process-global and a library that resets it must not be able to reopen the flood.
import logging as _logging
import warnings as _warnings

DICOM_NOISE_PATTERNS = ("Invalid value for VR UI",)


class _DicomNoiseFilter(_logging.Filter):
    def filter(self, record):
        try:
            msg = record.getMessage()
        except Exception:
            return True
        return not any(p in msg for p in DICOM_NOISE_PATTERNS)


def quiet_expected_dicom_noise(verbose=True):
    """Turn off VR validation at source, and filter the message on both paths as a backstop.

    Idempotent, and safe to call from any cell -- the notebook can be run out of order and the
    RSNA path must not depend on an earlier cell having been executed for its output to stay
    readable.
    """
    mode = "message filters only"
    try:
        import pydicom
        from pydicom import config as _pdcfg
        # IGNORE stops the validator before it formats anything. Reading only: WRITING validation
        # is deliberately left alone, because this notebook does not write DICOMs and a future
        # cell that did should not inherit a silent one.
        _pdcfg.settings.reading_validation_mode = _pdcfg.IGNORE
        # A truncated 32-bit length inside an element header otherwise raises
        # BytesLengthException, which is not a pydicom error type anyone catches by name. As a
        # UN element it becomes a value we can see and reject, instead of an exception class we
        # have to remember.
        _pdcfg.convert_wrong_length_to_UN = True
        mode = f"pydicom {pydicom.__version__} reading_validation_mode = IGNORE"
    except Exception as _e:
        mode = f"could not set pydicom validation mode ({type(_e).__name__}) -- filters only"

    for pat in DICOM_NOISE_PATTERNS:
        _warnings.filterwarnings("ignore", message=r".*" + pat + r".*")
    n = 0
    for name in ("pydicom", "pydicom.valuerep", "pydicom.dataelem", "pydicom.filereader"):
        lg = _logging.getLogger(name)
        if not any(isinstance(f, _DicomNoiseFilter) for f in lg.filters):
            lg.addFilter(_DicomNoiseFilter())
            n += 1
    # A filter on a logger does not apply to records that reach the root logger's handlers from
    # a child, so the root handlers get one too. Colab installs a stderr handler on root.
    for h in _logging.getLogger().handlers:
        if not any(isinstance(f, _DicomNoiseFilter) for f in h.filters):
            h.addFilter(_DicomNoiseFilter())
            n += 1
    if verbose:
        print(f"pydicom de-identification noise silenced: {mode}"
              + (f", plus {n} message filter(s)" if n else ""))
    return n


quiet_expected_dicom_noise()


# ---- the Drive mount: it fills your disk, and it dies --------------------------------------------
#
# Two documented failure modes, both of which end an overnight run, and neither of which is a bug
# in this notebook.
#
# 1. THE CACHE THAT IS NOT A CACHE.  DriveFS caches every byte it reads under
#    /root/.config/Google/DriveFS/<id>/content_cache. colabtools#960 reports that it "has no
#    control over its size, it does not delete or replace anything, it just accumulates data"
#    until the local disk is full. This notebook streams a 194.7 GB archive past the mount TWICE
#    -- once to index headers, once to extract the wanted slices -- against a Colab disk that is
#    around 170 GB and is not guaranteed to start empty (colabtools#5435: a runtime came up with
#    38 GB already gone). If the cache really does keep everything, the disk fills long before the
#    62 GB of extracted DICOMs does, and the run dies of ENOSPC somewhere unhelpful.
#
# 2. THE MOUNT DIES.  OSError [Errno 107] Transport endpoint is not connected means the DriveFS
#    helper process is gone; every open file descriptor is invalid and RETRYING THE READ CANNOT
#    WORK. The fix is drive.mount(force_remount=True) and reopening handles. colabtools#5970
#    (May 2026) is a Pro+ session where this happened and the job burned seven hours doing
#    nothing, because nothing noticed.
#
# So: measure the cache, prune it when it grows (it is a cache -- every byte in it is
# re-fetchable, and this notebook never reads the same byte twice anyway), keep a floor of free
# disk, and treat a dead mount as something to repair rather than retry.
import os as _os
import shutil as _shutil
import subprocess as _subprocess
import time as _time

DRIVE_MOUNT_POINT = P["drive_mount"]
DRIVE_CACHE_DIRS = ("/root/.config/Google/DriveFS",
                    "/root/.cache/Google/DriveFS")
# Prune the DriveFS cache once it passes this, and refuse to continue below the free-disk floor.
DRIVE_CACHE_PRUNE_GB = 20.0
DISK_FREE_FLOOR_GB = 25.0
DRIVE_HEALTH_STATS = {"checks": 0, "prunes": 0, "bytes_pruned": 0, "remounts": 0}


def _free_gb(path="/content"):
    try:
        st = _os.statvfs(path)
        return st.f_bavail * st.f_frsize / 1e9
    except Exception:
        return float("nan")


def drive_cache_gb():
    """Size of the DriveFS content cache, in GB. 0.0 if it is not there."""
    total = 0
    for root in DRIVE_CACHE_DIRS:
        if not _os.path.isdir(root):
            continue
        try:
            out = _subprocess.run(["du", "-sb", root], capture_output=True, text=True, timeout=60)
            if out.returncode == 0 and out.stdout.strip():
                total += int(out.stdout.split()[0])
        except Exception:
            for dirpath, _d, files in _os.walk(root):
                for f in files:
                    try:
                        total += _os.path.getsize(_os.path.join(dirpath, f))
                    except OSError:
                        pass
    return total / 1e9


def prune_drive_cache(older_than_s=120, verbose=True):
    """Delete cached content blobs that DriveFS is not actively filling.

    Every byte here is re-fetchable from Drive, and this notebook reads the archive strictly
    forward, so nothing it has already passed will ever be wanted again -- a cache hit is not
    possible even in principle. Files touched in the last `older_than_s` seconds are left alone
    so an in-flight fetch is not pulled out from under the reader.

    Never raises: a cache that cannot be pruned is a warning, not the end of the run.
    """
    freed = 0
    now = _time.time()
    for root in DRIVE_CACHE_DIRS:
        if not _os.path.isdir(root):
            continue
        for dirpath, _dirs, files in _os.walk(root):
            if "content_cache" not in dirpath:
                continue
            for f in files:
                p = _os.path.join(dirpath, f)
                try:
                    st = _os.stat(p)
                    if now - st.st_mtime < older_than_s:
                        continue
                    _os.remove(p)
                    freed += st.st_size
                except OSError:
                    pass
    DRIVE_HEALTH_STATS["prunes"] += 1
    DRIVE_HEALTH_STATS["bytes_pruned"] += freed
    if verbose and freed:
        print(f"    pruned {freed/1e9:.1f} GB from the DriveFS content cache "
              f"({_free_gb():.0f} GB free now)")
    return freed


def remount_drive(verbose=True):
    """Repair a dead mount. Returns True if the mount looks alive afterwards."""
    try:
        from google.colab import drive as _gdrive
    except Exception:
        if verbose:
            print("    not running in Colab -- cannot remount")
        return False
    try:
        _gdrive.mount(DRIVE_MOUNT_POINT, force_remount=True)
        DRIVE_HEALTH_STATS["remounts"] += 1
        if verbose:
            print(f"    Drive remounted (remount #{DRIVE_HEALTH_STATS['remounts']})")
        return _os.path.isdir(_os.path.join(DRIVE_MOUNT_POINT, "MyDrive"))
    except Exception as e:
        if verbose:
            print(f"    remount FAILED: {type(e).__name__}: {e}")
        return False


def is_dead_mount_error(exc):
    """True for the errors that mean the mount is gone, as opposed to a transient read failure.

    Errno 107 (ENOTCONN) and errno 103 mean the FUSE endpoint is dead: every open descriptor is
    invalid and no number of retries will help. Errno 5 (EIO) is the ambiguous one -- usually
    transient, but it is also what a dying mount returns, which is why the caller escalates to a
    remount only after several in a row.
    """
    return isinstance(exc, OSError) and getattr(exc, "errno", None) in (103, 107)


def drive_health_check(probe_path=None, verbose=True, prune_gb=None, floor_gb=None):
    """One call that answers: is the mount alive, is the disk filling, and is the cache to blame?

    Returns a dict. Raises RuntimeError only when free disk is below the floor AND pruning could
    not recover it -- stopping there is better than dying of ENOSPC halfway through a parquet
    write, because everything in this notebook is checkpointed and a clean stop resumes.
    """
    prune_gb = DRIVE_CACHE_PRUNE_GB if prune_gb is None else prune_gb
    floor_gb = DISK_FREE_FLOOR_GB if floor_gb is None else floor_gb
    DRIVE_HEALTH_STATS["checks"] += 1
    info = {"free_gb": _free_gb(), "cache_gb": drive_cache_gb(), "alive": True, "pruned_gb": 0.0}

    if probe_path is not None:
        try:
            _os.stat(probe_path)
        except OSError as e:
            info["alive"] = False
            if verbose:
                print(f"    the Drive mount is not answering ({type(e).__name__}: {e}) -- "
                      f"remounting")
            info["alive"] = remount_drive(verbose=verbose)

    if info["cache_gb"] > prune_gb or info["free_gb"] < floor_gb * 2:
        info["pruned_gb"] = prune_drive_cache(verbose=verbose) / 1e9
        info["free_gb"] = _free_gb()
        info["cache_gb"] = drive_cache_gb()

    if info["free_gb"] < floor_gb:
        raise RuntimeError(
            f"only {info['free_gb']:.1f} GB free on /content, below the {floor_gb:.0f} GB floor, "
            f"and the DriveFS cache is {info['cache_gb']:.1f} GB after pruning.\n"
            f"Stopping cleanly rather than failing partway through a write -- everything so far "
            f"is checkpointed, so re-running continues from here. Free some space (the local "
            f"DICOM cache is at {globals().get('RSNA_LOCAL_DICOM_DIR', 'LOCAL_ZIP_CACHE')}) or "
            f"lower RSNA_MAX_SERIES."
        )
    return info


print(f"Drive health helpers ready -- {_free_gb():.0f} GB free, "
      f"DriveFS cache {drive_cache_gb():.1f} GB "
      f"(pruned above {DRIVE_CACHE_PRUNE_GB:.0f} GB, hard floor {DISK_FREE_FLOOR_GB:.0f} GB)")


# ---- a heartbeat, because "is it stuck or is it disconnected?" is not answerable from a tab -----
#
# Colab Pro does NOT have background execution -- that is a Pro+ feature (Google's own comparison
# lists it only there) -- so a closed tab is subject to an idle shutdown whose duration Google
# explicitly declines to publish. And when the frontend disconnects from a backend that is still
# working, the cell shows a spinner and nothing else, whether the kernel is busy, dead, or waiting
# on a mount that died an hour ago.
#
# Neither state is visible from the notebook. So every long pass writes a line to a file on Drive
# instead: stage, timestamp, and what it was doing. Open it from anywhere -- another Colab
# session, the Drive web UI, your phone -- and the answer is the file's modification time.
import json as _json

HEARTBEAT_NAME = "run_heartbeat.jsonl"
_HEARTBEAT = {"path": None, "last": 0.0}


def heartbeat(stage, detail="", every_s=0.0, **fields):
    """Append one line saying where the run is. Never raises; a dead heartbeat is not a reason
    to lose a night's work."""
    now = _time.time()
    if every_s and (now - _HEARTBEAT["last"]) < every_s:
        return False
    out_dir = globals().get("OUTPUT_DIR")
    if not out_dir:
        return False
    path = _HEARTBEAT["path"] or _os.path.join(out_dir, HEARTBEAT_NAME)
    rec = {"t": now, "iso": _time.strftime("%Y-%m-%d %H:%M:%S", _time.localtime(now)),
           "stage": stage, "detail": str(detail)[:400],
           "free_gb": round(_free_gb(), 1), **fields}
    try:
        _os.makedirs(out_dir, exist_ok=True)
        with open(path, "a") as f:
            f.write(_json.dumps(rec) + "\n")
        _HEARTBEAT["path"] = path
        _HEARTBEAT["last"] = now
        return True
    except Exception:
        return False


def read_heartbeat(out_dir=None, n=5):
    """The last few heartbeat lines, for a human asking 'is it still alive?'."""
    out_dir = out_dir or globals().get("OUTPUT_DIR")
    path = _os.path.join(out_dir, HEARTBEAT_NAME) if out_dir else None
    if not path or not _os.path.exists(path):
        return []
    try:
        with open(path) as f:
            lines = f.readlines()[-n:]
        return [_json.loads(l) for l in lines if l.strip()]
    except Exception:
        return []

In [ ]:
# 3. Colab keep-alive (optional -- best effort only)
#
# Helps with the common "idle browser tab" disconnect. It does NOT beat a sleeping laptop, lost
# wifi, or Colab's hard session cap. The real protection is that every stage below checkpoints to
# Drive, so a disconnect costs minutes of rework, not the run.
try:
    from IPython.display import Javascript, display

    display(Javascript(r"""
    function ClickConnect(){
        var btn = document.querySelector("colab-connect-button");
        if (btn) { btn.click(); }
    }
    if (window._colabKeepAliveInterval) { clearInterval(window._colabKeepAliveInterval); }
    window._colabKeepAliveInterval = setInterval(ClickConnect, 60000);
    console.log("colab keep-alive interval installed (60s)");
    """))
    print("Keep-alive injected (best effort).")
except Exception as e:
    print(f"Keep-alive not installed (not in Colab's browser UI?): {e}")

print()
print("Long-run tips:")
print("  - laptop awake, on power, on wifi")
print("  - if it disconnects: reconnect and Run all again. Every stage checkpoints to Drive, so")
print("    nothing already finished is recomputed.")

In [ ]:
# 4. CONFIGURATION -- the only cell you should normally need to edit
import os
import torch

# ---------------------------------------------------------------------------------------------
# WHICH DATASET
# ---------------------------------------------------------------------------------------------
# "india" : the matched Indian government-hospital scans (Part A) -- day-zips on Drive, labels
#             from matched radiology reports.
# "rsna"    : RSNA 2019 Intracranial Hemorrhage Detection (Part B) -- a split archive on Drive,
#             expert per-slice labels aggregated to series level.
# Everything after the manifest is shared, so a difference in results between the two is a
# difference in the data, not in the pipeline.
DATA_SOURCE = "india"         # NELLORE runs on the "india" path

# "nellore" was the previous name for the India source. It was wrong: the matched scans are 97%
# Government General Hospital KADAPA (28,027 series), with GGH Nellore (~957) and RIMS Ongole (648)
# making up the rest -- the name came from a filename, not from the data. Accepted as an alias so a
# half-finished run does not break, but the accurate name is "india".
if DATA_SOURCE == "nellore":
    print("NOTE: DATA_SOURCE 'nellore' is the old name for 'india' (the data is mostly Kadapa, "
          "not Nellore) -- continuing as 'india'.")
    DATA_SOURCE = "india"

# ---------------------------------------------------------------------------------------------
# UNATTENDED RUN
# ---------------------------------------------------------------------------------------------
# True: the preflight cell treats every failed check as fatal and stops, because there is nobody
# watching to notice a warning scroll past. This is the right setting for an overnight run.
# False: warnings only -- what you want while poking at things interactively.
UNATTENDED_RUN = False
REQUIRE_GPU = True               # a CPU overnight run finishes a few dozen of several thousand
PREFLIGHT_DOWNLOAD_MODEL = True  # pull the weights during preflight so a 403 surfaces in minute 2

# Repo ids, duplicated here (cheaply) so the preflight cell can check gated access before the full
# registry in cell 15 is defined. Cell 15 asserts these agree with its own table, so they cannot
# drift apart.
PILLAR0_REGISTRY_REPOS = {
    "pillar0_headct": "YalaLab/Pillar0-HeadCT",
    "pillar0_chestct": "YalaLab/Pillar0-ChestCT",
    "pillar0_abdomenct": "YalaLab/Pillar0-AbdomenCT",
}

# ---------------------------------------------------------------------------------------------
# INPUTS
# ---------------------------------------------------------------------------------------------
# The combined matched-scans CSV (metadata + report findings). ~800 MB; cell 7 reads it once,
# column-selectively, and caches a much smaller parquet next to the outputs so later sessions
# never pay that cost again.
CSV_PATH = P["external_matched_csv"]

# Root of the day-zip tree on Drive. Layout is CT_ROOT/<MM>/<DD>.zip -- e.g.
# ".../2025/01/05.zip". Confirmed by listing the Drive folder directly.
# Verified: this folder's children are the month folders. Months present are 01, 03, 05, 06, 07,
# 08, 09, 10, 11 (242 day-zips); 02, 04 and 12 are absent entirely, and 08/20 and 11/17 are missing
# individually. Cell 10's coverage report is the authority -- this note is just so a number that
# looks low can be recognised as expected rather than as a bug.
CT_ROOT = P["external_ct_root"]

# --- RSNA (only used when DATA_SOURCE == "rsna") ----------------------------------------------
# The folder holding the split archive; cell B1 expects RSNA_ROOT/parts/part_00000 ...
# Verified: kaggle/rsna-ich holds parts/ (91 files, part_00000..part_00090, 193.4 GB) and
# rsna-intracranial-hemorrhage-detection/stage_2_train.csv.
# DO NOT point this at <drive_root>/RSNA_ICH -- that folder has a doubled subfolder name,
# only stage_2_test, no stage_2_train.csv and no parts/. It is the obvious wrong turn here.
RSNA_ROOT = P["pillar0_rsna_archive_dir"]
RSNA_PARTS_PATTERN = r"^part_\d+$"      # change if your split used e.g. 'xa*'
RSNA_SPLITS = ("stage_2_train",)         # only stage_2_train carries labels
RSNA_LABEL_CSV_NAME = "stage_2_train.csv"
# stage_2_train.csv also sits on Drive as a standalone 120 MB file next to the parts, so read it
# from there rather than pulling it out of a 194 GB archive. Set to "" to read it from inside the
# archive instead.
RSNA_LABEL_CSV_PATH = os.path.join(P["pillar0_rsna_archive_dir"],
                                   "rsna-intracranial-hemorrhage-detection", "stage_2_train.csv")
RSNA_REBUILD_LABELS = False

# OPTIONAL, and worth the five minutes it costs. There is a published CC0 extraction of the same
# 752,803 DICOM headers, made independently with pydicom over the unzipped files:
#     kaggle datasets download -d carlolepelaars/rsna-stage-2-metadata-ihd-2019
# Point this at stage_2_train_with_metadata.csv and cell B6 joins it to this notebook's own index
# on SOPInstanceUID and compares every geometry field. Two independent readings of the same
# archive either agree or they do not -- which is stronger evidence that the forward walk is
# correct than any test in this repository, because it comes from outside it. Empty = skip.
RSNA_METADATA_XCHECK_PATH = ""

# Header-scan tuning (cell B2). The cost is per-request latency against Drive, not bandwidth, so
# threads help a lot. Lower RSNA_INDEX_WORKERS if Drive starts returning errors.
RSNA_HEADER_BYTES = 16384                # enough for the header; pixel data is never touched
RSNA_INDEX_CHUNK = 20000                 # members per resumable checkpoint
RSNA_INDEX_WORKERS = 12
# A chunk with more than this fraction of unreadable members is NOT cached, so it is retried
# instead of being silently recorded as done with a handful of rows. (Tar path only now; the zip
# path walks forward and retries individual members -- see RSNA_MAX_RETRY_MEMBERS.)
RSNA_CHUNK_MAX_BAD_FRACTION = 0.02
# The forward walk reads members in one sequential sweep. Any it could not read are retried
# individually afterwards, by their true offsets. That is per-member seeking -- the slow access
# pattern -- so it is only worth doing for a handful. Beyond this many, the index is left marked
# INCOMPLETE and the run says the archive or the mount needs looking at, rather than grinding
# through 100,000 nine-second reads.
RSNA_MAX_RETRY_MEMBERS = 5000
# A hard wall on the individual-member retry pass, for the same reason RSNA_VERIFY_MAX_SECONDS
# exists: a retry loop with no escape is how a night gets spent going nowhere.
RSNA_RETRY_MAX_SECONDS = 900

# ~21.7k volumes through Pillar-0 is a multi-night run. None = all of them; an integer takes a
# PROPORTIONAL stratified sample on the full label pattern, which preserves every subtype's
# prevalence (so average precision and PPV stay meaningful).
RSNA_MAX_SERIES = 4000

# Both are named here, not just the active one, so Part C can find the other half without a second
# place to keep in sync. Change these two lines and everything follows.
INDIA_OUTPUT_DIR = P["pillar0_external_output_dir"]   # FRESH - keeps the crop run intact
RSNA_OUTPUT_DIR = P["pillar0_rsna_output_dir"]
OUTPUT_DIR = INDIA_OUTPUT_DIR if DATA_SOURCE == "india" else RSNA_OUTPUT_DIR

# ---------------------------------------------------------------------------------------------
# WHICH SCANS
# ---------------------------------------------------------------------------------------------
# "head" is the only region Pillar0-HeadCT is meant for. "chest" / "abdomen" exist so the rest of
# the matched data can be run later against their own checkpoints -- see MODELS_TO_RUN below.
TARGET_REGION = "head"          # "head" | "chest" | "abdomen"  (RSNA is head-only)

# Which Pillar-0 checkpoint(s) to run. Every name must be registered in cell 15, and every one must
# match TARGET_REGION -- cell 15 refuses a head/chest mismatch rather than quietly producing
# meaningless embeddings.
MODELS_TO_RUN = ["pillar0_headct"]

# Drop obvious non-diagnostic series (scouts, dose reports, key images, secondary capture) before
# any expensive work. On this dataset 13,134 HEAD series fall to 9,852 at >= 20 slices, and the
# median "Topogram ... sag" scout has 1-2 slices, so this filter is doing exactly what it should.
# For RSNA this is also the floor on a usable volume; the dataset paper says 20-60 axial images
# per series, so 20 keeps essentially everything real while still dropping stubs.
MIN_SLICES_PER_SERIES = 20

# A series must also keep at least this FRACTION of its members. The absolute floor above is not
# enough on its own: a 300-slice study where 270 instances fail to decode still yields 30 usable
# slices, clears the floor, and gets logged as a success -- an embedding of a tenth of a head that
# is indistinguishable downstream from a genuinely short scan.
MIN_KEPT_SLICE_FRACTION = 0.5

# False (default): one primary series per study -- brain-ish description, soft-tissue kernel
#   preferred over bone kernel, then most slices. ~3,205 head volumes.
# True: every qualifying series. ~9,852 head volumes, ~3x the compute, but lets you compare
#   reconstructions and pool per study at classification time.
PROCESS_ALL_QUALIFYING_SERIES = False

# ---------------------------------------------------------------------------------------------
# HOW TO READ THE ZIPS
# ---------------------------------------------------------------------------------------------
# "stream": read members straight out of the zip on Drive's FUSE mount. Only the bytes belonging to
#   wanted series are transferred -- usually a small fraction of a 25 GB day-zip -- but each small
#   member read pays FUSE latency.
# "local_copy": copy the whole day-zip to Colab's local disk first, then read from it. Reads all
#   25 GB but sequentially and fast, and every subsequent member read is local.
# Which wins depends on your Drive throughput. Cell 17 prints per-zip timings for both, so run one
# zip each way and pick with real numbers rather than a guess.
# Verify a series' identity while its slices are being READ, not in a separate pass beforehand.
#
# The old pre-pass opened the first three members of every wanted series before any series was
# read in full. On a 20-30 GB Drive-mounted archive that meant sweeping the file twice per day-zip
# and doubling the number of cold seeks, and cold seeks are precisely what these archives will not
# serve. Cell 13 now checks SeriesInstanceUID on the slices it is already parsing, which is the
# same guarantee at no extra I/O. Set True to restore the old pre-pass.
VERIFY_IN_LOCATOR = False

ZIP_ACCESS_MODE = "stream"      # "auto" | "stream" | "local_copy"
#
# This was "auto" -- stream first, then copy any zip left incomplete to local disk and re-read it
# there -- on the theory that the failures were the Drive mount mishandling random access. The
# 2026-08-25 overnight run settled that question and the answer was no.
#
# That run copied 22 day-zips to local disk, 499 GB, about five of its nine hours. Every single
# member read from every one of those local copies still failed with BadZipFile, exactly as it
# had off Drive. 129 archives opened, 909 series attempted, ZERO extracted. A local copy is a
# byte-for-byte duplicate, so it reproduces a broken central directory perfectly: the offsets in
# these archives overflowed 32 bits at write time (cell 11b), and no amount of copying repairs a
# number that was written wrong.
#
# With cell 11b in place the offsets are reconstructed and streaming reads work -- verified on
# 09/03 (51,027 members, 0/6 readable before, 10/10 after) and 06/18 (101,279 members, same).
# So streaming is now both the correct and the fast path, and "auto" would only re-spend those
# five hours to arrive at the same place. The escalation machinery is kept, not deleted: set
# "auto" if a day-zip ever fails for a reason that IS the mount.

# Ceiling on how much Drive->local copying pass 2 may do in a single run, when ZIP_ACCESS_MODE is
# "auto". Zips beyond it are left pending and the next run picks them up. Irrelevant while the
# mode is "stream"; kept at a value that bounds the damage if someone switches back without
# reading the note above.
MAX_LOCAL_COPY_GB_PER_RUN = 100.0

LOCAL_COPY_ATTEMPTS = 3         # sequential-copy retries before a zip is given up on
LOCAL_COPY_RETRY_SLEEP = 10     # seconds, multiplied by the attempt number
LOCAL_ZIP_CACHE = "/content/local_zip_cache"
LOCAL_COPY_FREE_SPACE_FACTOR = 1.15   # require size * this much free before copying

# ---------------------------------------------------------------------------------------------
# PILLAR-0 PREPROCESSING
# Every value below is sourced, not guessed. See the notes cell for the citations.
# ---------------------------------------------------------------------------------------------
# arXiv 2511.17803 Table 7 and Appendix A.5: head CT -> 256 x 256 x 128 (HWD), 8x8x4 patches,
# 32k tokens, 11-channel multi-windowing.
TARGET_SPATIAL_SIZE = (256, 256)        # (H, W)
TARGET_DEPTH = 128                      # D
# From the shipped rave/configs/ct_brain.yaml: target_spacing [0.5, 0.5, 1.25] in (x, y, z) mm.
TARGET_SPACING_XYZ = (0.5, 0.5, 1.25)
ANATOMY_MODALITY = "CT"                 # rve's modality string for window selection

# ---------------------------------------------------------------------------------------------
# HOW THE IN-PLANE FIELD OF VIEW IS FITTED INTO 256 PIXELS
# ---------------------------------------------------------------------------------------------
# "crop"     -- resample to 0.5 mm and centre-crop to 256 px. Faithful to Pillar-0's published
#               configuration, and what every result in this project so far was produced with.
#               It is also 256 x 0.5 = 128 mm of field of view, against a 150-160 mm skull:
#               measured on a realistic 512 x 512 head at 0.449 mm, it discards 12.5% of the
#               head, and the part it discards is the lateral convexity where subdural and
#               epidural collections sit.
# "fit_fov"  -- choose the in-plane spacing so the WHOLE field of view lands in 256 px. Nothing
#               is lost; in-plane resolution roughly halves.
# "fit_head" -- crop to the head's bounding box first, then scale that to 256 px. Nothing of the
#               head is lost and the empty air margin is gone, so the brain gets MORE pixels than
#               either of the others. This is what the RSNA-2019 winning solutions do
#               (Anjum48 crops to the second-largest connected component; BrainScan crops
#               400 x 400 around the centre of mass; darraghdog autocrops to the head).
#
# The default stays "crop", because Pillar-0's own paper reports 96.7 study-level AUROC for
# "any ICH" on RSNA-2019 with a frozen linear probe under that configuration, and an argument is
# not a reason to depart from a published recipe. Every run MEASURES what its crop actually cost
# (cell 14's measure_head_retention, reported at the end of cell 18), so the decision to change
# this can be made from numbers on YOUR data. Changing it changes the embeddings, so it is part
# of the extraction signature and re-extraction is forced -- which is correct, and is why the
# A/B is worth running on a subsample rather than guessed at.
PILLAR0_INPLANE_MODE = "fit_head"       # was "crop": it discarded a median 29.4% of the head

# Mixed precision. ~2x faster on a T4 and the embeddings are near-identical -- cell 16b measures
# the actual fp32-vs-AMP cosine similarity on real volumes so you can see that rather than trust
# it. Every progress record logs which precision produced it, so a mixed run stays auditable.
USE_AMP = True

# ---------------------------------------------------------------------------------------------
# RUN BUDGET -- lets a Colab session stop cleanly instead of being interrupted
# ---------------------------------------------------------------------------------------------
MAX_SERIES_PER_RUN = None       # e.g. 200 for a smoke test; None = no cap
MAX_SECONDS_PER_RUN = None      # e.g. 10 * 3600; None = no cap
# None = the real run. Set this to a small integer ONLY for an interactive smoke test, and set it
# back before leaving anything running overnight -- the preflight cell refuses to start an
# unattended run with a trial cap in place, because "extracted 20 of 3,205 series, no failures" is
# indistinguishable from success until you look closely, and by then the night is gone.
TRIAL_RUN_MAX_SERIES = None

# How often the extraction loop consolidates .npy files into the .npz store and flushes the
# feature parquet. Lower = safer, more Drive writes.
CONSOLIDATE_EVERY_N_SERIES = 200

# A tqdm bar over a group this large is turned off and replaced by a printed line every
# BIG_GROUP_LOG_EVERY series. The India path groups by day-zip (10-30 series each) and keeps its
# bar; the RSNA path is one group of ~4,000, where a widget updating 4,000 times into a
# disconnected browser is the same output-volume problem as the pydicom flood.
BIG_GROUP_SERIES = 300
BIG_GROUP_LOG_EVERY = 100

# ---------------------------------------------------------------------------------------------
# CLASSIFICATION
# ---------------------------------------------------------------------------------------------
# Refuse to run the sweep if the feature table predates `person_key` (cell 9). PatientID in this
# export is a per-visit registration counter, so grouping on it leaks -- see cell 21. Setting this
# False lets an old table through, at the cost of numbers that are inflated and unreportable.
# How many findings (beyond the special cohorts) get a full operating-point + confusion-matrix
# block in cell 22. All of them are computed; this only bounds how much is printed.
MAX_TARGETS_WITH_OPERATING_POINTS = 12

# Refuse to build a feature table, or compute a result, from a cohort that is not the one this
# config describes. With this on, cell 18 drops any embedding whose series is not in the current
# manifest and then verifies that every remaining row is the configured region, an allowed head
# sub-anatomy, and carries a person_key; cell 20c turns its cohort-quality warnings into hard
# failures. Turning it off returns to reporting those as advisories.
#
# The specific mixtures it prevents, all of which were present in earlier runs:
#   * 273 dedicated sinus / facial-bone / inner-ear scans sitting in the negatives of a
#     hemorrhage classifier, where they are guaranteed negative and trivially separable
#     (worth about +0.02 AUROC on its own);
#   * embeddings extracted under a wider manifest surviving a later narrowing, because cell 18
#     merged metadata with a LEFT join and kept rows the manifest had dropped;
#   * MR series in a CT cohort -- this export carries 337 MR rows.
# How many times to redraw the holdout. One holdout at one seed is a single sample of a
# quantity whose standard error, at ~600 held-out scans and ~40 positives, is about 0.04 -- so
# the reported number moved with the draw more than with most of the bugs in this project's
# history. Repeats cost one refit each (seconds) and turn that into a measured spread. The first
# repeat is always RANDOM_SEED, so the headline split is unchanged and comparable across runs.
N_EVAL_REPEATS = 20

STRICT_COHORT = True

REQUIRE_PERSON_KEY = True

MIN_POSITIVES = 20              # was 30: at 30, b_sdh (25) and b_sah (24) were never evaluated
MAX_FINDINGS_TO_EVALUATE = None # None = all eligible, most-supported first
CV_FOLDS = 5
HOLDOUT_FRAC = 0.2
TARGET_SENSITIVITY = 0.85       # for the "high sensitivity" operating point
RANDOM_SEED = 12345
MAX_SECONDS_PER_CLASSIFIER_SWEEP = None

# ---------------------------------------------------------------------------------------------
# DERIVED PATHS -- nothing below here normally needs editing
# ---------------------------------------------------------------------------------------------
EMBED_DIR            = os.path.join(OUTPUT_DIR, "embeddings")
ZIP_INDEX_DIR        = os.path.join(OUTPUT_DIR, "zip_index")
TABLE_CACHE_PATH     = os.path.join(OUTPUT_DIR, "matched_table.parquet")
TABLE_CACHE_META     = os.path.join(OUTPUT_DIR, "matched_table.meta.json")
MANIFEST_PATH        = os.path.join(OUTPUT_DIR, "manifest.csv")
MANIFEST_META        = os.path.join(OUTPUT_DIR, "manifest.meta.json")
PROGRESS_LOG_PATH    = os.path.join(OUTPUT_DIR, "extraction_progress.jsonl")
ZIPS_DONE_PATH       = os.path.join(OUTPUT_DIR, "zips_done.json")
FAILURES_CSV_PATH    = os.path.join(OUTPUT_DIR, "failed_series.csv")
CLASSIFIER_RESULTS_PATH = os.path.join(OUTPUT_DIR, "classifier_results.jsonl")

for _d in (OUTPUT_DIR, EMBED_DIR, ZIP_INDEX_DIR, LOCAL_ZIP_CACHE):
    os.makedirs(_d, exist_ok=True)

if DATA_SOURCE not in ("india", "rsna"):
    raise ValueError(f"DATA_SOURCE must be 'india' or 'rsna', got {DATA_SOURCE!r}")
if DATA_SOURCE == "rsna" and TARGET_REGION != "head":
    raise ValueError("RSNA-ICH is head CT only; set TARGET_REGION = 'head'.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"DATA_SOURCE     : {DATA_SOURCE}")
print(f"UNATTENDED_RUN  : {UNATTENDED_RUN}")
print(f"OUTPUT_DIR      : {OUTPUT_DIR}")
print(f"data root       : {CT_ROOT if DATA_SOURCE == 'india' else RSNA_ROOT}")
if DATA_SOURCE == "india":
    print(f"labels CSV      : {CSV_PATH}")
else:
    print(f"labels CSV      : {RSNA_LABEL_CSV_PATH or '(read from inside the archive)'}")
print(f"TARGET_REGION   : {TARGET_REGION}")
print(f"MODELS_TO_RUN   : {MODELS_TO_RUN}")
print(f"ZIP_ACCESS_MODE : {ZIP_ACCESS_MODE}")
print(f"DEVICE          : {DEVICE}  (AMP={'on' if USE_AMP and DEVICE == 'cuda' else 'off'})")
if TRIAL_RUN_MAX_SERIES is not None:
    print(f"\n*** TRIAL RUN: extraction will stop after {TRIAL_RUN_MAX_SERIES} series. ***")
    print("    Set TRIAL_RUN_MAX_SERIES = None for the real run -- nothing from the trial is")
    print("    wasted, those series are logged 'ok' and skipped next time.")
if DEVICE == "cpu":
    print()
    print("*** WARNING: no GPU in this runtime. ***")
    print("A Pillar-0 forward pass is an 11-channel 256x256x128 volume through a 32k-token")
    print("transformer. On CPU expect minutes per volume, not seconds -- a ~3,200-volume run will")
    print("not finish overnight. Runtime > Change runtime type > GPU, then restart and re-run.")
    print("(The classifier sweep at the end is sklearn/CPU regardless -- a GPU only helps here.)")

In [ ]:
# ==============================================================================================
# 4b. TURN OFF THE PILLAR-0 MODEL PATH  [inserted -- this notebook now uses our backbones]
# ==============================================================================================
# The cell above still declares MODELS_TO_RUN = ["pillar0_headct"], and the PREFLIGHT below
# loops over that list to download a Pillar-0 checkpoint from HuggingFace for every entry.
# Those weights are no longer used by anything: cells 15-16c further down load SwinUNETR/SuPreM,
# SegResNet/SuPreM and ResNet-18/MedicalNet instead. Downloading them anyway costs several GB
# and several minutes, and the preflight fails outright if the repo is gated.
#
# Emptying the list is all it takes -- the preflight's other checks (codecs, Drive, day-zips,
# disk) are exactly the ones still worth running, and they do not touch it. Cell 16c sets
# MODELS_TO_RUN to our three backbones, well before the extraction loop reads it.
#
# rve is left installed. Nothing below uses it for windowing any more, but the install cell
# also pins the DICOM codec plugins the day-zip reader depends on.
MODELS_TO_RUN = []

# A HARD GUARANTEE, not a comment. This notebook began life as a Pillar-0 notebook and the
# inherited names are still all over it -- PILLAR0_REGISTRY_REPOS, preprocess_for_pillar0, the
# nellore_pillar0_* folders your data already lives in. None of them load a Pillar-0 model, but
# "none of them do" is the kind of claim that quietly stops being true. So it is checked here
# and again after the zoo loads: if any Pillar-0 checkpoint is ever in MODELS_TO_RUN, this
# raises instead of running.
PILLAR0_NAMES = tuple(globals().get("PILLAR0_REGISTRY_REPOS", {}))


def assert_no_pillar0(models, where):
    bad = [m for m in models if str(m).lower().startswith("pillar0")
           or str(m) in PILLAR0_NAMES]
    if bad:
        raise RuntimeError(
            f"{where}: {bad} would run a Pillar-0 checkpoint. This paper uses only the three "
            f"frozen 3D backbones loaded by the zoo cell. If you actually want Pillar-0, delete "
            f"this guard deliberately rather than letting it happen by accident.")
    return True


assert_no_pillar0(MODELS_TO_RUN, "preflight")
print("PILLAR-0 IS OFF. MODELS_TO_RUN is empty, so the preflight downloads no Pillar-0")
print("checkpoint and no Pillar-0 model is ever constructed in this runtime.")
print("The backbones this notebook uses, and the only ones:")
print("  B1_swinunetr_suprem     SwinUNETR swinViT trunk, SuPreM supervised weights")
print("  B2_segresnet_suprem     SegResNet encoder, SuPreM supervised weights")
print("  B3_resnet18_medicalnet  3D ResNet-18, MedicalNet weights")
print("They are loaded by '13. The frozen backbone zoo', which proves each load changed the")
print("weights. Inherited pillar0 names elsewhere are folder and function names only.")

In [ ]:
# 5. Verify DICOM codec plugins actually work in THIS runtime (fully offline, no network needed)
# Installing a package is not the same as it actually working -- some Colab users have reported
# gdcm/pylibjpeg installing successfully but still failing to decode pixel data in that specific
# runtime (see https://github.com/pydicom/pydicom/issues/1645). This round-trips a tiny synthetic
# image through a real JPEG-LS encoder/decoder and checks the result is bit-exact, so you find out
# right now whether decoding actually works -- not partway through an overnight extraction run.
import numpy as np
import pydicom
from pydicom.dataset import Dataset, FileMetaDataset
from pydicom.uid import ExplicitVRLittleEndian, JPEGLSLossless, generate_uid


def _codec_self_test():
    ds = Dataset()
    ds.file_meta = FileMetaDataset()
    ds.file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
    ds.file_meta.MediaStorageSOPClassUID = generate_uid()
    ds.file_meta.MediaStorageSOPInstanceUID = generate_uid()
    ds.is_little_endian = True
    ds.is_implicit_VR = False
    ds.Rows, ds.Columns = 8, 8
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"
    ds.BitsAllocated = 16
    ds.BitsStored = 16
    ds.HighBit = 15
    ds.PixelRepresentation = 1
    original = (np.arange(64, dtype=np.int16).reshape(8, 8) * 7) - 100
    ds.PixelData = original.tobytes()

    try:
        ds.compress(JPEGLSLossless)  # round-trip through a real JPEG-LS encoder
    except Exception as e:
        # No ENCODER available (pyjpegls missing/broken). That does not tell us decoding is
        # broken, so report it as inconclusive rather than crashing the notebook here.
        return {"(encoder unavailable)": f"could not build the test image: {type(e).__name__}: {e}"}

    outcomes = {}
    for plugin in ("gdcm", "pylibjpeg", "pyjpegls", ""):
        try:
            decoded = pydicom.pixels.pixel_array(ds, decoding_plugin=plugin)
            ok = np.array_equal(decoded, original)
            outcomes[plugin or "(auto)"] = "OK (bit-exact)" if ok else "DECODED BUT VALUES DIFFER"
        except Exception as e:
            outcomes[plugin or "(auto)"] = f"FAILED: {type(e).__name__}: {e}"
    return outcomes


print("Codec plugin self-test (JPEG-LS round-trip, fully offline):")
_codec_results = _codec_self_test()
for _plugin, _status in _codec_results.items():
    print(f"  {_plugin:12s} {_status}")

if any(k == "(encoder unavailable)" for k in _codec_results):
    print()
    print("INCONCLUSIVE: this runtime could not ENCODE a JPEG-LS test image, so decoding could")
    print("not be exercised. That usually means pyjpegls did not install. Decoding may still work")
    print("via gdcm/pylibjpeg -- extraction will tell you, and it degrades safely (an instance")
    print("that cannot be decoded is skipped and counted, never a crash).")
elif not any(v.startswith("OK") for v in _codec_results.values()):
    print()
    print("WARNING: no plugin could round-trip JPEG-LS compressed pixel data in this runtime.")
    print("If your real data uses a compressed transfer syntax, extraction will skip a lot of")
    print("instances/series (safely -- it will not crash -- but you'll get far fewer embeddings")
    print("than expected). Try: Runtime > Restart session, then re-run this notebook from the top")
    print("-- a fresh runtime has fixed similar plugin-registration issues for other Colab users.")
else:
    print()
    print("At least one codec plugin is working correctly in this runtime -- good to proceed.")

In [ ]:
# 6. PREFLIGHT -- fail in the next two minutes, or not at all tonight
#
# Everything below is something that CAN fail and, if it does, fails better now than at 3am with
# nine hours of GPU time already burnt. Each check is cheap; together they cover every external
# dependency the long run has: credentials, gated access, the model download, the data paths, the
# output directory, disk, and the GPU.
#
# With UNATTENDED_RUN = True every finding is fatal. With it False they are warnings, which is what
# you want while poking at things interactively.
import os
import shutil
import time
import traceback

import torch

_fatal, _warn, _info = [], [], []


def _check(name, fn, fatal=True):
    """Run one preflight check, announcing it BEFORE it runs.

    This used to print only on completion. A check that takes four minutes -- the model
    download, or a parts scan over tens of thousands of files on the Drive mount -- therefore
    produced no output at all while it ran, and the preflight looked hung at whatever the
    PREVIOUS check had printed. That is the third time this project has produced a silent wait
    that was indistinguishable from a hang, so the name goes out first and the result completes
    the line.
    """
    print(f"  [....] {name:<34s} ", end="", flush=True)
    t0 = time.time()
    try:
        msg = fn()
        print(f"\r  [ok]   {name:<34s} {msg}   ({time.time()-t0:.1f}s)", flush=True)
        return True
    except Exception as e:
        detail = f"{type(e).__name__}: {e}"
        # Label it by what it actually MEANS for this run. Printing [FAIL] for something the
        # config has explicitly downgraded to a warning reads as broken when it is not.
        print(f"\r  [{'FAIL' if fatal else 'warn'}] {name:<34s} {detail}", flush=True)
        (_fatal if fatal else _warn).append(f"{name}: {detail}")
        return False


print("=" * 92)
print(f"PREFLIGHT   DATA_SOURCE={DATA_SOURCE}   models={MODELS_TO_RUN}   "
      f"unattended={UNATTENDED_RUN}")
print("=" * 92)

# ---- 1. compute -------------------------------------------------------------------------------
def _gpu():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "no CUDA device. A Pillar-0 forward pass is an 11-channel 256x256x128 volume through a "
            "32k-token transformer; on CPU that is minutes per scan, so an overnight run would "
            "finish a few dozen of several thousand. Runtime > Change runtime type > GPU, then "
            "Runtime > Restart session and Run all. (Set REQUIRE_GPU = False to proceed anyway.)")
    p = torch.cuda.get_device_properties(0)
    return f"{p.name}, {p.total_memory/1e9:.1f} GB"


_check("GPU", _gpu, fatal=REQUIRE_GPU)

# ---- 2. credentials and gated access ----------------------------------------------------------
def _hf_identity():
    from huggingface_hub import HfApi, login
    login(token=HF_TOKEN, add_to_git_credential=False)
    who = HfApi().whoami(token=HF_TOKEN)
    return f"authenticated as {who.get('name', '?')}"


_check("Hugging Face token", _hf_identity)


def _make_gated_check(repo):
    def _fn():
        from huggingface_hub import HfApi
        # A metadata call is enough to distinguish "no access" from "no network", and costs nothing.
        HfApi().model_info(repo, token=HF_TOKEN)
        return "access granted"
    return _fn


for _m in MODELS_TO_RUN:
    _repo = PILLAR0_REGISTRY_REPOS.get(_m, _m)
    _check(f"gated access: {_repo}", _make_gated_check(_repo))

# ---- 3. the model download (the single biggest thing that can go wrong late) -------------------
def _make_dl(repo):
    def _fn():
        from huggingface_hub import snapshot_download
        p = snapshot_download(repo, token=HF_TOKEN)
        n = sum(len(f) for _, _, f in os.walk(p))
        sz = sum(os.path.getsize(os.path.join(r, f))
                 for r, _, fs in os.walk(p) for f in fs) / 1e9
        if not os.path.exists(os.path.join(p, "model.safetensors")):
            raise RuntimeError(f"model.safetensors is missing from {p}")
        return f"{n} files, {sz:.1f} GB, cached"
    return _fn


if PREFLIGHT_DOWNLOAD_MODEL:
    for _m in MODELS_TO_RUN:
        _check(f"download: {PILLAR0_REGISTRY_REPOS.get(_m, _m)}",
               _make_dl(PILLAR0_REGISTRY_REPOS.get(_m, _m)))
else:
    print("  [skip] model download                 (PREFLIGHT_DOWNLOAD_MODEL = False)")

# ---- 4. data paths ----------------------------------------------------------------------------
if DATA_SOURCE == "india":
    def _csv():
        if not os.path.exists(CSV_PATH):
            d = os.path.dirname(CSV_PATH)
            listing = sorted(os.listdir(d))[:12] if os.path.isdir(d) else "(directory missing)"
            raise FileNotFoundError(f"{CSV_PATH} not found. {d} contains: {listing}")
        return f"{os.path.getsize(CSV_PATH)/1e6:.0f} MB"

    _check("matched-scans CSV", _csv)

    def _zips():
        if not os.path.isdir(CT_ROOT):
            raise FileNotFoundError(f"{CT_ROOT} is not a directory")
        months = sorted(d for d in os.listdir(CT_ROOT) if os.path.isdir(os.path.join(CT_ROOT, d)))
        zips = []
        for m in months:
            zips += [os.path.join(CT_ROOT, m, f) for f in os.listdir(os.path.join(CT_ROOT, m))
                     if f.lower().endswith(".zip")]
        if not zips:
            raise FileNotFoundError(f"no .zip under {CT_ROOT} (months seen: {months})")
        # Actually open one, so a Drive path that lists but cannot be read is caught now.
        import zipfile
        with zipfile.ZipFile(sorted(zips)[0]) as zf:
            n = len(zf.namelist())
        return f"{len(months)} month(s), {len(zips)} day-zip(s); opened {os.path.basename(sorted(zips)[0])} ({n:,} entries)"

    _check("day-zips on Drive", _zips)

else:
    def _drive_retry(fn, what, attempts=4, base=1.0):
        """Retry a Drive operation that fails with EIO.

        The Drive FUSE mount returns OSError(errno 5, "Input/output error") for reasons that
        have nothing to do with the file: a rate limit, a token refresh, a transient backend
        error. It is frequently gone a second later. Failing the whole preflight on the first
        one -- which is what stopped an RSNA run at the last check before extraction -- turns a
        blip into a night not run.
        """
        last = None
        for k in range(attempts):
            try:
                return fn()
            except OSError as e:
                last = e
                if getattr(e, "errno", None) != 5:
                    raise
                if k < attempts - 1:
                    print(f"\n         {what}: Drive returned EIO, retrying in "
                          f"{base * (k + 1):.0f}s ({k + 1}/{attempts - 1}) ...",
                          end="", flush=True)
                    time.sleep(base * (k + 1))
        raise last

    def _parts():
        d = os.path.join(RSNA_ROOT, "parts")
        if not os.path.isdir(d):
            # Say what IS there. A wrong RSNA_ROOT and a Drive hiccup produce very different
            # listings, and the message used to make them look identical.
            here = []
            try:
                here = sorted(os.listdir(RSNA_ROOT))[:15] if os.path.isdir(RSNA_ROOT) else []
            except OSError as e:
                raise RuntimeError(f"{RSNA_ROOT} could not even be listed: {e}")
            raise FileNotFoundError(
                f"{d} is not a directory. {RSNA_ROOT} contains: {here}\n"
                f"         If the archive is not split into parts, RSNA_ROOT is pointing at the "
                f"wrong folder -- cell B1 expects RSNA_ROOT/parts/part_00000 ...")

        # os.scandir, not listdir + getsize: half the Drive round trips. Progress is printed
        # because this is the first step that touches the 194 GB.
        t_scan = time.time()
        entries, failed = [], []
        it = _drive_retry(lambda: list(os.scandir(d)), "listing parts/")
        for e in it:
            if not e.name.startswith("part_"):
                continue
            try:
                sz = _drive_retry(lambda e=e: e.stat().st_size, f"stat {e.name}")
            except OSError as err:
                # Attribute the failure to a FILE. "OSError: [Errno 5]" with no path was the
                # entire message this check produced, which is unactionable.
                failed.append((e.name, f"{type(err).__name__}: {err}"))
                continue
            entries.append((e.name, sz))
            if len(entries) % 2000 == 0:
                print(f"\n         scanned {len(entries):,} part(s) "
                      f"({time.time()-t_scan:.0f}s) ...", end="", flush=True)
        if failed:
            raise RuntimeError(
                f"{len(failed)} part file(s) could not be read after retries, e.g. "
                f"{failed[:3]}.\n"
                f"         {len(entries)} other part(s) read fine, so the mount itself is alive. "
                f"A file that lists but cannot be stat'd is usually still uploading, or is a "
                f"Drive shortcut rather than a real file.")
        if not entries:
            raise FileNotFoundError(
                f"no part_* files in {d} (it holds {len(it)} entr(ies): "
                f"{sorted(x.name for x in it)[:10]})")
        entries.sort()
        names = [n for n, _ in entries]
        sizes = [z for _, z in entries]
        bad = [(names[i], sizes[i]) for i in range(len(sizes) - 1) if sizes[i] != sizes[0]]
        if bad:
            raise RuntimeError(f"short part(s) not at the end: {bad[:3]} -- upload still running?")
        head = _drive_retry(lambda: open(os.path.join(d, names[0]), "rb").read(8),
                            f"read {names[0]}")
        kind = ("zip" if head[:4] == b"PK\x03\x04" else
                "gzip (NOT usable -- see cell B2)" if head[:2] == b"\x1f\x8b" else "tar/other")
        return (f"{len(names)} part(s), {sum(sizes)/1e9:.1f} GB, first bytes look like {kind}")

    _check("RSNA split archive", _parts)

    def _labels():
        if RSNA_LABEL_CSV_PATH:
            if os.path.exists(RSNA_LABEL_CSV_PATH):
                return f"{os.path.getsize(RSNA_LABEL_CSV_PATH)/1e6:.0f} MB (standalone on Drive)"
            raise FileNotFoundError(
                f"RSNA_LABEL_CSV_PATH points at {RSNA_LABEL_CSV_PATH}, which does not exist. "
                f"Either fix the path or set RSNA_LABEL_CSV_PATH = '' to read the CSV out of the "
                f"archive instead.")
        return "will be read from inside the archive (RSNA_LABEL_CSV_PATH is empty)"

    _check("RSNA labels CSV", _labels)

    # One slice of RSNA-ICH is 512 x 512 stored as 16-bit, plus a DICOM header:
    #
    #     512 * 512 * 2 bytes = 524,288 B = 0.524 MB,  call it 0.53 MB with the header
    #
    # Named as a constant, in MEGABYTES, with the unit in the name. The first version of this
    # check wrote `need_gb = n_series * 30 * 0.525` -- megabytes multiplied as though they were
    # gigabytes -- and reported that 4,000 series needed 63,000 GB. It then refused to start a
    # run that fits in 63 GB with 170 GB free, and did it on every attempt.
    RSNA_SLICE_MB = 0.53
    # 35, not 30. The dataset paper gives the RANGE (20-60 axial images per exam); Burduja et al.
    # give the average ("approximately 35 slices per scan"), and the real archive agrees -- the
    # header index reports a median of 33 and cell B5 asked for 138,845 slices across 4,003
    # series, which is 34.7 each. At 30 this estimate came in at 62 GB against an actual 72.9 GB:
    # a 17% underestimate, harmless with 200 GB free and not harmless at all with 80.
    RSNA_SLICES_PER_SERIES = 35

    def _rsna_scratch_gb(n_series):
        return n_series * RSNA_SLICES_PER_SERIES * RSNA_SLICE_MB / 1024.0

    def _rsna_scratch():
        """Cell B5 copies the wanted slices to local disk before extraction reads any of them.

        That is not an optimisation to trade away: reading members straight out of the archive
        measured 9.5 s each in cell B1b, and RSNA_MAX_SERIES = 4000 series is ~120,000 members,
        which is thirteen days. If the scratch is not there, the run does not get slower -- it
        does not finish. So the space is checked HERE, before a night is committed, rather than
        discovered by cell B5 four hours in.

        The estimate is ROUGH -- it assumes a median-length series. Cell B5 computes the real
        figure from the header index and refuses there rather than filling the disk, so this
        check exists to catch "obviously not enough", not to be exact. It therefore warns rather
        than fails unless the shortfall is large.
        """
        d = LOCAL_ZIP_CACHE
        os.makedirs(d, exist_ok=True)
        st = os.statvfs(d)
        free_gb = st.f_bavail * st.f_frsize / 1e9
        n_series = RSNA_MAX_SERIES if RSNA_MAX_SERIES is not None else 21_700
        need_gb = _rsna_scratch_gb(n_series)
        # The arithmetic is shown, not just its result. Had the first version printed its own
        # working -- "4,000 x 30 x 0.53 MB" next to "63000 GB" -- the units error would have been
        # visible on the first line of output instead of blocking three nights.
        msg = (f"{free_gb:.0f} GB free at {d}; {n_series:,} series x "
               f"{RSNA_SLICES_PER_SERIES} slices x {RSNA_SLICE_MB} MB = {need_gb:.0f} GB needed")
        fits = int(free_gb * 0.85 / (RSNA_SLICES_PER_SERIES * RSNA_SLICE_MB / 1024.0))
        if need_gb * 1.15 > free_gb:
            raise RuntimeError(
                msg + f".\n         Lower RSNA_MAX_SERIES (a proportional stratified sample, so "
                      f"prevalence is unaffected), or point LOCAL_ZIP_CACHE at a bigger disk. "
                      f"RSNA_MAX_SERIES = {fits:,} would fit.")
        if need_gb * 1.6 > free_gb:
            return (msg + f" -- that is tight. Cell B5 measures the real total and refuses "
                          f"rather than filling the disk; {fits:,} series would be comfortable.")
        return msg

    _check("RSNA local scratch for cell B5", _rsna_scratch)

    def _rsna_inodes():
        """Cell B5 writes ~120,000 files. Files are cheap; inodes are finite.

        The fan-out into 256 subdirectories removes the ext4 large-directory problem (readdir
        returns entries in hash order, so a listdir+stat loop over 120,000 files is near-random
        inode I/O). What it does not remove is running out of inodes, which fails as ENOSPC with
        gigabytes still free -- a confusing way to lose a night.
        """
        st = os.statvfs(LOCAL_ZIP_CACHE)
        free_i = st.f_favail
        n_series = RSNA_MAX_SERIES if RSNA_MAX_SERIES is not None else 21_700
        need_i = n_series * RSNA_SLICES_PER_SERIES + 300
        if free_i and need_i * 1.2 > free_i:
            raise RuntimeError(
                f"{free_i:,} free inode(s) at {LOCAL_ZIP_CACHE}, and cell B5 writes about "
                f"{need_i:,} files. Running out of inodes reports 'No space left on device' with "
                f"plenty of free bytes, which is a confusing way to lose a night. Lower "
                f"RSNA_MAX_SERIES.")
        return (f"{free_i:,} free inode(s) for ~{need_i:,} file(s), fanned out over 256 "
                f"subdirectories" if free_i else "inode count not reported by this filesystem")

    _check("RSNA local inodes", _rsna_inodes)

    def _drive_cache():
        """The DriveFS content cache shares the disk cell B5 writes to.

        colabtools#960: DriveFS caches every byte it reads and, per the report, never evicts. This
        run streams 194.7 GB past the mount twice. If the cache really does keep all of it, the
        disk fills long before the extracted DICOMs do. Both long passes prune it as they go; this
        is the starting reading.
        """
        _cg = globals().get("drive_cache_gb")
        if _cg is None:
            return "cell 2d has not run -- cache size unknown"
        gb = _cg()
        return (f"{gb:.1f} GB now; pruned automatically above "
                f"{globals().get('DRIVE_CACHE_PRUNE_GB', 20):.0f} GB during the long passes")

    _check("DriveFS content cache", _drive_cache)

# ---- 5. output directory, actually written to -------------------------------------------------
def _writable():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    probe = os.path.join(OUTPUT_DIR, ".preflight_write_probe")
    payload = b"x" * 4096
    with open(probe, "wb") as f:
        f.write(payload)
    with open(probe, "rb") as f:
        got = f.read()
    os.remove(probe)
    if got != payload:
        raise RuntimeError("wrote 4096 bytes to OUTPUT_DIR and read back something different")
    return OUTPUT_DIR


_check("OUTPUT_DIR writable", _writable)


def _localdisk():
    st = os.statvfs("/content")
    free = st.f_bavail * st.f_frsize / 1e9
    need = 8.0 if ZIP_ACCESS_MODE == "stream" else 40.0
    if free < need:
        raise RuntimeError(f"{free:.1f} GB free on /content, want >= {need:.0f} GB for "
                            f"ZIP_ACCESS_MODE={ZIP_ACCESS_MODE!r}")
    return f"{free:.1f} GB free on /content"


_check("local disk", _localdisk, fatal=False)

# ---- 6. is this run actually configured to finish anything? -----------------------------------
# A config that is internally consistent but produces almost no work is the failure mode with no
# error message: you come back to "20 series extracted, no failures" and everything downstream
# quietly reports nothing. Caught here, where it costs a second.
def _will_do_real_work():
    caps = []
    if TRIAL_RUN_MAX_SERIES is not None:
        caps.append(f"TRIAL_RUN_MAX_SERIES = {TRIAL_RUN_MAX_SERIES}")
    if MAX_SERIES_PER_RUN is not None:
        caps.append(f"MAX_SERIES_PER_RUN = {MAX_SERIES_PER_RUN}")
    if MAX_SECONDS_PER_RUN is not None:
        caps.append(f"MAX_SECONDS_PER_RUN = {MAX_SECONDS_PER_RUN}s")
    if UNATTENDED_RUN and TRIAL_RUN_MAX_SERIES is not None:
        raise RuntimeError(
            f"UNATTENDED_RUN is True but TRIAL_RUN_MAX_SERIES = {TRIAL_RUN_MAX_SERIES}. The run "
            f"would stop after {TRIAL_RUN_MAX_SERIES} series -- a few minutes of work -- and "
            f"every downstream cell would then quietly report almost nothing, because the "
            f"classifier sweep needs far more than that before it will evaluate anything. Set "
            f"TRIAL_RUN_MAX_SERIES = None for the overnight run."
        )
    return ", ".join(caps) if caps else "uncapped -- will run until everything is done"


_check("run is uncapped", _will_do_real_work)

# ---- 7. verdict -------------------------------------------------------------------------------
print()
if _fatal:
    print("=" * 92)
    for f in _fatal:
        print(f"  FATAL  {f}")
    print("=" * 92)
    if UNATTENDED_RUN:
        raise RuntimeError(
            f"{len(_fatal)} preflight check(s) failed. Stopping here on purpose: with "
            f"UNATTENDED_RUN = True there is nobody around to notice this later. Fix the items "
            f"above and re-run from cell 1. Set UNATTENDED_RUN = False to continue anyway."
        )
    print("UNATTENDED_RUN is False, so continuing despite the failures above.")
elif _warn:
    for w in _warn:
        print(f"  warn   {w}")
    print("\nPREFLIGHT PASSED with warnings -- safe to leave running.")
else:
    print("PREFLIGHT PASSED. Everything the overnight run depends on has been verified:")
    print("  credentials, gated access, the model weights, the data paths, the output directory,")
    print("  disk and the GPU. From here on nothing else needs you at the keyboard.")

if UNATTENDED_RUN:
    # Three things about Colab that this notebook cannot fix from inside itself, and that decide
    # whether a long run survives the night. Each is from Google's own documentation or from an
    # open colabtools issue, not from folklore.
    print()
    print("BEFORE YOU LEAVE IT RUNNING -- three things only you can do:")
    print("  1. Edit > Notebook settings > 'Omit code cell output when saving this notebook'.")
    print("     Colab autosaves cell OUTPUT into the Drive-backed .ipynb while a cell is busy.")
    print("     On a long, chatty cell that save fails, the frontend hangs, and reopening the")
    print("     notebook hangs again while the cell is still working (colabtools#535).")
    print("  2. Background execution is a Colab Pro+ feature, not Pro. On Pro, a CLOSED TAB is")
    print("     subject to an idle shutdown whose length Google does not publish. If you are on")
    print("     Pro, leave the tab open; if you cannot, expect to resume -- every pass in this")
    print("     notebook checkpoints, so resuming costs minutes, not the night.")
    print("  3. To check on it from anywhere without touching the tab, read the heartbeat:")
    print(f"       {os.path.join(OUTPUT_DIR, 'run_heartbeat.jsonl')}")
    print("     Its last line says which stage is running and when it last moved. A stale")
    print("     timestamp means the kernel is gone; a moving one means it is simply working.")

## Loading the matched-scans table without drowning in it

The combined CSV is ~800 MB, and most of that weight is free-text report columns that no part of
this pipeline reads. Loading it naively costs several GB of RAM and a few minutes, every session.

Cell 7 instead:

1. reads **only the header row** (`nrows=0`) to discover what columns actually exist — the column
   set is not hardcoded, because it has changed between exports;
2. keeps identifier/metadata columns plus every column that *looks like* a finding flag
   (`b_/a_/c_/s_/p_/g_/v_/f_/n_`-prefixed, or already 0/1) and drops long free-text columns;
3. reads the CSV once with that `usecols` list and downcasts the finding columns to `int8`;
4. writes the result to `matched_table.parquet` and records the source CSV's size + mtime in a
   sidecar. Later sessions load the parquet in a second or two, and it is rebuilt automatically if
   the source CSV changes.

If a column you want was dropped, add its name to `EXTRA_KEEP_COLUMNS` and re-run — the cache
key includes that list, so it rebuilds itself.

In [ ]:
# 7. Load the matched-scans table (memory-safe, cached to parquet)  [RESUMABLE]

if DATA_SOURCE != "india":
    print("DATA_SOURCE is {!r} -- skipping this cell (Part A / India only).".format(DATA_SOURCE))
else:
    import json
    import os
    import numpy as np
    import pandas as pd

    # Identifier / metadata columns the rest of the notebook needs by name.
    ID_COLUMNS = [
        "SourceFolder", "FullFolderPath", "DiskPath", "NumSlices",
        "PatientID", "PatientName", "PatientSex", "PatientAge", "PatientBirthDate",
        "AccessionNumber", "StudyInstanceUID", "SeriesInstanceUID",
        "StudyID", "StudyDate", "StudyDescription",
        "SeriesNumber", "SeriesDescription", "Modality", "BodyPartExamined",
        "InstitutionName", "Manufacturer", "ManufacturerModelName", "ProtocolName",
        "SliceThickness", "KVP", "Rows", "Columns", "Hospital",
        "MatchMethod", "MatchedReportName", "MatchedReportAge", "MatchedReportSex",
        "MatchedReportDate", "MatchedStudyType",
    ]

    # Report-derived finding flags in this export are prefixed by body system:
    #   b_ brain/head (44 cols)   a_ abdomen (28)   c_ chest (25)   s_ spine (17)
    #   p_ pelvis (15)            g_ (10)           v_ (6)          f_ (5)   n_ (5)
    FINDING_PREFIXES = ("b_", "a_", "c_", "s_", "p_", "g_", "v_", "f_", "n_")

    # Anything extra you want carried through (e.g. a report-text column for spot checks).
    EXTRA_KEEP_COLUMNS = []


    def _cache_key():
        st = os.stat(CSV_PATH)
        return {
            "csv_path": CSV_PATH,
            "csv_size": st.st_size,
            "csv_mtime": int(st.st_mtime),
            "id_columns": sorted(ID_COLUMNS),
            "finding_prefixes": list(FINDING_PREFIXES),
            "extra": sorted(EXTRA_KEEP_COLUMNS),
            "schema_version": 3,
        }


    def _atomic_write_json(path, obj):
        tmp = path + ".tmp"
        with open(tmp, "w") as f:
            json.dump(obj, f, indent=2)
        os.replace(tmp, path)


    def load_matched_table():
        key = _cache_key()
        if os.path.exists(TABLE_CACHE_PATH) and os.path.exists(TABLE_CACHE_META):
            try:
                with open(TABLE_CACHE_META) as f:
                    cached_key = json.load(f)
                if cached_key == key:
                    out = pd.read_parquet(TABLE_CACHE_PATH)
                    print(f"loaded cached table: {len(out):,} rows x {out.shape[1]} cols "
                          f"from {TABLE_CACHE_PATH}")
                    return out
                print("cache key changed (source CSV or column selection differs) -- rebuilding.")
            except Exception as e:
                print(f"cache unreadable ({e}) -- rebuilding.")

        print(f"reading header of {CSV_PATH} ...")
        header = pd.read_csv(CSV_PATH, nrows=0)
        all_cols = list(header.columns)
        print(f"  source has {len(all_cols)} columns")

        id_present = [c for c in ID_COLUMNS if c in all_cols]
        missing_ids = [c for c in ID_COLUMNS if c not in all_cols]
        finding_cols = [
            c for c in all_cols
            if c.lower().startswith(FINDING_PREFIXES) and c not in id_present
        ]
        extra_present = [c for c in EXTRA_KEEP_COLUMNS if c in all_cols]
        usecols = list(dict.fromkeys(id_present + finding_cols + extra_present))

        print(f"  keeping {len(id_present)} id/metadata + {len(finding_cols)} finding "
              f"+ {len(extra_present)} extra = {len(usecols)} columns "
              f"({len(all_cols) - len(usecols)} dropped, mostly free text)")
        if missing_ids:
            print(f"  NOTE: these expected id columns are absent from this export and will simply not "
                  f"be available downstream: {missing_ids}")
        if not finding_cols:
            print("  WARNING: no finding-flag columns matched the prefixes. The classification half of "
                  "this notebook will have nothing to train on. Check FINDING_PREFIXES against the "
                  "real column names before running the sweep.")

        print("  reading (this is the slow one -- a few minutes on an 800 MB CSV) ...")
        out = pd.read_csv(CSV_PATH, usecols=usecols, low_memory=False)

        # Findings are report-derived flags; store them compactly and make "missing" an explicit 0
        # rather than a NaN that later silently poisons a .astype(bool).
        for c in finding_cols:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).clip(0, 1).astype("int8")
        if "NumSlices" in out.columns:
            out["NumSlices"] = pd.to_numeric(out["NumSlices"], errors="coerce").fillna(0).astype("int32")

        out.to_parquet(TABLE_CACHE_PATH, index=False)
        _atomic_write_json(TABLE_CACHE_META, key)
        print(f"  cached to {TABLE_CACHE_PATH}")
        return out


    df = load_matched_table()
    FINDING_COLS_ALL = [c for c in df.columns if c.lower().startswith(FINDING_PREFIXES)]

    print()
    print(f"rows            : {len(df):,}")
    for c in ("StudyInstanceUID", "SeriesInstanceUID", "PatientID"):
        if c in df.columns:
            print(f"unique {c:18s}: {df[c].nunique():,}")
    print(f"finding columns : {len(FINDING_COLS_ALL)}")
    if "Modality" in df.columns:
        print(f"modalities      : {df['Modality'].value_counts().to_dict()}")

In [ ]:
# 8. Tag each series by what was actually scanned, and score it as a candidate volume
#
# Two separate questions get confused easily, so they are kept apart here:
#
#   (a) WHAT ANATOMY IS IN THE PIXELS?  Answered from the DICOM's own BodyPartExamined and
#       SeriesDescription. This decides which Pillar-0 checkpoint may legitimately see the volume.
#
#   (b) WHAT DOES THE REPORT SAY THE STUDY WAS?  That is MatchedStudyType, attached by report
#       matching. On this dataset the two disagree for a substantial minority of studies. The
#       disagreement is recorded per row here and used as an OPTIONAL filter at classification
#       time (cell 21) -- never at extraction time, because embeddings are expensive and label
#       trust is a downstream decision you may want to A/B.

if DATA_SOURCE != "india":
    print("DATA_SOURCE is {!r} -- skipping this cell (Part A / India only).".format(DATA_SOURCE))
else:
    import re

    # Three different reasons a series is not a usable volume, kept separate because they call for
    # different treatment. Lumping them together (as an earlier draft did) silently threw away 18
    # studies whose only surviving series were axial MPR reformats -- recoverable data.
    #
    #   HARD JUNK      never diagnostic pixel data at all -> excluded outright
    #   NON-AXIAL      a real volume, but reformatted into a plane Pillar-0's head recipe does not
    #                  expect (it center-crops in Z assuming axial slices) -> excluded for head
    #   REFORMAT       an axial MPR: real, usable, but a derived reconstruction, so it is ranked
    #                  BELOW a native acquisition and only used when a study has nothing else
    _JUNK_DESC_PATTERNS = [
        "topogram", "scout", "localizer", "localiser", "surview", "scanogram",
        "dose report", "dose_report", "patient protocol", "screen save", "screensave",
        "secondary capture", "key images", "key image", "summary", "report",
    ]
    _JUNK_DESC_REGEX = re.compile(r"\b(3d|vrt|mip|ssd|cpr)\b")
    _NONAXIAL_REGEX = re.compile(r"\b(sag|sagittal|cor|coronal|obl|oblique)\b")
    _REFORMAT_REGEX = re.compile(r"\b(mpr|recon|reformat)\b")

    _REGION_DESC_KEYWORDS = {
        "head":    ["brain", "head", "skull", "cranial", "crani", "sinus", "pns", "facial",
                     "orbit", "temporal bone", "inner ear", "petrous", "mastoid", "sella"],
        "chest":   ["chest", "thorax", "lung", "hrct", "pulmonary", "thoracic"],
        "abdomen": ["abdomen", "abdo", "kub", "liver", "pancreas", "renal", "kidney",
                     "pelvis", "urogram", "cect abdomen"],
    }

    # Within head, the sub-anatomy matters: Pillar0-HeadCT is a brain model. A dedicated inner-ear or
    # facial-bone acquisition is still "HEAD" by BodyPartExamined but is not a brain volume.
    _HEAD_SUBTYPE_KEYWORDS = {
        "brain":         ["brain", "head", "cranial", "crani", "skull"],
        "sinus_facial":  ["sinus", "pns", "facial", "orbit", "maxill", "mandible", "nasal"],
        "temporal_bone": ["temporal bone", "inner ear", "petrous", "mastoid", "iac"],
    }


    def _norm(s):
        return re.sub(r"\s+", " ", str(s if s is not None else "")).strip().lower()


    def is_junk_description(desc):
        """Hard exclusion: this series is not diagnostic pixel data."""
        d = _norm(desc)
        if not d:
            return False
        if any(p in d for p in _JUNK_DESC_PATTERNS):
            return True
        return bool(_JUNK_DESC_REGEX.search(d))


    def is_nonaxial_description(desc):
        """Sagittal/coronal/oblique reformat. Excluded for head: the Pillar-0 head recipe
        center-crops 128 slices along Z on the assumption that Z is the slice axis of an axial
        acquisition. Feeding it a coronal stack does not error -- it just produces an embedding of
        something the model has never seen in that orientation, which is worse."""
        d = _norm(desc)
        return bool(d) and bool(_NONAXIAL_REGEX.search(d))


    def is_reformat_description(desc):
        """Axial MPR / recon: usable, but derived. Ranked below native acquisitions."""
        d = _norm(desc)
        return bool(d) and bool(_REFORMAT_REGEX.search(d))


    def classify_region(body_part, series_desc, study_desc, matched_study_type):
        """Region from the pixels' own metadata. BodyPartExamined wins when it is present and
        unambiguous; SeriesDescription/StudyDescription are the fallback. MatchedStudyType is used
        only as a last resort, and the caller records whether it agreed."""
        bp = _norm(body_part).strip(",")
        if bp:
            if "head" in bp or "brain" in bp or "skull" in bp:
                return "head"
            if "chest" in bp or "thorax" in bp or "lung" in bp:
                return "chest"
            if "abdomen" in bp or "pelvis" in bp or "kub" in bp:
                return "abdomen"
            for region in ("neck", "cspine", "lspine", "tspine", "spine", "extremity",
                            "hip", "shoulder", "knee"):
                if region in bp:
                    return "other"
        for text in (series_desc, study_desc, matched_study_type):
            t = _norm(text)
            if not t:
                continue
            for region, kws in _REGION_DESC_KEYWORDS.items():
                if any(kw in t for kw in kws):
                    return region
        return "unknown"


    def classify_head_subtype(series_desc, study_desc):
        """Order matters: 'temporal bone' and sinus/facial are more specific than the generic
        brain/head words, and a description like 'Brain+PNS' should land as brain (it does contain a
        full brain acquisition), so brain is checked last only among ties."""
        t = _norm(series_desc) + " " + _norm(study_desc)
        if any(kw in t for kw in _HEAD_SUBTYPE_KEYWORDS["temporal_bone"]):
            return "temporal_bone"
        if any(kw in t for kw in _HEAD_SUBTYPE_KEYWORDS["brain"]):
            return "brain"
        if any(kw in t for kw in _HEAD_SUBTYPE_KEYWORDS["sinus_facial"]):
            return "sinus_facial"
        return "head_other"


    # Siemens-style reconstruction kernels appear in the description as e.g. "Hr40 S3", "Hr60 S3",
    # "Br40", "J45". The two-digit number is the sharpness: low = smooth/soft-tissue, high = sharp/bone.
    # On this dataset a head study typically ships {5.00 Hr40, 2.00 Hr40, 1.00 Hr60} -- soft, soft,
    # bone. For brain parenchyma (hemorrhage, infarct, edema) the soft kernel is the right one; a bone
    # kernel makes parenchymal contrast far noisier. 50 is the conventional split point.
    # The trailing [a-z]? is not cosmetic. Siemens writes kernels both ways -- "Hr40 S3" (this
    # export's style) and "B30f" / "H31s" / "J70h" / "B70f" (the more common one elsewhere). The
    # older pattern required a word boundary straight after the digits, so every trailing-letter
    # form failed to match, is_soft_kernel returned its neutral True for BOTH the soft and the bone
    # reconstruction, and the ranking fell through to "most slices" -- which on a typical study
    # {5.00 B30f, 2.00 B30f, 1.00 B70f} selects the 150-slice BONE series as primary. That is the
    # exact inversion this rule exists to prevent.
    _KERNEL_RE = re.compile(r"\b([a-z]{1,2}r?)(\d{2})[a-z]?\b")
    SHARP_KERNEL_THRESHOLD = 50


    def kernel_sharpness(series_desc):
        """Returns (found: bool, number: int). Not finding a kernel is not an error -- plenty of
        scanners never write one -- so callers treat 'not found' as neutral, never as sharp."""
        m = _KERNEL_RE.search(_norm(series_desc))
        if not m:
            return False, -1
        return True, int(m.group(2))


    def is_soft_kernel(series_desc):
        found, num = kernel_sharpness(series_desc)
        if not found:
            return True          # neutral -> do not penalise
        return num < SHARP_KERNEL_THRESHOLD


    def has_kernel(series_desc):
        """Whether a kernel could be read at all. Reported so that 'every series is soft' can be
        told apart from 'no series had a readable kernel and the preference did nothing'."""
        return kernel_sharpness(series_desc)[0]


    # ---- apply -----------------------------------------------------------------------------------
    _src = df["SourceFolder"].astype(str).str.split("/")
    df["zip_month"] = _src.str[0]
    df["zip_day"] = _src.str[1]
    df["source_study_uid"] = _src.str[2]
    df["source_series_uid"] = _src.str[3]

    _bad_depth = (_src.str.len() != 4).sum()
    if _bad_depth:
        print(f"WARNING: {_bad_depth} row(s) have a SourceFolder that is not "
              f"MM/DD/<study>/<series>. Those rows cannot be mapped to a day-zip and are excluded "
              f"from the manifest in cell 9.")

    for col, src_col in (("source_study_uid", "StudyInstanceUID"),
                          ("source_series_uid", "SeriesInstanceUID")):
        if src_col in df.columns:
            agree = (df[col].astype(str) == df[src_col].astype(str)).mean()
            print(f"SourceFolder component matches {src_col}: {agree:.4%}")
            if agree < 0.999:
                print(f"  *** The manifest in cell 9 assumes this is ~100%. It is not. Investigate "
                      f"before trusting any zip lookup. ***")

    df["region"] = [
        classify_region(bp, sd, std, mst)
        for bp, sd, std, mst in zip(
            df.get("BodyPartExamined", pd.Series([None] * len(df))),
            df.get("SeriesDescription", pd.Series([None] * len(df))),
            df.get("StudyDescription", pd.Series([None] * len(df))),
            df.get("MatchedStudyType", pd.Series([None] * len(df))),
        )
    ]
    df["head_subtype"] = [
        classify_head_subtype(sd, std) if r == "head" else ""
        for r, sd, std in zip(df["region"],
                               df.get("SeriesDescription", pd.Series([None] * len(df))),
                               df.get("StudyDescription", pd.Series([None] * len(df))))
    ]
    _desc_series = df.get("SeriesDescription", pd.Series([None] * len(df)))
    df["is_junk_series"] = _desc_series.map(is_junk_description)
    df["is_nonaxial"] = _desc_series.map(is_nonaxial_description)
    df["is_reformat"] = _desc_series.map(is_reformat_description)
    df["is_soft_kernel"] = _desc_series.map(is_soft_kernel)
    df["has_kernel"] = _desc_series.map(has_kernel)

    # Does the report-side study type agree with what the pixels say? Recorded, not enforced.
    _report_region = [
        classify_region(None, None, None, mst)
        for mst in df.get("MatchedStudyType", pd.Series([None] * len(df)))
    ]
    df["report_region"] = _report_region
    df["studytype_agrees"] = (df["region"] == df["report_region"])

    # Which hospitals actually contributed. This matters for interpretation and nothing was
    # reporting it: the collection is dominated by one site and one scanner model, so an internal
    # validation number here is close to single-centre, whatever the row count suggests.
    for _col in ("Hospital", "InstitutionName"):
        if _col in df.columns:
            _vc = df[_col].fillna("(blank)").value_counts()
            print()
            print(f"{_col} ({_vc.shape[0]} distinct):")
            for _k, _v in _vc.head(6).items():
                print(f"  {str(_k)[:52]:<54s} {_v:>7,}  ({_v/len(df):5.1%})")
            if _vc.shape[0] > 6:
                print(f"  ... and {_vc.shape[0]-6} more")
    if "ManufacturerModelName" in df.columns:
        _m = df["ManufacturerModelName"].fillna("(blank)").value_counts()
        print()
        print(f"scanner model: {_m.index[0]} for {_m.iloc[0]/len(df):.1%} of series"
              + (f", {_m.shape[0]-1} other model(s)" if _m.shape[0] > 1 else " -- a single scanner"))

    print()
    print("region (from DICOM content):")
    print(df["region"].value_counts().to_string())
    print()
    _ct = df[df.get("Modality", pd.Series([""] * len(df))) == "CT"]
    if len(_ct):
        _tgt = _ct[_ct["region"] == TARGET_REGION]
        print(f"CT series in target region '{TARGET_REGION}': {len(_tgt):,} "
              f"across {_tgt['StudyInstanceUID'].nunique():,} studies")
        if TARGET_REGION == "head":
            print("  head subtypes:", _tgt["head_subtype"].value_counts().to_dict())
        _step1 = ~_tgt["is_junk_series"]
        _step2 = _step1 & ~_tgt["is_nonaxial"]
        _step3 = _step2 & (_tgt["NumSlices"] >= MIN_SLICES_PER_SERIES)
        print(f"  after hard-junk filter (scout/dose/VRT/3D/MIP) : {_step1.sum():,}")
        print(f"  after dropping sagittal/coronal reformats      : {_step2.sum():,}")
        print(f"  after NumSlices >= {MIN_SLICES_PER_SERIES:<3d}                        : {_step3.sum():,}")
        print(f"  of those, axial MPR reformats (ranked last)    : "
              f"{(_step3 & _tgt['is_reformat']).sum():,}")
        _kern = _tgt["has_kernel"].mean() if len(_tgt) else float("nan")
        print(f"  reconstruction kernel readable from the description : {_kern:.1%}")
        if _kern < 0.5:
            print("    ^ under half. The soft-vs-bone preference in cell 9 can only act on series")
            print("      whose kernel it can read; for the rest it is neutral and the ranking")
            print("      falls through to slice count -- which on a study whose thin series is the")
            print("      BONE reconstruction picks the wrong one. Check SeriesDescription before")
            print("      trusting is_primary.")
        _agree = _tgt["studytype_agrees"].mean() if len(_tgt) else float("nan")
        print(f"  MatchedStudyType agrees with pixel-derived region for {_agree:.1%} of these rows")
        print("  (recorded as `studytype_agrees`; cell 21 can filter on it, extraction never does)")

In [ ]:
# 9. Build the day-zip manifest  [RESUMABLE -- rebuilt only when its inputs change]
#
# No zip is opened here. SourceFolder already says MM/DD/<study>/<series>, and the archives are
# laid out as CT_ROOT/<MM>/<DD>.zip, so the mapping is arithmetic. Cell 10 then checks which of
# those zips actually exist on Drive.

if DATA_SOURCE != "india":
    print("DATA_SOURCE is {!r} -- skipping this cell (Part A / India only).".format(DATA_SOURCE))
else:
    import hashlib
    import json
    import os
    import re

    # Which head sub-anatomies may be picked as a study's primary volume. Pillar0-HeadCT is a BRAIN CT
    # model: a dedicated inner-ear or facial-bone acquisition is "HEAD" by BodyPartExamined but is not
    # a brain volume, and feeding one in produces an embedding that does not mean what you think.
    # Widen this only if you also intend to interpret the results that way.
    ALLOWED_HEAD_SUBTYPES = ("brain",)

    # False (default): a study with no brain series still contributes its best head series (a sinus or
    #   facial-bone acquisition, say) -- the manifest records `head_subtype` so you can filter later,
    #   and findings like b_sinusitis (934 positives here) genuinely live on those scans.
    # True: drop them entirely, so every extracted volume is a brain acquisition.
    RESTRICT_TO_ALLOWED_HEAD_SUBTYPES = True

    # Filename pattern for a day archive under CT_ROOT. Both zero-padded and bare day numbers are
    # tried, because real exports are inconsistent about it.
    def candidate_zip_paths(month, day):
        """Ordered, deterministic. The zero-padded form is first because that is what the Drive
        listing actually shows (2025/01/05.zip), so when NO candidate exists the path recorded in the
        manifest is still the one a human should go looking for. An earlier draft built these from a
        set, which made the fallback path non-deterministic between runs."""
        m, d = str(month), str(day)
        variants = [
            (m.zfill(2), d.zfill(2)),
            (m, d),
            (m.lstrip("0") or "0", d.lstrip("0") or "0"),
        ]
        out = []
        for mm, dd in variants:
            for ext in (".zip", ".ZIP"):
                out.append(os.path.join(CT_ROOT, mm, f"{dd}{ext}"))
        seen, uniq = set(), []
        for p in out:
            if p not in seen:
                seen.add(p)
                uniq.append(p)
        return uniq


    def select_series(table):
        """Rows that are legitimate candidate volumes for TARGET_REGION."""
        sel = table.copy()
        if "Modality" in sel.columns:
            _m = sel["Modality"].astype(str).str.upper()
            _drop = sel[_m != "CT"]
            if len(_drop):
                # This export contains 337 MR rows on a MAGNETOM Sempra and 12,051 structured
                # reports. Silently filtering them was correct and invisible; a cohort claim
                # needs the number, so it is printed.
                print(f"  modality filter: dropped {len(_drop):,} non-CT row(s) "
                      f"{dict(_drop['Modality'].astype(str).value_counts())}")
            sel = sel[_m == "CT"]
        sel = sel[sel["region"] == TARGET_REGION]
        sel = sel[~sel["is_junk_series"].fillna(False)]
        if TARGET_REGION == "head":
            # Pillar-0's head recipe crops 128 slices along Z assuming an axial stack.
            sel = sel[~sel["is_nonaxial"].fillna(False)]
            if RESTRICT_TO_ALLOWED_HEAD_SUBTYPES:
                sel = sel[sel["head_subtype"].isin(ALLOWED_HEAD_SUBTYPES)]
        sel = sel[sel["NumSlices"] >= MIN_SLICES_PER_SERIES]
        sel = sel[sel["SourceFolder"].astype(str).str.count("/") == 3]
        # One row per series -- the combined CSV carries one row per SOP instance for some exports.
        # Secondary key so a tie on NumSlices resolves the same way on every run -- the whole manifest
        # is built around being reproducible, and an unspecified tie-break quietly undermines that.
        sel = (sel.sort_values(["NumSlices", "SourceFolder"], ascending=[False, True])
                  .drop_duplicates("SeriesInstanceUID"))
        return sel


    def rank_within_study(sel):
        """Rank candidate series within each study, best first.

        Preference order, most important first:
          1. an allowed head sub-anatomy (brain, not inner-ear/facial) when TARGET_REGION == "head"
          2. a native acquisition over an axial MPR reformat
          3. a soft-tissue reconstruction kernel over a bone kernel
          4. more slices  (on this scanner that means thinner slices: 1.00 mm > 2.00 > 5.00)
          5. lower SeriesNumber, purely as a deterministic tie-break so reruns pick the same series
        """
        s = sel.copy()
        if TARGET_REGION == "head":
            s["_subtype_ok"] = s["head_subtype"].isin(ALLOWED_HEAD_SUBTYPES).astype(int)
        else:
            s["_subtype_ok"] = 1
        s["_native"] = (~s["is_reformat"].fillna(False)).astype(int)
        s["_soft"] = s["is_soft_kernel"].fillna(True).astype(int)
        s["_nslices"] = s["NumSlices"].astype(int)
        # .get with a scalar default returns a numpy scalar when the column is absent, and a
        # numpy scalar has no .fillna -- an AttributeError mid-manifest rather than a fallback.
        _sn = (s["SeriesNumber"] if "SeriesNumber" in s.columns
               else pd.Series(9999, index=s.index))
        s["_seriesnum"] = pd.to_numeric(_sn, errors="coerce").fillna(9999)
        s = s.sort_values(
            ["StudyInstanceUID", "_subtype_ok", "_native", "_soft", "_nslices", "_seriesnum"],
            ascending=[True, False, False, False, False, True],
        )
        s["priority_rank"] = s.groupby("StudyInstanceUID").cumcount()
        return s


    def refresh_zip_existence(m):
        """Re-stat every referenced archive, even on a cache hit.

        `zip_exists` is a fact about the filesystem RIGHT NOW, not about the manifest's inputs, so it
        has no business being frozen into the cache. Without this, uploading a missing month and
        re-running cell 9 hits the cache (the key is unchanged -- it contains the VALUE of CT_ROOT,
        not the set of files under it), the whole month stays `zip_exists=False`, and cell 10 keeps
        reporting data you just uploaded as missing. It is ~280 stat() calls; it costs nothing."""
        if not len(m):
            return m
        m = m.copy()
        resolved = {}
        for month, day in m[["zip_month", "zip_day"]].drop_duplicates().itertuples(index=False):
            found = ""
            for cand in candidate_zip_paths(month, day):
                if os.path.exists(cand):
                    found = cand
                    break
            resolved[(str(month), str(day))] = (found or candidate_zip_paths(month, day)[0], bool(found))
        keys = list(zip(m["zip_month"].astype(str), m["zip_day"].astype(str)))
        was = m["zip_exists"].astype(bool).to_numpy() if "zip_exists" in m.columns else None
        m["zip_path"] = [resolved[k][0] for k in keys]
        m["zip_exists"] = [resolved[k][1] for k in keys]
        if was is not None:
            gained = int((m["zip_exists"].to_numpy() & ~was).sum())
            lost = int((~m["zip_exists"].to_numpy() & was).sum())
            if gained or lost:
                print(f"  re-checked archives on disk: {gained} series became available, "
                      f"{lost} became unavailable since the manifest was cached.")
        return m


    def make_person_key(name, sex, birth_date, study_date, age, patient_id):
        """A stable identifier for a HUMAN, which PatientID in this export is not.

        PatientID here is a registration counter issued at each visit, not a medical record
        number. Two independent pieces of evidence, both from the matched-scans CSV:

          * Spearman correlation between numeric PatientID and StudyDate is 0.78. A stable MRN
            has no reason to track the calendar.
          * The per-month minimum PatientID rises steadily through the year, by about 2,000 a
            month, which is a counter incremented once per registration.

        And directly: 189 people appear under two or more PatientIDs across 401 studies.
        In one such case a single person (same name, sex and birth year) holds two consecutive
        PatientIDs issued on the same day -- registered twice at one visit.

        This matters because every split in cell 21 groups on it. Two scans of one person under
        two PatientIDs are two groups to GroupShuffleSplit, so the same skull -- same atrophy,
        same chronic infarcts, often the same acute finding on a follow-up -- lands in train and
        test. Simulated through the real evaluate_target at this cohort's size, that is worth
        +0.01 to +0.03 AUROC. Real leakage, and invisible to a leakage audit that checks only
        that no *patient_id* spans the split.

        The key is name + sex + birth year. Birth year comes from PatientBirthDate when present
        and is otherwise implied from StudyDate minus PatientAge, because this export fills one
        or the other. Names are upper-cased and stripped to letters, so punctuation and spacing
        variants collapse. Single-token names are NOT merged -- a lone given name is too common to treat
        as an identity, and a false merge is worse than a missed one: it would pull unrelated
        scans into one group and could push a whole finding out of a fold.

        Falls back to the PatientID when the name is unusable, so a row is never silently
        merged into a giant catch-all group.
        """
        nm = re.sub(r"[^A-Z]+", " ", str(name or "").upper()).strip()
        if len(nm.split()) < 2:
            return f"pid:{patient_id}"          # not enough to identify a person -- keep it alone
        byr = None
        bd = re.sub(r"[^0-9]", "", str(birth_date or ""))
        if len(bd) >= 4:
            y = int(bd[:4])
            if 1900 <= y <= 2100:
                byr = y
        if byr is None:
            sd = re.sub(r"[^0-9]", "", str(study_date or ""))
            ag = re.match(r"0*(\d+)", str(age or ""))
            if len(sd) >= 4 and ag:
                byr = int(sd[:4]) - int(ag.group(1))
        if byr is None:
            return f"pid:{patient_id}"
        return f"{nm}|{str(sex or '').upper()[:1]}|{byr}"


    def merge_name_variants(m, ratio=0.90, max_studies_per_person=12):
        """Merge person_keys that are the same human recorded under a spelling variant.

        make_person_key already collapses punctuation and spacing. What it cannot see is a name
        recorded with an initial or caste suffix on one visit and without it on the next --
        `<GIVEN NAME> F` and `<GIVEN NAME> Y F`, same sex, same birth year. In this dataset there are
        357 such pairs, and each unmerged pair is a person who can straddle the split.

        The asymmetry decides the design. A FALSE MERGE puts two unrelated people in one split
        group: the split is slightly more constrained, and nothing is inflated. A MISSED MERGE is
        leakage. So the threshold is set to merge, and it is only bounded to stop a cascade.

        Two things were tried and rejected against the real data, recorded here so they are not
        re-attempted:

          * Unioning on the matched REPORT identity. 481 report identities in this export are
            shared by different people (the report matcher is name+age+sex and does not enforce
            the date), so unioning on it chains person -> report -> person and collapses 1,120
            studies -- 18% of the cohort -- into a single group. Different people sharing a
            report is a MISLABEL, not leakage: the images differ, so there is nothing to leak.
          * Unioning on AccessionNumber. 75 accession numbers in this export span more than one
            person, so it over-merges for no gain; it is absent on 40% of studies anyway.

        The cap exists only as a backstop. Measured on the real table it never binds -- the
        largest group is 10 studies with the cap removed entirely -- but a future export with
        many same-name same-age people could cascade without it, and a runaway group would
        quietly starve a fold of positives.
        """
        import difflib

        if "person_key" not in m.columns or not len(m):
            return m
        st = m.drop_duplicates("study_uid")[["study_uid", "person_key", "patient_name",
                                              "patient_sex"]].copy()
        st["nm"] = st["patient_name"].map(
            lambda n: re.sub(r"[^A-Z]+", " ", str(n).upper()).strip())
        st["byr"] = st["person_key"].astype(str).str.rsplit("|", n=1).str[-1]
        n_studies = st.groupby("person_key").size().to_dict()

        parent, size = {}, {}

        def find(x):
            parent.setdefault(x, x)
            size.setdefault(x, n_studies.get(x, 1))
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        n_merged = n_refused = 0
        for (_sex, _byr), grp in st.groupby([st["patient_sex"].astype(str), st["byr"]]):
            if len(grp) < 2 or not str(_byr).isdigit():
                continue                       # a pid: fallback key carries no birth year
            uniq = grp[["nm", "person_key"]].drop_duplicates("person_key").to_numpy()
            if len(uniq) > 60:
                continue                       # pathological bucket; O(k^2) is not worth it
            for a in range(len(uniq)):
                for b in range(a + 1, len(uniq)):
                    ra, rb = find(uniq[a][1]), find(uniq[b][1])
                    if ra == rb:
                        continue
                    if difflib.SequenceMatcher(None, uniq[a][0], uniq[b][0]).ratio() < ratio:
                        continue
                    if size[ra] + size[rb] > max_studies_per_person:
                        n_refused += 1
                        continue
                    parent[ra] = rb
                    size[rb] = size[ra] + size[rb]
                    n_merged += 1

        if n_merged:
            m["person_key"] = [find(k) for k in m["person_key"]]
            print(f"  person identity: {n_merged} spelling variant(s) merged"
                  + (f", {n_refused} refused by the size cap" if n_refused else ""))
        _n_people = m.drop_duplicates("study_uid")["person_key"].nunique()
        _n_pid = m.drop_duplicates("study_uid")["patient_id"].nunique()
        print(f"  person identity: {_n_people:,} people across "
              f"{m['study_uid'].nunique():,} studies "
              f"({_n_pid - _n_people:,} visit-IDs merged into an existing person -- "
              f"PatientID here is a per-visit counter, not an MRN)")
        return m


    MANIFEST_COLUMNS = [
        "series_uid", "study_uid", "patient_id", "accession",
        "zip_month", "zip_day", "zip_key", "zip_path", "zip_exists",
        "region", "head_subtype", "series_desc", "body_part", "is_reformat",
        "n_slices_expected", "slice_thickness", "is_soft_kernel",
        "priority_rank", "is_primary", "studytype_agrees", "matched_study_type",
        "match_method", "study_date", "patient_sex", "patient_age",
        # `person_key` is what every split groups on. `patient_id` is kept beside it because it
        # is what the DICOM says, but it must not be used for grouping -- see person_key() below.
        "person_key", "patient_name", "patient_birth_date",
        # Kept so a site can be held out, or used as a covariate, without re-deriving it. These
        # were in the row dicts and NOT in this list, and pd.DataFrame(rows, columns=...) selects
        # rather than unions -- so they were silently dropped on construction for the whole
        # project, and the comment beside them described behaviour the code did not have.
        "hospital", "institution",
    ]


    def _manifest_key():
        with open(TABLE_CACHE_META) as f:
            table_key = json.load(f)
        return {
            "table": table_key,
            "ct_root": CT_ROOT,
            "target_region": TARGET_REGION,
            "min_slices": MIN_SLICES_PER_SERIES,
            "all_series": bool(PROCESS_ALL_QUALIFYING_SERIES),
            "allowed_head_subtypes": list(ALLOWED_HEAD_SUBTYPES),
            "restrict_head_subtypes": bool(RESTRICT_TO_ALLOWED_HEAD_SUBTYPES),
            "sharp_kernel_threshold": SHARP_KERNEL_THRESHOLD,
            # The selection heuristics in cell 8 are editable and DO change membership. Hash them in,
            # so tweaking a keyword list rebuilds the manifest instead of silently reusing the old one.
            "selection_rules_sha1": hashlib.sha1("|".join([
                repr(sorted(_JUNK_DESC_PATTERNS)),
                _JUNK_DESC_REGEX.pattern, _NONAXIAL_REGEX.pattern, _REFORMAT_REGEX.pattern,
                _KERNEL_RE.pattern, str(SHARP_KERNEL_THRESHOLD),
                repr({k: sorted(v) for k, v in _REGION_DESC_KEYWORDS.items()}),
                repr({k: sorted(v) for k, v in _HEAD_SUBTYPE_KEYWORDS.items()}),
            ]).encode()).hexdigest(),
            # The schema version used to be a hand-maintained integer, and the one time it
            # mattered it was not bumped: `person_key` and `hospital`/`institution` were added
            # to MANIFEST_COLUMNS, the key did not change, the cached manifest was reused, and
            # the new columns never appeared no matter how many times cell 9 ran. Cell 21 then
            # refused to start, correctly, and there was no way out of it from the notebook.
            #
            # Hashing the column list removes the human step: adding, removing or renaming a
            # manifest column changes the key by construction.
            "schema_columns_sha1": hashlib.sha1(
                "|".join(MANIFEST_COLUMNS).encode()).hexdigest()[:16],
            "schema_version": 6,
        }


    def build_manifest():
        key = _manifest_key()
        if os.path.exists(MANIFEST_PATH) and os.path.exists(MANIFEST_META):
            try:
                with open(MANIFEST_META) as f:
                    if json.load(f) == key:
                        m = pd.read_csv(MANIFEST_PATH, dtype={"zip_month": str, "zip_day": str})
                        print(f"loaded cached manifest: {len(m):,} rows from {MANIFEST_PATH}")
                        return refresh_zip_existence(m)
                print("manifest inputs changed -- rebuilding.")
            except Exception as e:
                print(f"manifest cache unreadable ({e}) -- rebuilding.")

        sel = select_series(df)
        print(f"candidate series after region/junk/slice filters: {len(sel):,} "
              f"across {sel['StudyInstanceUID'].nunique():,} studies")
        ranked = rank_within_study(sel)
        ranked["is_primary"] = ranked["priority_rank"] == 0
        if not PROCESS_ALL_QUALIFYING_SERIES:
            ranked = ranked[ranked["is_primary"]]
            print(f"PROCESS_ALL_QUALIFYING_SERIES=False -> one primary series per study: "
                  f"{len(ranked):,} series")
        else:
            print(f"PROCESS_ALL_QUALIFYING_SERIES=True -> every qualifying series: {len(ranked):,}")

        # Resolve each row's day-zip once per (month, day) -- 284 stat() calls, not 10,000.
        zip_cache = {}
        def resolve(month, day):
            k = (month, day)
            if k not in zip_cache:
                found = ""
                for cand in candidate_zip_paths(month, day):
                    if os.path.exists(cand):
                        found = cand
                        break
                zip_cache[k] = found or candidate_zip_paths(month, day)[0]
                zip_cache[(k, "exists")] = bool(found)
            return zip_cache[k], zip_cache[(k, "exists")]

        rows = []
        for r in ranked.itertuples(index=False):
            zpath, zexists = resolve(r.zip_month, r.zip_day)
            rows.append({
                "series_uid": r.SeriesInstanceUID,
                "study_uid": r.StudyInstanceUID,
                "patient_id": getattr(r, "PatientID", ""),
                "accession": getattr(r, "AccessionNumber", ""),
                "zip_month": r.zip_month,
                "zip_day": r.zip_day,
                "zip_key": f"{r.zip_month}/{r.zip_day}",
                "zip_path": zpath,
                "zip_exists": zexists,
                "region": r.region,
                "head_subtype": r.head_subtype,
                "series_desc": getattr(r, "SeriesDescription", ""),
                "body_part": getattr(r, "BodyPartExamined", ""),
                "is_reformat": bool(getattr(r, "is_reformat", False)),
                "n_slices_expected": int(r.NumSlices),
                "slice_thickness": getattr(r, "SliceThickness", ""),
                "is_soft_kernel": bool(r.is_soft_kernel),
                "priority_rank": int(r.priority_rank),
                "is_primary": bool(r.is_primary),
                "studytype_agrees": bool(r.studytype_agrees),
                "matched_study_type": getattr(r, "MatchedStudyType", ""),
                "match_method": getattr(r, "MatchMethod", ""),
                "study_date": getattr(r, "StudyDate", ""),
                "patient_sex": getattr(r, "PatientSex", ""),
                "patient_age": getattr(r, "PatientAge", ""),
                "patient_name": getattr(r, "PatientName", ""),
                "patient_birth_date": getattr(r, "PatientBirthDate", ""),
                "person_key": make_person_key(
                    getattr(r, "PatientName", ""), getattr(r, "PatientSex", ""),
                    getattr(r, "PatientBirthDate", ""), getattr(r, "StudyDate", ""),
                    getattr(r, "PatientAge", ""), getattr(r, "PatientID", "")),
                # Kept so a site can be held out, or used as a covariate, without re-deriving it.
                "hospital": getattr(r, "Hospital", ""),
                "institution": getattr(r, "InstitutionName", ""),
            })

        m = pd.DataFrame(rows, columns=MANIFEST_COLUMNS)
        m = merge_name_variants(m)
        m = m.sort_values(["zip_month", "zip_day", "study_uid", "priority_rank", "series_uid"]).reset_index(drop=True)
        _tmp = MANIFEST_PATH + ".tmp"
        m.to_csv(_tmp, index=False)
        os.replace(_tmp, MANIFEST_PATH)      # every other checkpoint is tmp+rename; so is this one
        _atomic_write_json(MANIFEST_META, key)
        print(f"manifest written: {MANIFEST_PATH}")
        return m


    manifest = build_manifest()
    print()
    print(f"manifest rows        : {len(manifest):,}")
    print(f"unique studies       : {manifest['study_uid'].nunique():,}")
    print(f"unique patients      : {manifest['patient_id'].nunique():,}")
    print(f"distinct day-zips    : {manifest['zip_key'].nunique()}")

In [ ]:
# 10. Manifest coverage report -- which day-zips are actually on Drive
#
# Worth looking at before starting a long run. When this notebook was written, the Drive tree held
# months 01, 03, 05, 06, 07, 08, 09, 10, 11 -- months 02 and 04 were absent, while the matched CSV
# covers 01-11. Series in a missing month are not failures; they are data that has not been
# uploaded, and the difference matters when you are reading a completion percentage later.

if DATA_SOURCE != "india":
    print("DATA_SOURCE is {!r} -- skipping this cell (Part A / India only).".format(DATA_SOURCE))
else:
    _zip_summary = (
        manifest.groupby(["zip_month", "zip_day"])
        .agg(series=("series_uid", "size"),
             studies=("study_uid", "nunique"),
             exists=("zip_exists", "first"),
             path=("zip_path", "first"))
        .reset_index()
        .sort_values(["zip_month", "zip_day"])
    )

    _present = _zip_summary[_zip_summary["exists"]]
    _absent = _zip_summary[~_zip_summary["exists"]]

    print(f"day-zips referenced by the manifest : {len(_zip_summary)}")
    print(f"  present on Drive                  : {len(_present)}  "
          f"({_present['series'].sum():,} series, {_present['studies'].sum():,} studies)")
    print(f"  NOT found                         : {len(_absent)}  "
          f"({_absent['series'].sum():,} series, {_absent['studies'].sum():,} studies)")

    if len(_absent):
        print()
        print("missing day-zips, grouped by month:")
        for month, grp in _absent.groupby("zip_month"):
            days = ", ".join(sorted(grp["zip_day"]))
            print(f"  month {month}: {len(grp)} day(s) -> {days}")
        print()
        print("  If a whole month is missing it has probably just not been uploaded yet -- upload it")
        print("  and re-run cell 9 (the manifest cache key includes CT_ROOT, so it re-resolves).")
        print("  If only scattered days are missing, check the exact filenames: cell 9 tries")
        print(f"  {os.path.join(CT_ROOT, 'MM', 'DD.zip')} with and without zero padding, and .ZIP.")

    print()
    print("per-month coverage:")
    _by_month = (
        _zip_summary.groupby("zip_month")
        .agg(days=("zip_day", "size"), days_present=("exists", "sum"),
             series=("series", "sum"), studies=("studies", "sum"))
        .reset_index()
    )
    print(_by_month.to_string(index=False))

    # Only rows whose zip is actually there can be worked on this session.
    manifest_runnable = manifest[manifest["zip_exists"]].reset_index(drop=True)
    print()
    print(f"runnable this session: {len(manifest_runnable):,} series across "
          f"{manifest_runnable['zip_key'].nunique()} day-zip(s)")
    if len(manifest_runnable) == 0:
        print()
        print("*** Nothing to run. No manifest row resolved to an existing zip. ***")
        print(f"    Check CT_ROOT: {CT_ROOT}")
        try:
            print(f"    It contains: {sorted(os.listdir(CT_ROOT))[:20]}")
        except Exception as e:
            print(f"    ...and it could not even be listed: {e}")

---

# Part B — RSNA 2019 Intracranial Hemorrhage Detection

Set `DATA_SOURCE = "rsna"` in the config cell and the cells below build the same manifest schema
the India path builds, so everything downstream — preprocessing, Pillar-0, the feature table, QC,
the classifier sweep — is **literally the same code**. That is the point: a difference in results
between the two datasets is then a difference in the data, not a difference in the pipeline.

## Why this is a genuinely useful second dataset

RSNA-ICH labels the same five hemorrhage compartments the India reports do:

| RSNA subtype | India report column |
|---|---|
| `subdural` | `b_sdh` |
| `epidural` | `b_edh` |
| `subarachnoid` | `b_sah` |
| `intraventricular` | `b_ivh` |
| `intraparenchymal` | `b_ich` |
| `any` | derived `any_intracranial_hemorrhage` cohort |

So it is a real external check on the head-CT half of this project — ~21.7k exams of expert-labelled
head CT against ~3.2k report-matched ones.

## Three things about this dataset that change the code

**1. It ships as loose 2D slices in one flat folder, with no directory hierarchy.** Filenames are
`ID_<9 hex>.dcm` and equal the `SOPInstanceUID`. Nothing in the path says which study or series a
slice belongs to — so unlike the Indian-hospital day-zips, where `SourceFolder` hands you the answer, the
grouping has to be read out of ~874k DICOM headers. Cell B3 does that once and caches it.

**2. The labels are PER SLICE, not per study.** `stage_2_train.csv` is long-format: six rows per
image (`ID_<hex>_<subtype>`), `Label` ∈ {0,1}. Pillar-0 consumes a volume, so the labels have to be
aggregated — a series is positive for a subtype if **any** of its slices is. That is the aggregation
the labelling protocol implies (annotators marked the slices on which a bleed was visible), and it
is what the RSNA dataset paper's own exam-level labels correspond to. It also makes the task
strictly easier than the competition's per-slice task, so **competition log-loss numbers are not
comparable to anything this notebook prints.**

**3. `stage_2_train` = `stage_1_train_images` ∪ `stage_1_test_images`.** 674,258 + 78,545 =
752,803, and 752,803 + 121,232 (`stage_2_test`) = 874,035, which is exactly the image count in the
RSNA dataset paper. Only `stage_2_train` has labels, so that is what this notebook uses.

## The 12-bit pixel bug — non-optional

**333,443 of the 752,803 training files (44.3%) store 12-bit unsigned pixel data**
(`BitsStored == 12`, `PixelRepresentation == 0`) — measured on the archive, not quoted from a
kernel. Some files carry a `RescaleIntercept` near 0 instead of ≈ −1024, and *those* have wrapped
modulo 4096: air comes back at about **+3096 HU instead of −1000**. Every window in the
11-channel stack is then computed on nonsense, and nothing errors — you just get confident
embeddings of garbage.

**How many actually need the fix: 123 slices.** Not "a large subset of the 333k", which is what
this section used to claim. The 12-bit unsigned population is enormous and almost all of it
already carries a sane intercept. Cell B2 prints both numbers so this stays a measurement.

**And the two published fix conditions turn out to agree exactly.** Jeremy Howard's requires all
three tags; shimacos37's 10th-place solution drops the `BitsStored` clause and triggers on
`PixelRepresentation == 0 and RescaleIntercept >= −100`. On this archive both select **the same
123 slices**, because every unsigned slice with a high intercept is also 12-bit. There is no
ambiguity to resolve.

One loose end worth knowing: **5,386 slices (0.7%) have `RescaleIntercept > −100` but are
`PixelRepresentation == 1`** — signed data, where the modulo-4096 wrap does not apply and no
published solution corrects them. If any of them are nevertheless wrong, the tag cannot say so;
the pixels can. Cell 13 records `hu_p01` for every series and flags IMPLAUSIBLE HU RANGE when air
does not come back near −1000, which is the check to trust.

The correction (originating in Jeremy Howard's `cleaning-the-data-for-rapid-prototyping-fastai`
kernel, and copied verbatim into several medal-winning solutions) is:

```python
if BitsStored == 12 and PixelRepresentation == 0 and int(RescaleIntercept) > -100:
    x = pixel_array + 1000
    x[x >= 4096] -= 4096
    intercept = -1000
```

Cell B2 applies exactly this. **Note that `fastai.medical.imaging` does not do it for you** — its
`scaled_px` is a naive `px * slope + intercept`; the fix has always been kernel-side.

`ID_6431af929.dcm` has corrupt `PixelData` and is excluded by name — that one file is confirmed
independently by several solution repos.

## The archive is split into 2 GB parts, and that is fine

`rsna-ich/parts/part_00000 … part_000NN`, 2 GiB each, from a `split -b 2G`. Reassembling them needs
~180 GB (zip) or ~431 GB (tar) of scratch, and Colab has ~80–110 GB. So they are never reassembled:
cell B1 exposes the parts as **one seekable file-like object**, and `zipfile` / a tar offset walk
read members out of it by offset. Same principle as the day-zip path — random access, nothing
materialised.

## What a correct run should score — a pre-registered target, not a hope

This matters more than any of the plumbing above, because it is the only way to tell a working
pipeline from a broken one that still prints a number.

Pillar-0's own paper ([arXiv 2511.17803](https://arxiv.org/abs/2511.17803)) runs **exactly this
setup** on **exactly this dataset**: frozen encoder, one linear classifier per task over the
embeddings, study level, 21,744 RSNA exams split patient-wise into 15,166 / 3,261 / 3,317. Its
reported AUROC for **any ICH**, by training-data fraction:

| 2.5% (~380 exams) | 5% | 10% | 25% | 50% | 100% (15,166) |
|---|---|---|---|---|---|
| 95.0 | 95.5 | 95.9 | 96.5 | 96.8 | **96.7** |

Baselines under the same frozen-linear-probe protocol at 100% data: Swin3D-t 95.9, Merlin 95.6,
3D ResNet-18 (Kinetics-400) 95.1, RadImageNet ResNet-50 92.1, MedicalNet 90.4.

**So: with `RSNA_MAX_SERIES = 4000` this notebook should land around 0.95–0.96 AUROC on
`rsna_any`.** The curve is nearly flat past 10% of the data, which is why 4,000 series is a
sensible sample rather than a compromise — going to all 21.7k buys perhaps a point.

If the run comes back materially below 0.95, **that is a pipeline defect, not a model limit**, and
the things to suspect, in order: the 12-bit intercept fix not firing (check the count cell B2
prints and the `hu_p01` values in the progress log — air must be near −1000); slice ordering
(check `order_source` is "slice normal" for essentially all series); the in-plane crop
(`PILLAR0_INPLANE_MODE`, and the head-retention figure cell 17 reports); and grouping
(`person_key` must be `PatientID`, which for RSNA is a real per-person hash — unlike the India
export, where it is a per-visit counter).

For orientation on the other end: the competition's best published private-LB score is 0.04345
weighted log-loss ([Swin-B + sequential transformer, arXiv 2205.07556](https://arxiv.org/abs/2205.07556)),
just ahead of SeuTao's winning 0.04383. Those are **per-slice** numbers from fully fine-tuned CNN
ensembles with slice-sequence models on top, and they are not comparable to a frozen linear probe
on study-level labels. The number to compare against is 96.7.

## What the published solutions do about the archive, and why this notebook now does the same

Every top solution digests the DICOMs **once**, up front, and never reads the originals again:
SeuTao's `2DNet/src/prepare_data.py` writes 8-bit PNGs; darraghdog's `scripts/prepare_meta_dicom.py`
writes 3-channel JPEGs straight out of the zip; Anjum48's `data_prep.py` takes *12–15 hours per
image set* and he says plainly it exists because "processing DICOM images on the fly was too slow
to keep 2 GPUs busy"; BrainScan writes `int16` HU `.npy` and measures it as **>3× faster** than
DICOM. The two teams that did read DICOMs every epoch (appian42 11th, shimacos37 10th) were reading
them off **local disk**, not off a network mount.

Cells B1b/B2/B5 are that same architecture, adapted to the constraint that the archive lives on
Drive and cannot be unzipped: one sequential forward pass to build the header index, one sequential
pass to pull the wanted slices onto local disk, and everything after that reads local files.

In [ ]:
# B1. Read a split archive without ever reassembling it  [RSNA]
#
# `split -b 2G` leaves part_00000 … part_000NN. Concatenating them back would need ~180 GB (zip) or
# ~431 GB (tar) of scratch; Colab has ~80-110 GB. So instead: present the parts as ONE seekable
# byte stream and let zipfile / the tar walker do random access over it. Nothing is materialised,
# and only the bytes actually wanted ever cross the network.

if DATA_SOURCE != "rsna":
    print("DATA_SOURCE is {!r} -- skipping this cell (Part B / RSNA only).".format(DATA_SOURCE))
else:
    import io
    import json
    import os
    import re
    import time
    import struct


    # Counters, so a run can SAY how much the mount misbehaved instead of the operator guessing
    # from how long it took.
    MPF_STATS = {"errors": 0, "transient": 0, "dead_mount": 0}


    class MultiPartFile(io.RawIOBase):
        """A read-only, seekable file over an ordered list of part files.

        Deliberately raw (unbuffered) by default. Wrapping this in a big BufferedReader is exactly the
        wrong move for archive indexing: a tar header is 512 bytes every ~500 KB, so a 32 MB read-ahead
        would stream the entire 431 GB to collect a few dozen headers. Ranged reads are what we want.
        """

        def __init__(self, paths):
            super().__init__()
            if not paths:
                raise ValueError("no parts given")
            self._paths = list(paths)
            self._sizes = [os.path.getsize(p) for p in self._paths]
            self._starts, acc = [], 0
            for sz in self._sizes:
                self._starts.append(acc)
                acc += sz
            self._total = acc
            self._pos = 0
            self._fh = None
            self._fh_idx = -1

        # -- introspection ---------------------------------------------------------------------
        @property
        def size(self):
            return self._total

        def readable(self):
            return True

        def seekable(self):
            return True

        def writable(self):
            return False

        # -- positioning -----------------------------------------------------------------------
        def tell(self):
            return self._pos

        def seek(self, offset, whence=io.SEEK_SET):
            if whence == io.SEEK_SET:
                new = offset
            elif whence == io.SEEK_CUR:
                new = self._pos + offset
            elif whence == io.SEEK_END:
                new = self._total + offset
            else:
                raise ValueError(f"bad whence {whence}")
            if new < 0:
                raise OSError("negative seek")
            self._pos = new           # seeking past the end is legal; the read there returns b""
            return self._pos

        def _part_for(self, pos):
            # Parts are equal-sized except possibly the last, so a direct index is right almost
            # always; the loop is the fallback for an uneven split.
            lo, hi = 0, len(self._paths) - 1
            while lo < hi:
                mid = (lo + hi + 1) // 2
                if self._starts[mid] <= pos:
                    lo = mid
                else:
                    hi = mid - 1
            return lo

        def _ensure(self, idx):
            if self._fh_idx != idx:
                if self._fh is not None:
                    try:
                        self._fh.close()
                    except Exception:
                        pass
                self._fh = open(self._paths[idx], "rb")
                self._fh_idx = idx
            return self._fh

        def reopen(self):
            """Drop the cached handle so the next read opens a fresh one.

            After drive.mount(force_remount=True) every descriptor opened against the old mount
            is invalid, and reads through it keep failing however many times they are retried.
            The handle has to be thrown away, not retried.
            """
            if self._fh is not None:
                try:
                    self._fh.close()
                except Exception:
                    pass
            self._fh = None
            self._fh_idx = -1

        # -- reading ---------------------------------------------------------------------------
        # THE ONE PLACE EVERY RSNA BYTE COMES THROUGH.
        #
        # So it is also the only sensible place to handle the two ways a Drive mount fails under a
        # multi-hour read:
        #
        #   errno 5   (EIO)    -- usually transient. Retry the same read.
        #   errno 107 (ENOTCONN, "Transport endpoint is not connected") -- the DriveFS helper
        #                        process is GONE. Every open descriptor is invalid, and retrying
        #                        is guaranteed to fail forever. The mount must be re-established
        #                        and the handle reopened. colabtools#5970 is a session that spent
        #                        seven hours retrying this and achieved nothing.
        #
        # A run that cannot tell those apart either gives up on a blip or spins on a corpse.
        # The ladder, in the shape the two failures actually have:
        #   a TRANSIENT error (errno 5) -- reopen the handle and try again, with backoff.
        #   a DEAD MOUNT (errno 107)    -- remount, then start the transient budget over,
        #                                  because a successful remount is progress and should
        #                                  not be charged against the retries for a blip.
        # Bounded on both axes, so a permanently dead mount raises in seconds rather than
        # spinning for the seven hours colabtools#5970 describes.
        MPF_TRANSIENT_ATTEMPTS = 4
        MPF_REMOUNT_ATTEMPTS = 2
        MPF_BACKOFF = (0.5, 2.0, 5.0, 12.0)

        def _read_chunk(self, fh, idx, local, n):
            """One physical read, with the mount-failure ladder around it."""
            last = None
            remounts_left = self.MPF_REMOUNT_ATTEMPTS
            attempt = 0
            while True:
                try:
                    if fh.tell() != local:
                        fh.seek(local)
                    return fh.read(n), fh
                except OSError as e:
                    last = e
                    MPF_STATS["errors"] += 1
                    dead = getattr(e, "errno", None) in (103, 107)
                    if dead:
                        if remounts_left <= 0:
                            break
                        remounts_left -= 1
                        attempt = 0                     # a remount resets the transient budget
                        MPF_STATS["dead_mount"] += 1
                        _rm = globals().get("remount_drive")
                        if _rm is not None:
                            _rm(verbose=True)
                        self.reopen()
                        time.sleep(1.0)
                    else:
                        if attempt >= self.MPF_TRANSIENT_ATTEMPTS - 1:
                            break
                        MPF_STATS["transient"] += 1
                        self.reopen()
                        time.sleep(self.MPF_BACKOFF[min(attempt, len(self.MPF_BACKOFF) - 1)])
                        attempt += 1
                    fh = self._ensure(idx)
            raise last

        def readinto(self, b):
            want = len(b)
            if want == 0 or self._pos >= self._total:
                return 0
            got = 0
            mv = memoryview(b)
            while got < want and self._pos < self._total:
                idx = self._part_for(self._pos)
                fh = self._ensure(idx)
                local = self._pos - self._starts[idx]
                chunk, fh = self._read_chunk(fh, idx, local,
                                             min(want - got, self._sizes[idx] - local))
                if not chunk:
                    break                       # short/truncated part -- stop rather than spin
                mv[got:got + len(chunk)] = chunk
                got += len(chunk)
                self._pos += len(chunk)
            return got

        def read(self, n=-1):
            if n is None or n < 0:
                n = max(0, self._total - self._pos)
            buf = bytearray(n)
            got = self.readinto(buf)
            return bytes(buf[:got])

        # Context-manager protocol, so a MultiPartFile can stand in for open(path, "rb")
        # wherever the code expects a file. The offset repair (cell 11b) probes the archive
        # through `with reader() as fp:` -- without these it could not be applied to the RSNA
        # archive at all, which is exactly why a 194.7 GB zip whose offsets wrap the 32-bit
        # field 45 times was being read unrepaired.
        def __enter__(self):
            return self

        def __exit__(self, *exc):
            return False

        def close(self):
            if self._fh is not None:
                try:
                    self._fh.close()
                finally:
                    self._fh = None
                    self._fh_idx = -1
            super().close()


    def discover_parts(parts_dir, pattern=r"^part_\d+$"):
        rx = re.compile(pattern)
        names = sorted(n for n in os.listdir(parts_dir) if rx.match(n))
        if not names:
            raise FileNotFoundError(
                f"no files matching {pattern!r} in {parts_dir}. Set RSNA_PARTS_PATTERN if your split "
                f"used a different naming scheme (e.g. 'xa*' from a bare `split`)."
            )
        paths = [os.path.join(parts_dir, n) for n in names]
        # Retry EIO. The Drive FUSE mount returns errno 5 for rate limits, token refreshes and
        # transient backend errors -- nothing to do with the file, and usually gone a second
        # later. discover_parts runs at the start of B1b and B2 as well as the preflight, so a
        # single blip here used to abort a whole night's run at its very last check.
        def _size(path, attempts=4):
            last = None
            for k in range(attempts):
                try:
                    return os.path.getsize(path)
                except OSError as e:
                    last = e
                    if getattr(e, "errno", None) != 5:
                        raise
                    if k < attempts - 1:
                        time.sleep(1.0 * (k + 1))
            raise OSError(
                f"{path} could not be stat'd after {attempts} attempts ({last}). Every other "
                f"part read fine, so the mount is alive -- this file is most likely still "
                f"uploading, or is a Drive shortcut rather than a real file.")

        sizes = [_size(p) for p in paths]
        # `split` produces equal-sized parts except the last. A short part in the MIDDLE means an
        # upload is still in flight or one part is truncated -- reading past it would silently produce
        # a corrupt archive, so refuse rather than guess.
        if len(sizes) > 1:
            head = sizes[0]
            bad = [(names[i], sizes[i]) for i in range(len(sizes) - 1) if sizes[i] != head]
            if bad:
                raise RuntimeError(
                    f"these part(s) are not the same size as part 0 ({head:,} bytes) but are not "
                    f"last: {bad[:5]}. Either an upload is still running or a part is truncated. "
                    f"Wait for the upload to finish, or re-upload those parts, before continuing -- "
                    f"reading past a short part would corrupt every member after it."
                )
        return paths, sizes


    def detect_archive_format(mpf):
        """Read the first 512 bytes and say what this is. Cheap, and it decides everything after."""
        mpf.seek(0)
        head = mpf.read(512)
        if head[:4] in (b"PK\x03\x04", b"PK\x05\x06", b"PK\x06\x06"):
            return "zip"
        if len(head) >= 262 and head[257:262] == b"ustar":
            return "tar"
        if head[:2] == b"\x1f\x8b":
            return "gzip"
        if head[:6] == b"\xfd7zXZ\x00":
            return "xz"
        if head[:3] == b"BZh":
            return "bzip2"
        if head[:6] == b"7z\xbc\xaf\x27\x1c":
            return "7z"
        if head[:4] == b"Rar!":
            return "rar"
        return "unknown"


    # ---- tar: a resumable header walk ------------------------------------------------------------
    # Written by hand rather than handed to `tarfile`, for two reasons. It gives exact control over
    # where the walk is (so a Colab session that dies at member 600,000 resumes from there instead of
    # restarting a multi-hour scan), and it reads exactly one 512-byte block per member and SEEKS over
    # the data -- no read-ahead, which is the difference between transferring ~450 MB of headers and
    # streaming the whole 431 GB.
    def _tar_octal(field):
        """Numeric tar header field.

        GNU tar switches to a base-256 encoding (high bit of the first byte set) for values that do
        not fit the octal field -- sizes over 8 GB, large uids. `int(s, 8)` raises on those, and
        the caller used to swallow that with a bare `except: break` and then mark the whole index
        `complete`, i.e. one unusual header silently truncated the archive index forever. Both
        encodings are handled here so that path cannot be reached by accident."""
        if field and field[0] & 0x80:
            v = field[0] & 0x7F
            for b in field[1:]:
                v = (v << 8) | b
            return v
        s = field.split(b"\x00")[0].split(b" ")[0].strip()
        if not s:
            return 0
        return int(s, 8)


    def walk_tar_members(mpf, start_offset=0, on_progress=None, progress_every=20000,
                          stop_after=None):
        """Yields (name, data_offset, size, next_offset) for each regular file member."""
        offset = start_offset
        total = mpf.size
        seen = 0
        pending_long_name = None
        while offset + 512 <= total:
            mpf.seek(offset)
            hdr = mpf.read(512)
            if len(hdr) < 512:
                break
            if hdr == b"\x00" * 512:
                # Two consecutive zero blocks mark end-of-archive.
                mpf.seek(offset + 512)
                if mpf.read(512) == b"\x00" * 512:
                    break
                offset += 512
                continue
            name = hdr[0:100].split(b"\x00")[0].decode("utf-8", "replace")
            try:
                size = _tar_octal(hdr[124:136])
            except Exception as e:
                # A header we cannot parse is NOT end-of-archive. Say so loudly: the caller marks
                # the index complete when this generator finishes, and a silent break here would
                # freeze a truncated index in place permanently.
                raise RuntimeError(
                    f"unparseable tar header at byte {offset:,} (size field "
                    f"{hdr[124:136]!r}): {type(e).__name__}: {e}. The index is NOT complete; do "
                    f"not treat it as such."
                ) from e
            typeflag = hdr[156:157]
            prefix = hdr[345:500].split(b"\x00")[0].decode("utf-8", "replace")
            if prefix:
                name = f"{prefix}/{name}"
            data_offset = offset + 512
            next_offset = data_offset + ((size + 511) // 512) * 512

            if typeflag in (b"L", b"K"):          # GNU long name / long link
                mpf.seek(data_offset)
                pending_long_name = mpf.read(size).split(b"\x00")[0].decode("utf-8", "replace")
                offset = next_offset
                continue
            if typeflag in (b"x", b"g"):
                # pax extended header. It is not enough to skip it: for a path longer than 100
                # characters, the REAL name lives in this record as "path=..." and the following
                # ustar header carries a truncated one. Skipping it leaves the member indexed under
                # a name that does not exist, so it can never be opened.
                mpf.seek(data_offset)
                blob = mpf.read(size)
                try:
                    pos = 0
                    while pos < len(blob):
                        sp = blob.index(b" ", pos)
                        reclen = int(blob[pos:sp])
                        rec = blob[pos:pos + reclen]
                        kv = rec[sp - pos + 1:].rstrip(b"\n")
                        if kv.startswith(b"path="):
                            pending_long_name = kv[5:].decode("utf-8", "replace")
                        pos += reclen
                except Exception:
                    pass                          # malformed pax record: fall back to the ustar name
                offset = next_offset
                continue
            if pending_long_name is not None:
                name = pending_long_name
                pending_long_name = None
            if typeflag in (b"0", b"\x00", b" "):
                yield name, data_offset, size, next_offset
                seen += 1
                if on_progress and seen % progress_every == 0:
                    on_progress(seen, offset, total)
                if stop_after is not None and seen >= stop_after:
                    return
            offset = next_offset


    class TarContainer:
        """Duck-types the little bit of ZipFile the rest of this notebook uses: `.open(name)` and
        `.namelist()`. Backed by an offset index, so member reads are direct seeks."""

        def __init__(self, mpf, index):
            self._mpf = mpf
            self._index = index          # name -> [data_offset, size]

        def namelist(self):
            return list(self._index)

        def open(self, name, max_bytes=None):
            """A file-like view of one member. `max_bytes` reads only a prefix.

            Without max_bytes this read the ENTIRE member and handed back a BytesIO. The header
            scan in cell B2 then did fh.read(RSNA_HEADER_BYTES) on it -- 16 KB out of an
            already-transferred ~500 KB slice. So the pass whose whole design is "read 8-16 KB
            per member, never touch pixel data" moved the complete 194 GB archive across the
            Drive mount instead of about 12 GB. Sixteen times the intended traffic, no error, no
            warning: the progress bar simply advanced at a sixteenth of the expected rate and
            the run did not finish inside a Colab session.

            The zip path never had this problem, because ZipExtFile reads lazily -- which is
            exactly why it went unnoticed.
            """
            off, size = self._index[name]
            n = size if max_bytes is None else min(size, int(max_bytes))
            self._mpf.seek(off)
            return io.BytesIO(self._mpf.read(n))

        def close(self):
            pass


    class ZipMemberOpener:
        """Same duck-type over a ZipFile, so the caller never branches on archive format."""

        def __init__(self, zf):
            self._zf = zf

        def namelist(self):
            return self._zf.namelist()

        def open(self, name, max_bytes=None):
            return self._zf.open(name)          # max_bytes is meaningless here: ZipExtFile
                                                #   already reads lazily, so a prefix read
                                                #   transfers only the prefix.

        def close(self):
            self._zf.close()


    RSNA_PARTS_DIR = os.path.join(RSNA_ROOT, "parts")

    print("multi-part archive reader ready "
          "(MultiPartFile / detect_archive_format / walk_tar_members / TarContainer)")
    print(f"parts directory: {RSNA_PARTS_DIR}")

In [ ]:
# B1b. Read the archive FORWARD, never through the central directory  [RSNA]
#
# ===============================================================================================
# WHY THIS EXISTS -- the arithmetic that should have been done first
# ===============================================================================================
# Cell B1b measured the cost of reading one member of the RSNA archive by seeking to the offset
# the central directory gives:
#
#     50/364 checked, 471s   ->  9.5 seconds per member
#
# There are 752,803 members in stage_2_train. At 9.5 s each that is 82 DAYS. Repairing the
# offsets and retrying the I/O errors were both real fixes and neither touched this: random
# access into a 194 GB zip spread across 91 files on a Drive FUSE mount is not slow because of a
# bug, it is slow because of the access pattern. The broken offsets and the EIO storms are
# symptoms of the same thing.
#
# Every published solution to this dataset does the opposite. SeuTao's 1st-place pipeline
# converts DICOM to PNG in one pass; appian42's create_dataset.py opens a pickle because the
# DICOMs were digested upstream; Anjum48 and shimacos37 do the same. Nobody reads the originals
# twice, and nobody reads them off network storage.
#
# ===============================================================================================
# THE OBSERVATION THAT MAKES IT CHEAP
# ===============================================================================================
# A zip does not have to be read through its central directory. Every member is preceded by a
# LOCAL FILE HEADER on the wire:
#
#     PK\x03\x04 | version | flags | method | time | date | crc32 | csize | usize | nlen | elen
#                | filename (nlen bytes) | extra (elen) | ...data (csize bytes)...
#
# so the archive can be walked FORWARD from byte 0: read a header, take the name and the
# compressed size, consume what you need of the data, skip the rest, land exactly on the next
# header. The central directory is never consulted, which means the 32-bit offset overflow --
# 194,671,162,157 bytes is 45.3 wraps of that field -- stops mattering entirely. There is
# nothing to repair if nothing reads the broken numbers.
#
# And it is cheap in the way that counts. A DICOM header is the first ~16 KB of a member's
# UNCOMPRESSED stream. So the pass reads about 8-16 KB per member and SEEKS OVER the other
# ~200 KB:
#
#     752,803 members x ~12 KB  =  ~9 GB actually read, forward, in one pass
#
# against 194 GB of random seeking. Sequential-with-skips is the access pattern Drive is built
# for, and it is also far more resistant to EIO, because there is no seek storm to trigger it.
#
# The central directory is still useful for what it gets RIGHT -- names, sizes and CRC32s are
# all correct in this archive; only the offsets wrapped. So it is used as a fallback for
# data-descriptor members and as a cross-check, and the WALK is what supplies position.
#
# ===============================================================================================
# THE THREE THINGS THAT MAKE A FORWARD WALK GO WRONG, AND WHAT IS DONE ABOUT EACH
# ===============================================================================================
# 1. DATA DESCRIPTORS. If general-purpose bit 3 is set, the local header's csize/usize are ZERO
#    and the true values follow the data in a 12/16/20/24-byte trailer. Without the size there
#    is nothing to skip by. Handled two ways: the central directory's (correct) sizes are used
#    when available, and the trailer itself is stepped over by _resync below.
# 2. ZIP64. A 194 GB archive necessarily uses the 0x0001 extra field for sizes that exceed
#    0xFFFFFFFF. Parsed.
# 3. LANDING WRONG. Any of the above being slightly off desynchronises the whole rest of the
#    pass, and a desynchronised walk does not error -- it silently invents members. So every
#    step VERIFIES that it landed on a signature, and if it did not, _resync tries the known
#    trailer lengths and then a BOUNDED byte-scan (64 KB, not 194 GB) for the next local header.
#    If none of that works the walk STOPS and says where. It never guesses past a desync.
# ===============================================================================================
if DATA_SOURCE != "rsna":
    print("DATA_SOURCE is {!r} -- skipping this cell (Part B / RSNA only).".format(DATA_SOURCE))
else:
    import struct
    import time
    import zlib

    LFH = b"PK\x03\x04"               # local file header
    CDH = b"PK\x01\x02"               # central directory header  (= end of the member area)
    DDS = b"PK\x07\x08"               # optional data-descriptor signature
    EOC = b"PK\x05\x06"               # end of central directory
    E64 = b"PK\x06\x06"               # zip64 end of central directory
    _LFH_FMT = "<HHHHHIIIHH"          # everything after the 4-byte signature
    ZIP64_EXTRA_ID = 0x0001
    _STOP_SIGS = (CDH, EOC, E64)

    # A local header is 30 bytes plus name plus extra. RSNA names are ~70 chars; 4 KB of slack
    # covers any real extra field, and _parse_local_header asks for more if it is ever wrong.
    LFH_SLACK = 4096
    # Two different searches, two different windows, and confusing them is a real bug.
    #
    # RESYNC_WINDOW is for recovering from a DESYNC, where the walk is a few bytes off a record
    # boundary -- a data-descriptor trailer, some padding. 64 KB is generous for that and small
    # enough that a genuinely corrupt archive costs 64 KB, not the archive.
    #
    # ENTRY_WINDOW is for entering the archive at an ARBITRARY byte, which is what a sampled
    # verification does. A random byte lands inside a member's data essentially always, and an
    # RSNA member is ~250 KB, so the next local header is a quarter of a megabyte away on
    # average and half a megabyte away at worst. Searching 64 KB for it finds nothing about
    # three times in four -- so the first version of the sampled verifier would have reported
    # roughly 30 of its 40 probes as failures on a perfectly healthy archive.
    RESYNC_WINDOW = 65536
    ENTRY_WINDOW = 8 << 20            # comfortably past any plausible single-member size

    def _parse_local_header(buf, base=0):
        """(name, header_len, csize, usize, method, flags, crc32) from a local file header.

        `header_len` is the distance from the start of the header to the start of the data, so a
        caller at absolute position `pos` finds the data at `pos + header_len`.

        Returns None if `buf` does not start with a local header -- which is how the walk knows
        it has reached the central directory -- or the string "SHORT" if the buffer ended inside
        the name/extra and the caller must read more.
        """
        if len(buf) - base < 30:
            return "SHORT" if buf[base:base + 4] == LFH or len(buf) - base < 4 else None
        if buf[base:base + 4] != LFH:
            return None
        (_ver, flags, method, _t, _d, crc,
         csize, usize, nlen, elen) = struct.unpack_from(_LFH_FMT, buf, base + 4)
        need = 30 + nlen + elen
        if len(buf) - base < need:
            return "SHORT"
        name = buf[base + 30: base + 30 + nlen]
        # ZIP64: the 32-bit size fields hold 0xFFFFFFFF and the real values live in an extra
        # field. This archive is 194 GB, so ZIP64 records are expected somewhere in it.
        if csize == 0xFFFFFFFF or usize == 0xFFFFFFFF:
            ex = buf[base + 30 + nlen: base + 30 + nlen + elen]
            i = 0
            while i + 4 <= len(ex):
                hid, hsz = struct.unpack_from("<HH", ex, i)
                if hid == ZIP64_EXTRA_ID:
                    vals, j = [], i + 4
                    while j + 8 <= min(i + 4 + hsz, len(ex)):
                        vals.append(struct.unpack_from("<Q", ex, j)[0])
                        j += 8
                    # Order in the extra field is uncompressed size then compressed size, and
                    # only the fields that were 0xFFFFFFFF are present.
                    if usize == 0xFFFFFFFF and vals:
                        usize = vals.pop(0)
                    if csize == 0xFFFFFFFF and vals:
                        csize = vals.pop(0)
                    break
                i += 4 + hsz
        try:
            nm = name.decode("utf-8" if flags & 0x800 else "cp437")
        except Exception:
            nm = name.decode("latin-1")
        return (nm, need, csize, usize, method, flags, crc)

    class ForwardWindow:
        """A read-through buffer that walks the archive front-to-back and never seeks backwards.

        WHY A BUFFER AND NOT PLAIN SEEKS
        --------------------------------
        The naive forward walk issues three reads per member -- the local header, the data
        prefix, and a peek at the next record -- which on 752,803 members is 2.3 MILLION
        round trips to a Drive FUSE mount. Latency, not bandwidth, is what this archive
        charges for: cell B1b measured NINE AND A HALF SECONDS to read one member through a
        seek, and a large part of that is MultiPartFile closing one part file and opening
        another every time the position jumps.

        Write down the two costs. With per-read latency L, bandwidth B, N = 752,803 members,
        S = 194.7 GB, and ~28 KB actually wanted per member:

            skip-and-seek :  N.L + 21.6 GB / B
            stream it all :  (S / block).L + S / B

        They cross at L = 3.9 ms for B = 60 MB/s. Drive FUSE is nowhere near that fast per
        request, so STREAMING WINS -- by a lot, and by more the worse the mount behaves. It is
        also the access pattern that keeps one file handle open and lets the FUSE layer read
        ahead, which is precisely what the 9.5 s/member measurement was missing.

        So: read forward in large blocks, serve every request out of the buffer, and skip over a
        member's pixel data by CONSUMING it rather than seeking past it. A real seek happens
        only for a resume (`start=`) or a skip so large that reading through it would cost more
        than the round trip -- `seek_threshold`, which is where the arithmetic above flips.
        """

        def __init__(self, mpf, block=8 << 20, seek_threshold=32 << 20):
            self._mpf = mpf
            self._block = int(block)
            self._seek_threshold = int(seek_threshold)
            self._base = 0
            self._buf = b""
            self.n_reads = 0
            self.bytes_read = 0
            self.n_seeks = 0
            self.size = mpf.size
            self._pos = 0

        def _fill_to(self, end):
            """Extend the buffer forward until it covers `end`, reading contiguously."""
            while self._base + len(self._buf) < end:
                want = max(self._block, end - (self._base + len(self._buf)))
                self._mpf.seek(self._base + len(self._buf))
                chunk = self._mpf.read(want)
                self.n_reads += 1
                self.bytes_read += len(chunk)
                if not chunk:
                    return False
                self._buf += chunk
                # Keep the buffer bounded: everything before the current window start has been
                # consumed and will never be asked for again (the walk only moves forward).
                if len(self._buf) > 4 * self._block:
                    drop = len(self._buf) - 2 * self._block
                    self._base += drop
                    self._buf = self._buf[drop:]
            return True

        def read_at(self, pos, n):
            pos = int(pos)
            if pos < self._base or pos - (self._base + len(self._buf)) > self._seek_threshold:
                # Backwards, or so far ahead that reading through it is the more expensive
                # option. Restart the window there.
                self._base = pos
                self._buf = b""
                self.n_seeks += 1
            if pos < self._base:
                self._base = pos
                self._buf = b""
            self._fill_to(min(pos + n, self.size))
            off = pos - self._base
            return self._buf[off:off + n]

        # -- file-like face, so the helpers below take a window or a raw MultiPartFile ---------
        def seek(self, pos, whence=0):
            if whence != 0:
                raise ValueError("a forward window only seeks absolutely")
            self._pos = int(pos)
            return self._pos

        def tell(self):
            return self._pos

        def read(self, n=-1):
            if n is None or n < 0:
                n = max(0, self.size - self._pos)
            b = self.read_at(self._pos, n)
            self._pos += len(b)
            return b

        def advance(self, pos):
            """Tell the window the walk has moved on, so consumed bytes can be released."""
            if pos > self._base:
                drop = min(pos - self._base, len(self._buf))
                self._base += drop
                self._buf = self._buf[drop:]


    def _sig_at(mpf, pos):
        """The 4-byte signature at `pos`, or b'' past the end."""
        mpf.seek(pos)
        return mpf.read(4)

    def _resync(mpf, pos, had_descriptor, window=None):
        """Return a position that really is the next record, or None.

        Called when `pos` -- computed as data_offset + csize -- is not a signature. The two
        legitimate reasons are a data-descriptor trailer (which is not counted in csize) and a
        few bytes of alignment padding; the illegitimate one is a wrong size, which must not be
        papered over. So: try the four standard trailer lengths, then scan a BOUNDED 64 KB
        window for a local header whose own fields parse. Anything beyond that returns None and
        the caller stops the walk.
        """
        cands = []
        if had_descriptor:
            # with signature: 16 (32-bit sizes) or 24 (zip64);  without: 12 or 20
            cands = [16, 24, 12, 20]
        cands = [0] + cands
        for d in cands:
            s = _sig_at(mpf, pos + d)
            if s[:4] in _STOP_SIGS or s[:4] == LFH:
                return pos + d
        mpf.seek(pos)
        buf = mpf.read(int(window or RESYNC_WINDOW))
        i = 0
        while True:
            i = buf.find(LFH, i)
            if i < 0:
                break
            # Only accept a hit whose own header parses -- "PK\x03\x04" occurs by chance inside
            # compressed data roughly once every 4 GB, and a false accept silently corrupts
            # everything after it.
            if _parse_local_header(buf, i) not in (None, "SHORT"):
                return pos + i
            i += 4
        j = buf.find(CDH)
        if j >= 0:
            return pos + j
        return None

    def find_next_local_header(mpf, pos, window=None):
        """The first local file header at or after `pos`, or None within `window` bytes.

        This is what lets a check START anywhere in a 194 GB archive instead of at byte 0, so it
        searches ENTRY_WINDOW by default rather than the much smaller desync window -- a random
        byte is inside a member's data, and that member can be half a megabyte long.

        It only accepts a candidate whose own header fields parse, because the four bytes
        PK\x03\x04 occur by chance inside compressed data about once every 4 GB -- roughly 45
        times in this archive.
        """
        return _resync(mpf, pos, False, window=window or ENTRY_WINDOW)


    def _inflate_prefix(mpf, data_off, csize, method, want_bytes):
        """Decompress the first `want_bytes` of a member without reading all of it.

        The compressed prefix needed for N uncompressed bytes is not known in advance, so this
        reads a guess and grows it while the output is still short and the input is not
        exhausted. DICOM headers are mostly incompressible text and short binary fields, so the
        first guess is usually enough; the loop is what makes it correct rather than usually
        correct.
        """
        if method == 0:
            mpf.seek(data_off)
            return mpf.read(min(csize, want_bytes))
        take = min(csize, want_bytes + 8192)
        for _ in range(4):
            mpf.seek(data_off)
            raw = mpf.read(take)
            try:
                out = zlib.decompressobj(-zlib.MAX_WBITS).decompress(raw, want_bytes)
            except zlib.error:
                return None
            if len(out) >= want_bytes or take >= csize:
                return out
            take = min(csize, take * 3)
        return out

    def scan_archive_forward(mpf, sizes_by_name=None, header_bytes=16384,
                             want=None, start=0, on_member=None, name_filter=None,
                             progress_every=25000, deadline=None, verbose=True,
                             on_checkpoint=None, checkpoint_every=0, on_failure=None,
                             window_block=8 << 20, should_stop=None):
        """Walk the archive from `start`, reading what each member needs and skipping the rest.

        sizes_by_name : {name: compressed_size} from the central directory, OR a zero-argument
                        callable returning one. Used ONLY when a member sets the data-descriptor
                        flag, where the local header carries zeros. The directory's SIZES are
                        correct in this archive -- it is the OFFSETS that wrapped -- so this is a
                        sound fallback. Pass a callable and it is never invoked unless such a
                        member actually turns up: parsing this directory costs ~7 s of CPU,
                        90-190 MB off Drive and 600-800 MB resident, and an archive written to a
                        seekable file (which is every Kaggle release) has no data descriptors at
                        all, so the common case should not pay for it.
        name_filter   : callable(name) -> bool. Members that fail it are skipped without any
                        data being read at all; the walk still steps over them correctly.
        want          : set of member names -- or a callable(name) -> bool -- whose FULL data
                        should be produced instead of just the header prefix. None means
                        header-only for every member. A callable lets a caller take "the first N
                        members from here", which a set cannot express before the walk has run.
        on_member     : callback(name, data_offset, csize, usize, method, payload). `payload` is
                        the decompressed prefix, or the whole member for a name in `want`, or
                        None if `on_member` only wants positions (header_bytes=0).
        on_checkpoint : callback(next_offset, n_seen) every `checkpoint_every` members, for a
                        caller that persists progress. Called with a position that is always a
                        valid resume point.
        should_stop   : callable() -> bool, checked after each member. A caller that only wants
                        the first N members from a starting point uses this to stop, instead of
                        walking to a deadline -- 40 sampled probes each streaming until a 15 s
                        timer expires is ten minutes of reading nobody asked for.
        on_failure    : callback(name, data_offset, csize, usize, method) for a member whose data
                        could not be read or whose CRC did not match. It exists so a caller can
                        RETRY those members later instead of quietly ending up with a hole: a
                        pass that finishes with unreadable members and calls itself complete is
                        the same defect as a chunk written with 3 of 20,000 rows.

        Returns a dict: next_offset (a resume point that always lands on a record), n_seen,
        n_read, n_failed, stopped ('eof' | 'central-directory' | 'deadline' | 'desync' |
        'no-size' | 'overrun' | 'asked-to-stop'). Only 'central-directory' and 'eof' mean the
        archive was walked to its end; everything else means the index is INCOMPLETE and says
        so.
        """
        # One buffered, forward-only view of the archive for the whole pass. Everything below --
        # the header reads, the prefix inflation, the resync scans -- goes through it, so the
        # archive is touched in large contiguous blocks and a member's pixel data is skipped by
        # being consumed rather than by a round trip.
        win = mpf if isinstance(mpf, ForwardWindow) else ForwardWindow(mpf, block=window_block)
        mpf = win
        pos = int(start)
        n_seen = n_read = n_failed = n_resync = n_crc_bad = 0
        resync_gaps = []
        crc_bad = []
        t0 = time.time()
        total = mpf.size
        stopped = "eof"
        while pos < total:
            mpf.seek(pos)
            head = mpf.read(30 + LFH_SLACK)
            parsed = _parse_local_header(head)
            if parsed == "SHORT":
                mpf.seek(pos)
                head = mpf.read(30 + RESYNC_WINDOW)
                parsed = _parse_local_header(head)
            if parsed is None:
                if head[:4] in _STOP_SIGS:
                    stopped = "central-directory"
                elif not head:
                    stopped = "eof"
                else:
                    # Not a local header, not the central directory, and not the end of the
                    # file: the walk is somewhere it should not be.
                    if verbose:
                        print(f"    stopped at byte {pos:,} after {n_seen:,} members: the bytes "
                              f"there are neither a local header nor a central-directory record "
                              f"({head[:4]!r}).")
                    stopped = "desync"
                break
            name, hlen, csize, usize, method, flags, crc = parsed
            data_off = pos + hlen
            had_dd = bool(flags & 0x08)
            if had_dd and not csize:
                if callable(sizes_by_name):
                    if verbose:
                        print(f"    a data-descriptor member turned up ({name!r}) -- reading the "
                              f"central directory for sizes after all")
                    try:
                        sizes_by_name = sizes_by_name() or {}
                    except Exception as _e:
                        if verbose:
                            print(f"      and it could not be read: {type(_e).__name__}: {_e}")
                        sizes_by_name = {}
                csize = (sizes_by_name or {}).get(name, 0)
                if not csize:
                    # No size from either source. _resync's byte-scan can still find the next
                    # header, but only within its window, and a member's data is ~200 KB. Stop
                    # rather than silently mis-walk the rest of the archive.
                    if verbose:
                        print(f"    stopped at {name!r}: data-descriptor member and the central "
                              f"directory has no size for it either")
                    stopped = "no-size"
                    break
            if on_member is not None and (name_filter is None or name_filter(name)):
                full = want is not None and (want(name) if callable(want) else name in want)
                payload = None
                if header_bytes or full:
                    try:
                        if full:
                            mpf.seek(data_off)
                            raw = mpf.read(csize)
                            payload = (raw if method == 0
                                       else zlib.decompressobj(-zlib.MAX_WBITS).decompress(raw))
                            # A whole member has been read, so its stored CRC32 can be checked
                            # for free -- and the local header carries it, so this needs no
                            # central directory either. This is the integrity check cell B1b
                            # used to make by seeking to 400 directory offsets.
                            if crc and zlib.crc32(payload) & 0xFFFFFFFF != crc:
                                n_crc_bad += 1
                                if len(crc_bad) < 20:
                                    crc_bad.append(name)
                                payload = None
                        else:
                            payload = _inflate_prefix(mpf, data_off, csize, method, header_bytes)
                    except Exception:
                        payload = None
                if payload is None and (header_bytes or full):
                    n_failed += 1
                    if on_failure is not None:
                        try:
                            on_failure(name, data_off, csize, usize, method)
                        except Exception:
                            pass
                else:
                    n_read += 1
                    try:
                        on_member(name, data_off, csize, usize, method, payload)
                    except Exception:
                        n_failed += 1
            n_seen += 1
            nxt = data_off + csize
            if nxt > total:
                # A size that runs off the end of the archive is corrupt, full stop. Left
                # unchecked the loop simply exits because pos >= total and reports "eof", which
                # is indistinguishable from a clean finish -- a truncated index that LOOKS
                # complete is worse than one that says it failed.
                if verbose:
                    print(f"    OVERRAN the archive after {n_seen:,} members: {name!r} claims "
                          f"{csize:,} bytes at {data_off:,}, past the {total:,}-byte end.")
                stopped = "overrun"
                pos = nxt
                break
            sig = _sig_at(mpf, nxt)
            if sig[:4] != LFH and sig[:4] not in _STOP_SIGS and nxt < total:
                fixed = _resync(mpf, nxt, had_dd)
                if fixed is None:
                    if verbose:
                        print(f"    DESYNCHRONISED after {n_seen:,} members, at byte {nxt:,} "
                              f"(last member {name!r}). Stopping rather than inventing members.")
                    stopped = "desync"
                    pos = nxt
                    break
                # RECOVERED -- but recovery is not the same as correctness. Whatever lies
                # between the bad record and the header we resynchronised onto has been SKIPPED,
                # and members skipped here are members missing from the index. A walk that
                # repairs itself and says nothing is how a 5% hole ends up inside a published
                # AUROC, so every gap is counted, the first few are printed, and the caller gets
                # the count back and decides what it means.
                n_resync += 1
                if len(resync_gaps) < 50:
                    resync_gaps.append((int(nxt), int(fixed - nxt), name))
                if verbose and n_resync <= 5:
                    print(f"    resynchronised after {name!r}: skipped {fixed-nxt:,} byte(s) at "
                          f"{nxt:,} to reach the next local header")
                nxt = fixed
            pos = nxt
            win.advance(pos)
            if verbose and progress_every and n_seen % progress_every == 0:
                el = time.time() - t0
                frac = pos / max(total, 1)
                msg = (f"    {n_seen:,} members, {pos/1e9:.1f} of {total/1e9:.1f} GB "
                       f"({frac:.1%}) in {el/60:.1f} min, {win.n_reads:,} read(s)")
                if frac > 0.01:
                    msg += f", ~{el / frac * (1 - frac) / 60:.0f} min left"
                print(msg, flush=True)
            if on_checkpoint is not None and checkpoint_every and n_seen % checkpoint_every == 0:
                on_checkpoint(pos, n_seen)
            if should_stop is not None and should_stop():
                stopped = "asked-to-stop"
                break
            if deadline is not None and time.time() > deadline:
                if verbose:
                    print(f"    stopping at the time budget with {n_seen:,} members done; "
                          f"resume from byte {pos:,}")
                stopped = "deadline"
                break
        if verbose and n_resync:
            print(f"    {n_resync:,} resynchronisation(s) during this pass -- "
                  f"{sum(g[1] for g in resync_gaps):,}+ bytes of the archive were skipped and "
                  f"the members inside them are NOT in the index.")
        return {"next_offset": pos, "n_seen": n_seen, "n_read": n_read,
                "n_failed": n_failed, "n_resync": n_resync, "resync_gaps": resync_gaps,
                "n_crc_bad": n_crc_bad, "crc_bad": crc_bad,
                "n_reads": win.n_reads, "bytes_read": win.bytes_read, "n_seeks": win.n_seeks,
                "stopped": stopped, "clean": stopped in ("central-directory", "eof")
                                              and n_resync == 0 and n_failed == 0
                                              and n_crc_bad == 0}

    print("forward archive scanner ready (scan_archive_forward) -- reads local file headers, "
          "never the central directory")

In [ ]:
# B1c. Verify the split archive before trusting it  [RSNA]
#
# There is no checksum file alongside the parts, so integrity has to be established from the
# archive itself. Three checks, cheapest first:
#
#   1. PART GEOMETRY -- `split` produces equal-sized parts with only the last one short. A short
#      part anywhere else means a truncated or still-uploading file, and everything after it is
#      garbage. (discover_parts already refuses this; here it is reported explicitly.)
#   2. STRUCTURE -- the central directory (zip) or the full header chain (tar) must parse end to
#      end. This is what catches a missing or reordered part: offsets stop lining up.
#   3. CRC ON A SAMPLE -- zip stores a CRC32 per member. Verifying a random sample decompresses
#      real bytes and checks them against a checksum written before upload, which is genuine
#      end-to-end verification of everything in the path. Verifying ALL of them would read the
#      whole 195 GB; a few hundred spread across the archive catches any localised corruption,
#      because a damaged part shows up as a run of consecutive failures.

if DATA_SOURCE != "rsna":
    print("DATA_SOURCE is {!r} -- skipping this cell (Part B / RSNA only).".format(DATA_SOURCE))
else:
    import hashlib
    import os
    import random
    import time
    import zipfile

    import numpy as np

    # Roughly this many members are CRC-checked, spread evenly so every byte range of the
    # archive is covered. At ~250 KB a member that is ~100 MB read from Drive.
    # 0 skips the check entirely.
    RSNA_VERIFY_SAMPLE = 400
    RSNA_VERIFY_SEED = 20250825
    # A HARD wall on the whole verification. This cell once ran for 2,378 seconds and was still
    # going, on its way to reporting 249 failures that did not exist -- a night spent verifying
    # instead of extracting. Whatever else changes, it now stops.
    RSNA_VERIFY_MAX_SECONDS = 600


    def _tar_offsets_from_cache(archive_size):
        """The completed tar header walk cell B2 already checkpointed, if there is one.

        B1b and B2 do the same walk over 752,803 members. B2's is resumable and cached; B1b's
        was neither, and ran unconditionally. Reading B2's cache here means the walk happens
        once per archive rather than once per execution of this cell.
        """
        path = globals().get("RSNA_TAR_INDEX_PATH")
        if not path or not os.path.exists(path):
            return None
        try:
            with open(path) as f:
                st = json.load(f)
        except Exception:
            return None
        if st.get("archive_size") != archive_size or not st.get("complete"):
            return None
        return st.get("offsets") or None


    def verify_rsna_archive(sample=None, seed=RSNA_VERIFY_SEED):
        sample = RSNA_VERIFY_SAMPLE if sample is None else sample
        paths, sizes = discover_parts(RSNA_PARTS_DIR, RSNA_PARTS_PATTERN)
        total = sum(sizes)
        print(f"1. PARTS")
        print(f"   {len(paths)} part(s), {total:,} bytes ({total/1e9:.2f} GB)")
        print(f"   first: {os.path.basename(paths[0])}   last: {os.path.basename(paths[-1])}")
        nums = []
        for p in paths:
            digits = "".join(ch for ch in os.path.basename(p) if ch.isdigit())
            nums.append(int(digits) if digits else -1)
        gaps = [n for n in range(min(nums), max(nums) + 1) if n not in set(nums)] if nums else []
        if gaps:
            raise RuntimeError(
                f"MISSING PART NUMBER(S) {gaps[:20]}. Every byte after the first gap belongs to the "
                f"wrong offset, so nothing downstream can be trusted. Re-upload the missing part(s)."
            )
        print(f"   numbering contiguous {min(nums)}..{max(nums)}, no gaps")
        short = [(os.path.basename(paths[i]), sizes[i]) for i in range(len(sizes) - 1)
                 if sizes[i] != sizes[0]]
        if short:
            raise RuntimeError(f"short part(s) not at the end: {short[:5]}")
        print(f"   all parts {sizes[0]:,} bytes except the last ({sizes[-1]:,}) -- as `split` produces")

        mpf = MultiPartFile(paths)
        fmt = detect_archive_format(mpf)
        print(f"\n2. STRUCTURE  (format: {fmt})")
        t0 = time.time()
        if fmt == "zip":
            zf = zipfile.ZipFile(mpf)
            # REPAIR BEFORE VERIFYING.
            #
            # This archive is 194.7 GB, so its member offsets span the 32-bit central-directory
            # field 45 times over. Without ZIP64 offsets every one past 4 GiB is wrong, and a
            # CRC check then reads garbage: a real run got 249 failures out of the first 250
            # members and would have reported the archive as corrupt. It is not corrupt -- the
            # index into it is, in exactly the way cell 11b was written to fix, and that repair
            # was wired into the India path only.
            _ens = globals().get("ensure_zip_readable")
            if _ens is not None:
                try:
                    _ens(zf, parts_dir, reader_factory=lambda: MultiPartFile(paths),
                         size=mpf.size)
                except Exception as _e:
                    print(f"   offset-repair check skipped: {type(_e).__name__}: {_e}")
            infos = zf.infolist()
            files = [i for i in infos if not i.is_dir()]
            print(f"   central directory parsed: {len(files):,} member(s) in {time.time()-t0:.1f}s")
            by_top = {}
            for i in files:
                top = i.filename.split("/")[0] + "/" if "/" in i.filename else "(archive root)"
                by_top[top] = by_top.get(top, 0) + 1
            for k, v in sorted(by_top.items(), key=lambda kv: -kv[1])[:6]:
                print(f"     {k:<24s} {v:,} member(s)")
            uncompressed = sum(i.file_size for i in files)
            print(f"   uncompressed size would be {uncompressed/1e9:.1f} GB "
                  f"(vs {total/1e9:.1f} GB compressed) -- this is why it is never extracted")
            _dcm = [i for i in files if i.filename.lower().endswith(".dcm")]
            print(f"   .dcm members: {len(_dcm):,}")
            if len(_dcm):
                print(f"   RSNA reference counts: stage_2_train 752,803 / stage_2_test 121,232 "
                      f"(= 874,035 total)")

            print(f"\n3. CRC32 ON A STRATIFIED SAMPLE, READ FORWARD")
            if sample <= 0:
                print("   skipped (RSNA_VERIFY_SAMPLE = 0)")
                return {"format": fmt, "n_members": len(files), "verified": 0, "failed": []}
            # ===================================================================================
            # WHY THIS NO LONGER USES THE CENTRAL DIRECTORY'S OFFSETS
            # ===================================================================================
            # It used to pick members from the directory and call zf.open(info) on each. On the
            # real archive that produced, verbatim:
            #
            #     50/364 checked, 471s, 49 failure(s)
            #     250/364 checked, 2378s, 249 failure(s)
            #
            # Both numbers are the same defect. 249 of 250 "corrupt" because a 194,671,162,157-
            # byte zip wraps the 32-bit offset field 45 times, so nearly every offset points at
            # the wrong place; and 9.5 seconds each because seeking randomly around 91 parts on
            # a Drive FUSE mount is the slowest thing this notebook can possibly do. A check
            # that takes longer than the work it is checking, and reports corruption that is not
            # there, is worse than no check.
            #
            # So it is done the way cell B1c does everything: the archive is entered at evenly
            # spaced BYTE positions, the next real local file header is found from each, and a
            # few members are read FORWARD from there. The CRC32 comes out of the local header
            # itself, so the directory is not consulted for this either, and a member that reads
            # back with the right CRC proves the landing was correct as well as the data.
            #
            # It is also bounded. RSNA_VERIFY_MAX_SECONDS is a hard wall: this cell can never
            # again spend a night verifying instead of extracting.
            # ===================================================================================
            n_strata = max(1, min(int(sample), 40))
            per = max(1, int(sample) // n_strata)
            budget = float(globals().get("RSNA_VERIFY_MAX_SECONDS", 600))
            print(f"   entering the archive at {n_strata} evenly spaced byte position(s), "
                  f"reading {per} member(s) forward from each "
                  f"(hard budget {budget:.0f}s; the central directory's offsets are NOT used)")
            failed, nbytes, checked, t1 = [], 0, 0, time.time()
            _hdr = globals().get("find_next_local_header")
            if _hdr is None:
                print("   cell B1c is not loaded -- skipping (run the cells in order).")
                return {"format": fmt, "n_members": len(files), "verified": 0, "failed": []}
            _zones = []
            for si in range(n_strata):
                _zones.append(int(total * si / n_strata))
            for si, zstart in enumerate(_zones):
                if time.time() - t1 > budget:
                    print(f"   budget reached after {si} of {n_strata} zone(s) -- stopping. "
                          f"What was checked is reported below.")
                    break
                start_at = _hdr(mpf, zstart) if zstart else 0
                if start_at is None:
                    failed.append((f"<zone {si} at byte {zstart:,}>",
                                   "no local file header within 64 KB of this position"))
                    continue
                # A stateful `want`: read the first `per` members forward from this landing
                # point in full (so their CRC32 can be verified) and only the headers of the
                # rest. Expressed as a callable because the names are not known until the walk
                # is under way -- there is no directory being consulted.
                _budget_left = {"n": per, "bytes": 0}

                def _take(_name, _b=_budget_left):
                    if _b["n"] <= 0:
                        return False
                    _b["n"] -= 1
                    return True

                def _acct(_n, _o, _c, _u, _m, _p, _b=_budget_left):
                    if _p is not None:
                        _b["bytes"] += len(_p)

                r = scan_archive_forward(
                    mpf, header_bytes=0, start=start_at, verbose=False,
                    want=_take, on_member=_acct,
                    should_stop=lambda _b=_budget_left: _b["n"] <= 0,
                    deadline=time.time() + max(10.0, budget / n_strata))
                _n_here = per - _budget_left["n"]
                checked += _n_here - r["n_crc_bad"] - r["n_failed"]
                nbytes += _budget_left["bytes"]
                for _n in r["crc_bad"]:
                    failed.append((_n, "CRC mismatch"))
                if r["n_failed"]:
                    failed.append((f"<zone {si}>", f"{r['n_failed']} member(s) unreadable"))
                if r["stopped"] in ("desync", "overrun", "no-size"):
                    failed.append((f"<zone {si} at byte {zstart:,}>",
                                   f"the walk could not stay in step ({r['stopped']})"))
                if (si + 1) % 5 == 0 or si == n_strata - 1 or failed:
                    print(f"   zone {si+1}/{n_strata} at {zstart/1e9:.1f} GB: "
                          f"{checked} member(s) verified, {nbytes/1e6:.0f} MB, "
                          f"{time.time()-t1:.0f}s, {len(failed)} failure(s)", flush=True)
            dt = time.time() - t1
            print(f"   {checked} member(s), {nbytes/1e6:.0f} MB read in {dt:.0f}s "
                  f"({nbytes/1e6/max(dt,1e-6):.1f} MB/s from Drive)")
            if failed:
                print(f"   *** {len(failed)} FAILURE(S) ***")
                for fn, why in failed[:10]:
                    print(f"     {fn}: {why}")
                print("   Failures clustered in consecutive zones point at one bad part;")
                print("   scattered ones point at a flaky connection -- re-run to tell them apart.")
            elif checked:
                print(f"   all {checked} verified byte-for-byte against the CRC32 stored in "
                      f"their own local file headers, at {n_strata} positions spread across the "
                      f"whole archive -- so a single corrupt part could not hide.")
            return {"format": fmt, "n_members": len(files), "verified": checked, "failed": failed}

        elif fmt == "tar":
            # This used to walk all 752,803 members purely to count them, checkpoint nothing, and
            # run unconditionally on every execution of the cell. Over the Drive mount that is
            # hours, it printed nothing while it ran, and a Colab session death threw all of it
            # away -- so on a long run this cell could never finish and B2 was never reached.
            #
            # B2 performs the identical walk and DOES checkpoint it. If that cache exists and
            # matches this archive, the walk has already been done and verified; re-doing it
            # proves nothing new.
            _cached = _tar_offsets_from_cache(mpf.size) if "_tar_offsets_from_cache" in globals() else None
            if _cached:
                print(f"   header chain already walked and cached by cell B2: "
                      f"{len(_cached):,} member(s) -- not repeated here")
                return {"format": fmt, "n_members": len(_cached), "verified": 0, "failed": []}
            n = 0
            _t_last = time.time()
            for _name, _off, _sz, _nxt in walk_tar_members(mpf):
                n += 1
                if time.time() - _t_last > 30:
                    print(f"      walked {n:,} member(s) ({_off/1e9:.1f} GB of "
                          f"{mpf.size/1e9:.1f} GB) ...", flush=True)
                    _t_last = time.time()
            print(f"   header chain walked end to end: {n:,} member(s) in {time.time()-t0:.0f}s")
            print("   A tar carries no per-member checksum, so step 3 is not available. The fact that")
            print("   the header chain reaches the end-of-archive marker is itself strong evidence:")
            print("   every member's size field has to be right for the next header to land on a")
            print("   512-byte boundary, so a missing or reordered part derails it almost immediately.")
            return {"format": fmt, "n_members": n, "verified": 0, "failed": []}
        else:
            raise RuntimeError(f"cannot verify a {fmt!r} archive -- see cell B2's error message.")


    RSNA_VERIFY = verify_rsna_archive()

In [ ]:
# B2. Open the RSNA archive and index every slice's header  [RSNA, RESUMABLE]
#
# The DICOMs sit in one flat folder named by SOPInstanceUID, so nothing in a path says which study
# or series a slice belongs to. The grouping only exists inside the headers, which means one pass
# over ~752,803 members reading ~8 KB each. That is the expensive, unavoidable step of using this
# dataset volumetrically, so it is: read once, cached to Drive as parquet chunks, resumable at
# chunk granularity, and threaded (the cost is per-request latency against Drive, not bandwidth).

if DATA_SOURCE != "rsna":
    print("DATA_SOURCE is {!r} -- skipping this cell (Part B / RSNA only).".format(DATA_SOURCE))
else:
    import hashlib
    import io
    import json
    import os
    import time
    import warnings
    import zipfile

    import numpy as np
    import pandas as pd
    import pydicom
    from tqdm.auto import tqdm

    # RSNA's SOPInstanceUID is literally "ID_<9 hex>" -- not a valid DICOM UI value, so pydicom
    # emits "Invalid value for VR UI" for essentially every one of the ~752,803 files. That is a
    # property of how the dataset was de-identified, not a problem with any individual file, and
    # left unfiltered it buries every real message under three quarters of a million warnings.
    #
    # This line alone was NOT enough, and the way it failed is worth remembering: pydicom's
    # warn_and_log calls LOGGER.warning() AND warnings.warn(), so a warnings filter silences
    # exactly half of it. A real run reached 100,000 members with 918 KB of
    # "WARNING:pydicom:Invalid value for VR UI" already in the output. Cell 2d installs a logging
    # filter for the other half; it is re-asserted here so this cell is correct even if the
    # notebook is run out of order.
    warnings.filterwarnings("ignore", message=r"Invalid value for VR UI.*")
    _q = globals().get("quiet_expected_dicom_noise")
    if _q is not None:
        _q(verbose=False)
    else:
        import logging as _lg
        _lg.getLogger("pydicom").setLevel(_lg.ERROR)
        print("  NOTE: cell 2d has not run, so pydicom noise is silenced by level rather than "
              "by message. Run cell 2d for the narrower filter.")

    RSNA_INDEX_DIR = os.path.join(OUTPUT_DIR, "rsna_index")
    RSNA_TAR_INDEX_PATH = os.path.join(OUTPUT_DIR, "rsna_tar_offsets.json")
    os.makedirs(RSNA_INDEX_DIR, exist_ok=True)

    # ---- the documented 12-bit pixel-representation defect ----------------------------------------
    RSNA_CORRUPT_SOPS = {"ID_6431af929"}    # confirmed corrupt PixelData; every solution repo drops it


    def rsna_needs_pixel_fix(ds):
        """The three conditions, all required, from Jeremy Howard's `cleaning-the-data-for-rapid-
        prototyping-fastai` kernel and copied verbatim into several medal-winning solutions."""
        try:
            return (int(getattr(ds, "BitsStored", 0)) == 12
                    and int(getattr(ds, "PixelRepresentation", 1)) == 0
                    and float(getattr(ds, "RescaleIntercept", -1024)) > -100)
        except Exception:
            return False


    def rsna_fix_pixels(arr, ds):
        """Undo the modulo-4096 wrap and restore the real intercept.

            x = pixel_array + 1000;  x[x >= 4096] -= 4096;  intercept = -1000

        Without this, air in these files reads as roughly +3096 HU instead of -1000. Nothing
        raises -- every one of the 11 windows is simply computed on nonsense, and the model
        returns a confident, unit-norm embedding of garbage.

        What the correction is really asserting is that the stored values are SIGNED HU written
        into unsigned 12 bits, so negatives wrapped:

            HU = px          when px <  3096
            HU = px - 4096   when px >= 3096

        Worked through: HU -1000 -> stored 3096 -> +1000 = 4096 -> -4096 = 0 -> -1000. Correct.
        HU 35 -> stored 35 -> 1035 -> 35. Correct.

        The consequence worth knowing: the recoverable range is exactly [-1000, +3095]. Anything
        originally BELOW -1000 (a scanner that writes air at -1024, say) wraps to a value the
        correction turns into a large POSITIVE number instead. This is inherent to the accepted
        fix, not something added here -- and it is self-consistent, because a 4096-wide window
        starting at -1000 is precisely what an intercept of -1000 means. Volume assembly records
        the 1st/50th/99th HU percentiles of every series and flags anything whose air is not near
        -1000, so if this ever bites you, the progress log says so rather than leaving you to
        wonder. Returns (array, intercept)."""
        x = arr.astype(np.int32) + 1000
        px_mode = 4096
        x[x >= px_mode] -= px_mode
        return x, -1000.0


    # ---- archive access ---------------------------------------------------------------------------
    _RSNA_STATE = {}


    class _LazyZipContainer:
        """A by-name member reader that does not parse the central directory until asked.

        Cell B2 walks the archive forward and never asks. That is the point: on the real RSNA
        archive the directory is 45-times-wrapped and reading a member through it measured 9.5
        seconds. Anything that DOES ask (the label CSV fallback) gets a repaired ZipFile, once.
        """

        def __init__(self, paths, mpf):
            self._paths = list(paths)
            self._mpf = mpf
            self._inner = None

        def _get(self):
            if self._inner is None:
                print("  (parsing the zip central directory -- something asked for a member by "
                      "name; the forward scan never does)")
                zf = zipfile.ZipFile(MultiPartFile(self._paths))
                _ens = globals().get("ensure_zip_readable")
                if _ens is not None:
                    try:
                        _ens(zf, RSNA_ROOT,
                             reader_factory=lambda: MultiPartFile(self._paths),
                             size=self._mpf.size)
                    except Exception as _e:
                        print(f"  offset-repair check skipped: {type(_e).__name__}: {_e}")
                self._inner = ZipMemberOpener(zf)
            return self._inner

        def namelist(self):
            return self._get().namelist()

        def open(self, *a, **k):
            return self._get().open(*a, **k)

        @property
        def parsed(self):
            return self._inner is not None



    def open_rsna_archive(verbose=True):
        """Returns (container, mpf, fmt). Cached for the session -- for zip this re-reads a large
        central directory, and for tar it would otherwise redo the offset walk."""
        if "container" in _RSNA_STATE:
            return _RSNA_STATE["container"], _RSNA_STATE["mpf"], _RSNA_STATE["fmt"]

        paths, sizes = discover_parts(RSNA_PARTS_DIR, RSNA_PARTS_PATTERN)
        total = sum(sizes)
        if verbose:
            print(f"{len(paths)} part(s), {total/1e9:.1f} GB total, "
                  f"{paths[0]} … {os.path.basename(paths[-1])}")
        mpf = MultiPartFile(paths)
        fmt = detect_archive_format(mpf)
        if verbose:
            print(f"archive format: {fmt}")

        if fmt == "zip":
            # DELIBERATELY LAZY. Parsing this central directory costs ~7 s of CPU, 90-190 MB off
            # Drive and 600-800 MB resident, and the forward scan (B1c/B2) does not need it at
            # all. The only things that still want a by-name reader are the label CSV fallback
            # and the tar-shaped test harnesses, so it is built on first use and not before.
            container = _LazyZipContainer(paths, mpf)
        elif fmt == "tar":
            offsets = _load_or_build_tar_offsets(mpf)
            container = TarContainer(mpf, offsets)
        elif fmt in ("gzip", "xz", "bzip2"):
            raise RuntimeError(
                f"the archive is {fmt}-compressed, which has no random access -- a member can only be "
                f"reached by decompressing everything before it. Reassembling and decompressing needs "
                f"~431 GB of scratch, which Colab does not have.\n"
                f"Re-upload as an UNCOMPRESSED tar or a zip (DICOM pixel data barely compresses "
                f"anyway, so you lose almost nothing):  cat part_* | gunzip > all.tar  then re-split."
            )
        else:
            raise RuntimeError(
                f"could not identify the archive (first bytes look like {fmt!r}). Expected a zip or an "
                f"uncompressed tar. Check RSNA_PARTS_PATTERN and that part_00000 really is the first "
                f"part of the archive."
            )

        _RSNA_STATE.update(container=container, mpf=mpf, fmt=fmt, paths=paths)
        return container, mpf, fmt


    def _load_or_build_tar_offsets(mpf):
        """Walk the tar's headers to get every member's data offset. Resumable: the walk checkpoints
        its position, so a session that dies partway continues instead of restarting."""
        state = {"offsets": {}, "next_offset": 0, "archive_size": mpf.size}
        if os.path.exists(RSNA_TAR_INDEX_PATH):
            try:
                with open(RSNA_TAR_INDEX_PATH) as f:
                    loaded = json.load(f)
                if loaded.get("archive_size") == mpf.size:
                    state = loaded
                    if state.get("complete"):
                        print(f"  tar offsets: {len(state['offsets']):,} members (cached)")
                        return state["offsets"]
                    print(f"  tar offsets: resuming from byte {state['next_offset']:,} "
                          f"({len(state['offsets']):,} members so far)")
                else:
                    print("  tar offset cache was built for a different archive size -- rebuilding.")
            except Exception as e:
                print(f"  tar offset cache unreadable ({e}) -- rebuilding.")

        offsets = state["offsets"]
        t0 = time.time()

        def _save(complete=False):
            tmp = RSNA_TAR_INDEX_PATH + ".tmp"
            with open(tmp, "w") as f:
                json.dump({"offsets": offsets, "next_offset": state["next_offset"],
                            "archive_size": mpf.size, "complete": complete}, f)
            os.replace(tmp, RSNA_TAR_INDEX_PATH)

        print("  walking tar headers (one 512-byte read per member, data is seeked over) ...")
        try:
            _walk_finished = False
            for name, off, size, nxt in walk_tar_members(mpf, start_offset=state["next_offset"]):
                offsets[name.lstrip("./")] = [off, size]
                state["next_offset"] = nxt
                if len(offsets) % 50000 == 0:
                    _save()
                    el = time.time() - t0
                    print(f"    {len(offsets):,} members, {state['next_offset']/1e9:.1f}/"
                          f"{mpf.size/1e9:.1f} GB, {el/60:.1f} min")
            _walk_finished = True
            _save(complete=True)
        except KeyboardInterrupt:
            _save()
            print(f"    interrupted at {len(offsets):,} members -- re-run this cell to continue.")
            raise
        except Exception as e:
            _save(complete=False)
            print(f"    tar walk FAILED at byte {state['next_offset']:,} after "
                  f"{len(offsets):,} members: {type(e).__name__}: {e}")
            print("    The index is saved but marked INCOMPLETE, so it will resume rather than be "
                  "mistaken for a finished walk.")
            raise
        print(f"  tar offsets: {len(offsets):,} members in {(time.time()-t0)/60:.1f} min")
        return offsets


    # THREADED PER-MEMBER READS ARE GONE, DELIBERATELY.
    #
    # This is where _new_container_handle() used to live: one independent ZipFile handle per
    # worker thread, twelve at a time, so that twelve members could be fetched at once. It went
    # through three versions -- a full directory re-parse per worker (8-11 GB resident, the
    # session died in chunk 0 every time), then a shared directory with per-worker file handles,
    # then per-worker locks.
    #
    # Every one of them was solving the wrong problem. Threading a per-member random read makes
    # it twelve times less catastrophic; it does not make it right. Cell B1b measured 9.5
    # seconds per member, so 752,803 members threaded twelve ways is still SEVEN DAYS, and the
    # thread pool is what turned Drive's occasional EIO into a storm of them.
    #
    # The header index is now built by ONE sequential forward walk (cell B1b's
    # scan_archive_forward), which reads the archive in 8 MB blocks -- about twenty thousand
    # requests instead of two and a quarter million, with no concurrency to tune and nothing to
    # collide. RSNA_INDEX_WORKERS no longer has any effect on the zip path, and the tar path
    # reads serially through its own exact offset index.


    # ---- the header scan --------------------------------------------------------------------------
    # x and y are stored, not just z: the slice-ordering key is the projection of the FULL
    # position onto the slice normal, and for a tilted gantry the x/y terms are exactly the part
    # that matters. Keeping only z reduces the "projection" to a monotone rescaling of z, i.e. a
    # no-op -- and flips the order outright when the normal's z component is negative.
    #
    # offset/csize/usize/method are stored too. They are what the SECOND pass (cell B5) uses to
    # pull only the wanted series out of the archive without re-walking every header, and they
    # come from the forward walk, so they are TRUE positions -- not the wrapped values the
    # central directory reports.
    RSNA_INDEX_FIELDS = ["member", "sop", "study_uid", "series_uid", "patient_id",
                         "x", "y", "z", "iop", "rows", "cols", "px_y", "px_x",
                         "bits_stored", "pixel_repr", "slope", "intercept", "needs_pixel_fix",
                         "offset", "csize", "usize", "method"]

    RSNA_FWD_STATE_PATH = os.path.join(RSNA_INDEX_DIR, "_forward_state.json")


    class TruncatedHeader(Exception):
        """The prefix did not contain a whole DICOM header. RETRYABLE with more bytes."""


    # The 21 elements every RSNA-ICH file carries (the set ambrzeski's create_dataframe.py
    # enumerates). PixelData sits at about byte 682 in these files, so RSNA_HEADER_BYTES = 16384
    # is roughly 24x the margin needed -- but "roughly 24x" is a statement about the files anyone
    # has looked at, not about all 752,803 of them.
    RSNA_REQUIRED_TAGS = ("SOPInstanceUID", "StudyInstanceUID", "SeriesInstanceUID",
                          "Rows", "Columns")


    def validate_dicom_header(ds, name=""):
        """Return None if the header is whole, or a reason string if it is not.

        ===========================================================================================
        WHY THIS IS NOT PARANOIA
        ===========================================================================================
        Handing pydicom a PREFIX of a file has four documented outcomes, and only one of them is
        an exception:

          1. the prefix ends cleanly between elements -> the generator simply returns. You get a
             Dataset that is MISSING later elements, with zero warnings and zero log records.
          2. the prefix ends INSIDE a value -> fp.read(length) returns short WITHOUT raising, and
             a RawDataElement is yielded holding the truncated value. Measured on a real file:
             ImagePositionPatient came back as [-125.000000, ''] -- two of three coordinates, one
             of them an empty string. Nothing raises. Nothing warns.
          3. the prefix ends inside an element HEADER -> struct.error or BytesLengthException,
             neither of which is a pydicom exception type, so `except Exception` is mandatory.
          4. the prefix ends inside an UNDEFINED-LENGTH element -> EOFError is caught internally
             and read_dataset discards EVERY element it had already parsed, because it builds its
             dict in one comprehension. You get a Dataset with ZERO elements.

        Case 2 is the one that matters here. A truncated ImagePositionPatient does not fail: it
        produces a plausible number, which sorts the slices into the wrong order, which assembles
        a scrambled volume, which embeds without complaint and lands in the classifier as a
        confident 1152-dimensional vector of nonsense. Case 1 and case 4 look, to the code that
        used to be here, exactly like "this file has no SeriesInstanceUID" -- and that was
        silently counted and dropped.

        So every parsed header is checked for wholeness, and anything that fails is RETRIED with
        more bytes rather than dropped or believed.
        """
        try:
            n_elems = len(ds)
        except Exception:
            return "dataset is unreadable"
        if n_elems == 0:
            # Case 4: read_dataset threw away everything it had.
            return "zero elements -- the prefix ended inside an undefined-length element"
        for tag in RSNA_REQUIRED_TAGS:
            v = getattr(ds, tag, None)
            if v is None or (isinstance(v, str) and not v.strip()):
                return f"missing {tag}"
        # Multi-valued geometry is where a mid-value truncation shows itself: the right number of
        # components, all finite, or the header did not finish.
        for tag, want in (("ImagePositionPatient", 3), ("ImageOrientationPatient", 6),
                          ("PixelSpacing", 2)):
            v = getattr(ds, tag, None)
            if v is None:
                continue                      # genuinely absent is a data property, not truncation
            # A truncation that leaves ONE component does not leave a one-element MultiValue --
            # pydicom returns a bare DSfloat, which is not iterable. Iterating it raises
            # TypeError, and a TypeError escaping this validator is the validator failing at
            # exactly the case it exists to catch. Found by sweeping every prefix length.
            seq = v if isinstance(v, (list, tuple)) or hasattr(v, "__len__") else [v]
            try:
                vals = [float(x) for x in seq]
            except (TypeError, ValueError):
                return f"{tag} has a non-numeric component {list(seq)!r} -- truncated mid-value"
            if len(vals) != want:
                return f"{tag} has {len(vals)} of {want} component(s) -- truncated mid-value"
            if not all(np.isfinite(vals)):
                return f"{tag} has a non-finite component {vals!r}"
        try:
            if int(ds.Rows) <= 0 or int(ds.Columns) <= 0:
                return f"implausible image size {ds.Rows}x{ds.Columns}"
        except Exception:
            return "Rows/Columns are not integers"
        return None


    def _row_from_header(name, blob, data_off, csize, usize, method):
        """Parse one DICOM header out of the bytes the forward walk already read.

        Raises TruncatedHeader if the prefix did not contain a whole header -- which the caller
        retries with more bytes. Returns None only for a file that parsed completely and genuinely
        carries no SeriesInstanceUID, which is a property of that file, not of how it was read.
        """
        try:
            ds = pydicom.dcmread(io.BytesIO(blob), stop_before_pixels=True, force=True)
        except Exception as e:
            # struct.error and BytesLengthException both land here and are neither pydicom types
            # nor subclasses of anything useful.
            raise TruncatedHeader(f"{type(e).__name__}: {e}")
        why = validate_dicom_header(ds, name)
        if why is not None:
            raise TruncatedHeader(why)
        suid = getattr(ds, "SeriesInstanceUID", None)
        if not suid:
            return None
        ipp = getattr(ds, "ImagePositionPatient", None)
        iop = getattr(ds, "ImageOrientationPatient", None)
        px = getattr(ds, "PixelSpacing", None)
        # Everything below coerces a tag to int or float. `convert_wrong_length_to_UN` turns a
        # truncated fixed-length element into a UN element holding RAW BYTES, so int(ds.BitsStored)
        # becomes int(b'\x10') -- a ValueError, from a field the validator above does not check
        # because it is not part of the geometry. Found by sweeping every prefix length: four
        # lengths escaped as ValueError instead of being retried. A field that will not coerce is
        # a header that did not finish, so it joins the retry queue like any other short read.
        try:
            return _build_row(ds, name, ipp, iop, px, suid, data_off, csize, usize, method)
        except TruncatedHeader:
            raise
        except Exception as e:
            raise TruncatedHeader(f"a header field would not coerce ({type(e).__name__}: {e}) "
                                  f"-- the prefix ended inside it")


    def _build_row(ds, name, ipp, iop, px, suid, data_off, csize, usize, method):
        return {
            "member": name,
            "x": float(ipp[0]) if ipp is not None and len(ipp) == 3 else np.nan,
            "y": float(ipp[1]) if ipp is not None and len(ipp) == 3 else np.nan,
            "sop": str(getattr(ds, "SOPInstanceUID", "") or ""),
            "study_uid": str(getattr(ds, "StudyInstanceUID", "") or ""),
            "series_uid": str(suid),
            "patient_id": str(getattr(ds, "PatientID", "") or ""),
            "z": float(ipp[2]) if ipp is not None and len(ipp) == 3 else np.nan,
            "iop": ",".join(f"{float(v):.6g}" for v in iop) if iop is not None and len(iop) == 6 else "",
            "rows": int(getattr(ds, "Rows", 0) or 0),
            "cols": int(getattr(ds, "Columns", 0) or 0),
            "px_y": float(px[0]) if px is not None and len(px) == 2 else np.nan,
            "px_x": float(px[1]) if px is not None and len(px) == 2 else np.nan,
            "bits_stored": int(getattr(ds, "BitsStored", 0) or 0),
            "pixel_repr": int(getattr(ds, "PixelRepresentation", -1)),
            "slope": float(getattr(ds, "RescaleSlope", 1) or 1),
            "intercept": float(getattr(ds, "RescaleIntercept", 0) or 0),
            "needs_pixel_fix": bool(rsna_needs_pixel_fix(ds)),
            "offset": int(data_off),
            "csize": int(csize),
            "usize": int(usize),
            "method": int(method),
        }


    def _read_one_header(container, member):
        """Kept for the tar path and for tests: read one member's header through the container.

        The ZIP path does NOT use this any more. Reading members one at a time through their
        central-directory offsets is what measured 9.5 s each in cell B1b -- 82 days for the
        752,803 members of stage_2_train -- and no amount of threading fixes an access pattern.
        """
        with container.open(member, max_bytes=RSNA_HEADER_BYTES) as fh:
            blob = fh.read(RSNA_HEADER_BYTES)
        return _row_from_header(member, blob, -1, -1, -1, -1)


    def _member_in_splits(name):
        if name.endswith("/") or not name.lower().endswith(".dcm"):
            return False
        return any(f"/{sp}/" in f"/{name}" or name.startswith(f"{sp}/") for sp in RSNA_SPLITS)


    def _forward_partition_key(archive_size):
        return {
            "mode": "forward-local-header-scan",
            "archive_size": int(archive_size),
            "splits": list(RSNA_SPLITS),
            "fields_sha1": hashlib.sha1("|".join(RSNA_INDEX_FIELDS).encode()).hexdigest(),
            "header_bytes": int(RSNA_HEADER_BYTES),
            "flush_rows": int(RSNA_INDEX_CHUNK),
            "schema_version": 4,
        }


    def build_rsna_member_index_forward(mpf):
        """One sequential pass over the archive, reading local file headers.

        RESUMABILITY. The walk checkpoints a BYTE OFFSET, not a chunk number, and every offset it
        reports is a position it verified lands on a record. A shard of rows is written and
        renamed into place BEFORE the state file naming its end offset is updated, so a session
        that dies between the two re-reads a shard's worth of members and produces duplicate
        rows -- which the concat deduplicates on `member`. The other order would LOSE rows
        silently, which is the failure this whole notebook keeps being rewritten to avoid.
        """
        key = _forward_partition_key(mpf.size)
        state = {"next_offset": 0, "n_seen": 0, "shard": 0, "key": key}
        if os.path.exists(RSNA_FWD_STATE_PATH):
            try:
                with open(RSNA_FWD_STATE_PATH) as f:
                    loaded = json.load(f)
                if loaded.get("key") == key:
                    state = loaded
                    print(f"  resuming the forward scan from byte {state['next_offset']:,} "
                          f"({state['n_seen']:,} members walked, {state['shard']} shard(s) written)")
                    if state.get("complete"):
                        print("  the forward scan is already COMPLETE for this archive.")
                else:
                    print("  the cached forward scan was built with different settings "
                          "(splits, schema, header size or archive size) -- restarting it.")
                    for f_ in os.listdir(RSNA_INDEX_DIR):
                        if f_.startswith("fwd_"):
                            os.remove(os.path.join(RSNA_INDEX_DIR, f_))
            except Exception as e:
                print(f"  forward-scan state unreadable ({e}) -- restarting the scan.")

        # The central directory is NOT needed to walk the archive, and on this archive reading
        # it is expensive (~7 s, 90-190 MB off Drive, 600-800 MB resident) and may simply fail.
        # It is needed only if a member turns up with the data-descriptor flag set, which an
        # archive written to a seekable file never has. So it is passed as a CALLABLE: the walk
        # invokes it if and only if it actually meets such a member, and prints when it does.
        def _sizes_from_directory():
            zf_ = zipfile.ZipFile(MultiPartFile(_RSNA_STATE["paths"]))
            out = {zi.filename: zi.compress_size for zi in zf_.filelist}
            print(f"    central directory read for SIZES only ({len(out):,} names); its "
                  f"OFFSETS are ignored entirely")
            return out

        buf = []
        stats = {"parsed": 0, "no_series": 0, "unreadable": 0, "truncated": 0}
        truncation_reasons = []
        bad = {}                       # member -> [offset, csize, usize, method]

        def _flush(next_offset, n_seen, complete=False):
            # The checkpoint is also the health check. It fires every ~20,000 members, which on
            # this archive is roughly every 5 GB read -- often enough to catch the DriveFS cache
            # filling the disk, rare enough to cost nothing.
            _hb = globals().get("heartbeat")
            if _hb is not None:
                _hb("B2 forward header index", f"{n_seen:,} members walked", every_s=60,
                    next_offset=int(next_offset), rows=int(stats["parsed"]))
            _hc = globals().get("drive_health_check")
            if _hc is not None and n_seen:
                try:
                    _h = _hc(probe_path=RSNA_PARTS_DIR, verbose=True)
                    if _h["cache_gb"] > 5 or _h["free_gb"] < 60:
                        print(f"    disk: {_h['free_gb']:.0f} GB free, DriveFS cache "
                              f"{_h['cache_gb']:.1f} GB")
                except RuntimeError:
                    # Out of disk. Save what we have -- the caller's shard and offset are still
                    # valid -- and let it propagate so the run stops here rather than corrupting
                    # a half-written parquet.
                    _save_shard_and_state(next_offset, n_seen, complete=False)
                    raise
            _save_shard_and_state(next_offset, n_seen, complete=complete)

        def _save_shard_and_state(next_offset, n_seen, complete=False):
            if buf:
                df = pd.DataFrame(buf, columns=RSNA_INDEX_FIELDS)
                tmp = os.path.join(RSNA_INDEX_DIR, f".fwd_{state['shard']:05d}.tmp.parquet")
                df.to_parquet(tmp, index=False)
                os.replace(tmp, os.path.join(RSNA_INDEX_DIR, f"fwd_{state['shard']:05d}.parquet"))
                state["shard"] += 1
                buf.clear()
            state["next_offset"] = int(next_offset)
            state["n_seen"] = int(n_seen)
            state["key"] = key
            state["complete"] = bool(complete)
            state.setdefault("failures", {})
            tmp = RSNA_FWD_STATE_PATH + ".tmp"
            with open(tmp, "w") as f:
                json.dump(state, f, indent=2)
            os.replace(tmp, RSNA_FWD_STATE_PATH)

        def _on_member(name, data_off, csize, usize, method, payload):
            if payload is None:
                stats["unreadable"] += 1
                bad[name] = [int(data_off), int(csize), int(usize), int(method)]
                return
            try:
                row = _row_from_header(name, payload, data_off, csize, usize, method)
            except TruncatedHeader as e:
                # NOT a bad file -- a short read. Queue it for a retry with more bytes, and keep
                # the first few reasons so the log says WHY rather than just how many.
                stats["truncated"] += 1
                if len(truncation_reasons) < 10:
                    truncation_reasons.append(f"{name}: {e}")
                bad[name] = [int(data_off), int(csize), int(usize), int(method)]
                return
            except Exception:
                stats["unreadable"] += 1
                bad[name] = [int(data_off), int(csize), int(usize), int(method)]
                return
            if row is None:
                stats["no_series"] += 1
                return
            stats["parsed"] += 1
            buf.append(row)

        def _on_failure(name, data_off, csize, usize, method):
            stats["unreadable"] += 1
            bad[name] = [int(data_off), int(csize), int(usize), int(method)]

        def _retry_failures():
            """Read the members the sweep could not, one at a time, by their TRUE offsets.

            Per-member seeking is the access pattern this whole cell exists to avoid -- but a
            handful of retries is not a seek storm, and the alternative is a pass that finishes
            with holes and calls itself complete. That is the same defect as a chunk written
            with 3 of 20,000 rows, and it is the one that keeps costing nights.

            If the list is not a handful, the pass is NOT marked complete: something is wrong
            with the archive or the mount, and grinding through 100,000 nine-second reads is not
            the answer to it.
            """
            if not bad:
                return 0
            carry = dict(state.get("failures") or {})
            carry.update(bad)
            if len(carry) > RSNA_MAX_RETRY_MEMBERS:
                print(f"    {len(carry):,} member(s) were unreadable -- far too many to retry "
                      f"one at a time. NOT marking the index complete. Check the mount and the "
                      f"parts before re-running.")
                state["failures"] = carry
                return len(carry)
            print(f"    retrying {len(carry):,} unreadable member(s) individually ...")
            still = dict(carry)
            # Two rounds, deliberately. The first has no backoff at all: a Drive EIO that has
            # already cleared costs nothing to re-read, and on a healthy mount this recovers
            # everything in seconds. Only what survives that gets the patient treatment, and
            # that round is bounded by a wall clock -- 5,000 members x 3 sleeping attempts is
            # four hours, which is not a retry, it is another lost night.
            _t_retry = time.time()
            for _round, _sleep in ((0, 0.0), (1, 0.35)):
                if not still:
                    break
                if _round and time.time() - _t_retry > RSNA_RETRY_MAX_SECONDS:
                    print(f"      retry budget ({RSNA_RETRY_MAX_SECONDS:.0f}s) reached with "
                          f"{len(still):,} still unread -- stopping.")
                    break
                _todo = sorted(still.items(), key=lambda kv: kv[1][0])
                _now = {}
                with MultiPartFile(_RSNA_STATE["paths"]) as _m2:
                    for _nm, (_off, _cs, _us, _mt) in _todo:
                        if time.time() - _t_retry > RSNA_RETRY_MAX_SECONDS:
                            _now[_nm] = [_off, _cs, _us, _mt]
                            continue
                        _row = None
                        # ESCALATING PREFIX. A member queued because its header was TRUNCATED will
                        # be truncated again at the same size, however many times it is retried --
                        # retrying an unchanged read is not a retry. So each attempt asks for more,
                        # ending at the whole member, which cannot be short by construction.
                        for _try, _mult in enumerate((1, 8, 0)):
                            _want = _cs if _mult == 0 else min(_cs, RSNA_HEADER_BYTES * _mult)
                            try:
                                _blob = _inflate_prefix(_m2, _off, _cs, _mt, _want)
                            except Exception:
                                _blob = None
                            if _blob:
                                try:
                                    _row = _row_from_header(_nm, _blob, _off, _cs, _us, _mt)
                                except TruncatedHeader:
                                    _row = None          # try again with more bytes
                                except Exception:
                                    _row = None
                            if _row is not None:
                                break
                            if _sleep:
                                time.sleep(_sleep * (_try + 1))
                        if _row is None:
                            _now[_nm] = [_off, _cs, _us, _mt]
                        else:
                            buf.append(_row)
                            stats["parsed"] += 1
                print(f"      round {_round + 1}: {len(still) - len(_now):,} recovered, "
                      f"{len(_now):,} still unread")
                still = _now
            state["failures"] = still
            print(f"    {len(carry) - len(still):,} recovered, {len(still):,} still unreadable "
                  f"in {time.time() - _t_retry:.0f}s")
            return len(still)

        if not state.get("complete"):
            t0 = time.time()
            deadline = (t0 + MAX_SECONDS_PER_RUN) if MAX_SECONDS_PER_RUN else None
            print(f"  walking local file headers forward from byte {state['next_offset']:,} "
                  f"of {mpf.size/1e9:.1f} GB ...")
            res = scan_archive_forward(
                mpf, sizes_by_name=_sizes_from_directory, header_bytes=RSNA_HEADER_BYTES,
                start=state["next_offset"], name_filter=_member_in_splits,
                on_member=_on_member, on_failure=_on_failure, on_checkpoint=_flush,
                checkpoint_every=max(1000, RSNA_INDEX_CHUNK), deadline=deadline)
            _residual = _retry_failures()
            _done = res["stopped"] in ("central-directory", "eof") and _residual == 0
            if _residual:
                # The index is NOT complete, and re-running will not fix it by itself: the walk
                # resumes at the end of the archive. Say what to do instead of implying a retry
                # will help.
                print(f"    *** {_residual:,} member(s) could not be read even on retry. The "
                      f"index is marked INCOMPLETE and their names are recorded in "
                      f"{os.path.basename(RSNA_FWD_STATE_PATH)}. Re-running retries exactly "
                      f"those; if they keep failing, the archive part(s) covering their offsets "
                      f"need re-uploading. ***")
            if res.get("n_resync"):
                # The walk repaired itself around a bad record. That is better than dying, and
                # it is NOT the same as a complete index -- the bytes it stepped over held
                # members, and those members are absent. Said plainly here, and again in the
                # summary, because a self-repairing pass that stays quiet is how a hole gets
                # into a result.
                print(f"    *** {res['n_resync']:,} gap(s) were skipped to stay in step. The "
                      f"index is missing whatever was inside them. First few (offset, bytes "
                      f"skipped): {[(g[0], g[1]) for g in res['resync_gaps'][:5]]} ***")
            globals()["RSNA_SCAN_RESYNCS"] = int(globals().get("RSNA_SCAN_RESYNCS", 0)
                                                 + res.get("n_resync", 0))
            _flush(res["next_offset"], state["n_seen"] + res["n_seen"], complete=_done)
            el = time.time() - t0
            print(f"  walked {res['n_seen']:,} member(s) in {el/60:.1f} min "
                  f"({el/max(res['n_seen'],1)*1000:.1f} ms each), stopped: {res['stopped']}")
            print(f"    parsed {stats['parsed']:,} header(s); "
                  f"{stats['no_series']:,} with no SeriesInstanceUID; "
                  f"{stats['truncated']:,} truncated (retried with more bytes); "
                  f"{stats['unreadable']:,} unreadable")
            if truncation_reasons:
                print(f"    the {RSNA_HEADER_BYTES:,}-byte prefix was not enough for "
                      f"{stats['truncated']:,} member(s). First few reasons:")
                for _r in truncation_reasons[:5]:
                    print(f"      {_r}")
                print(f"    A truncated header does not raise -- it yields plausible, WRONG "
                      f"geometry (a two-component ImagePositionPatient, say), which would sort "
                      f"the slices wrongly and embed a scrambled volume. They are retried with "
                      f"8x the prefix and then with the whole member.")
            if res["stopped"] in ("desync", "overrun", "no-size"):
                print(f"    *** The walk STOPPED early ({res['stopped']}). It did not continue "
                      f"past the bad record, so nothing above it is invented -- but the index is "
                      f"INCOMPLETE, and re-running resumes from the same byte and will stop "
                      f"there again. This needs a look, not another run. ***")
            elif res["stopped"] == "deadline":
                print("    time budget reached -- re-run this cell to continue from where it "
                      "stopped. Nothing is re-read.")
        elif state.get("failures"):
            # A previous session finished the walk but left members unread. Retry only those.
            print(f"  the walk is finished but {len(state['failures']):,} member(s) were never "
                  f"read -- retrying just those.")
            _residual = _retry_failures()
            _flush(state["next_offset"], state["n_seen"], complete=(_residual == 0))
        complete = bool(state.get("complete"))

        parts = sorted(f for f in os.listdir(RSNA_INDEX_DIR) if f.startswith("fwd_"))
        if not parts:
            return pd.DataFrame(columns=RSNA_INDEX_FIELDS), complete, 0
        idx = pd.concat([pd.read_parquet(os.path.join(RSNA_INDEX_DIR, f)) for f in parts],
                        ignore_index=True)
        n_before = len(idx)
        idx = idx.drop_duplicates(subset="member", keep="last").reset_index(drop=True)
        if len(idx) != n_before:
            print(f"  {n_before - len(idx):,} duplicate row(s) dropped -- expected when a previous "
                  f"session died between writing a shard and recording it.")
        return idx, complete, state.get("n_seen", 0)


    def build_rsna_member_index_random_access(container):
        """The old path: ask the container for each member by name. Used for tar only.

        A tar's offset walk is itself sequential and its index is exact, so per-member reads
        there are ordinary seeks into a file, not a 45-times-wrapped 32-bit offset field.
        """
        members = [n for n in container.namelist() if _member_in_splits(n)]
        members.sort()
        print(f"{len(members):,} DICOM member(s) under {RSNA_SPLITS}")
        if not members:
            raise RuntimeError(
                f"no .dcm members matched RSNA_SPLITS={RSNA_SPLITS}. A sample of what IS in the "
                f"archive:\n  " + "\n  ".join(container.namelist()[:12])
            )
        chunks = [members[i:i + RSNA_INDEX_CHUNK] for i in range(0, len(members), RSNA_INDEX_CHUNK)]
        partition_key = {
            "mode": "random-access", "chunk": int(RSNA_INDEX_CHUNK), "splits": list(RSNA_SPLITS),
            "n_members": len(members),
            "members_sha1": hashlib.sha1("\n".join(members).encode()).hexdigest(),
            "fields_sha1": hashlib.sha1("|".join(RSNA_INDEX_FIELDS).encode()).hexdigest(),
            "header_bytes": int(RSNA_HEADER_BYTES), "schema_version": 4,
        }
        key_path = os.path.join(RSNA_INDEX_DIR, "_partition.json")
        old_key = None
        if os.path.exists(key_path):
            try:
                with open(key_path) as f:
                    old_key = json.load(f)
            except Exception:
                old_key = None
        if old_key is not None and old_key != partition_key:
            stale = [f for f in os.listdir(RSNA_INDEX_DIR) if f.startswith("chunk_")]
            print(f"the chunk partition changed. Discarding {len(stale)} cached chunk file(s) "
                  f"rather than concatenating two different partitions into duplicate rows.")
            for f in stale:
                try:
                    os.remove(os.path.join(RSNA_INDEX_DIR, f))
                except Exception:
                    pass
        with open(key_path, "w") as f:
            json.dump(partition_key, f, indent=2)
        done_chunks = {int(f.split('_')[1].split('.')[0])
                       for f in os.listdir(RSNA_INDEX_DIR) if f.startswith("chunk_")}
        todo = [i for i in range(len(chunks)) if i not in done_chunks]
        print(f"{len(done_chunks)} chunk(s) already indexed, {len(todo)} to go")
        t0 = time.time()
        for ci in tqdm(todo, desc="indexing headers"):
            batch = chunks[ci]
            rows = []
            for name in batch:
                try:
                    r = _read_one_header(container, name)
                except Exception:
                    r = None
                if r is not None:
                    rows.append(r)
            n_bad = len(batch) - len(rows)
            if n_bad / max(len(batch), 1) > RSNA_CHUNK_MAX_BAD_FRACTION:
                print(f"  chunk {ci}: {n_bad:,}/{len(batch):,} unreadable -- NOT saved, retried "
                      f"next run.")
                continue
            df_chunk = pd.DataFrame(rows, columns=RSNA_INDEX_FIELDS)
            tmp = os.path.join(RSNA_INDEX_DIR, f".chunk_{ci:05d}.tmp.parquet")
            df_chunk.to_parquet(tmp, index=False)
            os.replace(tmp, os.path.join(RSNA_INDEX_DIR, f"chunk_{ci:05d}.parquet"))
            if (MAX_SECONDS_PER_RUN is not None) and (time.time() - t0 > MAX_SECONDS_PER_RUN):
                print(f"  time budget reached -- re-run this cell to continue.")
                break
        parts = sorted(f for f in os.listdir(RSNA_INDEX_DIR) if f.startswith("chunk_"))
        idx = pd.concat([pd.read_parquet(os.path.join(RSNA_INDEX_DIR, f)) for f in parts],
                        ignore_index=True) if parts else pd.DataFrame(columns=RSNA_INDEX_FIELDS)
        return idx, len(parts) == len(chunks), len(members)


    def build_rsna_member_index():
        container, mpf, fmt = open_rsna_archive()
        if fmt == "zip":
            idx, complete, n_walked = build_rsna_member_index_forward(mpf)
            n_expected = n_walked
        else:
            idx, complete, n_expected = build_rsna_member_index_random_access(container)
        globals()["RSNA_INDEX_COMPLETE"] = bool(complete)
        globals()["RSNA_INDEX_EXPECTED_MEMBERS"] = int(n_expected)
        print(f"\nindexed {len(idx):,} slice header(s) "
              f"({'COMPLETE' if complete else 'INCOMPLETE -- re-run this cell to continue'})")
        _rs = int(globals().get("RSNA_SCAN_RESYNCS", 0))
        if _rs:
            print(f"  NOTE: the forward walk resynchronised {_rs:,} time(s) around damaged "
                  f"record(s). It reached the end of the archive, but some members were stepped "
                  f"over and are not indexed. Treat coverage as approximate.")
        _dupes = int(idx.duplicated(subset="member").sum()) if len(idx) else 0
        if _dupes:
            print(f"  *** {_dupes:,} DUPLICATE member row(s) survived deduplication. That should "
                  f"be impossible -- delete {RSNA_INDEX_DIR} and re-index. ***")
        if not complete:
            print("  Everything downstream will cover only the series indexed so far, and will "
                  "say so.")
        return idx


    RSNA_INDEX = build_rsna_member_index()

    if len(RSNA_INDEX):
        print()
        print(f"studies : {RSNA_INDEX['study_uid'].nunique():,}")
        print(f"series  : {RSNA_INDEX['series_uid'].nunique():,}")
        print(f"patients: {RSNA_INDEX['patient_id'].nunique():,}")
        _spc = RSNA_INDEX.groupby("series_uid").size()
        print(f"slices per series: median {_spc.median():.0f}, "
              f"range {_spc.min()}-{_spc.max()} "
              f"(the dataset paper says 20-60 axial images, 3-5 mm)")
        # The 12-bit correction, broken into its three conditions rather than reported as one
        # number. The notebook used to assert that "a large subset" of the 12-bit unsigned files
        # carry a bad intercept; on the real archive that turned out to be 123 slices of 752,803.
        # Both halves are worth seeing: a large 12-bit population with almost none of it needing
        # the fix is a different fact from a small population, and only one of them was written
        # down before anyone measured it.
        _12u = ((RSNA_INDEX["bits_stored"] == 12) & (RSNA_INDEX["pixel_repr"] == 0))
        _fix = int(RSNA_INDEX["needs_pixel_fix"].sum())
        _n12u = int(_12u.sum())
        print(f"12-bit unsigned slices (BitsStored=12, PixelRepresentation=0): {_n12u:,} "
              f"({_n12u/max(len(RSNA_INDEX),1):.1%})")
        print(f"  of which RescaleIntercept > -100, so the modulo-4096 correction applies: "
              f"{_fix:,} ({_fix/max(_n12u,1):.2%} of them, "
              f"{_fix/max(len(RSNA_INDEX),1):.2%} of all slices)")
        print(f"  -- applied automatically in volume assembly; the other {_n12u - _fix:,} "
              f"12-bit files already carry a sane intercept and are left alone")
        if _n12u and not _fix:
            print("  NOTE: no slice meets all three conditions. Either this export was already "
                  "corrected, or the condition is wrong for it -- check hu_p01 in the progress "
                  "log during extraction: air must come back near -1000, not near +3096.")
        _noz = int(RSNA_INDEX["z"].isna().sum())
        if _noz:
            print(f"slices with no ImagePositionPatient: {_noz:,} (their series fall back to "
                  f"filename order and are flagged)")
        _iop = RSNA_INDEX["iop"].value_counts()
        print(f"distinct ImageOrientationPatient values: {len(_iop)}"
              + (f"  most common: {_iop.index[0]}" if len(_iop) else ""))
        if len(_iop) > 1:
            print("  more than one orientation is present -- slice ordering uses the slice normal "
                  "(cross product of the two direction cosines), not raw z, which is what makes a "
                  "tilted-gantry study sort correctly.")

In [ ]:
# B3. RSNA labels: per-slice -> per-series  [RSNA]
#
# stage_2_train.csv is long-format, six rows per image:
#     ID,Label
#     ID_63eb1e259_epidural,0
#     ID_63eb1e259_intraparenchymal,0
#     ...
#     ID_63eb1e259_any,0
# The `ID_<9 hex>` prefix is the SOPInstanceUID and the DICOM filename stem, which is the join key.
#
# Two things to get right:
#   * SPLIT ON THE LAST UNDERSCORE. A plain 3-way split only works because the hash happens to
#     contain no underscore; rsplit(n=1) is correct regardless.
#   * DEDUPLICATE BEFORE PIVOTING. stage_1_train.csv has 4,045,572 rows for 674,258 images --
#     4,045,548 expected, so 24 extra: four images duplicated. `pivot` raises on duplicate index
#     entries, and `pivot_table` would silently average them.

if DATA_SOURCE != "rsna":
    print("DATA_SOURCE is {!r} -- skipping this cell (Part B / RSNA only).".format(DATA_SOURCE))
else:
    import io
    import os

    import numpy as np
    import pandas as pd

    RSNA_SUBTYPES = ["epidural", "intraparenchymal", "intraventricular",
                     "subarachnoid", "subdural", "any"]
    RSNA_LABEL_CACHE = os.path.join(OUTPUT_DIR, "rsna_series_labels.parquet")


    def _find_label_member(container):
        cands = [n for n in container.namelist()
                 if os.path.basename(n).lower() == RSNA_LABEL_CSV_NAME.lower()]
        if not cands:
            csvs = [n for n in container.namelist() if n.lower().endswith(".csv")]
            raise FileNotFoundError(
                f"{RSNA_LABEL_CSV_NAME!r} is not in the archive. CSVs that ARE present: {csvs[:10]}\n"
                f"Set RSNA_LABEL_CSV_NAME, or point RSNA_LABEL_CSV_PATH at a copy on Drive."
            )
        return sorted(cands, key=len)[0]


    def load_rsna_slice_labels():
        # A missing standalone CSV is not fatal: the same file is inside the archive. Degrading
        # here rather than raising matters for an unattended run -- a path that was right last week
        # and is wrong tonight should cost a slower read, not the whole night.
        if RSNA_LABEL_CSV_PATH and os.path.exists(RSNA_LABEL_CSV_PATH):
            print(f"reading labels from {RSNA_LABEL_CSV_PATH}")
            raw = pd.read_csv(RSNA_LABEL_CSV_PATH)
        else:
            if RSNA_LABEL_CSV_PATH:
                print(f"NOTE: RSNA_LABEL_CSV_PATH ({RSNA_LABEL_CSV_PATH}) does not exist -- "
                      f"falling back to the copy inside the archive.")
            container, _, _ = open_rsna_archive(verbose=False)
            member = _find_label_member(container)
            print(f"reading labels from inside the archive: {member}")
            with container.open(member) as fh:
                raw = pd.read_csv(io.BytesIO(fh.read()))

        if set(raw.columns) != {"ID", "Label"}:
            print(f"  NOTE: expected columns ['ID','Label'], got {list(raw.columns)}. Continuing with "
                  f"the first two.")
            raw = raw.iloc[:, :2]
            raw.columns = ["ID", "Label"]

        n_raw = len(raw)
        raw = raw.drop_duplicates(subset="ID", keep="first")
        n_dup = n_raw - len(raw)

        parts = raw["ID"].astype(str).str.rsplit("_", n=1)
        raw = raw.assign(sop=parts.str[0], subtype=parts.str[1])
        raw["Label"] = pd.to_numeric(raw["Label"], errors="coerce").fillna(0).astype(np.int8)

        wide = raw.pivot(index="sop", columns="subtype", values="Label").fillna(0).astype(np.int8)
        wide = wide.reset_index()
        missing = [s for s in RSNA_SUBTYPES if s not in wide.columns]
        for s in missing:
            wide[s] = np.int8(0)

        print(f"  {n_raw:,} rows -> {len(wide):,} unique images x {len(RSNA_SUBTYPES)} subtypes")
        if n_dup:
            print(f"  {n_dup} duplicate ID row(s) dropped ({n_dup // len(RSNA_SUBTYPES)} duplicated "
                  f"image(s)) -- expected; stage_1_train.csv has exactly 4 of these.")
        if missing:
            print(f"  WARNING: these subtypes were absent from the CSV and set to 0: {missing}")

        # `any` should be the OR of the five compartments. The dataset paper does not promise this
        # (an annotator could mark "hemorrhage present, subtype unclear"), so check rather than assume.
        sub5 = [s for s in RSNA_SUBTYPES if s != "any"]
        derived = (wide[sub5].to_numpy() > 0).any(axis=1).astype(np.int8)
        disagree_any_only = int(((wide["any"].to_numpy() == 1) & (derived == 0)).sum())
        disagree_sub_only = int(((wide["any"].to_numpy() == 0) & (derived == 1)).sum())
        print(f"  'any' vs OR(5 subtypes): {disagree_any_only:,} image(s) marked 'any' with no named "
              f"subtype, {disagree_sub_only:,} with a subtype but 'any'=0")
        if disagree_any_only or disagree_sub_only:
            print("    -> they are NOT identical. `any` is used as given (it is the labelled column), "
                  "not recomputed.")
        return wide


    def aggregate_labels_to_series(slice_labels, index):
        """A series is positive for a subtype if ANY of its slices is.

        That is the aggregation the labelling protocol implies -- annotators marked the slices on
        which a bleed was visible, so a study with one positive slice is a study with a bleed. It also
        makes this task strictly EASIER than the competition's per-slice task, so no number this
        notebook prints is comparable to a competition log-loss.

        `*_n_pos_slices` is kept alongside: a single-positive-slice series is a much weaker positive
        than a twelve-slice one, and you may want to require a minimum before trusting the label."""
        merged = index.merge(slice_labels, left_on="sop", right_on="sop", how="left")
        n_unlabelled = int(merged["any"].isna().sum())
        # Count the label matches BEFORE fillna, otherwise every slice looks labelled. This is the
        # one column that would reveal a broken filename<->SOPInstanceUID join, so it has to be a
        # real count of non-null label rows, not a group size.
        merged["_has_label"] = merged["any"].notna()
        for s in RSNA_SUBTYPES:
            merged[s] = pd.to_numeric(merged[s], errors="coerce").fillna(0).astype(np.int8)

        agg = merged.groupby("series_uid").agg(
            study_uid=("study_uid", "first"),
            patient_id=("patient_id", "first"),
            n_slices=("member", "size"),
            n_labelled=("_has_label", "sum"),
            **{f"rsna_{s}": (s, "max") for s in RSNA_SUBTYPES},
            **{f"rsna_{s}_n_pos_slices": (s, "sum") for s in RSNA_SUBTYPES},
        ).reset_index()

        print(f"  {n_unlabelled:,} indexed slice(s) had no label row "
              f"({'expected if RSNA_SPLITS includes stage_2_test, which is unlabelled' if n_unlabelled else 'none'})")
        _partial = agg[(agg["n_labelled"] > 0) & (agg["n_labelled"] < agg["n_slices"])]
        _none = agg[agg["n_labelled"] == 0]
        print(f"  {len(agg):,} series")
        if len(_partial) or len(_none):
            print(f"  {len(_partial):,} series are only PARTIALLY labelled and {len(_none):,} have "
                  f"no labelled slice at all. On stage_2_train every image should have a label "
                  f"row, so anything here points at a filename<->SOPInstanceUID join problem, not "
                  f"at the data.")
        for s in RSNA_SUBTYPES:
            n = int(agg[f"rsna_{s}"].sum())
            med = agg.loc[agg[f"rsna_{s}"] > 0, f"rsna_{s}_n_pos_slices"].median()
            print(f"    rsna_{s:<18s} {n:6,} positive series ({n/max(len(agg),1):5.1%})"
                  + (f", median {med:.0f} positive slice(s) each" if n else ""))
        return agg


    # The cache key is the INDEX it was built from, not just "a file exists".
    #
    # Cell B2 is explicitly allowed to finish partially (a time budget, or an interrupt) and
    # execution continues straight into this cell, which would then cache labels for whatever
    # fraction of the index existed at that moment. Next session B2 completes -- and this cell
    # loads the stale cache. B4 inner-joins on it, so the manifest can never contain a series
    # indexed after the cache was written. With RSNA_MAX_SERIES subsampling both cases down to the
    # same number, the printed headline count is IDENTICAL either way, which is what makes the
    # data loss invisible. A 10-of-38-chunk first session would silently cap the whole project at
    # ~5,800 of 21,700 series.
    RSNA_LABEL_CACHE_META = RSNA_LABEL_CACHE + ".meta.json"

    def _label_cache_key():
        return {
            "n_index_rows": int(len(RSNA_INDEX)),
            "n_index_series": int(RSNA_INDEX["series_uid"].nunique()) if len(RSNA_INDEX) else 0,
            "index_complete": bool(globals().get("RSNA_INDEX_COMPLETE", False)),
            # A walk that resynchronised around damaged records reached the end of the archive
            # but stepped over members. That is a DIFFERENT index from a clean one with the same
            # row count, so it must not share a cache entry with it.
            "index_resyncs": int(globals().get("RSNA_SCAN_RESYNCS", 0)),
            "splits": list(RSNA_SPLITS),
            "label_csv": RSNA_LABEL_CSV_PATH or RSNA_LABEL_CSV_NAME,
            "schema_version": 3,
        }

    _key = _label_cache_key()
    _cached_ok = False
    if os.path.exists(RSNA_LABEL_CACHE) and os.path.exists(RSNA_LABEL_CACHE_META) \
            and not RSNA_REBUILD_LABELS:
        try:
            with open(RSNA_LABEL_CACHE_META) as _f:
                _cached_ok = (json.load(_f) == _key)
        except Exception:
            _cached_ok = False
        if not _cached_ok:
            print("the series-label cache was built from a DIFFERENT header index "
                  "(cell B2 has since indexed more, or the config changed) -- rebuilding.")

    if _cached_ok:
        RSNA_SERIES_LABELS = pd.read_parquet(RSNA_LABEL_CACHE)
        print(f"loaded cached series labels: {len(RSNA_SERIES_LABELS):,} series "
              f"(set RSNA_REBUILD_LABELS = True to force a rebuild)")
    else:
        _slice_labels = load_rsna_slice_labels()
        RSNA_SERIES_LABELS = aggregate_labels_to_series(_slice_labels, RSNA_INDEX)
        RSNA_SERIES_LABELS.to_parquet(RSNA_LABEL_CACHE, index=False)
        with open(RSNA_LABEL_CACHE_META, "w") as _f:
            json.dump(_key, _f, indent=2)
        print(f"cached to {RSNA_LABEL_CACHE}")

    if not globals().get("RSNA_INDEX_COMPLETE", False):
        print()
        print("*** The header index is INCOMPLETE. These labels, and the manifest built from them,")
        print("    cover only the slices indexed so far. Finish cell B2 before reading any result")
        print("    as covering the dataset. ***")

In [ ]:
# B4. RSNA manifest -- same schema the India path produces  [RSNA]
#
# From here on every downstream cell is shared. The manifest carries the same columns; `zip_key`
# and `zip_path` name the container (there is only one, the split archive) instead of a day-zip.
# The per-series member list lives in RSNA_MEMBERS_BY_SERIES, which stands in for the zip locator.

if DATA_SOURCE != "rsna":
    print("DATA_SOURCE is {!r} -- skipping this cell (Part B / RSNA only).".format(DATA_SOURCE))
else:
    import numpy as np
    import pandas as pd

    RSNA_MANIFEST_PATH = os.path.join(OUTPUT_DIR, "rsna_manifest.csv")


    def _slice_normal_from_iop(iop_str):
        """Unit normal of the imaging plane = cross(row cosines, column cosines).

        Named for its argument: cell 13 has `_slice_normal_from_ds(ds)`, which does the same
        arithmetic from a pydicom Dataset. They used to share the name `_slice_normal` and
        silently overwrite each other -- see the note in cell 13."""
        if not iop_str:
            return None
        try:
            v = [float(t) for t in iop_str.split(",")]
        except Exception:
            return None
        if len(v) != 6:
            return None
        n = np.cross(np.array(v[:3]), np.array(v[3:6]))
        norm = np.linalg.norm(n)
        if not np.isfinite(norm) or norm < 1e-6:
            return None
        return n / norm


    def _slice_order_key(sub):
        """Order slices along the acquisition axis: position projected onto the slice normal.

        Sorting on raw ImagePositionPatient[2] is the usual shortcut, and it breaks whenever the
        slice normal is not the patient z-axis -- a tilted gantry, or a reformat. The projection
        pos . n is the physical distance along the stack and is correct in every orientation; for
        an ordinary axial study (IOP = 1,0,0,0,1,0, n = (0,0,1)) it reduces exactly to raw z.

        An earlier version returned `z * n[2]`, which LOOKED like a projection and was not: it
        drops the x and y terms, which are precisely the ones a tilt contributes, so it was a
        monotone rescaling of z (a no-op) -- and for n_z < 0 it silently REVERSED the stack, while
        for a coronal plane (n_z = 0) it collapsed to a single constant, leaving the slices in
        input order while still reporting "ordered by slice normal".

        Returns (key, source). `source` says honestly which path was taken.
        """
        n = _slice_normal_from_iop(sub["iop"].iloc[0] if len(sub) else "")
        if n is None:
            z = sub["z"].to_numpy(dtype=float)
            if np.isnan(z).any():
                return None, "filename (no usable position or orientation)"
            return z, "raw z (no ImageOrientationPatient)"
        pos = np.column_stack([
            sub.get("x", pd.Series(np.zeros(len(sub)))).to_numpy(dtype=float),
            sub.get("y", pd.Series(np.zeros(len(sub)))).to_numpy(dtype=float),
            sub["z"].to_numpy(dtype=float),
        ])
        if np.isnan(pos).any():
            z = sub["z"].to_numpy(dtype=float)
            if np.isnan(z).any():
                return None, "filename (no usable position or orientation)"
            return z, "raw z (incomplete ImagePositionPatient)"
        key = pos @ n
        if np.ptp(key) < 1e-6:
            # Every slice projects to the same point -- the normal cannot be the stacking axis.
            return None, "filename (degenerate projection: normal is not the stacking axis)"
        return key, "slice normal"


    def build_rsna_members_by_series(index):
        """series_uid -> ordered member list, plus per-series geometry."""
        by_series, geom = {}, []
        skipped_corrupt = 0
        idx = index[~index["sop"].isin(RSNA_CORRUPT_SOPS)]
        skipped_corrupt = len(index) - len(idx)
        for suid, sub in idx.groupby("series_uid", sort=False):
            key, order_src = _slice_order_key(sub)
            if key is None:
                sub = sub.sort_values("member")
            else:
                sub = sub.iloc[np.argsort(key, kind="stable")]
            by_series[suid] = sub["member"].tolist()
            # Spacing is measured along the ORDERING axis, not along z -- for a tilted gantry
            # those differ, and it is the former that the resample needs.
            if key is not None:
                axis = np.sort(np.asarray(key, dtype=float))
            else:
                axis = sub["z"].to_numpy(dtype=float)
            zs = sub["z"].to_numpy(dtype=float)
            gaps = np.abs(np.diff(axis)) if len(axis) > 1 else np.array([])
            gaps = gaps[gaps > 1e-6]
            geom.append({
                "series_uid": suid,
                "n_slices": len(sub),
                "z_span": float(np.nanmax(zs) - np.nanmin(zs)) if len(zs) > 1 else 0.0,
                "sz_measured": float(np.median(gaps)) if gaps.size else np.nan,
                "px_x": float(sub["px_x"].iloc[0]), "px_y": float(sub["px_y"].iloc[0]),
                "rows": int(sub["rows"].iloc[0]), "cols": int(sub["cols"].iloc[0]),
                "n_needs_pixel_fix": int(sub["needs_pixel_fix"].sum()),
                "order_source": order_src,
            })
        if skipped_corrupt:
            print(f"  excluded {skipped_corrupt} slice(s) of known-corrupt SOP(s) {RSNA_CORRUPT_SOPS}")
        return by_series, pd.DataFrame(geom)


    def build_rsna_manifest():
        members_by_series, geom = build_rsna_members_by_series(RSNA_INDEX)
        m = RSNA_SERIES_LABELS.merge(geom.drop(columns=["n_slices"]), on="series_uid", how="inner")

        before = len(m)
        m = m[m["n_slices"] >= MIN_SLICES_PER_SERIES]
        print(f"series with >= {MIN_SLICES_PER_SERIES} slices: {len(m):,} of {before:,}")

        _byorder = geom["order_source"].value_counts()
        print("  slice ordering source:")
        for _src, _n in _byorder.items():
            print(f"    {_n:6,}  {_src}")
        _fallback = int((~geom["order_source"].str.startswith(("slice normal", "raw z"))).sum())
        if _fallback:
            print(f"  {_fallback:,} series fell back to filename order -- their through-plane "
                  f"spacing is an estimate, and their slice order is not physically verified")

        out = pd.DataFrame({
            "series_uid": m["series_uid"],
            "study_uid": m["study_uid"],
            "patient_id": m["patient_id"],
            # RSNA's PatientID is a de-identification hash assigned once per person, so unlike
            # the India export -- where PatientID is a per-visit registration counter and had to
            # be replaced (cell 9) -- it is already a valid grouping key. The column is emitted
            # under the same name so cell 21 groups both datasets the same way and neither has a
            # silent fallback path.
            "person_key": m["patient_id"],
            "accession": "",
            "zip_month": "rsna", "zip_day": "rsna", "zip_key": "rsna",
            "zip_path": RSNA_PARTS_DIR, "zip_exists": True,
            "region": "head", "head_subtype": "brain",
            "series_desc": "RSNA-ICH axial head CT", "body_part": "HEAD",
            "is_reformat": False,
            "n_slices_expected": m["n_slices"].astype(int),
            "slice_thickness": m["sz_measured"],
            "is_soft_kernel": True,
            "priority_rank": 0, "is_primary": True,
            "studytype_agrees": True,
            "matched_study_type": "RSNA-ICH", "match_method": "rsna_slice_labels",
            "study_date": "", "patient_sex": "", "patient_age": "",
        })
        for c in RSNA_SERIES_LABELS.columns:
            if c.startswith("rsna_"):
                out[c] = m[c].to_numpy()

        # ---- optional stratified subsample ---------------------------------------------------
        # ~21.7k volumes through Pillar-0 is a multi-night run. A PROPORTIONAL stratified sample on
        # the full 6-bit label pattern keeps every subtype's prevalence intact, which is what makes
        # average precision and PPV mean anything. Oversampling the rare classes would give prettier
        # AUROCs for epidural and a prevalence that matches no real population.
        if RSNA_MAX_SERIES is not None and len(out) > RSNA_MAX_SERIES:
            pat = out[[f"rsna_{s}" for s in RSNA_SUBTYPES]].astype(str).agg("".join, axis=1)
            frac = RSNA_MAX_SERIES / len(out)
            rng = np.random.default_rng(RANDOM_SEED)
            keep = []
            for _p, g in out.groupby(pat, sort=True):
                k = int(round(len(g) * frac))
                k = min(len(g), max(k, 1 if len(g) else 0))
                keep.append(g.iloc[rng.permutation(len(g))[:k]])
            out = pd.concat(keep).sort_values("series_uid").reset_index(drop=True)
            print(f"\nstratified subsample: {len(out):,} of {len(m):,} series "
                  f"(RSNA_MAX_SERIES={RSNA_MAX_SERIES}), prevalence preserved per label pattern")

        out = out.sort_values("series_uid").reset_index(drop=True)
        out.to_csv(RSNA_MANIFEST_PATH, index=False)
        return out, members_by_series


    manifest, RSNA_MEMBERS_BY_SERIES = build_rsna_manifest()
    manifest_runnable = manifest.copy()

    print()
    print(f"RSNA manifest: {len(manifest):,} series, "
          f"{manifest['study_uid'].nunique():,} studies, "
          f"{manifest['patient_id'].nunique():,} patients -> {RSNA_MANIFEST_PATH}")
    print(f"slices per series: median {manifest['n_slices_expected'].median():.0f}, "
          f"range {manifest['n_slices_expected'].min()}-{manifest['n_slices_expected'].max()}")
    print()
    print("label prevalence in the manifest actually being run:")
    for _s in RSNA_SUBTYPES:
        _c = f"rsna_{_s}"
        if _c in manifest.columns:
            _n = int(manifest[_c].sum())
            _flag = "   <-- below MIN_POSITIVES, will be skipped" if _n < MIN_POSITIVES else ""
            print(f"  {_c:<28s} {_n:6,}  ({_n/max(len(manifest),1):5.1%}){_flag}")
    print()
    print("Sanity check before you spend GPU hours: after resampling to 1.25 mm through-plane, a")
    print("20-slice 5 mm series becomes 80 slices and a 40-slice 3 mm series becomes 96 -- both under")
    print("the 128 Pillar-0 expects, so most RSNA volumes are PADDED in z rather than cropped. That is")
    print("a real distribution difference from the Indian data (median 185 slices at 2 mm), and it is")
    print("the first thing to suspect if RSNA results come out systematically lower.")
    _est = (manifest["n_slices_expected"] * manifest["slice_thickness"].fillna(4.0) / 1.25)
    print(f"estimated resampled depth: median {_est.median():.0f} slices, "
          f"{(_est < 128).mean():.0%} of series below 128 (padded), "
          f"{(_est > 128).mean():.0%} above (centre-cropped)")

In [ ]:
# B5. Pull the wanted series out of the archive in ONE forward sweep  [RSNA]
#
# ===============================================================================================
# THE SECOND HALF OF THE SAME ARITHMETIC
# ===============================================================================================
# Cell B1c fixed the INDEX pass: walk local file headers forward instead of seeking to 752,803
# central-directory offsets. That leaves the EXTRACTION pass, which had exactly the same defect
# and would have been discovered the same way -- at 3am, having burned the night.
#
# Volume assembly (cells 17-19) asks the container for one member at a time, by name. On the
# India day-zips that is fine: a day-zip is a few GB on Drive with a working directory. On the
# RSNA archive it is the 9.5-seconds-per-member access pattern again, and RSNA_MAX_SERIES = 4000
# series x ~30 slices = ~120,000 members = 13 DAYS.
#
# So the members are extracted first, in one sweep, in ASCENDING OFFSET ORDER -- which is
# sequential access with skips, the thing Drive is good at -- and written to local disk. Every
# cell after this one reads local files at local-disk speed and does not know the difference.
# This is what every published solution to this dataset does: SeuTao, appian42, Anjum48 and
# shimacos37 all digest the DICOMs once, up front, and never touch the originals again.
#
# The offsets used here are the TRUE ones the forward walk measured, not the wrapped values in
# the central directory, so no repair is involved and none is needed.
#
# COST. ~120,000 members at ~250 KB each is ~30 GB, against ~200 GB of local scratch. The budget
# is checked before a byte is written, and if it does not fit the MANIFEST is trimmed (keeping
# label prevalence) rather than the extraction failing halfway.
# ===============================================================================================

if DATA_SOURCE != "rsna":
    print("DATA_SOURCE is {!r} -- skipping this cell (Part B / RSNA only).".format(DATA_SOURCE))
else:
    import hashlib
    import json
    import os
    import shutil
    import time
    import zlib

    import numpy as np
    import pandas as pd

    RSNA_LOCAL_DICOM_DIR = os.path.join(LOCAL_ZIP_CACHE, "rsna_dicom")
    RSNA_LOCAL_STATE_PATH = os.path.join(RSNA_LOCAL_DICOM_DIR, "_state.json")


    def _rsna_local_name(member):
        """A flat, collision-free local filename for an archive member.

        The archive is a flat folder of ID_<hex>.dcm, so the basename is already unique -- but
        that is a property of THIS dataset, not of the code, so the full member path is encoded
        rather than assumed. os.sep is not legal in a filename, which is what makes it safe.
        """
        return member.replace("/", "__").replace("\\", "__")


    # ---- 256-way fan-out, because 120,000 files in one directory is a cliff ---------------------
    #
    # ext4 with dir_index handles millions of entries per directory for LOOKUP by name -- that
    # part is fine. What is not fine is any linear scan: readdir returns entries in HASH order,
    # which is uncorrelated with inode order, so a listdir-then-stat loop over 120,000 files
    # generates near-random inode-table I/O, and the directory blocks themselves grow to tens of
    # megabytes that every glob, ls or tab-completion re-reads.
    #
    # Two hex characters of a hash gives 256 subdirectories of ~470 files each. It is what git,
    # ccache and every object store does, it costs one mkdir, and it removes the whole class of
    # problem. The layout is derived from the member name, so it needs no index of its own and
    # the resume check stays a single os.path.getsize.
    RSNA_LOCAL_FANOUT = 256


    def _rsna_local_path(out_dir, member):
        flat = _rsna_local_name(member)
        shard = hashlib.sha1(flat.encode()).hexdigest()[:2]
        return os.path.join(out_dir, shard, flat)


    def _rsna_local_shards(out_dir):
        return [f"{i:02x}" for i in range(RSNA_LOCAL_FANOUT)]


    def _local_free_gb(path):
        """Free space at `path`, creating it first so statvfs cannot fail on a missing directory.

        This used to be called `_free_gb(path)` -- the same name cell 2d gives its own
        `_free_gb(path="/content")`. Cell B5 is defined LATER, so it silently replaced cell 2d's,
        and every call cell 2d made to `_free_gb()` with no argument then raised
        `TypeError: _free_gb() missing 1 required positional argument`. That is exactly what
        killed the first real RSNA extraction, four hours into the night, from inside a health
        check whose entire job was to keep the run alive."""
        os.makedirs(path, exist_ok=True)
        # Cross-cell helpers are looked up, never assumed -- the same pattern this notebook uses
        # for ensure_zip_readable, drive_health_check and heartbeat. A cell that raises NameError
        # because an earlier cell was skipped is a cell that cannot be run on its own, and "run
        # every cell in order or it breaks" is not a property worth having in a notebook that is
        # resumed as often as this one.
        _f = globals().get("_free_gb")
        if _f is not None:
            return _f(path)
        st = os.statvfs(path)
        return st.f_bavail * st.f_frsize / 1e9


    class LocalDicomContainer:
        """The same duck-type as ZipMemberOpener / TarContainer, over a local directory."""

        def __init__(self, root):
            self._root = root

        def namelist(self):
            # Deliberately walks the fan-out rather than listing one huge directory. Nothing in
            # the RSNA path actually calls this -- the member list comes from the header index --
            # but a duck-type that lies about its contents is worse than one that is slow.
            out = []
            for shard in os.listdir(self._root):
                d = os.path.join(self._root, shard)
                if os.path.isdir(d):
                    out.extend(f for f in os.listdir(d) if not f.startswith("_"))
            return out

        def open(self, name, max_bytes=None):
            return open(_rsna_local_path(self._root, name), "rb")

        def close(self):
            pass


    def rsna_materialise(index, wanted_members, out_dir, deadline=None):
        """Copy `wanted_members` out of the archive to `out_dir`, in ascending offset order.

        Resumable at member granularity: a member whose local file already has the expected
        uncompressed size is skipped. Partial writes go to `.tmp` and are renamed, so an
        interrupted write is never mistaken for a finished one -- the failure mode that produced
        "extraction complete, 0 series" the first time round.
        """
        os.makedirs(out_dir, exist_ok=True)
        rows = index[index["member"].isin(wanted_members)].copy()
        missing = set(wanted_members) - set(rows["member"])
        if missing:
            print(f"  {len(missing):,} wanted member(s) are not in the header index and cannot "
                  f"be located -- their series will be short and are reported as such.")
        if "offset" not in rows.columns or (rows["offset"] < 0).any():
            raise RuntimeError(
                "the header index carries no true member offsets. That means it was built by the "
                "old random-access path (or a tar). Delete the index directory and re-run cell B2, "
                "which walks the archive forward and records real positions."
            )
        rows = rows.sort_values("offset", kind="stable").reset_index(drop=True)

        need_gb = float(rows["usize"].sum()) / 1e9
        free_gb = _local_free_gb(out_dir)
        print(f"  {len(rows):,} member(s) to extract, {need_gb:.1f} GB uncompressed; "
              f"{free_gb:.1f} GB free on local disk")
        if need_gb * 1.10 > free_gb:
            raise RuntimeError(
                f"not enough local scratch: {need_gb:.1f} GB needed (plus 10% headroom) against "
                f"{free_gb:.1f} GB free. Lower RSNA_MAX_SERIES (currently {RSNA_MAX_SERIES}) -- "
                f"the sample stays proportional, so prevalence is unaffected."
            )

        # The fan-out directories are created only AFTER the budget check above has passed.
        # Creating them first meant a run that was refused for lack of disk still left 256
        # directories behind -- harmless in itself, and exactly the kind of "it did something
        # before deciding not to" that makes a refusal hard to trust.
        for _sh in _rsna_local_shards(out_dir):
            os.makedirs(os.path.join(out_dir, _sh), exist_ok=True)

        paths = _RSNA_STATE["paths"]
        n_done = n_new = n_failed = 0
        t0 = time.time()
        bytes_written = 0
        # The same buffered forward window the index pass uses, for the same reason: the members
        # are visited in ascending offset order, so the gaps between wanted ones are small and
        # reading through them costs less than a round trip. seek_threshold is where that flips
        # -- a gap bigger than it is jumped rather than streamed, which matters when only a small
        # subsample of series is being taken.
        with MultiPartFile(paths) as _raw:
            mpf = ForwardWindow(_raw, block=8 << 20, seek_threshold=16 << 20)
            for i, r in enumerate(rows.itertuples(index=False)):
                dst = _rsna_local_path(out_dir, r.member)
                try:
                    if os.path.getsize(dst) == int(r.usize):
                        n_done += 1
                        continue
                except OSError:
                    pass
                try:
                    mpf.seek(int(r.offset))
                    raw = mpf.read(int(r.csize))
                    blob = (raw if int(r.method) == 0
                            else zlib.decompressobj(-zlib.MAX_WBITS).decompress(raw))
                    if int(r.usize) and len(blob) != int(r.usize):
                        raise ValueError(f"got {len(blob)} bytes, header said {int(r.usize)}")
                    tmp = dst + ".tmp"
                    with open(tmp, "wb") as fh:
                        fh.write(blob)
                    os.replace(tmp, dst)
                    n_new += 1
                    bytes_written += len(blob)
                except Exception as e:
                    n_failed += 1
                    if n_failed <= 5:
                        print(f"    {r.member}: {type(e).__name__}: {e}")
                mpf.advance(int(r.offset) + int(r.csize))
                if (i + 1) % 2000 == 0:
                    # This pass writes tens of gigabytes to the same disk the DriveFS cache is
                    # filling, so it is the one most likely to run out. Checked often, and it
                    # raises rather than failing partway through a write -- every member already
                    # on disk is kept, so a stop here resumes.
                    _hc = globals().get("drive_health_check")
                    if _hc is not None:
                        _hc(probe_path=paths[0], verbose=True)
                    _hb = globals().get("heartbeat")
                    if _hb is not None:
                        _hb("B5 extracting slices to local disk",
                            f"{i+1:,}/{len(rows):,}", every_s=60,
                            gb_written=round(bytes_written / 1e9, 1))
                if (i + 1) % 5000 == 0:
                    el = time.time() - t0
                    frac = (i + 1) / len(rows)
                    print(f"    {i+1:,}/{len(rows):,} ({frac:.1%}), {bytes_written/1e9:.1f} GB "
                          f"written, {mpf.bytes_read/1e9:.1f} GB read in {mpf.n_reads:,} "
                          f"request(s), {el/60:.1f} min"
                          + (f", ~{el/frac*(1-frac)/60:.0f} min left" if frac > 0.01 else ""),
                          flush=True)
                if deadline is not None and time.time() > deadline:
                    print(f"    time budget reached at {i+1:,}/{len(rows):,} -- re-run this cell; "
                          f"finished members are skipped, nothing is re-read.")
                    break
        el = time.time() - t0
        print(f"  extracted {n_new:,} new, {n_done:,} already local, {n_failed:,} failed "
              f"in {el/60:.1f} min ({bytes_written/1e6/max(el,1e-6):.0f} MB/s written)")
        print(f"    archive traffic: {mpf.bytes_read/1e9:.1f} GB in {mpf.n_reads:,} request(s) "
              f"and {mpf.n_seeks:,} seek(s) -- "
              f"{mpf.n_reads/max(len(rows),1):.3f} request(s) per member")
        return {"n_new": n_new, "n_cached": n_done, "n_failed": n_failed,
                "n_wanted": len(rows), "n_missing_from_index": len(missing)}


    _fmt = _RSNA_STATE.get("fmt")
    if _fmt != "zip":
        # A tar's member index is exact and its reads are ordinary seeks into a file, so there is
        # nothing here to fix. Left explicit rather than implied, so the skip is visible in the
        # log instead of looking like the cell did nothing.
        RSNA_LOCAL_READY = False
        print(f"the archive is {_fmt!r}, not a zip -- members are read directly and no local "
              f"extraction pass is needed.")
    elif not len(manifest):
        RSNA_LOCAL_READY = False
        print("the manifest is empty -- nothing to extract.")
    else:
        _wanted = sorted({m for uid in manifest["series_uid"]
                          for m in RSNA_MEMBERS_BY_SERIES.get(uid, [])})
        print(f"materialising {len(_wanted):,} slice(s) for {len(manifest):,} series to "
              f"{RSNA_LOCAL_DICOM_DIR}")
        _dl = (time.time() + MAX_SECONDS_PER_RUN) if MAX_SECONDS_PER_RUN else None
        _res = rsna_materialise(RSNA_INDEX, _wanted, RSNA_LOCAL_DICOM_DIR, deadline=_dl)
        _have = _res["n_new"] + _res["n_cached"]
        RSNA_LOCAL_READY = (_have >= _res["n_wanted"] and _res["n_failed"] == 0)
        with open(RSNA_LOCAL_STATE_PATH, "w") as _f:
            json.dump({"complete": bool(RSNA_LOCAL_READY), **_res,
                       "n_series": int(len(manifest))}, _f, indent=2)
        if RSNA_LOCAL_READY:
            print(f"\nlocal DICOM cache COMPLETE: {_have:,} slice(s). Extraction reads local "
                  f"files from here on -- the archive is not touched again.")
        else:
            # Do NOT let a partial cache masquerade as a finished one. A series missing a third
            # of its slices still assembles into a volume, still embeds, and still produces a
            # number -- one that quietly means nothing.
            print(f"\n*** The local DICOM cache is INCOMPLETE ({_have:,} of "
                  f"{_res['n_wanted']:,} slice(s), {_res['n_failed']:,} failed). Re-run this "
                  f"cell to finish. Extraction below will fall back to reading the archive "
                  f"directly, which on this archive is prohibitively slow -- so finish this "
                  f"first. ***")
        # Series that are not fully local are dropped from the runnable manifest rather than
        # silently embedded short.
        if not RSNA_LOCAL_READY:
            _ok_uids = []
            for _uid in manifest["series_uid"]:
                _ms = RSNA_MEMBERS_BY_SERIES.get(_uid, [])
                if _ms and all(os.path.exists(_rsna_local_path(RSNA_LOCAL_DICOM_DIR, _m))
                               for _m in _ms):
                    _ok_uids.append(_uid)
            manifest_runnable = manifest[manifest["series_uid"].isin(_ok_uids)].reset_index(
                drop=True)
            print(f"    {len(manifest_runnable):,} of {len(manifest):,} series are fully local "
                  f"and will be run; the rest wait for the next pass.")

In [ ]:
# B6. Check the index against an INDEPENDENT extraction, and against the label  [RSNA]
#
# ===============================================================================================
# WHY THIS CELL EXISTS
# ===============================================================================================
# Cell B2 builds the header index by walking local file headers and decompressing a 16 KB prefix
# of each member. That is an unusual way to read a zip, it is the load-bearing step of the whole
# RSNA path, and until now the only thing checking it was this notebook's own tests against
# synthetic archives. "My code agrees with my code" is not verification.
#
# There is a published, CC0 extraction of the same headers, made by an entirely different method
# (pydicom over the unzipped files) by someone with no connection to this notebook:
#
#     kaggle datasets download carlolepelaars/rsna-stage-2-metadata-ihd-2019
#     -> stage_2_train_with_metadata.csv   (~266 MB)
#
# Joining it to RSNA_INDEX on SOPInstanceUID and comparing every geometry field is the strongest
# check available: two independent readings of 752,803 files either agree or they do not. Set
# RSNA_METADATA_XCHECK_PATH to that CSV and this cell runs; leave it empty and it says how to get
# it and moves on. It is not required for the run.
#
# NOTE on that dataset: its ID / Label / Sub_type columns come from a merge on PatientID against
# an unpivoted long-format label frame, which is not a sound join. The DICOM columns are what is
# used here; the labels are taken from stage_2_train.csv as always.
#
# ===============================================================================================
# THE SECOND HALF: does the GEOMETRY predict the label?
# ===============================================================================================
# A public kernel (tomooinubushi, "image position in DICOM has predictive information") showed
# that on this dataset slice Z-POSITION carries label information: hemorrhages sit near the middle
# of the volume and the top and bottom slices are usually normal, because patients are positioned
# with the suspicious region at the centre of the field of view.
#
# That is a per-slice finding. The study-level analogue is the one that matters here: if scan
# LENGTH, slice COUNT, or field of view differ between hemorrhage and non-hemorrhage studies --
# because a trauma protocol is not a headache protocol -- then those numbers carry label
# information on their own, and part of any AUROC this notebook reports is the model reading
# acquisition protocol rather than anatomy.
#
# So each one is scored on its own, as a univariate AUROC, and printed next to the model's. This
# is not a fix; it is a number that has to be on the table when the result is read.

if DATA_SOURCE != "rsna":
    print("DATA_SOURCE is {!r} -- skipping this cell (Part B / RSNA only).".format(DATA_SOURCE))
elif not len(globals().get("RSNA_INDEX", [])):
    print("no header index in memory -- run cell B2 first.")
else:
    import numpy as np
    import pandas as pd

    # ---- 1. cross-check against the published extraction ---------------------------------------
    _xp = globals().get("RSNA_METADATA_XCHECK_PATH", "")
    if not _xp:
        print("index cross-check SKIPPED -- RSNA_METADATA_XCHECK_PATH is empty.")
        print("  To run it, put the CC0 published extraction on Drive and point at it:")
        print("    kaggle datasets download -d carlolepelaars/rsna-stage-2-metadata-ihd-2019")
        print("  It is an independent reading of the same 752,803 headers, so agreeing with it")
        print("  is real evidence that the forward walk read the archive correctly. Not required.")
    elif not os.path.exists(_xp):
        print(f"index cross-check SKIPPED -- {_xp} does not exist.")
    else:
        print(f"cross-checking the header index against {os.path.basename(_xp)} ...")
        _use = ["SOPInstanceUID", "StudyInstanceUID", "SeriesInstanceUID", "PatientID",
                "Rows", "Columns", "BitsStored", "PixelRepresentation",
                "RescaleIntercept", "RescaleSlope",
                "ImagePositionPatient_1", "ImagePositionPatient_2", "ImagePositionPatient_3",
                "PixelSpacing_1", "PixelSpacing_2"] + \
               [f"ImageOrientationPatient_{i}" for i in range(1, 7)]
        try:
            _head = pd.read_csv(_xp, nrows=5)
            _cols = [c for c in _use if c in _head.columns]
            _ref = pd.read_csv(_xp, usecols=_cols, low_memory=False)
        except Exception as _e:
            _ref = None
            print(f"  could not read it ({type(_e).__name__}: {_e}) -- skipping the cross-check.")
        if _ref is not None and "SOPInstanceUID" not in _ref.columns:
            # A file that is not the expected extraction must SKIP, not crash. This cell is an
            # optional verification; it has no business ending a run because someone pointed it
            # at the wrong CSV.
            print(f"  {os.path.basename(_xp)} has no SOPInstanceUID column "
                  f"(found: {list(_ref.columns)[:6]}...) -- this is not the published extraction. "
                  f"Skipping the cross-check.")
            _ref = None
        if _ref is not None:
            _ref = _ref.drop_duplicates(subset="SOPInstanceUID", keep="first")
            _mine = RSNA_INDEX.drop_duplicates(subset="sop", keep="last")
            _j = _mine.merge(_ref, left_on="sop", right_on="SOPInstanceUID", how="inner")
            print(f"  {len(_ref):,} row(s) published, {len(_mine):,} indexed here, "
                  f"{len(_j):,} joined on SOPInstanceUID")
            _only_mine = len(_mine) - len(_j)
            _only_ref = len(_ref) - len(_j)
            if _only_mine or _only_ref:
                print(f"  {_only_mine:,} SOP(s) only in this index, {_only_ref:,} only in the "
                      f"published one. Some difference is expected -- the published extraction "
                      f"covers stage_2_train and this run may be indexing a subset (RSNA_SPLITS "
                      f"= {RSNA_SPLITS}) -- but a large one-sided gap is worth understanding.")

            def _agree(mine_col, ref_col, tol=None):
                if ref_col not in _j.columns or mine_col not in _j.columns:
                    return None
                a, b = _j[mine_col], _j[ref_col]
                if tol is None:
                    ok = (a.astype(str).str.strip() == b.astype(str).str.strip())
                else:
                    a = pd.to_numeric(a, errors="coerce")
                    b = pd.to_numeric(b, errors="coerce")
                    ok = (a.isna() & b.isna()) | (np.abs(a - b) <= tol)
                return float(ok.mean()), int((~ok).sum())

            _checks = [("study_uid", "StudyInstanceUID", None),
                       ("series_uid", "SeriesInstanceUID", None),
                       ("patient_id", "PatientID", None),
                       ("rows", "Rows", 0), ("cols", "Columns", 0),
                       ("bits_stored", "BitsStored", 0),
                       ("pixel_repr", "PixelRepresentation", 0),
                       ("intercept", "RescaleIntercept", 1e-6),
                       ("slope", "RescaleSlope", 1e-6),
                       ("x", "ImagePositionPatient_1", 1e-4),
                       ("y", "ImagePositionPatient_2", 1e-4),
                       ("z", "ImagePositionPatient_3", 1e-4),
                       ("px_y", "PixelSpacing_1", 1e-6),
                       ("px_x", "PixelSpacing_2", 1e-6)]
            _worst, _all_ok = 1.0, True
            for _mc, _rc, _tol in _checks:
                _r = _agree(_mc, _rc, _tol)
                if _r is None:
                    print(f"    {_mc:<14s} (not in the published file)")
                    continue
                _frac, _n_bad = _r
                _worst = min(_worst, _frac)
                _flag = "" if _frac >= 0.9999 else f"   <-- {_n_bad:,} DISAGREE"
                print(f"    {_mc:<14s} agrees on {_frac:7.4%} of joined rows{_flag}")
                if _frac < 0.9999:
                    _all_ok = False
                    _bad = _j.loc[_j[_mc].astype(str) != _j[_rc].astype(str),
                                  ["member", _mc, _rc]].head(3)
                    for _, _row in _bad.iterrows():
                        print(f"      e.g. {str(_row['member'])[-28:]}: "
                              f"index={_row[_mc]!r} published={_row[_rc]!r}")
            # ImageOrientationPatient is stored here as one string; compare component-wise.
            if all(f"ImageOrientationPatient_{i}" in _j.columns for i in range(1, 7)):
                _iop_ref = _j[[f"ImageOrientationPatient_{i}" for i in range(1, 7)]].apply(
                    lambda r: ",".join(f"{float(v):.6g}" for v in r), axis=1)
                _iop_ok = float((_j["iop"] == _iop_ref).mean())
                _worst = min(_worst, _iop_ok)
                print(f"    {'iop':<14s} agrees on {_iop_ok:7.4%} of joined rows"
                      + ("" if _iop_ok >= 0.9999 else "   <-- DISAGREE"))
                _all_ok = _all_ok and _iop_ok >= 0.9999
            print()
            if _all_ok and len(_j) > 0:
                print(f"  CROSS-CHECK PASSED. Every geometry field agrees with an independent "
                      f"extraction on all {len(_j):,} joined slices. The forward walk read the "
                      f"archive correctly -- this is evidence from outside this notebook, not "
                      f"from its own tests.")
            elif len(_j):
                print(f"  *** CROSS-CHECK FAILED on at least one field (worst agreement "
                      f"{_worst:.4%}). Do NOT read any result from this index until that is "
                      f"understood: a disagreement here means the forward walk and pydicom "
                      f"disagree about what is in the archive. ***")
            globals()["RSNA_XCHECK_AGREEMENT"] = float(_worst)

    # ---- 2. does the geometry alone predict the label? -----------------------------------------
    _lab = globals().get("RSNA_SERIES_LABELS")
    if _lab is None or not len(_lab):
        print("\ngeometry-vs-label check skipped -- cell B3 has not run.")
    else:
        print("\nDOES THE SCAN GEOMETRY ALONE PREDICT HEMORRHAGE?")
        print("  A public kernel showed slice Z-position carries label information on this")
        print("  dataset: bleeds sit near the middle of the volume because patients are")
        print("  positioned that way. The study-level version of that question is whether scan")
        print("  length, slice count or field of view differ between positive and negative")
        print("  studies -- if they do, part of any AUROC below is acquisition protocol.")
        # Prefixed, because RSNA_SERIES_LABELS already carries an `n_slices` of its own and a
        # colliding merge silently produces n_slices_x / n_slices_y -- so the line that reads
        # `_m["n_slices"]` raises, or worse, reads the wrong one.
        _g = RSNA_INDEX.groupby("series_uid")
        _geom = pd.DataFrame({
            "geom_n_slices": _g.size(),
            "geom_z_span": _g["z"].max() - _g["z"].min(),
            "geom_rows": _g["rows"].first(),
            "geom_px_x": _g["px_x"].first(),
        }).reset_index()
        _geom["geom_fov_mm"] = _geom["geom_rows"] * _geom["geom_px_x"]
        # Checked by set intersection, not by looking for _x/_y suffixes -- `geom_px_x` ends in
        # "_x" all by itself, and a collision test that fires on a legitimate column name is a
        # test that gets deleted the first time it is wrong.
        _clash = (set(_lab.columns) & set(_geom.columns)) - {"series_uid"}
        if _clash:
            raise RuntimeError(f"the geometry frame collides with label column(s) {sorted(_clash)} "
                               f"-- rename them before merging, or the wrong one is read")
        _m = _lab.merge(_geom, on="series_uid", how="inner")
        _y = (pd.to_numeric(_m.get("rsna_any"), errors="coerce").fillna(0) > 0).to_numpy()
        if _y.sum() < 20 or (~_y).sum() < 20:
            print(f"  too few of one class to measure ({int(_y.sum())} positive of {len(_y)}).")
        else:
            print(f"  {int(_y.sum()):,} positive of {len(_m):,} series\n")
            _flagged = []
            for _c, _lbl in (("geom_n_slices", "slice count"),
                             ("geom_z_span", "scan length (mm)"),
                             ("geom_fov_mm", "in-plane FOV (mm)"),
                             ("geom_px_x", "pixel spacing (mm)")):
                _v = pd.to_numeric(_m[_c], errors="coerce").to_numpy(dtype=float)
                _ok = np.isfinite(_v)
                if _ok.sum() < 50 or _y[_ok].sum() < 10:
                    continue
                # Mann-Whitney U as an AUROC: what does this ONE number achieve on its own?
                from scipy.stats import mannwhitneyu
                _a, _b = _v[_ok][_y[_ok]], _v[_ok][~_y[_ok]]
                _auc = mannwhitneyu(_a, _b, alternative="two-sided").statistic / (len(_a) * len(_b))
                _bad = abs(_auc - 0.5) > 0.06
                if _bad:
                    _flagged.append((_lbl, _auc))
                print(f"    {_lbl:<22s} positives median {np.median(_a):9.2f}  "
                      f"negatives {np.median(_b):9.2f}   AUROC {_auc:.3f}"
                      + ("   <-- CONFOUNDED" if _bad else ""))
            print()
            if _flagged:
                print("  *** At least one geometric number separates the classes on its own.")
                print("      Report it beside the model's AUROC: a classifier with access to the")
                print("      whole volume can read the same thing, and to that extent the result")
                print("      is measuring how the scan was acquired, not what is in the head. ***")
            else:
                print("  None of them separates the classes (all within 0.06 of chance), so the")
                print("  geometry is not carrying the label and the model's AUROC is not")
                print("  inflated by acquisition protocol.")

## Getting at the pixels inside a 25 GB day-zip

Two independent problems, solved separately below.

**Where is the series inside the archive?** Not by decoding headers. `ZipFile.namelist()` returns
the central directory — every member path, without reading a single member — and the series UID is
a path component (`.../<StudyUID>/<SeriesUID>/<file>`). Cell 12 matches on path components (never a
bare substring, which would let `…096` also swallow `…0961`), falls back to a substring pass only
for series that came up empty, and then **verifies** by reading exactly one member's header and
checking its `SeriesInstanceUID`. One header read per series, not per file.

The result is cached to `zip_index/<mm>_<dd>.json`, so a re-run of the same zip is instant.

**How do we read the bytes?** `ZIP_ACCESS_MODE` picks:

* `"stream"` — read members directly off the Drive FUSE mount. Transfers only the wanted series
  (a few hundred MB of a 25 GB archive) but pays FUSE latency on every small member.
* `"local_copy"` — copy the whole archive to Colab local disk first, then read locally. Moves all
  25 GB but sequentially, and every member read after that is fast.

There is no universally right answer — it depends on your Drive throughput on the night. Cell 17
prints per-zip wall-clock split into *locate / read / preprocess / model*, so after one zip in each
mode you can decide from measurements. The local copy is resumable (`.part` file plus a size
check), reuses an existing complete copy, and refuses to start unless there is
`LOCAL_COPY_FREE_SPACE_FACTOR ×` the archive size free.

In [ ]:
# 11. Container access -- one interface over a Indian-hospital day-zip and the RSNA split archive
#
# Everything downstream only ever needs two things from a container: `.namelist()` and
# `.open(name)`. `zipfile.ZipFile` provides both natively; `TarContainer` (cell B1) duck-types
# them. So this layer hands back an object and the extraction loop never branches on which dataset
# it is looking at -- which is what makes "the same pipeline ran on both" a true statement rather
# than an aspiration.
import contextlib
import os
import shutil
import time
import zipfile

# Set by the extraction loop for the duration of one day-zip; None means "use ZIP_ACCESS_MODE".
_ACTIVE_ZIP_MODE = None


def _free_bytes(path):
    st = os.statvfs(path)
    return st.f_bavail * st.f_frsize


def ensure_local_copy(zip_path, cache_dir=None, chunk_mb=64):
    """Copy a day-zip to local disk, resumably. Returns the local path.

    Resumability here is deliberately conservative. A `.part` file from an interrupted copy is
    NOT resumed byte-wise (a torn write from a killed kernel is indistinguishable from a clean
    prefix) -- it is discarded and the copy restarted. What IS reused is a *complete* copy, which
    is verified by size against the source before being trusted.
    """
    cache_dir = cache_dir or LOCAL_ZIP_CACHE
    os.makedirs(cache_dir, exist_ok=True)
    src_size = os.path.getsize(zip_path)
    rel = os.path.relpath(zip_path, CT_ROOT).replace(os.sep, "__")
    dst = os.path.join(cache_dir, rel)
    part = dst + ".part"

    if os.path.exists(dst):
        if os.path.getsize(dst) == src_size:
            return dst
        print(f"    local copy of {rel} is the wrong size "
              f"({os.path.getsize(dst):,} vs {src_size:,}) -- recopying.")
        os.remove(dst)

    if os.path.exists(part):
        os.remove(part)

    need = int(src_size * LOCAL_COPY_FREE_SPACE_FACTOR)
    have = _free_bytes(cache_dir)
    if have < need:
        # Evict other cached zips (they are re-copyable) before giving up.
        for other in sorted(os.listdir(cache_dir)):
            if other == os.path.basename(dst):
                continue
            with contextlib.suppress(Exception):
                os.remove(os.path.join(cache_dir, other))
        have = _free_bytes(cache_dir)
    if have < need:
        raise RuntimeError(
            f"not enough local disk for {rel}: need ~{need/1e9:.1f} GB, have {have/1e9:.1f} GB. "
            f"Set ZIP_ACCESS_MODE = 'stream' to read straight from Drive instead."
        )

    # Retry the copy. A day-zip here is 9-22 GB and the read is a single sequential pass off a
    # FUSE-mounted Drive; a transient stall partway through is a real and recoverable event, and
    # the size check below is what turns it from silent corruption into a retry. This mirrors the
    # earlier notebook that extracted this same dataset successfully.
    t0 = time.time()
    chunk = chunk_mb * 1024 * 1024
    last_err = None
    for attempt in range(1, LOCAL_COPY_ATTEMPTS + 1):
        copied = 0
        try:
            with open(zip_path, "rb") as fsrc, open(part, "wb") as fdst:
                while True:
                    buf = fsrc.read(chunk)
                    if not buf:
                        break
                    fdst.write(buf)
                    copied += len(buf)
            if copied != src_size:
                raise IOError(f"short copy: {copied:,} of {src_size:,} bytes "
                              f"(truncated Drive transfer)")
            break
        except (OSError, IOError) as e:
            last_err = e
            with contextlib.suppress(Exception):
                os.remove(part)
            print(f"    copy attempt {attempt}/{LOCAL_COPY_ATTEMPTS} of {rel} failed: {e}")
            if attempt < LOCAL_COPY_ATTEMPTS:
                time.sleep(LOCAL_COPY_RETRY_SLEEP * attempt)
    else:
        raise RuntimeError(f"could not copy {rel} after {LOCAL_COPY_ATTEMPTS} attempt(s): {last_err}")
    os.replace(part, dst)
    dt = time.time() - t0
    print(f"    copied {rel} locally: {src_size/1e9:.1f} GB in {dt/60:.1f} min "
          f"({src_size/1e6/max(dt, 1e-6):.0f} MB/s)")
    return dst


def release_local_copy(zip_path):
    """Delete a cached local copy. Called once a day-zip is finished so the next one has room.

    Unconditional for India: it is called from the loop's `finally`, which cannot know whether
    this particular zip was streamed or copied (under "auto" that varies per zip), and deleting a
    file that was never created is already a no-op. Gating it on ZIP_ACCESS_MODE == "local_copy",
    as it once was, leaked every fallback copy and filled the disk after a few zips.
    A genuine no-op for RSNA, which is always streamed."""
    if DATA_SOURCE == "rsna":
        return
    rel = os.path.relpath(zip_path, CT_ROOT).replace(os.sep, "__")
    for p in (os.path.join(LOCAL_ZIP_CACHE, rel), os.path.join(LOCAL_ZIP_CACHE, rel + ".part")):
        with contextlib.suppress(Exception):
            if os.path.exists(p):
                os.remove(p)


@contextlib.contextmanager
def open_container(container_path):
    """Yields (container, effective_path).

    India: a day-zip, optionally copied to local disk first, opened and closed per call.
    RSNA: the single split archive, opened once per session and NOT closed here -- reopening it
    would re-read a very large zip central directory (or redo the tar offset walk) on every group.
    """
    if DATA_SOURCE == "rsna":
        # Cell B5 extracts every wanted slice to local disk in one forward sweep, precisely so
        # that this loop never seeks into the 194 GB archive. Reading members here through their
        # central-directory offsets measured 9.5 s EACH in cell B1b -- 120,000 members would be
        # 13 days -- so the local cache is not an optimisation, it is the only way this finishes.
        _local = globals().get("RSNA_LOCAL_DICOM_DIR")
        if globals().get("RSNA_LOCAL_READY") and _local and os.path.isdir(_local):
            yield LocalDicomContainer(_local), _local
            return
        if _local and os.path.isdir(_local):
            print("    WARNING: the local RSNA DICOM cache is INCOMPLETE (cell B5). Falling back "
                  "to reading the archive directly, which is prohibitively slow on this archive "
                  "-- finish cell B5 instead of leaving this running.")
        container, _mpf, _fmt = open_rsna_archive(verbose=False)
        yield container, container_path
        return

    effective = container_path
    # `_ACTIVE_ZIP_MODE` is set by the extraction loop just before each zip is opened, so a single
    # zip can be retried through a local copy without disturbing the run-wide setting. It is None
    # everywhere else, and then the configured mode applies.
    _mode = _ACTIVE_ZIP_MODE or ZIP_ACCESS_MODE
    if _mode == "auto":
        _mode = "stream"          # "auto" means: stream first, and the loop re-queues on failure
    if _mode == "local_copy":
        effective = ensure_local_copy(container_path)
    elif _mode != "stream":
        raise ValueError(f"zip access mode must be 'stream', 'local_copy' or 'auto', "
                          f"got {_mode!r}")
    _t_open = time.time()
    zf = zipfile.ZipFile(effective)
    _dt_open = time.time() - _t_open
    if _dt_open > 5:
        # Reading a 100,000-entry central directory off Drive is a genuinely slow operation with
        # no output of its own. Saying so turns "the cell looks hung" into "it is reading the
        # directory", which is the difference between waiting and interrupting.
        print(f"    read the central directory ({len(zf.infolist()):,} entries) in "
              f"{_dt_open:.0f}s")
    # Repair a 32-bit-overflowed central directory HERE, where the archive is opened, rather than
    # inside the locator.
    #
    # The locator returns early on a cached index, so a repair placed in it ran on the first pass
    # and was skipped on every resume -- the archive would come back unreadable the moment the
    # index cache was warm, which is precisely the run that matters. Opening is the one place
    # every reader goes through.
    # Cell 11b owns this; look it up rather than referencing it directly, so a harness that execs
    # this cell without cell 11b gets a quiet no-op instead of a NameError on every archive.
    _ensure = globals().get("ensure_zip_readable")
    if _ensure is not None:
        try:
            _ensure(zf, effective)
        except Exception as _e:
            print(f"    offset-repair check skipped: {type(_e).__name__}: {_e}")
    try:
        yield zf, effective
    finally:
        with contextlib.suppress(Exception):
            zf.close()


# Kept under the old name so nothing that referred to it breaks.
open_day_zip = open_container


if DATA_SOURCE == "rsna":
    print("container: the RSNA split archive (always streamed; local_copy does not apply)")
else:
    print(f"container: Indian-hospital day-zips, access mode {ZIP_ACCESS_MODE}")
if DATA_SOURCE != "rsna" and ZIP_ACCESS_MODE == "local_copy":
    print(f"  local cache   : {LOCAL_ZIP_CACHE} "
          f"({_free_bytes(LOCAL_ZIP_CACHE)/1e9:.1f} GB free)")

In [ ]:
# 11b. Repair day-zips whose central directory overflowed 32 bits
#
# ===============================================================================================
# WHAT IS ACTUALLY WRONG WITH THE ARCHIVES
# ===============================================================================================
# A probe on 09/03.zip (11.0 GB on Drive) reported member offsets running from 8.590 GB to
# 12.885 GB. Those two numbers are 2 x 2^32 and 3 x 2^32 to six figures, and they span exactly
# 2^32. An offset of 12.885 GB cannot exist in an 11.0 GB file.
#
# That is a 32-bit overflow. A zip's central directory stores each member's local-header offset in
# a 4-byte field; past 4 GiB the value has to move into a ZIP64 extra field. Whatever wrote these
# day-zips did not do that, so every offset past 4 GiB wrapped. Python then adds its own
# `concat` correction, computed from the same overflowed field in the end-of-central-directory
# record, and lands the whole set two wraps too high.
#
# The failure modes match exactly:
#   reported offset inside the file  -> lands on compressed data -> "Bad magic number for file header"
#   reported offset past EOF         -> nothing to read          -> "Truncated file header"
#   a member's data appearing to run past the directory -> "Overlapped entries (possible zip bomb)"
#
# It is NOT a Drive problem, NOT an access-order problem, and NOT something a local copy fixes: a
# byte-perfect copy has a byte-perfect broken directory. The member DATA is intact -- only the
# index pointing at it is wrong.
#
# ===============================================================================================
# HOW IT IS REPAIRED
# ===============================================================================================
# True offsets ascend in central-directory order, and only the low 32 bits survived. So walking
# the directory and incrementing a wrap counter every time the stored value jumps backwards
# reconstructs every offset arithmetically -- no I/O at all. One member is then probed to pin the
# absolute position, and a spread sample is probed to confirm the reconstruction before it is
# trusted. If the sample disagrees (an archive written out of order), it falls back to probing
# each wanted member individually.
#
# Cost: about a dozen 30-byte reads per archive, against 20-30 GB for a copy that would not have
# worked anyway.
# ===============================================================================================
import os
import struct
import zipfile

ZIP32_WRAP = 1 << 32

# Which archives needed repairing this session, and how many members each. Surfaced in cell 17's
# summary: "how many of the day-zips were broken" is the single number that says whether the
# overflow was the whole story or only part of it.
ZIP_OFFSET_REPAIRS = {}


def _open_reader(path, reader_factory=None):
    """A fresh seekable reader over the archive's bytes.

    The repair originally assumed a real filesystem path, because the India day-zips are single
    files. The RSNA archive is 91 concatenated parts behind a MultiPartFile, so there is no path
    to open -- which is why this repair, built and verified for exactly this defect, was applied
    to none of the RSNA path at all. A 194.7 GB zip spans the 32-bit offset field 45 times over;
    B1b read 250 members through the unrepaired directory and 249 of them failed.

    A factory rather than a shared handle: every probe seeks, and sharing a handle with the
    ZipFile that is mid-read corrupts both.
    """
    if reader_factory is not None:
        return reader_factory()
    return open(path, "rb")


def _true_cd_start(path, size, reader_factory=None):
    """Where the central directory really begins.

    ZipFile takes this from the EOCD's stored offset -- the same field that overflowed -- so its
    `start_dir` is wrong too, and the zip-bomb guard then rejects members near the end of the
    archive. The EOCD's *size* of the directory did not overflow, so subtracting it from the
    EOCD's own position gives the true start.
    """
    with _open_reader(path, reader_factory) as f:
        f.seek(max(0, size - 65557))
        tail = f.read()
    i = tail.rfind(b"PK\x05\x06")
    if i == -1:
        return None
    return (size - len(tail)) + i - int.from_bytes(tail[i + 12:i + 16], "little")


def _local_header_matches(fp, offset, info, size):
    """Is there a local file header for exactly this member at `offset`?"""
    if offset < 0 or offset + 30 > size:
        return False
    fp.seek(offset)
    hdr = fp.read(30)
    if len(hdr) != 30 or hdr[:4] != b"PK\x03\x04":
        return False
    nlen = int.from_bytes(hdr[26:28], "little")
    want = info.orig_filename.encode("utf-8" if info.flag_bits & 0x800 else "cp437")
    if nlen != len(want):
        return False
    return fp.read(nlen) == want


def _probe_true_offset(fp, info, size, wrap):
    """Find this member by trying every offset congruent to its stored value."""
    base = info.header_offset % wrap
    for j in range(0, int(size // wrap) + 2):
        cand = base + j * wrap
        if cand >= size:
            break
        if _local_header_matches(fp, cand, info, size):
            return cand
    return None


def zip_needs_offset_repair(zf, path, size=None, probes=8, reader_factory=None):
    """Is this archive's central directory pointing at the wrong places?

    An offset beyond the end of the file is proof on its own and costs nothing to check. But that
    test alone is not sufficient: whether a wrapped offset lands outside the file depends on how
    far past the 4 GiB boundary the archive goes, and an archive can be thoroughly broken with
    every offset still inside it. So when the free check is clean, a handful of 30-byte reads
    settle it -- is each sampled member's local header where the directory claims?

    This used to sample exactly two members, the first and the last, on the reasoning that a wrap
    misplaces everything after the first boundary so the ends bracket it. A real trial found an
    archive that passed both and whose members then would not read at all, so the reasoning was
    not sound -- Python's own `concat` correction shifts the whole set by a multiple of 2^32, and
    which members land correctly after that shift depends on where the boundaries fall. A spread
    of samples across the directory costs six more 30-byte reads and does not depend on being
    right about which members are the informative ones.

    A false negative here is no longer fatal in any case: cell 17 forces a repair when reads fail
    on an archive this cleared. This check is the cheap path, not the only one.
    """
    size = (os.path.getsize(path) if size is None else size)
    try:
        infos = [i for i in zf.infolist() if not i.filename.endswith("/")]
    except Exception:
        return False
    if not infos:
        return False
    if any(i.header_offset >= size for i in infos):
        return True
    n = len(infos)
    idx = sorted({0, n - 1} | {(k * (n - 1)) // max(1, probes - 1) for k in range(probes)})
    try:
        with _open_reader(path, reader_factory) as fp:
            for k in idx:
                if not _local_header_matches(fp, infos[k].header_offset, infos[k], size):
                    return True
    except OSError:
        return False
    return False


def repair_zip_offsets(zf, path, wrap=ZIP32_WRAP, sample=12, verbose=True,
                        reader_factory=None, size=None):
    """Rewrite this ZipFile's in-memory offsets so its members can be read. Returns n repaired."""
    size = (os.path.getsize(path) if size is None else size)
    infos = [i for i in zf.infolist() if not i.filename.endswith("/")]
    if not infos:
        return 0

    # ---- reconstruct, with no I/O: a backward jump in the stored value means one more wrap -----
    rel, j, prev = [], 0, -1
    for info in infos:
        base = info.header_offset % wrap
        if base < prev:
            j += 1
        prev = base
        rel.append(base + j * wrap)

    with _open_reader(path, reader_factory) as fp:
        anchor = _probe_true_offset(fp, infos[0], size, wrap)
        if anchor is None:
            if verbose:
                print("    offset repair: could not locate the first member -- leaving as is")
            return 0
        delta = anchor - rel[0]

        idx = sorted({int(k * (len(infos) - 1) / max(sample - 1, 1)) for k in range(sample)})
        agree = sum(_local_header_matches(fp, rel[k] + delta, infos[k], size) for k in idx)

    # The reconstruction is applied whether or not every sample point agreed, and the sample is
    # only ever used to say how much to trust it.
    #
    # The first version of this fell back to probing EVERY member when the sample was not perfect.
    # On a 100,000-member archive that is up to seven Drive seeks each -- hundreds of thousands of
    # round trips, silently, with no progress and no bound. A trial that should have taken twenty
    # minutes was still going after ninety. There is no version of "probe everything" that belongs
    # in a loop over 107 archives.
    #
    # Applying a partly-validated reconstruction is the better trade: the arithmetic is free, and
    # a member whose offset comes out wrong simply fails its own read and is counted like any other
    # unreadable member. The cost of being wrong is bounded and visible; the cost of exhaustive
    # probing was neither.
    for info, r in zip(infos, rel):
        info.header_offset = r + delta
    n_fixed = len(infos)

    if verbose:
        if agree == len(idx):
            print(f"    offset repair: 32-bit overflow corrected for all {n_fixed:,} members "
                  f"({len(idx) + 1} probes, {agree}/{len(idx)} sample checks agreed)")
        else:
            print(f"    offset repair: applied to {n_fixed:,} members, but only "
                  f"{agree}/{len(idx)} sample checks agreed -- this archive was probably not "
                  f"written in ascending order.")
            print(f"      Series that land on a wrong offset will fail their own read and be "
                  f"reported as failures below; nothing else is affected.")

    # ---- the two derived fields ZipFile also computed from the overflowed values ---------------
    cd = _true_cd_start(path, size, reader_factory)
    if cd is not None:
        zf.start_dir = cd
    # _end_offset is the zip-bomb guard's idea of where each member's data must stop: the next
    # member's header, or the directory for the last one. It was built from the broken offsets.
    end = zf.start_dir if cd is not None else size
    for info in sorted(infos, key=lambda i: i.header_offset, reverse=True):
        if hasattr(info, "_end_offset"):
            info._end_offset = end
        end = info.header_offset
    return n_fixed


def force_zip_offset_repair(zf, path, verbose=True, reader_factory=None, size=None,
                             wrap=ZIP32_WRAP):
    """Repair regardless of what detection concluded. Returns n repaired (0 if it could not).

    Called from cell 17 when member reads fail on an archive that `zip_needs_offset_repair`
    cleared. Detection is a heuristic over a sample; a failed read is a fact. When the two
    disagree, the fact wins.
    """
    if not isinstance(zf, zipfile.ZipFile):
        return 0
    n = repair_zip_offsets(zf, path, wrap=wrap, verbose=verbose,
                            reader_factory=reader_factory, size=size)
    zf._offsets_repaired = True
    zf._offsets_repaired_n = n
    if n:
        ZIP_OFFSET_REPAIRS[path] = n
    return n


def ensure_zip_readable(zf, path, verbose=True, reader_factory=None, size=None,
                         wrap=ZIP32_WRAP):
    """Repair `zf` if its directory overflowed. Safe and cheap to call on a healthy archive.

    The "already done" marker lives on the ZipFile object itself. A module-level dict keyed on
    id(zf) looked equivalent and was not: CPython reuses the id of a collected object, so a fresh
    ZipFile could inherit a previous one's entry and skip its repair -- rarely, unreproducibly,
    and only under memory pressure, which is the worst way for this to fail.
    """
    if getattr(zf, "_offsets_repaired", False):
        return getattr(zf, "_offsets_repaired_n", 0)
    if not isinstance(zf, zipfile.ZipFile):
        return 0                     # the RSNA tar container: nothing to repair
    size = (os.path.getsize(path) if size is None else size)
    if zip_needs_offset_repair(zf, path, size, reader_factory=reader_factory):
        n = repair_zip_offsets(zf, path, wrap=wrap, verbose=verbose,
                                reader_factory=reader_factory, size=size)
    else:
        n = 0
        # Say so. A silent pass here and a silent pass on a healthy archive looked identical in
        # the log, so when detection was wrong there was nothing to point at afterwards.
        if verbose:
            try:
                _f = [i for i in zf.infolist() if not i.filename.endswith("/")]
                print(f"    offsets look correct ({len(_f):,} members, "
                      f"{size/1e9:.1f} GB, sampled 8) -- no repair applied")
            except Exception:
                pass
    zf._offsets_repaired = True
    zf._offsets_repaired_n = n
    if n:
        ZIP_OFFSET_REPAIRS[path] = n
    return n


print("zip offset repair ready (32-bit central-directory overflow; detection costs 2 small "
      "reads, repair ~13 more)")

In [ ]:
# 12. Locate the wanted series inside a day-zip, from the central directory alone  [RESUMABLE]
#
# This replaces the "open every zip and dcmread every entry's header" scan. `namelist()` reads the
# zip's central directory -- a few MB at the end of the archive -- and gives every member path
# without touching a single member's contents.
#
# Matching is on PATH COMPONENTS, not a bare substring. A substring match would let a wanted UID
# ending "...096" also claim files belonging to "...0961", which is a silent cross-contamination
# bug of exactly the kind that never shows up until someone audits an embedding six months later.
# A substring pass still runs, but only for UIDs that got zero component hits (an export that
# encodes the UID into the filename rather than a folder), and anything it finds is flagged.
#
# Whatever the match mode, the result is VERIFIED: one member per series is header-read and its
# SeriesInstanceUID compared against what we were looking for. One header read per series -- not
# per file, and not per zip entry.
import hashlib
import json
import os

import pydicom


def _split_components(name):
    return [c for c in name.replace("\\", "/").split("/") if c]


def _zip_index_path(zip_key):
    return os.path.join(ZIP_INDEX_DIR, zip_key.replace("/", "_") + ".json")


def _zip_index_key(zip_path, wanted_uids):
    """Keyed on SIZE only, not mtime, and always on the ORIGINAL Drive path.

    Two reasons. Drive's FUSE mount reports mtimes that can shift without the file changing, which
    would silently invalidate the cache. And a local copy made by ZIP_ACCESS_MODE="local_copy" has
    its own mtime, so keying on the effective path would make the index cache useless the moment
    you switched access modes -- the archive is byte-identical either way."""
    try:
        sig = str(os.path.getsize(zip_path))
    except OSError:
        sig = "missing"
    h = hashlib.sha1("\n".join(sorted(wanted_uids)).encode()).hexdigest()
    # schema_version 4: entries now carry an "offset" and their "names" are ordered by position
    # in the archive rather than alphabetically. A version-3 cache holds name-ordered lists, and
    # reusing one would silently reinstate the jumpy read pattern this notebook exists to avoid --
    # the fix would appear to be in place and do nothing. Bumping the version discards those.
    return {"zip_sig": sig, "wanted_sha1": h, "n_wanted": len(wanted_uids), "schema_version": 4}


def locate_series_in_zip(zf, zip_path, zip_key, wanted_uids, verify=True):
    """Returns {series_uid: {"names": [...], "match_mode": ..., "verified": bool, "note": str}}."""
    cache_path = _zip_index_path(zip_key)
    key = _zip_index_key(zip_path, wanted_uids)
    if os.path.exists(cache_path):
        try:
            with open(cache_path) as f:
                blob = json.load(f)
            if blob.get("key") == key and blob.get("complete", False):
                return {k: v for k, v in blob["index"].items()}
            if blob.get("key") == key:
                print(f"    zip index for {zip_key} is cached but incomplete "
                      f"({len(blob.get('index', {}))}/{len(wanted_uids)} series located) -- "
                      f"rescanning so the missing ones get another chance")
        except Exception as e:
            print(f"    zip index cache unreadable ({e}) -- rebuilding for {zip_key}")

    wanted = set(wanted_uids)
    by_component = {uid: uid for uid in wanted}

    # A day-zip can hold half a million members. Scanning every wanted UID against every path
    # component is O(entries x components x uids) -- tens of millions of startswith() calls on a
    # real archive, which is minutes of pure Python. Bucket the UIDs by their first _PREFIX_LEN
    # characters instead, so a component costs one dict lookup and, in the rare case that lookup
    # hits, at most a handful of full comparisons.
    _PREFIX_LEN = 24
    by_prefix = {}
    for uid in wanted:
        by_prefix.setdefault(uid[:_PREFIX_LEN], []).append(uid)

    hits = {uid: [] for uid in wanted}
    n_entries = 0
    for name in zf.namelist():
        if name.endswith("/"):
            continue
        n_entries += 1
        for comp in _split_components(name):
            uid = by_component.get(comp)
            if uid is None and len(comp) > _PREFIX_LEN:
                # "<uid>.<sop-suffix>" / "<uid>_0001" style filenames. The separator check is what
                # stops UID "...096" from claiming files belonging to UID "...0961".
                for cand in by_prefix.get(comp[:_PREFIX_LEN], ()):
                    if (comp.startswith(cand) and len(comp) > len(cand)
                            and comp[len(cand)] in "._-"):
                        uid = cand
                        break
            if uid is not None:
                hits[uid].append(name)
                break

    # ---- order every member list by its position in the ARCHIVE, not by name -----------------
    #
    # This is the difference between reading these day-zips and not reading them.
    #
    # An earlier notebook read the same archives with no failures at all. What it did was iterate
    # zf.namelist() from beginning to end and open every member in turn -- that is, it walked the
    # file forwards. This notebook opens three members of one wanted series, then three of another
    # a few gigabytes back, and on a 20-30 GB Drive-mounted file those backward jumps are what
    # come back as "BadZipFile: Bad magic number for file header". Retrying does not help, because
    # a cold seek to the same place fails the same way; sorting by name does not help either,
    # because name order is not archive order across studies.
    #
    # `header_offset` comes from the central directory, which is already parsed -- so this costs
    # one pass over infolist() and nothing else.
    _offset_of = {}
    try:
        for _zi in zf.infolist():
            _offset_of[_zi.filename] = int(getattr(_zi, "header_offset", 0))
    except Exception:
        _offset_of = {}          # a container that is not a real zip (RSNA tar) -- name order it is

    def _in_archive_order(names):
        return sorted(names, key=lambda n: (_offset_of.get(n, 0), n))

    index = {}
    for uid, names in hits.items():
        if names:
            _ordered = _in_archive_order(names)
            index[uid] = {"names": _ordered, "match_mode": "component",
                           "verified": False, "note": "",
                           # Lets the extraction loop visit this zip's series in archive order too,
                           # so the whole day-zip is read in one forward sweep rather than a series
                           # of jumps backwards and forwards.
                           "offset": _offset_of.get(_ordered[0], 0)}

    # Fallback only for UIDs that found nothing by component.
    unmatched = [u for u in wanted if u not in index]
    if unmatched:
        names_all = [n for n in zf.namelist() if not n.endswith("/")]
        for uid in unmatched:
            found = [n for n in names_all if uid in n]
            if found:
                _ordered = _in_archive_order(found)
                index[uid] = {"names": _ordered, "match_mode": "substring",
                               "verified": False,
                               "note": "matched by substring, not path component -- verify",
                               "offset": _offset_of.get(_ordered[0], 0)}

    # Verify each located series by reading exactly one header.
    if verify:
        # Try up to VERIFY_ATTEMPTS members, not just the lexicographically first one.
        #
        # dcmread(force=True) turns a non-DICOM member -- a stray DICOMDIR, a thumbnail, a
        # truncated file -- into an empty Dataset with SeriesInstanceUID = None rather than an
        # exception. If that member happened to sort first, the whole series was marked unverified
        # and every one of its slices discarded, and because the negative result is CACHED, the
        # usual "fix it and re-run" did nothing until the zip index was also deleted. Volume
        # assembly already tolerates individual bad members; verification should too.
        VERIFY_ATTEMPTS = 3
        for uid, entry in index.items():
            attempts, last_err = [], ""
            for name in entry["names"][:VERIFY_ATTEMPTS]:
                try:
                    with zf.open(name) as fh:
                        ds = pydicom.dcmread(fh, stop_before_pixels=True, force=True)
                    actual = getattr(ds, "SeriesInstanceUID", None)
                except Exception as e:
                    attempts.append(f"{name}: read failed ({type(e).__name__})")
                    continue
                if actual == uid:
                    entry["verified"] = True
                    if attempts:
                        entry["note"] = (f"verified on member {name} after "
                                          f"{len(attempts)} unusable member(s)")
                    break
                attempts.append(f"{name}: SeriesInstanceUID={actual!r}")
                if actual is not None:
                    # A member that parsed cleanly and claims a DIFFERENT series is a genuine
                    # collision, not a damaged file. Stop and report it rather than hunting for
                    # a member that happens to agree.
                    last_err = f"member {name} belongs to series {actual!r}, not {uid!r}"
                    break
            if not entry["verified"]:
                entry["note"] = last_err or ("could not verify from the first "
                                              f"{min(VERIFY_ATTEMPTS, len(entry['names']))} "
                                              f"member(s): {'; '.join(attempts)}")

    # Cache POSITIVE results only.
    #
    # A `verified: False` entry is almost always a transient read failure -- a Drive FUSE hiccup
    # at 02:40 makes zf.open() throw for a few members, and 40 series get marked unverified.
    # Persisting that verdict makes it permanent: the next run returns from the cache before doing
    # any zip I/O, reproduces the identical failures, and the summary's promise that "all of these
    # are retried automatically on the next run" becomes false. The only cure would be deleting a
    # cache file nothing tells you about. So a failed lookup simply is not remembered, and the
    # cache is only treated as authoritative when it covers everything that was asked for.
    # With verification deferred to read time there is no negative result to persist -- locating
    # is a pure central-directory operation, so every entry is cacheable. With verification on,
    # only positives are cached, for the reason spelt out just above.
    if verify:
        cacheable = {uid: e for uid, e in index.items() if e.get("verified")}
    else:
        cacheable = dict(index)
    try:
        tmp = cache_path + ".tmp"
        with open(tmp, "w") as f:
            json.dump({"key": key, "index": cacheable,
                        "complete": len(cacheable) == len(wanted)}, f)
        os.replace(tmp, cache_path)
    except Exception as e:
        print(f"    WARNING: could not cache zip index for {zip_key}: {e}")

    n_found = len(index)
    n_sub = sum(1 for e in index.values() if e["match_mode"] == "substring")
    n_bad = sum(1 for e in index.values() if verify and not e["verified"])
    msg = (f"    located {n_found}/{len(wanted)} wanted series in {zip_key} "
           f"({n_entries:,} zip entries scanned)")
    if n_sub:
        msg += f"; {n_sub} via substring fallback"
    if n_bad:
        msg += f"; {n_bad} FAILED VERIFICATION"
    print(msg)
    if n_bad:
        for uid, e in list(index.items()):
            if not e["verified"]:
                print(f"      {uid}: {e['note']}")
    return index


def get_series_index(container, container_path, container_key, wanted_uids, verify=None):
    """Dispatch: how do we find a series' members inside this container?

    India -- match path components in the zip's central directory, then verify by header.
    RSNA    -- already known. Cell B2 read every header once to learn the grouping (the flat
               ID_<hex>.dcm layout carries no study/series information in the path), and cell B4
               ordered each series' members along the slice normal. That index IS the answer, and
               it was built from the headers themselves, so it needs no separate verification.
    """
    if DATA_SOURCE == "rsna":
        out = {}
        for uid in wanted_uids:
            names = RSNA_MEMBERS_BY_SERIES.get(uid)
            if names:
                out[uid] = {"names": names, "match_mode": "rsna_header_index",
                             "verified": True, "note": ""}
        return out
    _v = VERIFY_IN_LOCATOR if verify is None else verify
    return locate_series_in_zip(container, container_path, container_key, wanted_uids,
                                 verify=_v)


print("series locator ready "
      "(India: path-component matching + header verification, cached per zip; "
      "RSNA: the cell B2 header index)")

In [ ]:
# 12b. Why will a day-zip not stream? -- measure it, on one real archive  [DIAGNOSTIC, optional]
#
# The first overnight run lost 834 series to `BadZipFile: Bad magic number for file header`, and
# the natural reading of that message -- "the archive is corrupt" -- is wrong. An earlier notebook
# read these same day-zips with no failures at all. The difference was the ACCESS PATTERN: it
# walked zf.namelist() from the first member to the last, so the file was read forwards; this
# notebook jumps straight to the three members it wants, several gigabytes in.
#
# Rather than keep reasoning about which of those matters, this cell measures it. It reads a few
# hundred KB and takes under a minute, and it answers four questions:
#
#   1. can a member near the START of the archive be read?
#   2. can a member DEEP in the archive be read cold?
#   3. does reading the wanted members in ASCENDING OFFSET order work where mixed order does not?
#   4. how fast does this runtime read sequentially off Drive? (that sets the copy-time estimate)
#
# Set RUN_ZIP_DIAGNOSTIC = False once you have the answer.
RUN_ZIP_DIAGNOSTIC = True
DIAGNOSTIC_ZIP_KEY = None      # None = pick the first pending day-zip automatically

if not RUN_ZIP_DIAGNOSTIC:
    print("RUN_ZIP_DIAGNOSTIC is False -- skipping.")
elif DATA_SOURCE == "rsna":
    print("diagnostic applies to the India day-zips only; DATA_SOURCE is 'rsna'. Skipping.")
else:
    import json
    import random
    import time
    import zipfile

    def _pick_zip():
        if DIAGNOSTIC_ZIP_KEY:
            _m = manifest_runnable[manifest_runnable["zip_key"] == DIAGNOSTIC_ZIP_KEY]
            if not len(_m):
                raise SystemExit(f"no manifest rows for zip_key {DIAGNOSTIC_ZIP_KEY!r}")
            return DIAGNOSTIC_ZIP_KEY, _m["zip_path"].iloc[0], _m["series_uid"].tolist()
        # `DONE_OK` belongs to cell 17, which has not run yet when this cell is executed in
        # order. Read the same fact off the progress log instead, so this cell stands alone.
        _done = set()
        if os.path.exists(PROGRESS_LOG_PATH):
            with open(PROGRESS_LOG_PATH) as _f:
                for _line in _f:
                    if not _line.endswith("\n"):
                        break                      # a run may be appending right now
                    try:
                        _r = json.loads(_line)
                    except Exception:
                        continue
                    if _r.get("status") == "ok":
                        _done.add(_r.get("series_uid"))
        for _k, _g in manifest_runnable.groupby("zip_key", sort=True):
            _uids = [u for u in _g["series_uid"] if u not in _done]
            if _uids:
                return _k, _g["zip_path"].iloc[0], _uids
        return None, None, []

    _key, _path, _uids = _pick_zip()
    if _key is None:
        print("nothing pending -- no zip left to diagnose.")
    else:
        _size = os.path.getsize(_path)
        print(f"diagnosing {_key}  ({_size/1e9:.1f} GB)  {_path}")
        print(f"  {len(_uids)} pending series in this archive\n")

        _t0 = time.time()
        with zipfile.ZipFile(_path) as _zf:
            _infos = _zf.infolist()
            _cd_s = time.time() - _t0
            print(f"[1] central directory : {len(_infos):,} entries read in {_cd_s:.1f}s  -- OK")
            print(f"    (this is a read from the END of the file, and it already tells you the")
            print(f"     archive is not truncated and the mount can reach its last bytes)\n")

            _files = [i for i in _infos if not i.filename.endswith("/")]
            _files.sort(key=lambda i: i.header_offset)

            def _try(info, label):
                try:
                    with _zf.open(info.filename) as _fh:
                        _fh.read(512)
                    print(f"    OK    {label:<26s} offset {info.header_offset/1e9:7.2f} GB")
                    return True
                except Exception as _e:
                    print(f"    FAIL  {label:<26s} offset {info.header_offset/1e9:7.2f} GB  "
                          f"{type(_e).__name__}: {str(_e)[:54]}")
                    return False

            print("[2] one member from each region of the archive, read cold:")
            _probe = [(_files[0], "first member"),
                      (_files[len(_files)//4], "25% in"),
                      (_files[len(_files)//2], "50% in"),
                      (_files[3*len(_files)//4], "75% in"),
                      (_files[-1], "last member")]
            _region = [_try(i, lbl) for i, lbl in _probe]
            print()

            # the wanted members, both ways round
            _idx = get_series_index(_zf, _path, _key, _uids)
            _wanted_names = []
            for _u in _uids:
                for _n in _idx.get(_u, {}).get("names", [])[:2]:
                    _wanted_names.append(_n)
            _wi = [_zf.getinfo(_n) for _n in _wanted_names if _n in _zf.NameToInfo]
            if not _wi:
                print("[3] none of the pending series could be located -- nothing to probe.")
            else:
                _shuf = list(_wi)
                random.Random(0).shuffle(_shuf)
                print(f"[3] {len(_shuf)} wanted member(s) in SHUFFLED order "
                      f"(what a jumpy read looks like):")
                _n_shuf = sum(_try(i, i.filename.split('/')[-1][-22:]) for i in _shuf[:6])
                print()
                _asc = sorted(_wi, key=lambda i: i.header_offset)
                print(f"[4] the same members in ASCENDING OFFSET order (a forward sweep):")
                _n_asc = sum(_try(i, i.filename.split('/')[-1][-22:]) for i in _asc[:6])

        print()
        _t1 = time.time()
        with open(_path, "rb") as _f:
            _got = 0
            while _got < 200 * 1024 * 1024:
                _b = _f.read(16 * 1024 * 1024)
                if not _b:
                    break
                _got += len(_b)
        _sec = time.time() - _t1
        _mbs = _got / 1e6 / max(_sec, 1e-6)
        print(f"[5] sequential read speed off Drive: {_mbs:.0f} MB/s "
              f"({_got/1e6:.0f} MB in {_sec:.1f}s)")
        print(f"    a full copy of this {_size/1e9:.1f} GB archive would take "
              f"about {_size/1e6/_mbs/60:.0f} min")

        print("\n" + "=" * 92)
        print("READING")
        print("=" * 92)
        if all(_region):
            print("Every region of the archive read fine cold. The mount is healthy for this file,")
            print("and any failures you saw are about ORDER, not reachability.")
        elif _region[0] and not all(_region):
            print("Members near the start read; deeper ones do not. The mount is not serving cold")
            print("reads far into this file -- a forward sweep or a local copy is required.")
        else:
            print("Even the first member failed. That points at the archive itself rather than the")
            print("access pattern -- worth opening this one zip by hand.")
        if _wi:
            if _n_asc > _n_shuf:
                print(f"\nOffset order recovered {_n_asc - _n_shuf} member(s) that shuffled order")
                print("could not read. Ordering reads by archive position is the fix, and this")
                print("notebook now does that everywhere.")
            elif _n_asc == _n_shuf == len(_asc[:6]):
                print("\nBoth orders read fine, so this particular zip was never the problem.")
                print("Run the diagnostic again with DIAGNOSTIC_ZIP_KEY set to a day that failed.")
            else:
                print("\nOrder made no difference here: these members do not read either way.")
                print("This archive needs the local-copy fallback, which 'auto' mode will do.")
        print("=" * 92)

In [ ]:
# 13. Assemble a 3D HU volume for one series, straight out of the zip
#
# Carried over from the v6 notebook, plus four fixes for gaps that notebook listed as open:
#
#   1. SPACING IS RETURNED BY NAME, not as a bare 3-tuple. v6 returned (sz, sy, sx) while the
#      Pillar-0 preprocessing in notebook 14 wanted (sx, sy, sz). Both were individually correct
#      and the pair would have silently swapped the in-plane and through-plane axes. A dict makes
#      that mistake impossible to make quietly.
#   2. THROUGH-PLANE SPACING IS MEASURED, not trusted. The median gap between consecutive
#      ImagePositionPatient z-values is what the volume physically is. SpacingBetweenSlices /
#      SliceThickness are the fallback, and they disagree with reality whenever there is slice
#      overlap or a gap -- which then propagates straight into the resample.
#   3. DUPLICATE SOPInstanceUIDs are dropped (v6 listed this as a known open gap). Re-exported
#      PACS studies do contain them, and a duplicated slice both distorts the measured spacing and
#      doubles a slice's weight in the volume.
#   4. Pixel decoding prefers gdcm then pylibjpeg before pydicom's auto-selection, because
#      pydicom's own docs warn that Pillow "performs a number of undesirable operations on the
#      decoded images which are not always reversible" -- not acceptable when the HU values are
#      the entire signal. Each instance's decode is individually wrapped, so one bad file skips
#      one slice instead of failing the whole series (the bug that produced a 100% failure rate in
#      v6's last real run).
import numpy as np
import pydicom
import zipfile

_PREFERRED_DECODE_PLUGINS = ("gdcm", "pylibjpeg")


def _pixel_correction(arr, ds):
    """Dataset-specific correction applied to raw stored values BEFORE rescaling.

    Returns (array, intercept_override_or_None).

    For RSNA this is not cosmetic. Roughly 333k of the 752,803 training files store 12-bit
    unsigned data whose values have wrapped modulo 4096, tagged with a RescaleIntercept near 0
    instead of ~-1000. Air then reads as about +3096 HU instead of -1000, every one of the 11
    windows is computed on nonsense, and the model returns a perfectly healthy-looking unit-norm
    embedding of garbage. Nothing raises. The correction (Jeremy Howard's fastai cleaning kernel,
    copied verbatim into several medal-winning solutions) is applied here.
    """
    if DATA_SOURCE != "rsna":
        return arr, None
    fix_needed = globals().get("rsna_needs_pixel_fix")
    fix = globals().get("rsna_fix_pixels")
    if fix_needed is None or fix is None or not fix_needed(ds):
        return arr, None
    return fix(arr, ds)


def safe_float_attr(ds, name, default=None):
    """getattr(ds, name, default) returns the default only when the tag is ABSENT -- not when it
    is present but blank, which is a common PACS export quirk and used to crash on float(None)."""
    try:
        v = getattr(ds, name, default)
        if v is None or v == "":
            return default
        return float(v)
    except Exception:
        return default


def _decode_pixel_array(ds):
    for plugin in _PREFERRED_DECODE_PLUGINS:
        try:
            return pydicom.pixels.pixel_array(ds, decoding_plugin=plugin)
        except Exception:
            continue
    return ds.pixel_array


def _slice_normal_from_ds(ds):
    """Unit normal of the imaging plane, from the two ImageOrientationPatient direction cosines.
    None when the tag is absent or degenerate.

    NAMED FOR ITS ARGUMENT, deliberately. This used to be `_slice_normal(ds)` while cell B4 had
    its own `_slice_normal(iop_str)` -- same name, different argument type, both at notebook
    global scope. Cell 13 runs AFTER B4, so cell 13's won, and any re-run of B4 (which is what a
    resume does) then called this one with a comma-joined string. getattr("1,0,0,0,1,0",
    "ImageOrientationPatient", None) is None, so it returned None, so every RSNA series silently
    fell back from slice-normal ordering to raw z -- on a dataset with 139 distinct orientations,
    where slice-normal ordering is precisely the thing that matters. Nothing raised. The only
    trace was one line of the manifest report changing from "slice normal" to "raw z"."""
    iop = getattr(ds, "ImageOrientationPatient", None)
    if iop is None or len(iop) != 6:
        return None
    try:
        v = [float(t) for t in iop]
    except Exception:
        return None
    n = np.cross(np.array(v[:3]), np.array(v[3:6]))
    norm = float(np.linalg.norm(n))
    if not np.isfinite(norm) or norm < 1e-6:
        return None
    return n / norm


def _slice_position(ds, normal, fallback):
    """Returns (value, is_real_position).

    The physical stacking coordinate is the position projected onto the slice NORMAL, not the raw
    z coordinate. They agree for an ordinary axial study (normal = (0,0,1)) and they do not agree
    for a tilted gantry -- which is exactly the case where raw z stops being monotonic along the
    stack, so sorting on it interleaves slices.

    The two return paths are on completely different scales -- millimetres, or a bare ordinal from
    InstanceNumber -- so the caller must never mix them in one sort. See the `all_ipp` handling in
    assemble_volume: a series where only SOME slices carry a position is sorted by InstanceNumber
    for ALL of them, because sorting a mm value of 285.0 against an ordinal of 12 silently drops
    three mid-brain slices to the bottom of the stack, and the median-of-gaps spacing estimate
    survives that scrambling without complaining."""
    ipp = getattr(ds, "ImagePositionPatient", None)
    if ipp is not None and len(ipp) == 3:
        try:
            p = np.array([float(ipp[0]), float(ipp[1]), float(ipp[2])])
            if normal is not None:
                return float(p @ normal), True
            return float(p[2]), True
        except Exception:
            pass
    n = safe_float_attr(ds, "InstanceNumber", None)
    return (float(n) if n is not None else float(fallback)), False


def _in_plane_spacing(ds):
    px = getattr(ds, "PixelSpacing", None)
    if px is not None and len(px) == 2:
        try:
            # DICOM PixelSpacing is [row spacing (y), column spacing (x)].
            return float(px[1]), float(px[0])
        except Exception:
            pass
    return 1.0, 1.0


class ZipMembersUnreadable(RuntimeError):
    """The archive's index is wrong for these members -- not a per-slice decode problem.

    Distinct from ValueError so cell 17 can tell "this series has no usable pixel data" (a data
    problem, give up on it) from "this archive cannot be read at the offsets its directory
    claims" (an index problem, worth one forced repair and a retry).
    """


# How many consecutive unreadable members before a series is abandoned. Five is well past any
# plausible run of individually-corrupt slices and costs ~23 s at cell 2d's backoff, against the
# ~15 minutes it takes to walk a whole series that was never going to be read.
ABORT_SERIES_AFTER_CONSECUTIVE_READ_FAILURES = 5


def assemble_volume(names, zf, expect_uid=None):
    """Read every instance of one series and stack it into a (D, H, W) HU volume.

    Returns (volume, spacing, info) where spacing is {"sx","sy","sz"} in mm and info carries the
    per-series counts worth logging.

    `expect_uid` moves series verification to where the slices are ALREADY being read, instead of
    a separate pass that opens the first few members of every series before any of them is read.
    That pass doubled the number of seeks into a 20-30 GB Drive-mounted archive and, worse, made
    the read order sweep the file twice -- which is what a Drive mount handles badly. Checking the
    UID here costs nothing: the header is parsed regardless. A slice whose SeriesInstanceUID
    disagrees is dropped and counted, and a series where nothing agrees is a hard failure, which
    is exactly what the old pre-pass was protecting against.
    """
    decoded = []
    n_wrong_series = 0
    seen_sop = set()
    n_read_failed = n_undecodable = n_dup_sop = n_pixel_fixed = 0
    n_consecutive_fail = 0
    aborted_early = False

    for name in names:
        try:
            with zf.open(name) as fh:
                ds = pydicom.dcmread(fh, force=True)
            n_consecutive_fail = 0
        except Exception:
            n_read_failed += 1
            # ---- stop grinding through an archive that is not readable ----------------------
            #
            # Cell 2d retries every failed member 4 times with 0.4 + 1.2 + 3.0 s of backoff. That
            # was built for a transient mount, and against one it is right. Against an archive
            # whose directory is wrong it is a disaster: the failure is deterministic, so each
            # member costs the full 4.6 s of sleeping before failing anyway. A 200-slice series
            # then takes 15 minutes to arrive at a conclusion that the first five reads already
            # established. A real trial run spent 12.5 minutes on ONE series that way and printed
            # nothing while it did -- indistinguishable, from outside, from a hang.
            #
            # If the first few members in a row all fail, the series is not readable. Say so now.
            n_consecutive_fail += 1
            if n_consecutive_fail >= ABORT_SERIES_AFTER_CONSECUTIVE_READ_FAILURES and not decoded:
                aborted_early = True
                break
            continue
        if expect_uid is not None:
            _got = getattr(ds, "SeriesInstanceUID", None)
            if _got is not None and _got != expect_uid:
                n_wrong_series += 1
                continue
        sop = getattr(ds, "SOPInstanceUID", None)
        if sop is not None:
            if sop in seen_sop:
                n_dup_sop += 1
                continue
            seen_sop.add(sop)
        try:
            arr = _decode_pixel_array(ds)
        except Exception:
            n_undecodable += 1
            continue
        arr, icpt_override = _pixel_correction(arr, ds)
        arr = np.asarray(arr, dtype=np.float32)
        if icpt_override is not None:
            n_pixel_fixed += 1
        decoded.append((ds, arr, icpt_override))

    if aborted_early:
        raise ZipMembersUnreadable(
            f"{n_read_failed} member(s) in a row failed to read and none succeeded "
            f"-- stopped after {ABORT_SERIES_AFTER_CONSECUTIVE_READ_FAILURES} rather than "
            f"retrying all {len(names)}")
    if expect_uid is not None and not decoded and n_wrong_series:
        raise ValueError(
            f"every readable member belonged to a different series "
            f"({n_wrong_series} slice(s)) -- the path match for {expect_uid} was wrong")
    if not decoded:
        raise ValueError(
            f"no readable/decodable pixel data ({len(names)} member(s): "
            f"{n_read_failed} unreadable, {n_undecodable} undecodable, {n_dup_sop} duplicate SOP)"
        )

    notes = []
    info = {
        "n_members": len(names),
        "n_read_failed": n_read_failed,
        "n_undecodable": n_undecodable,
        "n_duplicate_sop": n_dup_sop,
        "n_pixel_corrected": n_pixel_fixed,
        "multiframe": False,
        "z_from_ipp": False,
    }

    # --- multi-frame: one file holding the whole volume ----------------------------------------
    ds0 = decoded[0][0]
    n_frames = getattr(ds0, "NumberOfFrames", None)
    if len(decoded) == 1 and n_frames and int(n_frames) > 1:
        _, arr, icpt_override = decoded[0]
        slope = safe_float_attr(ds0, "RescaleSlope", 1.0) or 1.0
        intercept = (icpt_override if icpt_override is not None
                     else (safe_float_attr(ds0, "RescaleIntercept", 0.0) or 0.0))
        vol = (arr * slope + intercept).astype(np.float32)
        sx, sy = _in_plane_spacing(ds0)
        sz = (safe_float_attr(ds0, "SpacingBetweenSlices", None)
              or safe_float_attr(ds0, "SliceThickness", None) or 1.0)
        info.update(multiframe=True, n_slices=int(vol.shape[0]))
        # Honest limitation: per-frame z positions live in PerFrameFunctionalGroupsSequence and
        # are not read here; a uniform spacing is assumed for multi-frame series.
        info["note"] = "multi-frame: uniform z spacing assumed, not read per frame"
        _ = notes
        return vol, {"sx": sx, "sy": sy, "sz": float(sz)}, info

    # --- ordinary per-slice series --------------------------------------------------------------
    _multi = [i for i, (_, a, _o) in enumerate(decoded) if a.ndim == 3]
    if _multi:
        # Several files each holding multiple frames. Stacking them as if each were one 2-D slice
        # builds a 4-D array that fails later inside SimpleITK with an opaque message; fail here
        # with one that says what actually happened.
        raise ValueError(
            f"series has {len(decoded)} file(s), {len(_multi)} of which contain multiple frames "
            f"(first has shape {decoded[_multi[0]][1].shape}). Multi-frame is only handled for a "
            f"single-file series; this layout is not supported."
        )

    # One normal for the whole series, taken from the first slice. A series whose slices disagree
    # on orientation is not a single stack; that is flagged rather than silently averaged.
    _normal = _slice_normal_from_ds(decoded[0][0])
    _iop_values = {tuple(round(float(t), 5) for t in getattr(ds, "ImageOrientationPatient", ()) or ())
                   for ds, _a, _o in decoded}
    if len(_iop_values) > 1:
        notes.append(f"slices disagree on ImageOrientationPatient ({len(_iop_values)} distinct "
                      f"values) -- this is not one coherent stack; ordering uses the first slice's "
                      f"orientation")
    info["slice_normal"] = ([round(float(v), 5) for v in _normal] if _normal is not None else None)

    zs, flags = [], []
    for i, (ds, _a, _o) in enumerate(decoded):
        z, from_ipp = _slice_position(ds, _normal, i)
        zs.append(z)
        flags.append(from_ipp)
    all_ipp = all(flags)
    if not all_ipp and any(flags):
        # Mixed: re-derive EVERY key on the ordinal scale so the sort is at least internally
        # consistent, and refuse to claim the spacing was measured from real geometry.
        notes.append(f"{len(flags) - sum(flags)}/{len(flags)} slice(s) had no "
                      f"ImagePositionPatient; the whole series was ordered by InstanceNumber "
                      f"instead of mixing millimetres with ordinals")
        zs = [safe_float_attr(ds, "InstanceNumber", i) or float(i)
              for i, (ds, _a, _o) in enumerate(decoded)]
    order = np.argsort(np.asarray(zs, dtype=np.float64), kind="stable")
    decoded = [decoded[i] for i in order]
    zs = [zs[i] for i in order]
    info["z_from_ipp"] = all_ipp

    slices, kept_z, kept_ds = [], [], []
    for (ds, arr, icpt_override), z in zip(decoded, zs):
        slope = safe_float_attr(ds, "RescaleSlope", 1.0) or 1.0
        intercept = (icpt_override if icpt_override is not None
                     else (safe_float_attr(ds, "RescaleIntercept", 0.0) or 0.0))
        slices.append(arr * slope + intercept)
        kept_z.append(z)
        kept_ds.append(ds)

    # Reference in-plane shape = the MOST COMMON one, not the first slice's.
    #
    # Found by a hostile test, and it is not a corner case: a real series can carry a couple of
    # odd-shaped frames (a stray localizer, a different reconstruction sharing the SeriesInstance-
    # UID). Slices are sorted by z before this point, so "the first slice" is whichever one
    # happens to sit at the bottom of the stack -- and if that one is the odd one out, taking it
    # as the reference throws away the entire real volume and keeps the junk. The test case (28 of
    # 30 slices at 256x256, 2 at 200x200, written in reverse z order) went from "2 usable slices,
    # series failed" to "28 usable slices, 2 dropped" with this one change.
    from collections import Counter
    shape_counts = Counter(s.shape for s in slices)
    ref_shape, _n_ref = shape_counts.most_common(1)[0]
    keep = [i for i, s in enumerate(slices) if s.shape == ref_shape]
    n_shape_dropped = len(slices) - len(keep)
    info["n_shape_mismatch"] = n_shape_dropped
    info["shape_counts"] = {str(k): v for k, v in shape_counts.items()}
    if n_shape_dropped:
        notes.append(f"{n_shape_dropped} slice(s) dropped for not matching the modal in-plane "
                      f"shape {ref_shape}; shapes seen: {dict(shape_counts)}")
    if not keep:
        raise ValueError("no slices share a common in-plane shape")
    # kept_ds is filtered in lockstep. This matters more than it looks: the spacing tags below are
    # read from kept_ds[0], and an odd-shaped frame (a screen capture, a stray reconstruction) is
    # exactly the one whose PixelSpacing differs from the real volume's. Reading spacing from a
    # slice that was just discarded silently rescales the whole volume -- e.g. 1.0 mm read off a
    # 200x200 capture instead of 0.449 mm from the 512x512 brain resamples to 1024 px wide instead
    # of 460, and the 256-px center crop then keeps a quarter of the field of view. No exception,
    # embedding saved, status "ok".
    slices = [slices[i] for i in keep]
    kept_z = [kept_z[i] for i in keep]
    kept_ds = [kept_ds[i] for i in keep]

    if len(slices) < MIN_SLICES_PER_SERIES:
        raise ValueError(f"only {len(slices)} usable slice(s), need >= {MIN_SLICES_PER_SERIES}")

    # An absolute floor is not enough. A 300-slice series where 270 instances fail to decode
    # yields a 30-slice volume, clears MIN_SLICES_PER_SERIES, and is logged `ok` -- an embedding of
    # a tenth of someone's head, indistinguishable downstream from a genuinely short scan. The
    # counts were already being recorded; nothing was comparing them.
    _kept_frac = len(slices) / max(len(names), 1)
    info["kept_slice_fraction"] = float(_kept_frac)
    if _kept_frac < MIN_KEPT_SLICE_FRACTION:
        raise ValueError(
            f"only {len(slices)} of {len(names)} member(s) survived reading "
            f"({_kept_frac:.0%} < MIN_KEPT_SLICE_FRACTION={MIN_KEPT_SLICE_FRACTION:.0%}): "
            f"{n_read_failed} unreadable, {n_undecodable} undecodable, {n_dup_sop} duplicate SOP, "
            f"{n_shape_dropped} shape mismatch. Refusing to embed a partial volume."
        )

    vol = np.stack(slices, axis=0).astype(np.float32)
    info["n_slices"] = int(vol.shape[0])

    ref_ds = kept_ds[0]
    sx, sy = _in_plane_spacing(ref_ds)

    # Measure z spacing from the geometry itself -- but only when EVERY slice carried a real
    # position, so the gaps being differenced are all in millimetres.
    sz = None
    if all_ipp and len(kept_z) >= 2:
        gaps = np.diff(np.asarray(kept_z, dtype=np.float64))
        gaps = gaps[np.abs(gaps) > 1e-6]
        if gaps.size:
            sz = float(np.median(np.abs(gaps)))
            info["z_spacing_source"] = ("measured along the slice normal" if _normal is not None
                                         else "measured from ImagePositionPatient z")
            tag_sz = (safe_float_attr(ref_ds, "SpacingBetweenSlices", None)
                      or safe_float_attr(ref_ds, "SliceThickness", None))
            if tag_sz and abs(tag_sz - sz) > max(0.15 * tag_sz, 0.2):
                notes.append(f"measured z spacing {sz:.3f}mm differs from the tag value "
                              f"{tag_sz:.3f}mm (overlap or gap in the acquisition); the measured "
                              f"value is used")
    if sz is None or not np.isfinite(sz) or sz <= 0:
        sz = (safe_float_attr(ref_ds, "SpacingBetweenSlices", None)
              or safe_float_attr(ref_ds, "SliceThickness", None) or 1.0)
        info["z_spacing_source"] = "SpacingBetweenSlices/SliceThickness tag"

    # HU plausibility. A head CT should have air near -1000 and brain parenchyma near +35. These
    # three numbers are cheap and they are the single best tripwire for a rescale/intercept
    # problem -- including one this notebook cannot fix (see below) -- because a wrong intercept
    # produces a volume that windows to nonsense without raising anything.
    _p = np.percentile(vol, [1, 50, 99])
    info["hu_p01"], info["hu_p50"], info["hu_p99"] = (float(_p[0]), float(_p[1]), float(_p[2]))
    if not (-1200.0 <= _p[0] <= -700.0):
        notes.append(
            f"IMPLAUSIBLE HU RANGE: 1st percentile is {_p[0]:.0f}, expected air near -1000 "
            f"(p50 {_p[1]:.0f}, p99 {_p[2]:.0f}). The rescale intercept is probably wrong for this "
            f"series -- the embedding will be of a wrongly-windowed volume, not of the anatomy."
        )

    info["note"] = " | ".join(notes)
    return vol, {"sx": float(sx), "sy": float(sy), "sz": float(sz)}, info


print("volume assembly ready")

In [ ]:
# ==============================================================================================
# 15. OUR BACKBONES INSTEAD OF PILLAR-0 -- configuration
# ==============================================================================================
# This cell and the three below REPLACE the Pillar-0 preprocessing, registry and loader. Every
# cell above (day-zip access, series tagging, assemble_volume) and every cell below (the
# extraction loop, consolidation, QC, the classifier sweep) is untouched and runs as it is.
#
# WHY IT DROPS IN CLEANLY. assemble_volume() already returns exactly what the RSNA pipeline's
# preprocessing consumes: a (D, H, W) HU volume plus spacing as {"sx","sy","sz"} in mm. So the
# only things that change are what happens to that volume and which model sees it.
#
# THE ONE THING THAT MATTERS SCIENTIFICALLY. External validation is only meaningful if the
# India embeddings live in the SAME feature space as the RSNA ones -- same grid, same spacing,
# same windows, same interpolation, same floor, same QC. Every value below is copied from the
# Phase-A run that produced features_B*.parquet, and the cell ENDS by reading the preprocessing
# signature that run recorded on Drive and refusing to continue if this notebook would produce
# a different one. A silent mismatch here does not raise anywhere downstream: it produces a
# perfectly healthy-looking table whose columns mean something different from RSNA's, and a
# domain-shift result that is really a preprocessing artefact.
import json
import os
import subprocess
import sys

# ---- verify, do NOT install ------------------------------------------------------------------
# MONAI and einops are installed by the install cell at the top of this notebook, which runs
# before numpy, torch and the DICOM codecs are imported. Installing them HERE instead is what
# killed the previous run: pip put new versions on disk while the old ones were still live in
# the kernel, and the session died silently at the next compiled import -- no traceback, no
# error cell, execution just stopped after the preprocessing self-test.
#
# So this only checks, and tells you exactly what to do if the check fails.
try:
    import monai
except Exception as _e:
    raise SystemExit(
        f"MONAI is not importable ({type(_e).__name__}: {_e}).\n"
        f"It is pinned in the install cell at the top of this notebook. Do this:\n"
        f"  1. Runtime > Restart session\n"
        f"  2. Run all\n"
        f"Do NOT pip install it from here -- installing into a kernel that has already imported\n"
        f"numpy and the DICOM codecs is what makes the session die without an error.")


def _ver_lt(v, want=(1, 4)):
    """MONAI ships versions like '1.4.0rc1' and '1.6.0+gitabc'. int('0rc1') raises, and a
    crash here would stop a run over a cosmetic check."""
    out = []
    for part in str(v).split(".")[:2]:
        digits = "".join(ch for ch in part if ch.isdigit())
        out.append(int(digits) if digits else 0)
    return tuple(out) < want


print(f"monai {monai.__version__}")
if _ver_lt(monai.__version__):
    print(f"  NOTE: monai {monai.__version__} predates the SwinUNETR constructor change. The zoo "
          f"falls back, but Phase A was built against >=1.4.")

RSNA_OUTPUT_DIR = P["rsna_output_dir"]   # where Phase A wrote its features
INDIA_OUTPUT_DIR = P["external_output_dir"]

BACKBONE_IDS = [
    "B1_swinunetr_suprem",
    "B2_segresnet_suprem",
    "B3_resnet18_medicalnet",
]

# ---- copied verbatim from Phase A. Do not "improve" any of it. --------------------------------
TARGET_SPACING_XYZ     = (1.0, 1.0, 3.0)
TARGET_SHAPE_DHW       = (64, 256, 256)
HU_WINDOWS             = [("brain", 40.0, 80.0), ("subdural", 80.0, 200.0), ("soft", 40.0, 380.0)]
RESAMPLE_INTERPOLATOR  = "linear"
# The extraction loop stamps this into its cache key. Pillar-0 fed 11 windows; we feed 3,
# so leaving the old value would make two genuinely different inputs share a key. It was
# defined by the Pillar-0 preprocessing cell this notebook no longer has, so without it
# the loop raises NameError on its first series -- minutes into a multi-hour run.
N_INPUT_CHANNELS       = len(HU_WINDOWS)

# ---- the geometry the extraction loop STAMPS INTO EVERY RECORD --------------------------------
# preproc_signature() in the extraction cell builds its string from TARGET_DEPTH,
# TARGET_SPATIAL_SIZE and PILLAR0_INPLANE_MODE -- Pillar-0's names, still holding Pillar-0's
# values (128 deep, fit_head). Our preprocessing produces 64 x 256 x 256 with a centred
# crop/pad, so without these three lines every embedding is written with a signature that
# describes a geometry it was not made with. The vectors are right; the label on them is a lie,
# and the label is the only provenance a reader of the parquet has.
#
# It also drives the head-retention report at the end of the loop, which was quoting
# "128 slices x 3.0 mm = 384 mm of coverage" for a run whose window is 192 mm.
TARGET_DEPTH           = TARGET_SHAPE_DHW[0]          # 64
TARGET_SPATIAL_SIZE    = tuple(TARGET_SHAPE_DHW[1:])  # (256, 256)
PILLAR0_INPLANE_MODE   = "crop"                       # centre crop/pad, not Pillar-0's fit_head
SKULL_STRIP            = False
CLAMP_HU_FLOOR         = True
HU_AIR_FLOOR           = -1024.0
APPLY_RSNA_PIXEL_FIX   = True      # the gate is RescaleIntercept > -100, so on well-formed
                                   # Indian DICOMs it never fires -- see the note below
SAVE_SPATIAL_MAP       = True
SPATIAL_MAP_SIZE       = 2
SPATIAL_MAP_DTYPE      = "float16"
QC_REJECT_ON_FAIL      = True
QC_HU_P01_RANGE        = (-1200.0, -700.0)
QC_HU_P99_MAX          = 3000.0
SERIES_MIN_SLICES      = 20
MIN_KEPT_SLICE_FRACTION = 0.5
# 1, NOT 3. This chunks the three HU windows through the backbone one at a time. It does
# not change a single feature value -- the windows are independent and the results are
# concatenated either way -- but it is the difference between SwinUNETR peaking at 6.5 GB
# and asking for 3.5 GB more than a T4 has. Phase A ran at 1; setting 3 here is what made
# B1 drop out with "CUDA out of memory" while B2 and B3 sailed through.
EXTRACT_BATCH_SIZE     = 1
USE_AMP                = True
# True, not "auto". "auto" keeps all three backbones on the GPU and only falls back AFTER
# something runs out of memory -- and on this runtime that fallback did not save B1, it
# was dropped. Holding ONE backbone on the GPU and swapping as the loop moves between
# them caps the peak at a single model (~6.5 GB for SwinUNETR) instead of the sum.
#
# This is NOT the same as extracting one backbone at a time over three passes. The volume
# is still read, assembled and preprocessed ONCE per series and then shown to each
# backbone in turn -- which matters here, because reading a series out of a Drive-mounted
# day-zip is the expensive part and three passes would pay it three times. The only cost
# is moving ~250 MB of weights across PCIe per swap, about 0.1 s per series.
GPU_ONE_AT_A_TIME      = True
DROP_BACKBONE_ON_FAILURE = True
MAX_SERIES             = None      # India: take everything that passes QC. Only the zoo's
                                   # ETA line reads this, and it is None-safe below.

FEATURE_DIR  = os.path.join(INDIA_OUTPUT_DIR, "features")
SPATIAL_DIR  = os.path.join(INDIA_OUTPUT_DIR, "spatial_maps")
CKPT_DIR     = os.path.join(RSNA_OUTPUT_DIR, "checkpoints")   # reuse the cached SuPreM weights
OUTPUT_DIR   = INDIA_OUTPUT_DIR
for _d in (INDIA_OUTPUT_DIR, FEATURE_DIR, SPATIAL_DIR, CKPT_DIR):
    os.makedirs(_d, exist_ok=True)
os.environ.setdefault("HF_HOME", os.path.join(RSNA_OUTPUT_DIR, "hf_cache"))

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
if DEVICE != "cuda":
    print("  WARNING: no GPU. SwinUNETR on CPU is ~145 s per volume against ~3 s on a T4.\n"
          "  Runtime > Change runtime type > GPU before starting a real extraction.")

FEATURE_DIMS = {"B1_swinunetr_suprem": 2304, "B2_segresnet_suprem": 384,
                "B3_resnet18_medicalnet": 1536}


def preproc_signature(backbone_id):
    """Byte-for-byte the string Phase A wrote. If this differs, the two cohorts are not
    comparable, and nothing downstream would notice."""
    return "|".join([
        "v1", f"backbone={backbone_id}", f"shape={tuple(TARGET_SHAPE_DHW)}",
        f"spacing={tuple(TARGET_SPACING_XYZ)}", f"windows={[(n, c, w) for n, c, w in HU_WINDOWS]}",
        f"interp={RESAMPLE_INTERPOLATOR}", f"skullstrip={bool(SKULL_STRIP)}",
        f"smap={bool(SAVE_SPATIAL_MAP)}x{int(SPATIAL_MAP_SIZE)}",
        f"pixfix={bool(APPLY_RSNA_PIXEL_FIX)}",
        f"floor={float(HU_AIR_FLOOR) if CLAMP_HU_FLOOR else None}",
        f"qc={bool(QC_REJECT_ON_FAIL)}{tuple(QC_HU_P01_RANGE)}{float(QC_HU_P99_MAX)}",
        f"minslices={int(SERIES_MIN_SLICES)}",
        f"minkept={float(MIN_KEPT_SLICE_FRACTION)}",
    ])


# ---- the check that makes external validation mean something ---------------------------------
print("\npreprocessing signature, this notebook:")
for _b in BACKBONE_IDS:
    print(f"  {preproc_signature(_b)}")

_rsna_sigs, _missing = {}, []
for _b in BACKBONE_IDS:
    _p = os.path.join(RSNA_OUTPUT_DIR, f"extraction_progress_{_b}.jsonl")
    if not os.path.exists(_p):
        _missing.append(_b)
        continue
    with open(_p) as _fh:
        for _line in _fh:
            try:
                _r = json.loads(_line)
            except Exception:
                continue
            if _r.get("status") == "ok" and _r.get("preproc_sig"):
                _rsna_sigs[_b] = _r["preproc_sig"]
if _missing:
    print(f"\n  no Phase-A progress log for {_missing} at {RSNA_OUTPUT_DIR}.")
    print("  Cannot verify that these embeddings will match RSNA's. Point RSNA_OUTPUT_DIR at the")
    print("  folder holding features_B*.parquet, or accept that the comparison is unverified.")
_bad = {b: (_rsna_sigs[b], preproc_signature(b))
        for b in _rsna_sigs if _rsna_sigs[b] != preproc_signature(b)}
if _bad:
    print("\n" + "=" * 94)
    print("PREPROCESSING MISMATCH -- India would NOT be in the same feature space as RSNA")
    print("=" * 94)
    for _b, (_rs, _here) in _bad.items():
        print(f"  {_b}")
        _ra = dict(p.split("=", 1) for p in _rs.split("|") if "=" in p)
        _ha = dict(p.split("=", 1) for p in _here.split("|") if "=" in p)
        for _k in sorted(set(_ra) | set(_ha)):
            if _ra.get(_k) != _ha.get(_k):
                print(f"    {_k}: RSNA={_ra.get(_k)!r}  here={_ha.get(_k)!r}")
    raise SystemExit("Align the constants at the top of this cell with Phase A and re-run.")
if _rsna_sigs:
    print(f"\nsignature matches Phase A for {len(_rsna_sigs)} backbone(s). The two cohorts will "
          f"be in the same feature space.")

# A NOTE ON pixfix. The RSNA 12-bit wraparound correction is gated on RescaleIntercept > -100,
# and this notebook's _pixel_correction() is additionally a no-op unless DATA_SOURCE == "rsna".
# So the flag reads True in both signatures and the correction simply never fires on Indian
# files -- the code path is identical, the data does not trigger it. That is what you want:
# had the flag been set to False here, the signature would differ and every comparison with
# RSNA would be against differently-preprocessed features.

In [ ]:
# 12. Preprocessing for the 3D backbone -- resample -> whole-brain crop/pad -> HU windowing
#
# Every choice below is sourced or reasoned, not picked by hand.
#
#   RESAMPLE to TARGET_SPACING_XYZ using each scan's OWN measured spacing (assemble_volume
#     measures z from the ImagePositionPatient gaps rather than trusting SliceThickness, which
#     disagrees with reality whenever there is slice overlap or a gap). SimpleITK, sitkLinear by
#     default: in plane this is a DOWNsample on this dataset (~0.45 mm native to the configured
#     TARGET_SPACING_XYZ, so about 2.2x at 1.0 mm), where B-spline buys nothing and its overshoot
#     at the skull/air boundary is a real artefact. (This note used to quote a 4x factor and a
#     2.0 mm target, left over from an earlier default -- the grid-fit report in cell 10 prints
#     the real factor measured on your own cohort.) Set RESAMPLE_INTERPOLATOR = "bspline" if you disagree; the preprocessing
#     signature tracks the choice, so cached features are re-extracted rather than mixed.
#
#   CROP/PAD to TARGET_SHAPE_DHW on a fixed whole-head grid, NOT a lesion-centred crop: negatives
#     and deployment-time cases, where no lesion location is known in advance, must be framed
#     identically to training positives, or the framing itself leaks the label. True centre
#     crop/pad on each axis: start = (current - target) // 2. Padding uses the volume's own
#     minimum, which for a head CT is air.
#
#   DUAL HU WINDOWING, brain + subdural, each clipped to [C - W/2, C + W/2] and
#     rescaled to [0, 1]. Then each window is z-scored over the whole volume, which is MedicalNet's
#     own intensity normalisation (Med3D, Chen et al. 2019) -- the pretrained weights expect
#     zero-mean unit-variance input, and feeding them [0, 1] instead shifts every activation in
#     the frozen stem before a single learned parameter is involved.
#
#   ONE FORWARD PASS PER WINDOW. MONAI refuses `pretrained=True` for any n_input_channels != 1
#     (see monai/networks/nets/resnet.py::_resnet), so a 2-channel input would mean inflating
#     conv1's weights -- which changes the pretrained function. Running the frozen backbone twice
#     and concatenating keeps the published weights bit-exact. Cost: 2x the forward passes, which
#     on a T4 is ~0.1 s per volume against ~30 s of zip reading. Not the bottleneck.
#
# The functions are in src/qct3d/preprocessing.py, moved verbatim; notebooks 01 (RSNA) and 05
# (external cohort) import the same file, so the two cohorts cannot drift apart. The settings
# chosen in this notebook's configuration cell are handed to that module here.
import numpy as np
import SimpleITK as sitk
import torch

import qct3d.preprocessing as _prep

_prep.configure(TARGET_SPACING_XYZ=TARGET_SPACING_XYZ, TARGET_SHAPE_DHW=TARGET_SHAPE_DHW,
                HU_WINDOWS=HU_WINDOWS, RESAMPLE_INTERPOLATOR=RESAMPLE_INTERPOLATOR,
                SKULL_STRIP=SKULL_STRIP)
from qct3d.preprocessing import (_SITK_INTERP, _preprocessing_self_test,  # noqa: E402,F401
                                 center_crop_or_pad, hu_window, preprocess_for_backbone,
                                 resample_to_spacing)

_rs, _os = _preprocessing_self_test()
_fov = tuple(round(TARGET_SHAPE_DHW[i] * TARGET_SPACING_XYZ[::-1][i]) for i in range(3))
print(f"windows: {[(n, f'C{c:.0f}/W{w:.0f}') for n, c, w in HU_WINDOWS]}")
print(f"grid   : {tuple(TARGET_SHAPE_DHW)} (D,H,W) at {TARGET_SPACING_XYZ} mm "
      f"-> {_fov[0]} x {_fov[1]} x {_fov[2]} mm field of view (D,H,W)")
print(f"self-test: anisotropic 40x320x384 @ (0.7,0.7,3.0) mm -> resampled {_rs} -> model input {_os}")
print("preprocessing self-test passed (axis order, shape, standardisation, short-volume padding, "
      "windows actually differ)")
if not SKULL_STRIP:
    print()
    print("NOTE: SKULL_STRIP is False. A frozen backbone can read skull thickness, scalp and")
    print("      scanner fingerprints instead of parenchyma, and the AUROC that results looks")
    print("      entirely normal. See the closing notes before reporting any of these numbers.")

In [ ]:
# 13. The frozen backbone zoo -- and the loading that PROVES itself
#
# THE FAILURE THIS CELL EXISTS TO PREVENT
# ---------------------------------------
# If one backbone is pretrained and another is silently randomly initialised, then the
# grid is not comparing architectures -- it is comparing "has weights" with "does not",
# and the pretrained one wins for a reason that has nothing to do with inductive bias.
# Nothing downstream complains: a random 3D CNN produces perfectly healthy-looking
# features, trains, and reports a plausible AUROC of noise.
#
# A key count cannot catch this (a state_dict that matched 3 buffers and nothing else
# still "loads"). The only check that cannot be fooled is behavioural: build a second,
# freshly-random copy and confirm the two produce DIFFERENT features. That is what
# verify_pretrained_loaded does, and it raises rather than warns.
#
# THE ZOO
#   B1  SwinUNETR encoder  hybrid CNN-Transformer  SuPreM, 2,100 CT, supervised
#   B2  SegResNet encoder  pure CNN                SuPreM, 2,100 CT, supervised
#   B3  ResNet-18          pure CNN (classic)      MedicalNet, 23 datasets
#   B4  SwinUNETR encoder  hybrid                  self-supervised, 5,050 CT  (optional)
#
# B1 and B2 share a pretraining corpus and supervision signal, so B1-vs-B2 isolates
# ARCHITECTURE. B4-vs-B1 holds architecture fixed and isolates the PRETRAINING REGIME.
import contextlib
import gc
import os

import numpy as np
import torch
import torch.nn as nn

SUPREM_URLS = {
    "suprem_swinunetr": "https://huggingface.co/MrGiovanni/SuPreM/resolve/main/supervised_suprem_swinunetr_2100.pth",
    "suprem_segresnet": "https://huggingface.co/MrGiovanni/SuPreM/resolve/main/supervised_suprem_segresnet_2100.pth",
    "ssl_swinunetr_5050": "https://huggingface.co/MrGiovanni/SuPreM/resolve/main/self_supervised_nv_swin_unetr_5050.pt",
}


def download_checkpoint(key, dest_dir):
    """Cached on Drive: the ~250 MB SwinUNETR download happens once, not once per session."""
    url = SUPREM_URLS[key]
    os.makedirs(dest_dir, exist_ok=True)
    path = os.path.join(dest_dir, os.path.basename(url))
    if os.path.exists(path) and os.path.getsize(path) > 1_000_000:
        print(f"  checkpoint cached: {os.path.basename(path)} "
              f"({os.path.getsize(path)/1e6:.0f} MB)")
        return path
    print(f"  downloading {os.path.basename(url)} ...")
    try:
        from huggingface_hub import hf_hub_download
        got = hf_hub_download(repo_id="MrGiovanni/SuPreM",
                              filename=os.path.basename(url), local_dir=dest_dir)
        print(f"  saved {got}")
        return got
    except Exception as e:
        print(f"  huggingface_hub failed ({type(e).__name__}); falling back to a direct GET")
        import urllib.request
        tmp = path + ".part"
        urllib.request.urlretrieve(url, tmp)
        os.replace(tmp, path)
        print(f"  saved {path} ({os.path.getsize(path)/1e6:.0f} MB)")
        return path


def _unwrap_state_dict(obj):
    for key in ("state_dict", "net", "model", "module", "student"):
        if isinstance(obj, dict) and key in obj and isinstance(obj[key], dict):
            return _unwrap_state_dict(obj[key])
    return obj


_STRIP_PREFIXES = ("module.", "backbone.", "model.", "net.", "swinViT.", "swin_vit.",
                   "encoder.", "_orig_mod.")


def _candidate_names(k):
    """Every name a checkpoint key `k` could correspond to in MY module tree.

    THE CASE THAT USED TO BE MISSING. A released checkpoint is saved from the authors'
    module tree, not from mine. SuPreM's SwinUNETR keys look like
    `module.swinViT.patch_embed...`, and my wrapper stores that trunk as `self.swinViT`,
    so simply stripping `module.` lands on the right name -- which is why B1 loaded at
    100%. Their SegResNet keys look like `module.backbone.convInit...`, and my wrapper
    stores it as `self.net`, so the right name is `net.convInit...`: their prefix has to
    come OFF and mine has to go ON, in that order.

    The old version generated only the strip-chain, plus each prefix prepended to the
    ORIGINAL key. It never combined the two, so that rename was unreachable and the load
    matched nothing -- 0% coverage, and the behavioural check below would have stopped the
    run at cell 13 with a message about prefix handling. Correct behaviour, hours before
    it needed to be, but only because the check exists.

    Now the candidates are the CROSS PRODUCT of {every stripped form} x {nothing, or each
    prefix put back on}. `_match_by_unique_suffix` covers whatever naming this still fails
    to anticipate.
    """
    out, seen = [], []

    def add(x):
        if x and x not in seen:
            seen.append(x)
            out.append(x)

    stripped = [k]
    cur = k
    for _ in range(6):
        changed = False
        for p in _STRIP_PREFIXES:
            if cur.startswith(p) and len(cur) > len(p):
                cur = cur[len(p):]
                if cur not in stripped:
                    stripped.append(cur)
                changed = True
        if not changed:
            break
    for s in stripped:            # the stripped forms first: an exact hit is preferred
        add(s)
    for s in stripped:            # then each of them with one of my own names put back on
        for p in _STRIP_PREFIXES:
            add(p + s)
    return out


def _match_by_unique_suffix(sd, target, already):
    """Last resort for a checkpoint whose module tree we did not anticipate.

    Matches a checkpoint key to a target key when one target ends with the checkpoint
    key's trailing path AND that target is the ONLY one that does AND the shapes agree.
    Ambiguity is refused rather than guessed: a wrong-but-plausible mapping is the failure
    mode this whole cell exists to prevent, so a tie matches nothing.
    """
    free = [t for t in target if t not in already]
    extra = {}
    for k, v in sd.items():
        parts = k.split(".")
        for i in range(len(parts)):
            tail = "." + ".".join(parts[i:])
            hits = [t for t in free
                    if ("." + t).endswith(tail) and tuple(target[t].shape) == tuple(v.shape)]
            if len(hits) == 1:
                extra[hits[0]] = v
                free.remove(hits[0])
                break
            if len(hits) > 1:
                break             # ambiguous at this depth; a shorter tail is worse
    return extra


def load_state_dict_forgiving(module, ckpt_path, verbose=True):
    """Match by name-with-prefix-stripping AND shape. Reports; does not assert -- the
    behavioural check below is the one that actually decides."""
    raw = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    sd = _unwrap_state_dict(raw)
    if not isinstance(sd, dict):
        raise RuntimeError(f"{ckpt_path} holds no state_dict (got {type(sd).__name__})")
    sd = {k: v for k, v in sd.items() if isinstance(v, torch.Tensor)}
    target = module.state_dict()
    matched, mism = {}, []
    for k, v in sd.items():
        for cand in _candidate_names(k):
            if cand in target:
                if tuple(target[cand].shape) == tuple(v.shape):
                    matched[cand] = v
                else:
                    mism.append((k, tuple(v.shape), tuple(target[cand].shape)))
                break
    n_by_name = len(matched)
    if n_by_name < len(target):
        # Nothing above anticipated this checkpoint's module tree. Fall back to matching on
        # the trailing path, accepting only unambiguous, shape-compatible hits.
        matched.update(_match_by_unique_suffix(sd, target, set(matched)))
    module.load_state_dict({**target, **matched}, strict=False)
    tot = sum(int(np.prod(t.shape)) for t in target.values())
    got = sum(int(np.prod(t.shape)) for t in matched.values())
    cov = got / max(tot, 1)
    if verbose:
        print(f"  checkpoint tensors : {len(sd):,}")
        print(f"  matched            : {len(matched):,} / {len(target):,} tensors"
              + (f"  ({len(matched)-n_by_name} by unique suffix)"
                 if len(matched) > n_by_name else ""))
        print(f"  PARAMETER COVERAGE : {cov:.1%}")
        if mism:
            print(f"  shape mismatches   : {len(mism)} (e.g. {mism[:2]})")
    return {"coverage": cov, "matched": len(matched), "target": len(target),
            "by_suffix": len(matched) - n_by_name, "mismatches": len(mism)}


def verify_load_changed_model(model, load_fn, probe_shape=(1, 1, 64, 96, 96), atol=1e-6):
    """Compare the model against ITSELF, before and after loading. Unfoolable.

    WHY NOT "compare against a fresh random twin", which is the obvious design: two
    SEPARATELY constructed models have DIFFERENT random initialisations, so their features
    differ whether or not the checkpoint loaded anything. That check reports "weights
    loaded" for a model with 0% parameter coverage -- i.e. for a completely random network.
    It was written that way first, tested, and found to fail silently.

    Measuring the model against its own pre-load state has no such hole: if the checkpoint
    changed nothing, the features are bit-identical, and that is a fact about the
    checkpoint rather than about the RNG.
    """
    model.eval()
    torch.manual_seed(1234)
    x = torch.randn(*probe_shape)
    with torch.no_grad():
        before = model(x).float().clone()
    before_sd = {k: v.detach().clone() for k, v in model.state_dict().items()}

    rep = load_fn(model)

    after_sd = model.state_dict()
    n_changed = sum(1 for k in before_sd
                    if k in after_sd and not torch.equal(before_sd[k], after_sd[k]))
    model.eval()
    with torch.no_grad():
        after = model(x).float()
    rel = float((after - before).norm() / (before.norm() + 1e-9))
    identical = bool(torch.allclose(before, after, atol=atol))
    print(f"  tensors changed by the load : {n_changed} / {len(before_sd)}")
    print(f"  feature change vs pre-load  : {rel:.4f}")
    # FINITENESS FIRST. allclose() is False whenever either side holds NaN, so a checkpoint
    # that loads garbage -- negative BatchNorm variances are the usual way -- "differs from
    # the pre-load state" exactly as a good one does and passes the test below. The only
    # visible sign is `rel` printing as nan, one line above the word "verified".
    if not torch.isfinite(after).all():
        raise RuntimeError(
            "THE LOADED CHECKPOINT PRODUCES NON-FINITE FEATURES. The load did change the "
            "model, so the difference test below would have passed it -- but the model now "
            "outputs NaN or Inf, which means the tensors that arrived are not usable "
            "weights (negative BatchNorm running_var is the usual cause). This is a "
            "corrupt or mismatched checkpoint, not a loading-path problem.")
    if n_changed == 0 or identical:
        raise RuntimeError(
            "THE PRETRAINED WEIGHTS DID NOT REACH THIS BACKBONE. Loading the checkpoint "
            "changed nothing -- the model's features are bit-identical to what they were "
            "before the load, so it is still randomly initialised. Every number this "
            "notebook goes on to produce would be the output of a random network, and "
            "nothing else would complain. Fix the checkpoint path or the prefix handling "
            "above before continuing. Do NOT extract features in this state.")
    print("  -> weights verified: the load actually changed the model")
    return rep


def verify_constructor_loaded(build_pretrained, build_random, probe_shape=(1, 1, 64, 96, 96),
                              atol=1e-6):
    """For weights loaded INSIDE the constructor (MONAI's pretrained=True), where there is
    no pre-load state to snapshot. Both models are built under the SAME seed, so their
    random initialisations are identical by construction and any feature difference can
    only come from the pretrained weights."""
    torch.manual_seed(4321)
    m_pre = build_pretrained()
    torch.manual_seed(4321)
    m_rnd = build_random()
    m_pre.eval()
    m_rnd.eval()
    torch.manual_seed(1234)
    x = torch.randn(*probe_shape)
    with torch.no_grad():
        a, b = m_pre(x).float(), m_rnd(x).float()
    rel = float((a - b).norm() / (b.norm() + 1e-9))
    print(f"  pretrained vs same-seed random init: relative difference {rel:.4f}")
    # Same hole as in verify_load_changed_model: NaN makes the two "differ".
    if not torch.isfinite(a).all():
        raise RuntimeError(
            "THE PRETRAINED BACKBONE PRODUCES NON-FINITE FEATURES. It does differ from the "
            "same-seed random init, so the difference test below would have passed it -- "
            "but the weights that arrived are not usable (negative BatchNorm running_var is "
            "the usual cause). Do NOT extract features in this state.")
    if bool(torch.allclose(a, b, atol=atol)):
        raise RuntimeError(
            "THE PRETRAINED WEIGHTS DID NOT REACH THIS BACKBONE: built with "
            "pretrained=True it is bit-identical to the same architecture built with "
            "pretrained=False under the same seed. Every number downstream would come "
            "from a random network.")
    print("  -> weights verified: pretrained differs from same-seed random init")
    return m_pre


# ---- encoder wrappers -------------------------------------------------------------------------
class SwinUNETREncoder(nn.Module):
    """SwinUNETR's swinViT trunk; deepest stage has 16 * feature_size channels."""

    def __init__(self, feature_size=48, spatial_map_size=None):
        super().__init__()
        from monai.networks.nets import SwinUNETR
        # `img_size` was a REQUIRED positional argument in MONAI 1.3, optional-and-ignored
        # in 1.4, and gone in 1.5+. Calling the 1.5 signature on 1.3 raises TypeError; the
        # fallback keeps this cell working on every version in between rather than pinning
        # the notebook to whatever MONAI Colab happened to ship the week it was written.
        _kw = dict(in_channels=1, out_channels=1, feature_size=feature_size,
                   spatial_dims=3)
        try:
            net = SwinUNETR(**_kw)
        except TypeError:
            import warnings as _w
            with _w.catch_warnings():
                _w.simplefilter("ignore")
                net = SwinUNETR(img_size=(96, 96, 96), **_kw)   # MONAI <= 1.3
        del _kw
        self.swinViT = net.swinViT
        self.feat_dim = 16 * feature_size
        self.spatial_map_size = spatial_map_size

    def _deep(self, x):
        return self.swinViT(x, normalize=True)[-1]

    def forward(self, x):
        return self._deep(x).flatten(2).mean(-1)

    def forward_map(self, x):
        h = self._deep(x)
        if self.spatial_map_size:
            s = self.spatial_map_size
            h = nn.functional.adaptive_avg_pool3d(h, (s, s, s))
        return h


class SegResNetEncoder(nn.Module):
    def __init__(self, init_filters=16, blocks_down=(1, 2, 2, 4), spatial_map_size=None):
        super().__init__()
        from monai.networks.nets import SegResNet
        self.net = SegResNet(spatial_dims=3, in_channels=1, out_channels=1,
                             init_filters=init_filters, blocks_down=blocks_down,
                             blocks_up=(1,) * (len(blocks_down) - 1))
        self.feat_dim = init_filters * (2 ** (len(blocks_down) - 1))
        self.spatial_map_size = spatial_map_size

    def forward(self, x):
        h, _ = self.net.encode(x)
        return h.flatten(2).mean(-1)

    def forward_map(self, x):
        h, _ = self.net.encode(x)
        if self.spatial_map_size:
            s = self.spatial_map_size
            h = nn.functional.adaptive_avg_pool3d(h, (s, s, s))
        return h


class MedicalNetResNetEncoder(nn.Module):
    """MONAI refuses pretrained=True unless spatial_dims==3, n_input_channels==1,
    feed_forward is False and (shortcut_type, bias_downsample) match the depth. Passed
    explicitly, because getting them wrong on a different MONAI version yields a randomly
    initialised backbone that looks entirely healthy."""

    def __init__(self, depth=18, pretrained=True, spatial_map_size=None):
        super().__init__()
        from monai.networks.nets import resnet18, resnet34, resnet50
        builders = {18: resnet18, 34: resnet34, 50: resnet50}
        self.net = builders[depth](
            pretrained=pretrained, spatial_dims=3, n_input_channels=1, feed_forward=False,
            shortcut_type=("A" if depth in (18, 34) else "B"),
            bias_downsample=(depth in (18, 34)))
        self.feat_dim = 512 if depth in (18, 34) else 2048
        self.spatial_map_size = spatial_map_size

    def forward(self, x):
        return self.net(x).flatten(1)

    def forward_map(self, x):
        n = self.net
        h = n.act(n.bn1(n.conv1(x)))
        if not n.no_max_pool:
            h = n.maxpool(h)
        h = n.layer4(n.layer3(n.layer2(n.layer1(h))))
        if self.spatial_map_size:
            s = self.spatial_map_size
            h = nn.functional.adaptive_avg_pool3d(h, (s, s, s))
        return h


_SMS = SPATIAL_MAP_SIZE if SAVE_SPATIAL_MAP else None

BACKBONE_REGISTRY = {
    "B1_swinunetr_suprem": {
        "family": "hybrid CNN-Transformer",
        "arch": "SwinUNETR swinViT trunk (feature_size=48)",
        "weights": "SuPreM supervised, 2,100 CT volumes",
        "ckpt_key": "suprem_swinunetr",
        "feat_dim_per_window": 768,
        "builder": lambda: SwinUNETREncoder(48, spatial_map_size=_SMS),
        "citation": "Tang et al. CVPR 2022 (architecture); Li et al. ICLR 2024 (SuPreM weights)",
    },
    "B2_segresnet_suprem": {
        "family": "pure CNN",
        "arch": "SegResNet encoder (init_filters=16, blocks_down=(1,2,2,4))",
        "weights": "SuPreM supervised, 2,100 CT volumes -- SAME corpus as B1",
        "ckpt_key": "suprem_segresnet",
        "feat_dim_per_window": 128,
        "builder": lambda: SegResNetEncoder(16, (1, 2, 2, 4), spatial_map_size=_SMS),
        "citation": "Myronenko 2018 (architecture); Li et al. ICLR 2024 (SuPreM weights)",
    },
    "B3_resnet18_medicalnet": {
        "family": "pure CNN (classic)",
        "arch": "3D ResNet-18",
        "weights": "MedicalNet, 23 medical datasets (MONAI pretrained=True)",
        "ckpt_key": None,
        "feat_dim_per_window": 512,
        "builder": lambda: MedicalNetResNetEncoder(18, True, spatial_map_size=_SMS),
        "citation": "Chen et al. 2019, Med3D / MedicalNet",
    },
    "B4_swinunetr_ssl5050": {
        "family": "hybrid CNN-Transformer",
        "arch": "SwinUNETR swinViT trunk (feature_size=48)",
        "weights": "self-supervised, 5,050 CT volumes",
        "ckpt_key": "ssl_swinunetr_5050",
        "feat_dim_per_window": 768,
        "builder": lambda: SwinUNETREncoder(48, spatial_map_size=_SMS),
        "citation": "Tang et al. CVPR 2022",
    },
}


def build_backbone(name):
    spec = BACKBONE_REGISTRY[name]
    print(f"backbone {name}")
    print(f"  family   : {spec['family']}")
    print(f"  arch     : {spec['arch']}")
    print(f"  weights  : {spec['weights']}")
    print(f"  citation : {spec['citation']}")
    if spec["ckpt_key"]:
        model = spec["builder"]()
        path = download_checkpoint(spec["ckpt_key"], CKPT_DIR)
        rep = verify_load_changed_model(
            model, lambda m: load_state_dict_forgiving(m, path))
        if rep["coverage"] < 0.50:
            print(f"  WARNING: only {rep['coverage']:.1%} of parameters matched. The load "
                  f"did change the model, so this is not fatal -- but check the missing "
                  f"keys above before trusting the features.")
    else:
        # MONAI loads inside the constructor, so there is no pre-load state to snapshot.
        model = verify_constructor_loaded(
            lambda: MedicalNetResNetEncoder(18, True, spatial_map_size=_SMS),
            lambda: MedicalNetResNetEncoder(18, False, spatial_map_size=_SMS))

    model.eval().to(DEVICE)
    for p in model.parameters():
        p.requires_grad_(False)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if n_train:
        raise RuntimeError(f"{name} still has {n_train:,} trainable parameters after "
                           f"freezing; the entire design assumes a frozen backbone.")
    if model.feat_dim != spec["feat_dim_per_window"]:
        raise RuntimeError(
            f"{name} produced {model.feat_dim}-d features but the registry (and therefore "
            f"FEATURE_DIM, and therefore every cached vector) says "
            f"{spec['feat_dim_per_window']}. Fix the registry before extracting.")
    print(f"  parameters: {sum(p.numel() for p in model.parameters()):,} (frozen, verified)")
    return model, spec


@torch.no_grad()
def extract_features(model, x, use_amp=None):
    """x: (n_windows, 1, D, H, W). Returns the concatenated pooled vector, and -- when
    SAVE_SPATIAL_MAP -- the stacked spatial maps for the Q3 arm."""
    use_amp = (USE_AMP if use_amp is None else use_amp) and DEVICE == "cuda"
    x = x.to(DEVICE, non_blocking=True)
    vecs, maps = [], []
    bs = max(1, int(EXTRACT_BATCH_SIZE))
    for i in range(0, x.shape[0], bs):
        chunk = x[i:i + bs]
        if use_amp:
            with torch.autocast("cuda", dtype=torch.float16):
                v = model(chunk)
                m = model.forward_map(chunk) if SAVE_SPATIAL_MAP else None
        else:
            v = model(chunk)
            m = model.forward_map(chunk) if SAVE_SPATIAL_MAP else None
        vecs.append(v.float().cpu())
        if m is not None:
            maps.append(m.float().cpu())
    feats = torch.cat(vecs, 0).reshape(-1).numpy().astype(np.float32)
    smap = None
    if SAVE_SPATIAL_MAP and maps:
        # (n_windows, C, s, s, s) -> concatenate along channels: (n_windows*C, s, s, s)
        smap = torch.cat(maps, 0).numpy()
        smap = smap.reshape(-1, *smap.shape[-3:]).astype(SPATIAL_MAP_DTYPE)
    return feats, smap


# Every selected backbone is held in memory at once. Combined that is ~100M frozen
# parameters (~400 MB fp32) for the default three -- comfortable on any GPU that can run
# the extraction at all, and the price of reading the archive only once.
import time as _time

LOADED, SPECS, DROPPED = {}, {}, {}
for _bid in list(BACKBONE_IDS):
    _t0 = _time.time()
    try:
        LOADED[_bid], SPECS[_bid] = build_backbone(_bid)
        print(f"  loaded in {_time.time()-_t0:.1f}s on {DEVICE}\n")
    except Exception as _e:
        if not DROP_BACKBONE_ON_FAILURE:
            raise
        DROPPED[_bid] = f"{type(_e).__name__}: {_e}"
        LOADED.pop(_bid, None)
        SPECS.pop(_bid, None)
        print(f"  *** {_bid} DID NOT LOAD -- dropping it and continuing ***")
        print(f"      {DROPPED[_bid]}\n")
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

BACKBONE_IDS = [b for b in BACKBONE_IDS if b in LOADED]
if not BACKBONE_IDS:
    raise RuntimeError(
        "not one backbone could be built, so there is nothing to extract. The reasons are "
        f"printed above: {DROPPED}")
BACKBONE_ID = BACKBONE_IDS[0]
_spec = SPECS[BACKBONE_ID]

# ---- one backbone on the GPU, or all of them? decided by measurement below --------------------
ONE_AT_A_TIME = (GPU_ONE_AT_A_TIME is True or str(GPU_ONE_AT_A_TIME).lower() == "true")
_GPU_RESIDENT = None


def _activate(bid):
    """LOADED[bid], guaranteed to be on the compute device. A no-op unless ONE_AT_A_TIME."""
    global _GPU_RESIDENT
    m = LOADED[bid]
    if not ONE_AT_A_TIME or DEVICE != "cuda":
        return m
    if _GPU_RESIDENT != bid:
        if _GPU_RESIDENT is not None and _GPU_RESIDENT in LOADED:
            LOADED[_GPU_RESIDENT].to("cpu")
        m.to(DEVICE)
        _GPU_RESIDENT = bid
        torch.cuda.empty_cache()
    return m


def _oom_escalate(why=""):
    """Fall back to one backbone on the GPU. Idempotent; safe to call from the hot loop."""
    global ONE_AT_A_TIME, _GPU_RESIDENT
    if ONE_AT_A_TIME or DEVICE != "cuda":
        return False
    ONE_AT_A_TIME = True
    for _o in list(LOADED):
        LOADED[_o].to("cpu")
    _GPU_RESIDENT = None
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  -> switched to ONE backbone on the GPU at a time{(' (' + why + ')') if why else ''}. "
          f"Costs ~0.1 s per series in host-to-device copies; nothing else changes.")
    return True


if ONE_AT_A_TIME:
    _oom_escalate.__wrapped__ = None          # marker: the fallback was requested, not measured
    for _o in list(LOADED):
        LOADED[_o].to("cpu")
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print("GPU_ONE_AT_A_TIME is on: backbones are held on the CPU and moved to the GPU in turn.")
print(f"{len(LOADED)} backbone(s) resident: "
      f"{sum(sum(p.numel() for p in m.parameters()) for m in LOADED.values()):,} "
      f"frozen parameters total")


# ---- prove the whole model path works before spending hours on it -----------------------------
# Synthetic head phantoms, not random noise: a soft-tissue ellipsoid inside a dense shell
# inside air. Random noise passes every shape assertion while telling you nothing about
# whether the windowing produced anything with structure in it.
def _phantom(D=160, H=512, W=512, seed=1):
    # np.ogrid, NOT np.mgrid. mgrid materialises three int64 arrays of D*H*W elements --
    # 1.0 GB for the default 160x512x512 -- and this runs once per backbone, so the
    # self-test alone allocated ~3 GB it never needed. ogrid returns broadcastable views
    # and the arithmetic below broadcasts identically, for a few hundred bytes.
    rng = np.random.default_rng(seed)
    zz, yy, xx = np.ogrid[0:D, 0:H, 0:W]
    r = ((zz - D / 2) / 62) ** 2 + ((yy - H / 2) / 190) ** 2 + ((xx - W / 2) / 175) ** 2
    vol = np.full((D, H, W), -1000.0, np.float32)
    vol[r < 1.00] = 35.0
    vol[(r >= 1.00) & (r < 1.14)] = 900.0
    vol += rng.standard_normal(vol.shape).astype(np.float32) * 8.0
    return vol, (zz, yy, xx, D, H, W)


def _backbone_self_test(model, spec):
    base, (zz, yy, xx, D, H, W) = _phantom()
    lesion = base.copy()
    lesion[((zz - D / 2) / 34) ** 2 + ((yy - H / 2 - 40) / 60) ** 2
           + ((xx - W / 2 - 30) / 60) ** 2 < 1.0] = 68.0
    noise = (np.random.default_rng(3).standard_normal((D, H, W)).astype(np.float32)
             * 300.0) - 200.0
    sp = {"sx": 0.45, "sy": 0.45, "sz": 1.0}
    # Time ONE real extraction (all windows, at the configured grid) -- this is the number
    # the extraction cell quotes as its ETA. Timing the whole self-test instead would fold
    # in phantom generation and resampling and overstate the forward-pass cost.
    _xb = preprocess_for_backbone(base, sp)
    import time as _t
    # WARM UP FIRST. The first forward pass through a model on CUDA pays for lazy context
    # creation, cuDNN algorithm selection and kernel autotuning -- routinely several
    # seconds that are never paid again. Timing that pass and calling it "seconds per
    # volume" overstates the steady-state cost by an order of magnitude, and this number
    # is what the CPU gate and every ETA in the extraction cell are computed from.
    # CUDA is also asynchronous: without a synchronize, the clock stops while kernels are
    # still queued. (extract_features ends in .cpu(), which does synchronise, but relying
    # on that is exactly the kind of accident that stops being true after an edit.)
    f_base, m_base = extract_features(model, _xb)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    _t0 = _t.time()
    f_base, m_base = extract_features(model, _xb)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    _sec_per_vol = _t.time() - _t0
    f_les, _ = extract_features(model, preprocess_for_backbone(lesion, sp))
    f_noise, _ = extract_features(model, preprocess_for_backbone(noise, sp))

    exp = spec["feat_dim_per_window"] * len(HU_WINDOWS)
    assert f_base.shape == (exp,), f"feature dim {f_base.shape} != expected ({exp},)"
    assert np.isfinite(f_base).all(), "non-finite features"
    assert float(np.std(f_base)) > 1e-6, "the backbone returned a constant vector"

    def rel(a, b):
        return float(np.linalg.norm(a - b) / max(np.linalg.norm(a), 1e-12))

    r_les, r_noise = rel(f_base, f_les), rel(f_base, f_noise)
    assert r_les > 1e-9, (
        "adding a hyperdense collection changed the features by less than 1e-9 relative -- "
        "two different volumes produced the same features. Suspect preprocess_for_backbone "
        "handing the model the same array twice.")
    assert r_noise > 2 * r_les, (
        f"pure noise moved the features only {r_noise/max(r_les,1e-12):.1f}x as much as an "
        f"added lesion ({r_noise:.4f} vs {r_les:.4f}) -- not enough separation to believe "
        f"the backbone is responding to image content.")
    return exp, f_base, r_les, r_noise, m_base, _sec_per_vol


# Time a REAL forward pass at the configured grid, so the extraction cell can quote a
# measured ETA instead of a hardcoded constant. The inherited "~2 s per volume" was
# measured for ResNet-18 on a smaller grid; at (64,256,256) with three windows the true
# CPU cost is ~26 s for ResNet-18 and ~106 s for SwinUNETR -- a 50x understatement for the
# heaviest backbone, which is exactly the number someone would rely on when deciding
# whether to set ALLOW_CPU_EXTRACTION.
MEASURED_SEC_PER_VOLUME, PEAK_GPU_GB = {}, {}


def _self_test_once(bid):
    """Run the self-test with peak GPU memory recorded. Raises whatever it raises."""
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    out = _backbone_self_test(_activate(bid), SPECS[bid])
    if DEVICE == "cuda":
        PEAK_GPU_GB[bid] = torch.cuda.max_memory_allocated() / 2 ** 30
    return out


for _bid in list(BACKBONE_IDS):
    _res = None
    try:
        _res = _self_test_once(_bid)
    except torch.cuda.OutOfMemoryError as _e:
        # The real deployment condition just ran out of memory. Fall back to one backbone on
        # the GPU and try again -- that is the whole point of measuring before extracting.
        gc.collect()
        torch.cuda.empty_cache()
        if _oom_escalate(f"{_bid} ran out of memory with {len(LOADED)} backbones resident"):
            try:
                _res = _self_test_once(_bid)
            except Exception as _e2:
                _e = _e2
        if _res is None:
            DROPPED[_bid] = f"{type(_e).__name__}: {_e}"
    except Exception as _e:
        DROPPED[_bid] = f"{type(_e).__name__}: {_e}"

    if _res is None:
        if not DROP_BACKBONE_ON_FAILURE:
            raise RuntimeError(f"{_bid} failed its self-test: {DROPPED.get(_bid)}")
        print(f"\n  *** {_bid} FAILED ITS SELF-TEST -- dropping it and continuing ***")
        print(f"      {DROPPED[_bid]}")
        LOADED.pop(_bid, None)
        SPECS.pop(_bid, None)
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        continue

    _exp, _f, _rl, _rn, _m, _sec = _res
    MEASURED_SEC_PER_VOLUME[_bid] = _sec
    if FEATURE_DIMS[_bid] != _exp:
        DROPPED[_bid] = (f"FEATURE_DIMS says {FEATURE_DIMS[_bid]} but the backbone produced "
                         f"{_exp}; the registry and the cached vectors would disagree")
        print(f"\n  *** {_bid}: {DROPPED[_bid]} -- dropping it ***")
        LOADED.pop(_bid, None)
        SPECS.pop(_bid, None)
        MEASURED_SEC_PER_VOLUME.pop(_bid, None)
        continue
    print(f"\n[{_bid}] self-test on synthetic head phantoms")
    print(f"  feature vector          : {_exp}-d ({len(HU_WINDOWS)} windows x "
          f"{SPECS[_bid]['feat_dim_per_window']}), all finite")
    print(f"  + hyperdense collection : relative change {_rl:.5f}")
    print(f"  vs pure noise           : relative change {_rn:.5f}")
    print(f"  -> responds {_rn/max(_rl,1e-12):.0f}x more to a different volume than to one "
          f"added lesion, which is the right ordering.")
    if _m is not None:
        print(f"  spatial map             : {_m.shape} -> {_m.nbytes/1024:.0f} KB per series")
    print(f"  MEASURED forward cost   : {_sec:.2f} s per volume on {DEVICE} "
          f"(warmed up; the first pass is not what you pay 4,000 times)")
    if _bid in PEAK_GPU_GB:
        _tot_gb = torch.cuda.get_device_properties(0).total_memory / 2 ** 30
        print(f"  PEAK GPU memory         : {PEAK_GPU_GB[_bid]:.2f} GB of {_tot_gb:.1f} GB"
              f"   ({PEAK_GPU_GB[_bid]/_tot_gb:.0%} of the card)")
BACKBONE_IDS = [b for b in BACKBONE_IDS if b in LOADED]
if not BACKBONE_IDS:
    raise RuntimeError(f"every backbone was dropped, so there is nothing to extract: {DROPPED}")
BACKBONE_ID = BACKBONE_IDS[0]
_spec = SPECS[BACKBONE_ID]
if DROPPED:
    print("\n" + "=" * 94)
    print(f"DROPPED {len(DROPPED)} BACKBONE(S) -- extraction continues without them")
    for _b, _why in DROPPED.items():
        print(f"  {_b}\n    {str(_why)[:300]}")
    print("  Re-run this notebook later with only these listed in BACKBONE_IDS; everything")
    print("  already extracted tonight is skipped, so it costs only what these need.")
    print("=" * 94)

# Written to Drive so nb3's RQ4 cell can quote a MEASURED cost per volume for the 3D
# arm without rebuilding these backbones (and re-downloading their checkpoints) just to
# time them. RQ4's claim is accuracy PER UNIT COST, so the cost half has to be real.
_cost = {
    "device": str(DEVICE), "n_windows": len(HU_WINDOWS),
    "grid": list(TARGET_SHAPE_DHW), "spacing": list(TARGET_SPACING_XYZ),
    "backbones": {b: {"sec_per_volume": float(MEASURED_SEC_PER_VOLUME[b]),
                      "parameters": int(sum(p.numel() for p in LOADED[b].parameters())),
                      "feature_dim": int(FEATURE_DIMS[b]),
                      "arch": SPECS[b]["arch"], "family": SPECS[b]["family"]}
                  for b in BACKBONE_IDS},
}
with contextlib.suppress(Exception):
    import json as _json
    # MERGE, NEVER REPLACE, AND KEY BY DEVICE. This file is read by nb3 to price the 3D arm
    # in RQ4's accuracy-per-cost claim. It used to be overwritten wholesale by whichever
    # notebook ran last -- so a CPU-only session replaced the T4 measurements and RQ4 quoted
    # 144.89 s per volume against the 3.24 s that was actually measured on the GPU.
    _cost_path = os.path.join(OUTPUT_DIR, "backbone_cost.json")
    _all = {}
    if os.path.exists(_cost_path):
        with contextlib.suppress(Exception):
            _all = _json.load(open(_cost_path))
    if "by_device" not in _all:                      # migrate the old flat layout
        _all = {"by_device": ({_all.get("device", "unknown"): _all} if _all else {})}
    _all["by_device"][str(DEVICE)] = _cost
    _all.update({k: v for k, v in _cost.items() if k != "backbones"})
    _all["backbones"] = _cost["backbones"]           # top level = this run, for compatibility
    with open(_cost_path, "w") as _fh:
        _json.dump(_all, _fh, indent=2)
    print(f"\nbackbone cost written -> {os.path.join(OUTPUT_DIR, 'backbone_cost.json')}")

_TOTAL_SEC_PER_VOL = sum(MEASURED_SEC_PER_VOLUME.values())
print(f"\nALL BACKBONES: {_TOTAL_SEC_PER_VOL:.2f} s of GPU per volume combined "
      f"({len(BACKBONE_IDS)} backbones x {len(HU_WINDOWS)} windows)")
# MAX_SERIES is None when the cohort is "everything that passes QC" (the India path), and
# both the multiplication and the thousands-separator format raise on None. This is an ETA
# line; it must not be able to end a run in which every backbone has just loaded correctly.
if MAX_SERIES:
    print(f"  -> {_TOTAL_SEC_PER_VOL*MAX_SERIES/3600:.1f} h of forward passes for "
          f"MAX_SERIES={MAX_SERIES:,}, before any Drive reads.")
else:
    print(f"  -> no series cap set; at {_TOTAL_SEC_PER_VOL:.2f} s/volume that is "
          f"{_TOTAL_SEC_PER_VOL*1000/3600:.1f} h per 1,000 series of forward passes.")
print(f"  Reading the archive is the dominant cost on this dataset, and it is now paid")
print(f"  ONCE for all {len(BACKBONE_IDS)} backbones rather than once each.")
if DEVICE == "cpu":
    print(f"\n  This is a CPU runtime. Phase A wants a GPU: Runtime > Change runtime type")
    print(f"  > GPU, then Runtime > Restart session. Nothing cached is lost.")
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# ==============================================================================================
# 16c. SHIM -- so the extraction loop below runs unchanged
# ==============================================================================================
# The loop in the next cell calls preprocess_for_pillar0(), extract_embedding(), LOADED and
# MODELS_TO_RUN. Rather than edit a 40 KB loop that already handles zip grouping, resumability,
# budgets and retries, those four names are re-pointed at our equivalents here.
#
# The one thing the loop does not know about is the SPATIAL MAP. Q3 (quanvolution) is the
# strongest arm on RSNA and it cannot run from pooled vectors, so the map is written as a side
# effect of extract_embedding() rather than by changing the loop's save path. It lands in the
# same layout Phase A used: spatial_maps/<backbone>/<series>.npy, float16, (C, 2, 2, 2).
import numpy as np

MODELS_TO_RUN = list(BACKBONE_IDS)
if "assert_no_pillar0" in globals():          # the guard set up in the gate cell above
    assert_no_pillar0(MODELS_TO_RUN, "extraction loop")
EMBED_DIR = FEATURE_DIR
PREPROC_CODE_VERSION = 1
PREPROC_SIG = preproc_signature(BACKBONE_IDS[0])
for _b in MODELS_TO_RUN:
    os.makedirs(os.path.join(FEATURE_DIR, _b), exist_ok=True)
    os.makedirs(os.path.join(SPATIAL_DIR, _b), exist_ok=True)

_CURRENT_SERIES = {"suid": None}


def _head_retention(volume, spacing):
    """What fraction of the head survives our fixed 192 x 256 x 256 mm window?

    The loop already logs head_retained / _z / _inplane out of the dict assemble_volume
    returns, and prints a summary at the end -- but the function that used to fill those keys
    lived in the Pillar-0 preprocessing cell and went with it. Left unfilled, that summary
    silently reports the PREVIOUS Pillar-0 run's numbers as if they described this one.
    This fills them for real, so the report is about the run that produced the embeddings.

    It matters more here than it did on RSNA. RSNA head CT is a median 33 slices and 155 mm
    of z; these scans are a median 173 slices. If they are thin-slice reconstructions the
    192 mm window holds them comfortably; if they are not, the vertex and the skull base go
    first -- and those are where subdural and epidural collections sit.

    Measured against a window centred on the VOLUME centre, which is what center_crop_or_pad
    actually does, not on the head centre.
    """
    try:
        head = volume > -300.0
        n = int(head.sum())
        if n == 0:
            return None
        D, H, W = volume.shape
        sx, sy, sz = float(spacing["sx"]), float(spacing["sy"]), float(spacing["sz"])
        hz = TARGET_SHAPE_DHW[0] * TARGET_SPACING_XYZ[2] / 2.0 / max(sz, 1e-6)
        hy = TARGET_SHAPE_DHW[1] * TARGET_SPACING_XYZ[1] / 2.0 / max(sy, 1e-6)
        hx = TARGET_SHAPE_DHW[2] * TARGET_SPACING_XYZ[0] / 2.0 / max(sx, 1e-6)
        zi = np.abs(np.arange(D) - D / 2.0 + 0.5) <= hz
        yi = np.abs(np.arange(H) - H / 2.0 + 0.5) <= hy
        xi = np.abs(np.arange(W) - W / 2.0 + 0.5) <= hx
        r_z = float(head[zi].sum()) / n
        sub = head[zi]
        r_xy = (float(sub[:, yi][:, :, xi].sum()) / max(float(sub.sum()), 1.0)) if sub.any() else 1.0
        return r_z * r_xy, r_z, r_xy
    except Exception:
        return None


def preprocess_for_pillar0(volume, spacing, target_spatial=None, target_depth=None,
                           info=None, **kwargs):
    """Phase A's preprocessing, behind the name the loop already calls."""
    if isinstance(info, dict) and "head_retained" not in info:
        _r = _head_retention(volume, spacing)
        if _r is not None:
            info["head_retained"], info["head_retained_z"], info["head_retained_inplane"] = _r
    return preprocess_for_backbone(volume, spacing)


def extract_embedding(model, x, backbone_id=None, use_amp=None):
    """Pooled feature vector, with the spatial map written alongside it.

    `backbone_id` is resolved by identity against LOADED because the loop passes only the
    model object. Matching on the object is exact; matching on feature width would confuse
    two backbones that happened to share one.
    """
    if backbone_id is None:
        backbone_id = next((b for b, m in LOADED.items() if m is model), None)
    # use_amp is forwarded because the mixed-precision comparison cell above calls this
    # with use_amp=True and use_amp=False to measure how much autocast moves the
    # embeddings. Dropping the argument made that cell raise TypeError.
    feats, smap = extract_features(_activate(backbone_id) if backbone_id else model, x,
                                   use_amp=use_amp)
    if SAVE_SPATIAL_MAP and smap is not None and backbone_id and _CURRENT_SERIES["suid"]:
        _p = os.path.join(SPATIAL_DIR, backbone_id, f"{_CURRENT_SERIES['suid']}.npy")
        _tmp = _p + ".partial"
        try:
            with open(_tmp, "wb") as _fh:
                np.save(_fh, smap.astype(SPATIAL_MAP_DTYPE))
            os.replace(_tmp, _p)
        except Exception as _e:
            print(f"    spatial map not written for {_CURRENT_SERIES['suid']} [{backbone_id}]: {_e}")
            if os.path.exists(_tmp):
                os.remove(_tmp)
    return feats


# The loop names the series `suid` in its own scope; this makes the current one visible to
# extract_embedding without touching the loop. Called by the patched loop below.
def _set_current_series(suid):
    _CURRENT_SERIES["suid"] = suid


# ---- a dropped backbone must never be something you discover in the morning ---------------
# The zoo drops a backbone that will not load or will not fit and carries on, which is right:
# two of three beats none of three. But B1 is the backbone the RSNA finalist adapter was
# trained on, so extracting India without it silently would produce a cohort that cannot be
# used for the transfer experiment at all. If any backbone is missing, say so loudly here and
# leave a record on Drive next to the features.
_DROPPED = {b: str(r) for b, r in globals().get("DROPPED", {}).items()}
if _DROPPED:
    print("=" * 94)
    print(f"!! {len(_DROPPED)} BACKBONE(S) MISSING FROM THIS RUN")
    for _b, _r in _DROPPED.items():
        print(f"   {_b}\n     {_r[:220]}")
    print("   Everything below extracts WITHOUT them. Re-run this notebook later with only the")
    print("   missing ones listed in BACKBONE_IDS -- series already done are skipped.")
    if "B1_swinunetr_suprem" in _DROPPED:
        print("   B1 is the backbone the RSNA finalist adapter was trained on. Without it the")
        print("   adapter-transfer experiment cannot run on this cohort.")
    print("=" * 94)
    with open(os.path.join(INDIA_OUTPUT_DIR, "dropped_backbones.json"), "w") as _fh:
        json.dump(_DROPPED, _fh, indent=2)
else:
    _p = os.path.join(INDIA_OUTPUT_DIR, "dropped_backbones.json")
    if os.path.exists(_p):
        os.remove(_p)      # a clean run must not leave a stale complaint behind

print(f"shim ready: {len(MODELS_TO_RUN)} backbone(s) -> {FEATURE_DIR}")
print(f"            spatial maps -> {SPATIAL_DIR}")
print("cells below (extraction loop, consolidation, QC) run unchanged.")

In [ ]:
# 16b. How much does mixed precision actually change the embeddings? (optional, ~1 min)
#
# USE_AMP roughly halves extraction time on a T4, which at a few thousand volumes is the
# difference between one overnight session and two. That is only a good trade if the embeddings
# are effectively the same, so measure it instead of assuming. Every progress record logs which
# precision produced it, so even a mixed run stays auditable afterwards.
import numpy as np
import torch

if DEVICE != "cuda":
    print("no GPU -- AMP is off regardless, nothing to compare.")
elif not USE_AMP:
    print("USE_AMP is False -- everything runs in fp32, nothing to compare.")
else:
    _rng = np.random.default_rng(11)
    _sims = {name: [] for name in LOADED}
    for _trial in range(3):
        _z, _y, _x = np.mgrid[0:80, 0:280, 0:280]
        _cz, _cy, _cx = 40 + _trial * 3, 140, 140
        _rr = (((_z - _cz) / 38.0) ** 2 + ((_y - _cy) / 120.0) ** 2 + ((_x - _cx) / 120.0) ** 2)
        _v = np.full(_rr.shape, -1000.0, dtype=np.float32)
        _v[_rr < 1.0] = 30.0 + _trial * 5
        _v[(_rr >= 0.9) & (_rr < 1.0)] = 850.0
        _v += _rng.standard_normal(_v.shape).astype(np.float32) * 10.0
        _t = preprocess_for_pillar0(_v, {"sx": 0.55, "sy": 0.55, "sz": 2.0})
        for _name, _b in LOADED.items():
            _a = extract_embedding(_b, _t, use_amp=True)
            _f = extract_embedding(_b, _t, use_amp=False)
            _sims[_name].append(
                float(np.dot(_a, _f) / (np.linalg.norm(_a) * np.linalg.norm(_f) + 1e-12)))
        del _t
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    print("cosine similarity, fp16-autocast vs fp32, on 3 synthetic volumes:")
    for _name, _vals in _sims.items():
        _m = float(np.mean(_vals))
        print(f"  {_name}: {['%.6f' % v for v in _vals]}  (mean {_m:.6f})")
        if _m > 0.999:
            print("    -> effectively identical; keep USE_AMP = True.")
        elif _m > 0.99:
            print("    -> very close. Fine for a linear probe, but note it in your results.")
        else:
            print("    -> NOT close enough. Set USE_AMP = False and re-run; the speedup is not")
            print("       worth embeddings that differ this much from the fp32 reference.")

## The extraction loop

Work is ordered **by day-zip**, not by series. Opening a 25 GB archive on Drive is the expensive
part; every wanted series inside it is then handled before moving on, and the archive is opened
exactly once.

Three things stop the loop cleanly, and all of them leave a resumable state:

* `TRIAL_RUN_MAX_SERIES` — a small end-to-end proof on real data. Trial series are logged `ok` like
  any other, so switching it to `None` afterwards costs nothing.
* `MAX_SERIES_PER_RUN` / `MAX_SECONDS_PER_RUN` — for fitting inside a Colab session.
* Ctrl-C — trapped, flushed, reported.

Only `status == "ok"` counts as done. A series that failed is retried on the next run, which is
what makes "fix the bug, re-run the cell" the correct response to any failure rather than a
question about how to un-stick things. A day-zip whose every wanted series is `ok` gets recorded in
`zips_done.json` and is never opened again.

Per-zip timing is printed split into **locate / read / preprocess / model**, because which of those
dominates is the only honest way to choose between `ZIP_ACCESS_MODE = "stream"` and `"local_copy"`.

In [ ]:
# ==============================================================================================
# 16d. RELABEL THE EMBEDDINGS ALREADY ON DRIVE  [defines a function; the loop calls it]
# ==============================================================================================
# The previous run produced 1,200 correct embeddings and stamped them with a signature that
# described Pillar-0's geometry (depth=128, inplane=fit_head) rather than ours. The vectors are
# right -- the zoo's self-test printed "model input (3, 1, 64, 256, 256)" and every series went
# through preprocess_for_backbone -- but the LABEL on them is wrong, and the loop admits
# embeddings by label. With the geometry corrected in cell 15, those 1,200 would now be judged
# stale and re-extracted from scratch: about five hours for vectors that already exist and are
# already correct.
#
# So this relabels rather than recomputes.
#
# WHY IT IS A FUNCTION AND NOT A CELL THAT RUNS NOW. The signature that matters is the one the
# EXTRACTION LOOP writes, and the loop builds it from its own preproc_signature() after setting
# PREPROC_CODE_VERSION = 4 -- a value that does not exist yet at this point in the notebook (the
# shim set 1). Anything computed here would be stamping records with a string the loop does not
# recognise, which is the same bug again with a different value in it. So the loop calls this
# with its own PREPROC_SIG, one line after it computes it and before it reads the progress log.
# There is exactly one signature in play and it is the loop's.
#
# WHAT IT WILL TOUCH. A record qualifies only if ALL of these hold:
#   * it is logged "ok" and its signature is NOT the current one;
#   * its signature says channels=3 -- our three HU windows. Pillar-0 wrote channels=11, so no
#     Pillar-0 embedding can ever match this, whatever else is in the log;
#   * its signature differs from the current one ONLY in the geometry terms this fix corrected
#     (depth and inplane). A record that also disagrees on spacing, windows, minslices or the
#     code version is a genuinely different embedding and is left to be re-extracted;
#   * the .npy is on disk AND its length equals FEATURE_DIMS for that backbone;
#   * the SPATIAL MAP is on disk too. Q3 quanvolution is the strongest arm on RSNA and it
#     cannot run from pooled vectors -- it reads spatial_maps/<backbone>/<series>.npy. A
#     relabelled series is never re-extracted, so a missing map would become a permanent,
#     silent hole in the one experiment the paper leads with. Cheaper to re-extract it.
# Anything else is left exactly as it is.
#
# The log is append-only and load_progress() takes the LAST record per (series, model), so a
# corrected record appended here supersedes the old one without rewriting or deleting anything,
# and every corrected record carries relabelled_from so the correction stays auditable.
import json as _json
import os as _os

import numpy as _np

# The only terms this migration is allowed to have changed. Everything else in the signature
# describes something that genuinely alters the vector, so a disagreement there is not a
# mislabel -- it is a different embedding, and re-extraction is the correct answer.
RELABEL_ALLOWED_KEYS = ("depth", "inplane")


def _sig_terms(sig):
    """Split a signature into {key: value}, keeping the bare version term under 'v'."""
    out = {}
    for t in str(sig).split("|"):
        if "=" in t:
            k, _, v = t.partition("=")
            out[k] = v
        elif t:
            out["v"] = t
    return out


def relabel_stale_embeddings(current_sig, log_path=None, verbose=True):
    """Append corrected records for embeddings that are right but labelled wrong.

    Returns the number of (series, model) pairs relabelled. Safe to call repeatedly: once a
    pair carries current_sig it no longer qualifies, so a second call relabels nothing."""
    log_path = log_path or PROGRESS_LOG_PATH
    cur_terms = _sig_terms(current_sig)
    cand = []
    skipped = {"wrong_channels": 0, "other_terms_differ": 0, "missing_npy": 0, "wrong_dim": 0,
               "missing_spatial_map": 0}

    if _os.path.exists(log_path):
        latest = {}
        with open(log_path) as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                try:
                    r = _json.loads(line)
                except Exception:
                    continue
                latest[(r.get("series_uid"), r.get("model_name"))] = r

        for (suid, m), r in latest.items():
            old = str(r.get("preproc_sig", ""))
            if r.get("status") != "ok" or old == current_sig:
                continue
            old_terms = _sig_terms(old)
            if old_terms.get("channels") != "3":
                skipped["wrong_channels"] += 1
                continue
            # every key on either side, compared -- a term PRESENT in one and ABSENT in the
            # other (inplane is omitted when it is the default) counts as a difference.
            differing = {k for k in set(old_terms) | set(cur_terms)
                         if old_terms.get(k) != cur_terms.get(k)}
            if not differing or (differing - set(RELABEL_ALLOWED_KEYS)):
                skipped["other_terms_differ"] += 1
                continue
            p = _os.path.join(EMBED_DIR, m, f"{suid}.npy")
            if not _os.path.exists(p):
                skipped["missing_npy"] += 1
                continue
            try:
                n = int(_np.load(p, mmap_mode="r").reshape(-1).shape[0])
            except Exception:
                skipped["missing_npy"] += 1
                continue
            if n != int(FEATURE_DIMS.get(m, -1)):
                skipped["wrong_dim"] += 1
                continue
            # Q3 reads this file and nothing re-creates it for a skipped series.
            if not _os.path.exists(_os.path.join(SPATIAL_DIR, m, f"{suid}.npy")):
                skipped["missing_spatial_map"] += 1
                continue
            cand.append((suid, m, r))

    if verbose:
        print("=" * 94)
        print("RELABEL EMBEDDINGS ALREADY ON DRIVE  [cell 16d]")
        print("=" * 94)
        print(f"  current signature : {current_sig}")

    if not cand:
        if verbose:
            print("  nothing to relabel -- no 'ok' record differs from the current signature in")
            print(f"  {list(RELABEL_ALLOWED_KEYS)} alone.")
            # Say WHAT is in the log instead. "Nothing to relabel" is the right answer on a
            # clean run and the wrong answer if a term drifted, and those look identical from
            # here -- so print the signatures that are actually on record and let the reader
            # see which term disagrees before a five-hour re-extraction starts underneath them.
            other = sorted({str(r.get("preproc_sig")) for r in latest.values()
                            if r.get("status") == "ok"
                            and str(r.get("preproc_sig", "")) != current_sig}) if _os.path.exists(log_path) else []
            for s in other[:6]:
                diff = sorted(k for k in set(_sig_terms(s)) | set(cur_terms)
                              if _sig_terms(s).get(k) != cur_terms.get(k))
                print(f"    on record : {s}")
                print(f"                differs in {diff} -> will be re-extracted")
            if not other:
                print("    every 'ok' record on this log already carries the current signature.")
    else:
        with open(log_path, "a") as fh:
            for suid, m, r in cand:
                new = dict(r)
                new["preproc_sig"] = current_sig
                new["relabelled_from"] = r.get("preproc_sig")
                new["relabelled_note"] = ("geometry corrected: the vectors were made with "
                                          "64 x 256 x 256 but stamped with Pillar-0's "
                                          "128 x 256 x 256 fit_head")
                fh.write(_json.dumps(new) + "\n")
        if verbose:
            for s in sorted({str(r.get("preproc_sig")) for _, _, r in cand}):
                print(f"  old signature     : {s}")
            print(f"  relabelled        : {len(cand):,} (series, model) pair(s) -- differing only "
                  f"in {list(RELABEL_ALLOWED_KEYS)},")
            print("                      dimension verified against FEATURE_DIMS and the Q3")
            print("                      spatial map present, nothing deleted")
            print("                      or recomputed, each carrying relabelled_from.")

    if verbose:
        if any(skipped.values()):
            print(f"  left alone        : {skipped}")
            print("                      (these are genuinely different embeddings and will be")
            print("                      re-extracted normally)")
        print("=" * 94)
    return len(cand)


print("relabel_stale_embeddings() defined. The extraction loop calls it with its own")
print("PREPROC_SIG, before it reads the progress log -- nothing runs here.")


In [ ]:
# 17. Main extraction loop -- zip-grouped, per-series resumable, budgeted  [RESUMABLE]
import contextlib
import gc
import json
import os
import time
import traceback
import zipfile

import numpy as np
import torch
from tqdm.auto import tqdm


# ---- progress log ----------------------------------------------------------------------------
def load_progress(path):
    """Returns (done_ok, last_status). Only 'ok' counts as done -- a previously failed pair is
    retried, so fixing a bug and re-running this cell is all that is ever needed."""
    done_ok, last = set(), {}
    if not os.path.exists(path):
        return done_ok, last
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue          # a torn last line from a killed kernel -- skip it, not fatal
            k = (rec.get("series_uid"), rec.get("model_name"))
            last[k] = rec
            if rec.get("status") == "ok":
                done_ok.add(k)
            else:
                done_ok.discard(k)
    return done_ok, last


def filter_by_signature(done_ok, last, current_sig):
    """Drop 'done' entries produced under a different preprocessing signature so they get redone.

    Records written before this field existed have no signature; they are kept (treating an
    unknown provenance as current would be worse, but so would silently redoing an entire finished
    run) -- and counted out loud so the mix is visible rather than assumed."""
    stale, legacy, kept = set(), 0, set()
    for k in done_ok:
        sig = last.get(k, {}).get("preproc_sig")
        if sig is None:
            legacy += 1
            kept.add(k)
        elif sig != current_sig:
            stale.add(k)
        else:
            kept.add(k)
    if stale:
        print(f"  {len(stale):,} embedding(s) were produced under a DIFFERENT preprocessing "
              f"signature and will be re-extracted (their .npy files are overwritten).")
        _other = {last[k].get("preproc_sig") for k in list(stale)[:200]}
        for o in list(_other)[:3]:
            print(f"    old: {o}")
        print(f"    new: {current_sig}")
    if legacy:
        print(f"  {legacy:,} record(s) predate signature tracking and are trusted as-is. If they "
              f"came from an older preprocessing setup, delete extraction_progress.jsonl and the "
              f"embeddings folder to force a clean rebuild.")
    return kept


def load_zips_done(path):
    if not os.path.exists(path):
        return set()
    try:
        with open(path) as f:
            return set(json.load(f))
    except Exception:
        return set()


def save_zips_done(path, done):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(sorted(done), f)
    os.replace(tmp, path)


# Embeddings produced during THIS run, kept in memory so the periodic consolidation does not have
# to re-read every .npy off Drive each time. Re-reading was O(n^2 / CONSOLIDATE_EVERY_N_SERIES)
# Drive reads over a 3,200-series run -- thousands of round-trips spent rebuilding a file that is
# only a convenience copy.
EMB_MEMORY = {m: {} for m in MODELS_TO_RUN}


def consolidate_embeddings(model_name, full_rescan=False):
    """Roll the per-series .npy files into one .npz. The individual files stay -- they are the
    source of truth and are what makes a partial run safe; the .npz is a convenience for loading
    thousands of vectors in one read instead of thousands of Drive round-trips.

    Incremental by default: merges this run's in-memory embeddings into whatever .npz already
    exists. `full_rescan=True` rebuilds from the .npy files on disk, which is what you want after
    a hard kernel kill lost the in-memory state."""
    d = os.path.join(EMBED_DIR, model_name)
    if not os.path.isdir(d):
        return 0
    store = {}
    out = os.path.join(EMBED_DIR, f"embeddings_{model_name}.npz")
    if full_rescan:
        for fn in os.listdir(d):
            if fn.endswith(".npy"):
                try:
                    store[fn[:-4]] = np.load(os.path.join(d, fn))
                except Exception:
                    continue
    else:
        if os.path.exists(out):
            try:
                with np.load(out) as z:
                    store = {k: z[k] for k in z.files}
            except Exception:
                store = {}
        store.update(EMB_MEMORY.get(model_name, {}))
    if not store:
        return 0
    tmp = out + ".tmp.npz"
    np.savez_compressed(tmp, **store)
    os.replace(tmp, out)
    return len(store)


def preproc_signature():
    """Everything that changes what an embedding IS, in one comparable string.

    The extraction cache key is (series_uid, model_name) -- which says nothing about the geometry,
    the precision, or the volume-assembly code that produced the vector. Cell 15 even MUTATES
    TARGET_SPACING_XYZ / TARGET_DEPTH / TARGET_SPATIAL_SIZE at runtime to match the checkpoint. So
    without this, changing the spacing (or fixing a bug in assemble_volume) and re-running leaves
    every existing embedding in place, marked `ok`, silently stale. "Failures are never sticky" is
    true; successes were permanently sticky, which is worse.

    Bump PREPROC_CODE_VERSION by hand whenever you change assemble_volume or
    preprocess_for_pillar0 in a way that alters the output."""
    return "|".join([
        f"v{PREPROC_CODE_VERSION}",
        f"spatial={tuple(TARGET_SPATIAL_SIZE)}",
        f"depth={int(TARGET_DEPTH)}",
        f"spacing={tuple(TARGET_SPACING_XYZ)}",
        f"channels={N_INPUT_CHANNELS}",
        f"minslices={int(MIN_SLICES_PER_SERIES)}",
        # How the in-plane field of view is fitted into 256 px. A non-default mode is a different
        # embedding of the same series, so it MUST flip the cache -- an A/B that silently reused
        # the old vectors would compare a mode against itself and conclude, correctly and
        # uselessly, that it made no difference.
        #
        # The DEFAULT contributes NOTHING to the signature, deliberately. Adding an
        # always-present `inplane=crop` term invalidated every embedding on Drive and cost a
        # 7.1-hour re-extraction of 2,705 series that produced byte-identical vectors, because
        # the behaviour it names had not changed. A cache key must encode what an embedding IS,
        # not which release computed it.
        *([] if globals().get("PILLAR0_INPLANE_MODE", "crop") == "crop"
          else [f"inplane={globals()['PILLAR0_INPLANE_MODE']}"]),
        # `amp` is deliberately NOT in the signature. Cell 16b exists to measure fp16-vs-fp32
        # agreement on real volumes and it is ~0.9999; making precision a hard cache key means
        # being handed a CPU-only Colab session for one night silently invalidates a multi-night
        # GPU run and starts redoing it at CPU speed. The precision that produced each embedding
        # is still recorded per-record (`amp`), so a mixed run stays fully auditable -- and if
        # cell 16b ever reports agreement below ~0.999 on your hardware, that is the signal to
        # bump PREPROC_CODE_VERSION deliberately rather than to have it happen by accident.
    ])


PREPROC_CODE_VERSION = 4
PREPROC_SIG = preproc_signature()
print(f"preprocessing signature: {PREPROC_SIG}")

# Embeddings that are correct but were stamped with Pillar-0's geometry by an earlier run are
# relabelled HERE -- with the signature computed one line above, and before load_progress()
# reads the log, so the corrected records are the ones this run sees. No-op when there is
# nothing to fix, and absent entirely on a Pillar-0 run. See cell 16d.
if callable(globals().get("relabel_stale_embeddings")):
    relabel_stale_embeddings(PREPROC_SIG)

def _save_npy_with_retry(path, arr, attempts=4, base_delay=2.0):
    """Write one embedding to Drive, retrying transient failures.

    Drive's FUSE mount throws intermittently under sustained small-file writes -- and this loop
    writes one file per scan for hours. Without a retry a rate-limit blip is recorded as a failed
    series; if the blip lasts a few minutes it is recorded as hundreds of failed series, and the
    whole night's work looks like a bug in the model rather than a hiccup in the filesystem. The
    embedding is already computed at this point, so retrying costs a few seconds, not a forward
    pass. Written to a temp file and renamed, so a crash mid-write cannot leave a truncated .npy
    that later looks like a valid embedding."""
    last = None
    for k in range(attempts):
        try:
            tmp = path + ".tmp.npy"
            np.save(tmp, arr)
            os.replace(tmp, path)
            if k:
                print(f"    (saved {os.path.basename(path)} on attempt {k+1})")
            return
        except Exception as e:
            last = e
            with contextlib.suppress(Exception):
                if os.path.exists(path + ".tmp.npy"):
                    os.remove(path + ".tmp.npy")
            if k < attempts - 1:
                time.sleep(base_delay * (2 ** k))
    raise RuntimeError(f"could not write {path} after {attempts} attempts: "
                        f"{type(last).__name__}: {last}")


DONE_OK, LAST_STATUS = load_progress(PROGRESS_LOG_PATH)
DONE_OK = filter_by_signature(DONE_OK, LAST_STATUS, PREPROC_SIG)
ZIPS_DONE = load_zips_done(ZIPS_DONE_PATH)
print(f"resuming: {len(DONE_OK):,} (series, model) pair(s) already ok, "
      f"{len(ZIPS_DONE)} day-zip(s) recorded complete")

if len(manifest_runnable) == 0:
    todo = manifest_runnable.copy()
    print("nothing runnable -- see cell 10's coverage report.")
else:
    _pending = [any((suid, m) not in DONE_OK for m in MODELS_TO_RUN)
                for suid in manifest_runnable["series_uid"]]
    todo = manifest_runnable[np.asarray(_pending)].copy()
    # NOTE: `zips_done.json` is deliberately NOT used to filter here.
    #
    # It used to be, and it was a trap. That file is keyed only by "MM/DD" -- not by the manifest,
    # not by MODELS_TO_RUN, not by the selection config. So the moment you widened the selection
    # (PROCESS_ALL_QUALIFYING_SERIES False -> True, a lower MIN_SLICES_PER_SERIES, a second
    # checkpoint), every newly-manifested series in an already-"done" zip was silently dropped and
    # the run reported "pending: 0 series ... no failures." -- which is exactly the workflow the
    # notes cell recommends.
    #
    # The filter was also redundant: `_pending` above already excludes every series whose work is
    # finished, so a genuinely complete zip contributes no rows and is never opened. The file is
    # still written below, as a human-readable record of which days are finished.
print(f"pending: {len(todo):,} series across {todo['zip_key'].nunique()} day-zip(s)")

_limit = TRIAL_RUN_MAX_SERIES if TRIAL_RUN_MAX_SERIES is not None else MAX_SERIES_PER_RUN
if TRIAL_RUN_MAX_SERIES is not None and MAX_SERIES_PER_RUN is not None:
    _limit = min(TRIAL_RUN_MAX_SERIES, MAX_SERIES_PER_RUN)
if _limit is not None:
    print(f"this run will stop after {_limit} series "
          f"({'TRIAL_RUN_MAX_SERIES' if TRIAL_RUN_MAX_SERIES is not None else 'MAX_SERIES_PER_RUN'})")

progress_f = open(PROGRESS_LOG_PATH, "a")


def log_progress(rec):
    progress_f.write(json.dumps(rec) + "\n")
    progress_f.flush()
    try:
        os.fsync(progress_f.fileno())
    except Exception:
        pass                      # fsync is not always meaningful on a FUSE-mounted Drive
    if rec.get("status") == "ok":
        DONE_OK.add((rec["series_uid"], rec["model_name"]))


run_t0 = time.time()
n_series_done = n_series_failed = 0
failures = []
zip_level_failures = []
stop_reason = None
timing = {"locate": 0.0, "read": 0.0, "preprocess": 0.0, "model": 0.0}
amp_active = bool(USE_AMP and DEVICE == "cuda")

# ----------------------------------------------------------------------------------------------
# The work queue.
#
# A plain `for ... in todo.groupby(...)` visits each day-zip exactly once, which is why the first
# overnight run finished with 834 series still unread: a zip whose members would not stream off
# Drive got one attempt and was never revisited. The queue below is the same iteration, except a
# zip that ends the pass with work still pending can put ITSELF back on the end with the access
# mode escalated to a verified local copy. Everything else about the body is unchanged.
#
# Re-visiting a zip is safe and cheap: the per-series loop already skips anything in DONE_OK, so
# the second visit reads only what the first one missed.
# ----------------------------------------------------------------------------------------------
# RSNA is a single split archive that open_container always streams -- a local copy is not
# even attempted there, so escalating an RSNA "zip" would just repeat the identical pass
# and double the run for nothing. Escalation is an India-path idea only.
_ESCALATE = (ZIP_ACCESS_MODE == "auto" and DATA_SOURCE != "rsna")
_first_mode = "stream" if ZIP_ACCESS_MODE == "auto" else ZIP_ACCESS_MODE
# Order the queue by how many pending series each GB of archive buys.
#
# When a run's copy budget is smaller than the pending work -- 500 GB against 2,100 GB of
# candidates -- the order decides what gets recovered. Chronological order spends the budget on
# whichever zip happens to come first: 03/11 is 25.7 GB and holds 3 series, while other days hold
# five times as many for the same bytes. Densest-first recovers several times more series from the
# same budget, and the remainder is still picked up by the next run.
def _zip_density(gdf):
    try:
        _g = os.path.getsize(gdf["zip_path"].iloc[0]) / 1e9
    except OSError:
        return 0.0
    return (gdf["series_uid"].nunique() / _g) if _g > 0 else 0.0


_groups = list(todo.groupby("zip_key", sort=True))
if _ESCALATE and len(_groups) > 1:
    _groups.sort(key=lambda kv: -_zip_density(kv[1]))
    print("queue ordered by pending series per GB (densest first), so a partial run recovers the "
          "most series")
_queue = [(k, gdf, _first_mode) for k, gdf in _groups]
_qi = 0
_copy_budget_gb = (float("inf") if MAX_LOCAL_COPY_GB_PER_RUN is None
                   else float(MAX_LOCAL_COPY_GB_PER_RUN))
_copied_gb = 0.0
_escalated, _budget_skipped = [], []
if _ESCALATE:
    # Upfront projection. These day-zips are 5-28 GB each, so "fall back to a local copy" is a
    # decision with hours attached to it, and the run should say so before it starts rather than
    # after. Sizes are read from Drive metadata only (a stat, not a read), which is instant.
    _sz = []
    for _k, _gdf in todo.groupby("zip_key", sort=True):
        try:
            _sz.append(os.path.getsize(_gdf["zip_path"].iloc[0]) / 1e9)
        except OSError:
            pass
    _tot_gb = sum(_sz)
    print(f"access mode 'auto': stream first, then re-read any zip left incomplete from a "
          f"verified local copy.")
    if _sz:
        print(f"  {len(_sz)} day-zip(s) pending, {_tot_gb:,.0f} GB in total "
              f"(mean {_tot_gb/len(_sz):.1f} GB, largest {max(_sz):.1f} GB)")
        print(f"  copy budget this run: "
              + (f"{_copy_budget_gb:,.0f} GB" if _copy_budget_gb != float('inf') else 'unlimited'))
        if _copy_budget_gb != float("inf") and _tot_gb > _copy_budget_gb:
            print(f"  If EVERY pending zip needed copying, that is more than one run's budget. "
                  f"Whatever is not reached stays pending and the next run continues -- nothing "
                  f"is lost or redone. Most zips are expected to stream fine and cost no copy at "
                  f"all; only the ones that fail are copied.")
        print(f"  A copy runs at roughly 30-80 MB/s off Drive, so budget about "
              f"{_copy_budget_gb/0.080/3600 if _copy_budget_gb != float('inf') else _tot_gb/0.080/3600:.0f}"
              f"-{_copy_budget_gb/0.030/3600 if _copy_budget_gb != float('inf') else _tot_gb/0.030/3600:.0f}"
              f" hours of copying if the budget is fully spent, on top of extraction time.")

try:
    while _qi < len(_queue):
        zip_key, group, _zip_mode = _queue[_qi]
        _qi += 1
        if stop_reason:
            break
        _ACTIVE_ZIP_MODE = _zip_mode
        zip_path = group["zip_path"].iloc[0]
        # Locate EVERY manifest series for this zip, not just the ones still pending. The zip
        # index is cached under a key that includes the wanted-UID set, so asking for a different
        # subset each run would invalidate the cache and force a full re-scan of the central
        # directory every time a few series are retried. Asking for the full set keeps the cache
        # stable, and it also makes the "is this zip finished?" check below the real question
        # rather than "did this run's subset finish?".
        all_for_zip = list(dict.fromkeys(
            manifest_runnable.loc[manifest_runnable["zip_key"] == zip_key, "series_uid"].tolist()))
        wanted = list(dict.fromkeys(group["series_uid"].tolist()))
        print(f"\n=== {zip_key}  ({len(wanted)} pending of {len(all_for_zip)} wanted)  {zip_path}")
        zt0 = time.time()
        zt = {"locate": 0.0, "read": 0.0, "preprocess": 0.0, "model": 0.0}

        try:
            with open_container(zip_path) as (zf, effective_path):
                t = time.time()
                index = get_series_index(zf, zip_path, zip_key, all_for_zip)
                zt["locate"] += time.time() - t

                # Visit this zip's series in ARCHIVE order. The index knows where each series
                # starts; manifest order is effectively random with respect to the file, and a
                # backward jump on a 20-30 GB Drive-mounted zip is what produces BadZipFile.
                # Ordering here turns the whole day-zip into a single forward sweep.
                _rows = list(group.itertuples(index=False))
                _rows.sort(key=lambda r: index.get(r.series_uid, {}).get("offset", 1 << 62))
                # ---- progress reporting that survives a disconnected browser -------------------
                #
                # tqdm.auto picks the ipywidgets bar in Colab, which updates a widget IN PLACE --
                # fine for the India path, where a day-zip group is 10-30 series. The RSNA path is
                # ONE group of ~4,000, and each update is a comm message that Colab buffers for
                # replay and writes into the notebook's saved widget state. Over 4,000 iterations
                # with the tab closed that is a lot of nothing, and it is the same output-volume
                # problem that the pydicom flood was.
                #
                # So: past BIG_GROUP_SERIES the bar is turned off and a plain line is printed on a
                # cadence instead. Cheap, readable in a saved .ipynb, and it still says how far in
                # it is when you come back.
                _big = len(_rows) > BIG_GROUP_SERIES
                _bar = tqdm(_rows, desc=zip_key, leave=False, mininterval=5.0,
                            disable=_big)
                _t_grp, _n_grp = time.time(), 0
                for row in _bar:
                    if _big:
                        _n_grp += 1
                        _hb = globals().get("heartbeat")
                        if _hb is not None:
                            _hb("cell 17 extraction", f"{_n_grp:,}/{len(_rows):,} series",
                                every_s=60, zip_key=str(zip_key))
                        if _n_grp % BIG_GROUP_LOG_EVERY == 0:
                            _el = time.time() - _t_grp
                            _fr = _n_grp / max(len(_rows), 1)
                            print(f"    {_n_grp:,}/{len(_rows):,} series ({_fr:.1%}) in "
                                  f"{_el/60:.0f} min"
                                  + (f", ~{_el/_fr*(1-_fr)/60:.0f} min left" if _fr > 0.01 else ""),
                                  flush=True)
                    suid = row.series_uid
                    pending_models = [m for m in MODELS_TO_RUN if (suid, m) not in DONE_OK]
                    if not pending_models:
                        continue

                    base = {"series_uid": suid, "study_uid": row.study_uid,
                            "patient_id": str(row.patient_id), "zip_key": zip_key,
                            "amp": amp_active, "preproc_sig": PREPROC_SIG,
                            "ts": int(time.time())}

                    entry = index.get(suid)
                    if entry is None:
                        for m in pending_models:
                            log_progress({**base, "model_name": m, "status": "failed",
                                          "detail": "series not present in this day-zip"})
                            failures.append((suid, m, "not present in day-zip"))
                        n_series_failed += 1
                        continue
                    # `verified` is False only when the locator was asked to verify and could
                    # not. With VERIFY_IN_LOCATOR off it is simply absent, and identity is checked
                    # by assemble_volume against the slices it reads.
                    if VERIFY_IN_LOCATOR and not entry.get("verified", False):
                        for m in pending_models:
                            log_progress({**base, "model_name": m, "status": "failed",
                                          "detail": f"zip index verification failed: {entry.get('note','')}"})
                            failures.append((suid, m, f"verification failed: {entry.get('note','')}"))
                        n_series_failed += 1
                        continue

                    try:
                        t = time.time()
                        try:
                            vol, spacing, vinfo = assemble_volume(entry["names"], zf,
                                                                  expect_uid=suid)
                        except ZipMembersUnreadable:
                            # ---- repair on demand, once per archive --------------------------
                            #
                            # Cell 11b decides whether an archive's directory overflowed by
                            # probing a sample of members. A real trial found an archive it
                            # cleared whose members then would not read at all -- so the
                            # heuristic can be wrong, and when it is, the run silently extracts
                            # nothing from that day-zip.
                            #
                            # The reads themselves are the better evidence: they just failed.
                            # So force the reconstruction and try the series once more. This
                            # costs one repair (about a dozen 30-byte reads) per archive, is
                            # attempted at most once, and turns a wrong detection into a
                            # recovered day-zip instead of a silent hole in the dataset.
                            _forced = globals().get("force_zip_offset_repair")
                            # The forced offset repair rewrites a ZipFile's central directory. For
                            # RSNA the container is a LOCAL DIRECTORY of already-extracted DICOMs
                            # (cell B5), which has no central directory and no offsets -- calling
                            # it there raises an AttributeError that gets logged as this series'
                            # failure, hiding whatever the real problem was.
                            if (_forced is None or getattr(zf, "_offsets_forced", False)
                                    or not isinstance(zf, zipfile.ZipFile)):
                                raise
                            zf._offsets_forced = True
                            print(f"      {suid[-24:]}: members unreadable although the archive "
                                  f"was not flagged -- forcing the offset repair and retrying")
                            _n_forced = _forced(zf, effective_path)
                            if not _n_forced:
                                raise
                            vol, spacing, vinfo = assemble_volume(entry["names"], zf,
                                                                  expect_uid=suid)
                        zt["read"] += time.time() - t

                        t = time.time()
                        _set_current_series(suid)   # the spatial map needs to know
                        x = preprocess_for_pillar0(vol, spacing, info=vinfo)
                        zt["preprocess"] += time.time() - t
                        del vol
                    except Exception as e:
                        msg = f"{type(e).__name__}: {e}"
                        for m in pending_models:
                            log_progress({**base, "model_name": m, "status": "failed",
                                          "detail": f"volume/preprocess: {msg}"})
                            failures.append((suid, m, msg))
                        n_series_failed += 1
                        continue

                    ok_any = False
                    for m in pending_models:
                        try:
                            t = time.time()
                            try:
                                emb = extract_embedding(LOADED[m], x)
                            except torch.cuda.OutOfMemoryError:
                                # An OOM leaves the allocator fragmented; without clearing it the
                                # NEXT scan usually OOMs too, and one unlucky volume turns into a
                                # cascade that empties the rest of the night. Clear, wait, retry
                                # once -- and if it fails again let it be recorded as this series'
                                # own failure rather than everyone else's.
                                print(f"    CUDA OOM on {suid[-12:]} -- clearing cache and "
                                      f"retrying once")
                                torch.cuda.empty_cache()
                                gc.collect()
                                time.sleep(2.0)
                                emb = extract_embedding(LOADED[m], x)
                            zt["model"] += time.time() - t
                            if not np.isfinite(emb).all():
                                raise ValueError("model returned non-finite values")
                            emb = emb.astype(np.float32)
                            _save_npy_with_retry(os.path.join(EMBED_DIR, m, f"{suid}.npy"), emb)
                            EMB_MEMORY[m][suid] = emb
                            log_progress({
                                **base, "model_name": m, "status": "ok", "detail": "",
                                "dim": int(emb.shape[0]),
                                "norm": float(np.linalg.norm(emb)),
                                "n_slices": int(vinfo.get("n_slices", 0)),
                                "spacing": [spacing["sx"], spacing["sy"], spacing["sz"]],
                                "z_spacing_source": vinfo.get("z_spacing_source", ""),
                                "n_undecodable": int(vinfo.get("n_undecodable", 0)),
                                "n_duplicate_sop": int(vinfo.get("n_duplicate_sop", 0)),
                                "n_pixel_corrected": int(vinfo.get("n_pixel_corrected", 0)),
                                "hu_p01": vinfo.get("hu_p01"), "hu_p50": vinfo.get("hu_p50"),
                                "n_shape_mismatch": int(vinfo.get("n_shape_mismatch", 0)),
                                # How much of the head survived the 256 x 256 centre crop at
                                # 0.5 mm (= 128 mm of field of view). See measure_head_retention
                                # in cell 14 for why this is measured rather than assumed.
                                "head_retained": vinfo.get("head_retained"),
                                "head_retained_z": vinfo.get("head_retained_z"),
                                "head_retained_inplane": vinfo.get("head_retained_inplane"),
                                "resampled_shape": vinfo.get("resampled_shape"),
                                "kept_fov_mm": vinfo.get("kept_fov_mm"),
                                "note": vinfo.get("note", ""),
                            })
                            ok_any = True
                        except Exception as e:
                            msg = f"{type(e).__name__}: {e}"
                            log_progress({**base, "model_name": m, "status": "failed",
                                          "detail": msg})
                            failures.append((suid, m, msg))

                    del x
                    if DEVICE == "cuda":
                        torch.cuda.empty_cache()

                    if ok_any:
                        n_series_done += 1
                    else:
                        n_series_failed += 1

                    if n_series_done and n_series_done % CONSOLIDATE_EVERY_N_SERIES == 0:
                        for m in MODELS_TO_RUN:
                            consolidate_embeddings(m)

                    if _limit is not None and n_series_done >= _limit:
                        stop_reason = f"series limit ({_limit}) reached"
                        break
                    if (MAX_SECONDS_PER_RUN is not None
                            and time.time() - run_t0 > MAX_SECONDS_PER_RUN):
                        stop_reason = f"time limit ({MAX_SECONDS_PER_RUN}s) reached"
                        break
        except KeyboardInterrupt:
            stop_reason = "interrupted by user"
        except Exception as e:
            # A whole-archive failure -- BadZipFile on a partially-synced Drive file, or the
            # out-of-disk RuntimeError from ensure_local_copy. No per-series record is written
            # (correct: nothing is marked done, so it all retries), but it must not vanish from
            # the summary, which used to print "no failures." right under the traceback.
            print(f"  ZIP-LEVEL FAILURE for {zip_key}: {type(e).__name__}: {e}")
            traceback.print_exc(limit=3)
            zip_level_failures.append((zip_key, zip_path, f"{type(e).__name__}: {e}"))
        finally:
            # Unconditional. Under "auto" this zip may have been copied even though the run-wide
            # setting is not "local_copy", and a leaked 9-22 GB copy fills the disk within a few
            # zips. Deleting a file that was never made costs nothing.
            release_local_copy(zip_path)

        for k in timing:
            timing[k] += zt[k]
        _tot = time.time() - zt0
        print(f"    {zip_key} took {_tot/60:.1f} min  "
              f"[locate {zt['locate']:.0f}s | read {zt['read']:.0f}s | "
              f"prep {zt['preprocess']:.0f}s | model {zt['model']:.0f}s]")

        # Mark the zip finished only if every series we wanted from it is now ok.
        _complete = all((s, m) in DONE_OK for s in all_for_zip for m in MODELS_TO_RUN)
        if _complete:
            ZIPS_DONE.add(zip_key)
            save_zips_done(ZIPS_DONE_PATH, ZIPS_DONE)
            print(f"    {zip_key} complete -- it will not be reopened.")
        elif _ESCALATE and _zip_mode == "stream":
            # Still incomplete after streaming. Escalate to a local copy -- but only if the file
            # fits the run's copy budget, and only once (the re-queued entry is "local_copy", so
            # this branch cannot fire for it again and the queue cannot grow without bound).
            try:
                _sz_gb = os.path.getsize(zip_path) / 1e9
            except OSError:
                _sz_gb = float("inf")
            _still = sum(1 for _s in all_for_zip
                         for _m in MODELS_TO_RUN if (_s, _m) not in DONE_OK)
            if _copied_gb + _sz_gb <= _copy_budget_gb:
                _copied_gb += _sz_gb
                # Insert as the NEXT item, not at the end of the queue.
                #
                # Appending looked harmless and was not: with 107 pending zips the run spent its
                # entire copy budget handing out promises during the streaming pass, then started
                # the first actual copy only after streaming all 107 -- so a session that died in
                # between recovered nothing at all, despite hours of work. Finishing each zip
                # before moving on makes the progress real and incremental.
                _queue.insert(_qi, (zip_key, group, "local_copy"))
                _escalated.append(zip_key)
                print(f"    {zip_key}: {_still} series still unread after streaming -- re-queued "
                      f"for a local copy ({_sz_gb:.1f} GB; {_copied_gb:.0f}/{_copy_budget_gb:.0f} "
                      f"GB of this run's copy budget used)")
            else:
                _budget_skipped.append(zip_key)
                print(f"    {zip_key}: {_still} series still unread, but a local copy needs "
                      f"{_sz_gb:.1f} GB and the run's {_copy_budget_gb:.0f} GB copy budget is "
                      f"spent. Left pending -- re-run this cell to continue.")

except KeyboardInterrupt:
    stop_reason = "interrupted by user"
finally:
    _ACTIVE_ZIP_MODE = None
    progress_f.close()
    for m in MODELS_TO_RUN:
        # A full rescan once at the end catches anything a previous hard kill left on disk but
        # never merged into the .npz.
        n_stored = consolidate_embeddings(m, full_rescan=True)
        print(f"consolidated {n_stored:,} embedding(s) for {m}")

print()
if _escalated:
    print(f"{len(_escalated)} day-zip(s) could not be streamed and were re-read from a local "
          f"copy ({_copied_gb:.0f} GB copied): {', '.join(_escalated[:12])}"
          + (f" ... and {len(_escalated) - 12} more" if len(_escalated) > 12 else ""))
if _budget_skipped:
    print(f"{len(_budget_skipped)} day-zip(s) needed a local copy but the run's "
          f"{_copy_budget_gb:.0f} GB budget was spent -- re-run this cell to finish them: "
          f"{', '.join(_budget_skipped[:12])}"
          + (f" ... and {len(_budget_skipped) - 12} more" if len(_budget_skipped) > 12 else ""))
print()
print(f"this run: {n_series_done:,} series extracted, {n_series_failed:,} failed, "
      f"{(time.time() - run_t0)/60:.1f} min elapsed")
if stop_reason:
    print(f"stopped early: {stop_reason}. Re-run this cell to continue -- nothing is lost.")
_tsum = sum(timing.values()) or 1.0
print("time split: " + " | ".join(
    f"{k} {v/60:.1f} min ({100*v/_tsum:.0f}%)" for k, v in timing.items()))
if timing["read"] > 2 * (timing["preprocess"] + timing["model"]):
    other = "local_copy" if ZIP_ACCESS_MODE == "stream" else "stream"
    print(f"  -> reading from the zip dominates. Worth trying ZIP_ACCESS_MODE = {other!r} "
          f"for one zip and comparing this line.")

if failures:
    import csv as _csv
    with open(FAILURES_CSV_PATH, "w", newline="") as f:
        w = _csv.writer(f)
        w.writerow(["series_uid", "model_name", "error"])
        w.writerows(failures)
    groups = {}
    for _s, _m, _e in failures:
        groups.setdefault(_e[:200], []).append(_s)
    print(f"\n{len(failures)} failure(s), {len(groups)} distinct message(s) "
          f"-- full list in {FAILURES_CSV_PATH}:")
    for _e, _ss in sorted(groups.items(), key=lambda kv: -len(kv[1]))[:10]:
        print(f"  [{len(_ss)}x] {_e}")
        print(f"        e.g. {_ss[0]}")
    print("  (all of these are retried automatically on the next run of this cell)")
else:
    print("no per-series failures.")

if zip_level_failures:
    print(f"\n{len(zip_level_failures)} WHOLE-ARCHIVE failure(s) -- every series in these days was "
          f"skipped and will be retried next run:")
    for _zk, _zp, _err in zip_level_failures[:10]:
        print(f"  {_zk}: {_err}")
        print(f"      {_zp}")
    if len(zip_level_failures) > 10:
        print(f"  ... and {len(zip_level_failures) - 10} more")

# Cell 11b owns this dict; look it up rather than referencing it directly, so a harness that
# execs this cell without cell 11b does not die on a NameError here.
_repairs = globals().get("ZIP_OFFSET_REPAIRS")
if _repairs:
    print(f"\nday-zips whose central directory had to be repaired: {len(_repairs)} "
          f"({sum(_repairs.values()):,} member offsets corrected)")
    print("  These are the archives written past 4 GiB without ZIP64. Nothing was copied and")
    print("  nothing on Drive was modified -- the offsets are corrected in memory each time an")
    print("  archive is opened.")
elif _repairs is not None:
    print("\nno day-zip needed a central-directory repair this run")

print(f"\ntotal completed (series, model) pairs: {len(DONE_OK):,}")

# ---- how much of the head each embedding actually saw -----------------------------------------
# The pipeline follows Pillar-0's own configuration: resample to 0.5 mm in plane, centre-crop to
# 256 x 256. That is 128 mm of field of view, and an adult skull is 150-160 mm across. Whether
# that clips anything depends on the source FOV, which varies by scanner and protocol -- so it is
# MEASURED, per series, and reported here rather than reasoned about.
#
# What to do with the number: if retention is ~1.0 the crop is harmless and this is noise. If it
# is materially below 1.0 the embeddings are of partial heads, and the lateral convexities are
# the first thing lost -- which is where subdural and epidural collections sit, so it would bite
# those targets hardest and 'any hemorrhage' least. That asymmetry is itself diagnostic.
def _progress_rows(path):
    """Every record in the progress log, tolerating a torn last line from a killed kernel."""
    if not os.path.exists(path):
        return
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception:
                continue


# FILTER BY THE CURRENT SIGNATURE. Without this the summary pools every "ok" record ever
# written to this log, including the Pillar-0 run at 128 x 256 x 256 -- and then prints those
# numbers under a heading naming the CURRENT geometry. On the last run that is exactly what
# happened: 1,200 rows of a 384 mm window reported as though they described a 192 mm one.
try:
    _rows = [r for r in _progress_rows(PROGRESS_LOG_PATH)
             if r.get("status") == "ok" and r.get("head_retained") is not None
             and r.get("preproc_sig") == PREPROC_SIG]
    _hr = [r["head_retained"] for r in _rows]
    _stale = sum(1 for r in _progress_rows(PROGRESS_LOG_PATH)
                 if r.get("status") == "ok" and r.get("head_retained") is not None
                 and r.get("preproc_sig") != PREPROC_SIG)
    if _stale:
        print(f"\n({_stale:,} head-retention record(s) from an older preprocessing signature are "
              f"excluded from the summary below -- they describe a different geometry.)")
except Exception:
    _hr, _rows = [], []
if _hr:
    import numpy as _np
    _a = _np.asarray([r["head_retained"] for r in _rows], dtype=float)
    _z = _np.asarray([r.get("head_retained_z") for r in _rows
                      if r.get("head_retained_z") is not None], dtype=float)
    _xy = _np.asarray([r.get("head_retained_inplane") for r in _rows
                       if r.get("head_retained_inplane") is not None], dtype=float)
    print(f"\nhead retained through the crop to {TARGET_DEPTH} x {TARGET_SPATIAL_SIZE[0]} x "
          f"{TARGET_SPATIAL_SIZE[1]} (n={len(_a):,}): "
          f"median {_np.median(_a):.3f}, 10th pct {_np.percentile(_a, 10):.3f}, "
          f"min {_a.min():.3f}")
    if len(_z) and len(_xy):
        print(f"  by axis -- z alone: median {_np.median(_z):.3f} "
              f"({TARGET_DEPTH} slices x {TARGET_SPACING_XYZ[2]} mm = "
              f"{TARGET_DEPTH * TARGET_SPACING_XYZ[2]:.0f} mm of coverage) | "
              f"in-plane alone: median {_np.median(_xy):.3f} "
              f"({TARGET_SPATIAL_SIZE[0]} px x {TARGET_SPACING_XYZ[0]} mm = "
              f"{TARGET_SPATIAL_SIZE[0] * TARGET_SPACING_XYZ[0]:.0f} mm of field of view)")
        _worse = "the Z crop" if _np.median(_z) < _np.median(_xy) else "the IN-PLANE crop"
        print(f"  -> {_worse} is doing more of the damage. PILLAR0_INPLANE_MODE only addresses "
              f"the in-plane half; a z shortfall means the scan is longer than "
              f"{TARGET_DEPTH * TARGET_SPACING_XYZ[2]:.0f} mm and the middle slice window may "
              f"not even be centred on the brain.")
    _clipped = float((_a < 0.98).mean())
    if _clipped > 0.02:
        print(f"  *** {_clipped:.0%} of series lost more than 2% of the head. The lateral "
              f"convexities and the vertex go first -- where subdural and epidural collections "
              f"live. Expect those targets to suffer more than 'any hemorrhage'. ***")
    else:
        print("  the centre crop is not clipping these heads -- the source volumes fit.")
gc.collect()

In [ ]:
# 18. Assemble the feature table  [RESUMABLE -- safe to run mid-extraction]
#
# PROVENANCE MATTERS HERE. A .npy sitting in EMBED_DIR does not by itself prove it came from the
# current version of this notebook -- EMBED_DIR lives on Drive and survives every code change. So
# only embeddings whose (series, model) pair is logged `ok` in extraction_progress.jsonl are
# admitted, and anything on disk without that record is reported separately rather than silently
# swept in. If you want the orphans anyway (e.g. the log was lost), cell 18b adopts them after
# checking them.
import json
import os

import numpy as np
import pandas as pd


def load_ok_records(path, require_sig=None):
    """Last record wins per (series, model) -- a retry supersedes a stale failure.

    `require_sig` is not optional in practice. Cell 17 drops stale-signature pairs from its own
    DONE_OK so they get re-extracted, but it does NOT rewrite their `ok` lines in the log or delete
    their .npy files -- and it only re-extracts as many as that session had budget for. Without the
    same filter here, a session that re-extracted 20 of 3,185 stale series still yields a 3,205-row
    feature table silently mixing two preprocessing generations. Nothing downstream can see it:
    both generations are unit-norm, non-degenerate and non-duplicate, so cell 19's QC passes and
    cell 21 trains on the mixture."""
    latest = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except Exception:
                    continue
                latest[(rec.get("series_uid"), rec.get("model_name"))] = rec
    ok = {k: v for k, v in latest.items() if v.get("status") == "ok"}
    if require_sig is None:
        return ok, 0, 0
    current, stale, legacy = {}, 0, 0
    for k, v in ok.items():
        sig = v.get("preproc_sig")
        if sig is None:
            legacy += 1              # predates signature tracking -- cell 17 trusts these too
            current[k] = v
        elif sig == require_sig:
            current[k] = v
        else:
            stale += 1
    return current, stale, legacy


def build_feature_table(model_name, ok_records, manifest_df):
    d = os.path.join(EMBED_DIR, model_name)
    wanted = {suid for (suid, m) in ok_records if m == model_name}
    on_disk = {fn[:-4] for fn in os.listdir(d) if fn.endswith(".npy")} if os.path.isdir(d) else set()

    usable = sorted(wanted & on_disk)
    logged_missing_file = sorted(wanted - on_disk)
    untracked = sorted(on_disk - wanted)

    vecs, uids, bad_dim = [], [], []
    dim = None
    for suid in usable:
        try:
            v = np.load(os.path.join(d, f"{suid}.npy")).astype(np.float32).reshape(-1)
        except Exception:
            logged_missing_file.append(suid)
            continue
        if dim is None:
            dim = v.shape[0]
        if v.shape[0] != dim or not np.isfinite(v).all():
            bad_dim.append(suid)
            continue
        vecs.append(v)
        uids.append(suid)

    print(f"[{model_name}]")
    print(f"  logged ok             : {len(wanted):,}")
    print(f"  .npy present on disk  : {len(on_disk):,}")
    print(f"  usable                : {len(uids):,} (dim {dim})")
    if logged_missing_file:
        print(f"  logged ok but file missing/unreadable: {len(logged_missing_file):,} "
              f"(e.g. {logged_missing_file[:2]}) -- they will be re-extracted on the next run "
              f"only if you also remove their 'ok' lines; otherwise delete the .npy and re-run.")
    if bad_dim:
        print(f"  wrong dimension or non-finite: {len(bad_dim):,} (excluded)")
    if untracked:
        print(f"  on disk with no 'ok' record: {len(untracked):,} (excluded -- see cell 18b)")
    if not uids:
        return None, {"untracked": untracked}

    X = np.vstack(vecs)
    feat_cols = [f"emb_{i:04d}" for i in range(X.shape[1])]
    feats = pd.DataFrame(X, columns=feat_cols)
    feats.insert(0, "series_uid", uids)

    meta_cols = [c for c in ["series_uid", "study_uid", "patient_id", "person_key", "zip_key", "region",
                              "head_subtype", "series_desc", "body_part", "is_reformat",
                              "n_slices_expected", "is_soft_kernel", "is_primary",
                              "studytype_agrees", "matched_study_type", "match_method",
                              "study_date", "patient_sex", "patient_age",
                              "hospital", "institution"]
                 if c in manifest_df.columns]
    # ---- INNER join, and it matters -----------------------------------------------------------
    #
    # This was how="left". An embedding whose series is no longer in the manifest survived the
    # merge with every metadata column NaN -- so tightening the cohort (brain-only, CT-only, a
    # stricter slice minimum) removed rows from the MANIFEST and left them in the FEATURE TABLE,
    # where cells 20 and 21 then trained on them. The cohort you configured and the cohort you
    # evaluated were different populations, and nothing said so.
    #
    # An inner join makes the manifest the single definition of who is in the study. A .npy on
    # disk is evidence that a scan was once extracted, not that it belongs.
    _before = len(feats)
    _man = manifest_df[meta_cols].drop_duplicates("series_uid")
    out = feats.merge(_man, on="series_uid", how="inner")
    _dropped = _before - len(out)
    if _dropped:
        _gone = set(feats["series_uid"]) - set(out["series_uid"])
        print(f"  {_dropped:,} embedding(s) on disk are NOT in the current manifest and are "
              f"EXCLUDED from the feature table.")
        print(f"    Their .npy files are untouched -- widen the manifest and re-run this cell to "
              f"bring them back.")
        print(f"    e.g. {sorted(_gone)[0][:44]}...")
    out.attrs["feature_columns"] = feat_cols

    # ---- and the cohort must be what the config says it is ------------------------------------
    # globals().get: a harness that execs this cell without cell 4 should get the permissive
    # default rather than a NameError from a cell it is not testing. The notebook always defines
    # it, so the real run is unaffected.
    if globals().get("STRICT_COHORT", False) and DATA_SOURCE == "india":
        _bad = []
        if "region" in out.columns:
            _r = out[out["region"].astype(str) != TARGET_REGION]
            if len(_r):
                _bad.append(f"{len(_r):,} scan(s) are not region {TARGET_REGION!r}: "
                            f"{dict(_r['region'].value_counts())}")
        if globals().get("RESTRICT_TO_ALLOWED_HEAD_SUBTYPES", False) and "head_subtype" in out.columns:
            _h = out[~out["head_subtype"].astype(str).isin(ALLOWED_HEAD_SUBTYPES)]
            if len(_h):
                _bad.append(f"{len(_h):,} scan(s) are not in ALLOWED_HEAD_SUBTYPES="
                            f"{tuple(ALLOWED_HEAD_SUBTYPES)}: "
                            f"{dict(_h['head_subtype'].value_counts())}")
        if "person_key" in out.columns:
            _p = out[out["person_key"].isna() | (out["person_key"].astype(str).str.len() == 0)]
            if len(_p):
                _bad.append(f"{len(_p):,} scan(s) have no person_key, so they cannot be grouped")
        if _bad:
            raise RuntimeError(
                "STRICT_COHORT is on and the feature table is not the cohort you configured:\n  "
                + "\n  ".join(_bad)
                + "\n\nThis is the check that stops a mixed cohort from producing a number. "
                  "Rebuild the manifest (cell 9) so it matches the config, then re-run this cell."
            )
        print(f"  cohort purity verified: every row is region={TARGET_REGION!r}"
              + (f", head_subtype in {tuple(ALLOWED_HEAD_SUBTYPES)}"
                 if globals().get("RESTRICT_TO_ALLOWED_HEAD_SUBTYPES", False) else "")
              + ", and carries a person_key")

    path = os.path.join(OUTPUT_DIR, f"features_{model_name}.parquet")
    _tmp = path + ".tmp.parquet"
    out.to_parquet(_tmp, index=False)
    os.replace(_tmp, path)      # a disconnect mid-write must not leave a half-parquet behind
    print(f"  -> {path}  ({out.shape[0]:,} rows x {len(feat_cols)} features)")
    return out, {"untracked": untracked, "feature_columns": feat_cols}


_sig = globals().get("PREPROC_SIG")
OK_RECORDS, _n_stale, _n_legacy = load_ok_records(PROGRESS_LOG_PATH, require_sig=_sig)
if _sig is None:
    print("WARNING: PREPROC_SIG is not defined (cell 17 has not run this session), so embeddings "
          "cannot be checked against the current preprocessing. Run cell 17 first.")
else:
    print(f"admitting embeddings with preprocessing signature: {_sig}")
    if _n_stale:
        # Break this down by model. Most of the stale records here belong to pillar0_headct from
        # the superseded Pillar-0 runs, and cell 17 will NEVER re-extract those -- that model is
        # not in MODELS_TO_RUN any more. Reporting one lump total made a finished cohort look
        # like a half-finished one on every single run.
        # LAST WINS, exactly as load_progress does. Counting every stale LINE instead of every
        # stale PAIR reported the 3,600 superseded rows the relabel had already corrected, so a
        # finished cohort printed "1,200 ARE pending" for all three backbones while the feature
        # tables below correctly held 1,200 rows each. The log is append-only; only the last
        # record for a (series, model) pair describes its current state.
        _latest_rec = {}
        for _r in _progress_rows(PROGRESS_LOG_PATH):
            _latest_rec[(_r.get("series_uid"), _r.get("model_name"))] = _r
        _stale_by_model = {}
        for (_su, _m), _r in _latest_rec.items():
            if _r.get("status") == "ok" and _r.get("preproc_sig") not in (None, _sig):
                _stale_by_model[_m] = _stale_by_model.get(_m, 0) + 1
        _n_stale = sum(_stale_by_model.values()) or _n_stale
        _pending = {m: n for m, n in _stale_by_model.items() if m in MODELS_TO_RUN}
        _inert = {m: n for m, n in _stale_by_model.items() if m not in MODELS_TO_RUN}
        print(f"  {_n_stale:,} record(s) on this log carry an older preprocessing signature.")
        for _m, _n in sorted(_inert.items()):
            print(f"    {_n:,} for {_m} -- NOT pending: that model is not in MODELS_TO_RUN, so "
                  f"nothing will re-extract it. Left on disk as history.")
        if _pending:
            for _m, _n in sorted(_pending.items()):
                print(f"    {_n:,} for {_m} -- these ARE pending; cell 17 will re-extract them.")
            print("    Until it does, this feature table is a SUBSET, not the full dataset -- do "
                  "not read the classifier results below as covering everything.")
        else:
            print("    None of them belong to a model this run extracts, so the table below is "
                  "complete for every backbone in MODELS_TO_RUN.")
    if _n_legacy:
        print(f"  {_n_legacy:,} record(s) predate signature tracking and are admitted as-is.")
FEATURES = {}
FEATURE_COLUMNS = {}
UNTRACKED = {}
for _m in MODELS_TO_RUN:
    _tab, _info = build_feature_table(_m, OK_RECORDS, manifest)
    UNTRACKED[_m] = _info.get("untracked", [])
    if _tab is not None:
        FEATURES[_m] = _tab
        FEATURE_COLUMNS[_m] = _info["feature_columns"]

if not FEATURES:
    print("\nNo feature table could be built yet -- run cell 17 first (or longer).")

In [ ]:
# 18b. Optional: adopt .npy files that have no "ok" record
#
# Cell 18 excludes any embedding whose (series, model) pair is not logged `ok`, because a file
# existing on Drive is not evidence that the current code produced it. That is the right default.
# But there is one legitimate case for overriding it: the progress log was lost or truncated while
# the embeddings survived.
#
# This cell does not simply trust them. It checks each orphan for the right dimension, finite
# values, non-degenerate variance and unit norm (which a real Pillar-0 output always has), then
# writes a genuine `ok` record for the ones that pass -- with `adopted: true` on the record, so the
# distinction is never lost. Anything failing a check is left alone and reported.
ADOPT_UNTRACKED = False   # set True, run once, then set back to False

import json
import os

import numpy as np

if not ADOPT_UNTRACKED:
    _n = sum(len(v) for v in UNTRACKED.values())
    print(f"ADOPT_UNTRACKED is False. {_n} untracked embedding file(s) are being ignored"
          + (" -- set it to True and re-run if you want them checked and adopted."
             if _n else "; nothing to adopt anyway."))
else:
    adopted = rejected = 0
    with open(PROGRESS_LOG_PATH, "a") as f:
        for model_name, uids in UNTRACKED.items():
            expected_dim = globals().get("EMBED_DIMS", {}).get(model_name)
            for suid in uids:
                p = os.path.join(EMBED_DIR, model_name, f"{suid}.npy")
                try:
                    v = np.load(p).astype(np.float32).reshape(-1)
                except Exception as e:
                    print(f"  reject {suid[:24]}...: unreadable ({e})")
                    rejected += 1
                    continue
                problems = []
                if expected_dim and v.shape[0] != expected_dim:
                    problems.append(f"dim {v.shape[0]} != {expected_dim}")
                if not np.isfinite(v).all():
                    problems.append("non-finite values")
                if v.std() < 1e-6:
                    problems.append("degenerate (near-zero variance)")
                n = float(np.linalg.norm(v))
                if abs(n - 1.0) > 0.05:
                    problems.append(f"L2 norm {n:.4f} -- Pillar-0 output is normalised to ~1")
                if problems:
                    print(f"  reject {suid[:24]}...: {'; '.join(problems)}")
                    rejected += 1
                    continue
                f.write(json.dumps({
                    "series_uid": suid, "model_name": model_name, "status": "ok",
                    "detail": "adopted from disk by cell 18b", "adopted": True,
                    "dim": int(v.shape[0]), "norm": n,
                }) + "\n")
                adopted += 1
    print(f"\nadopted {adopted}, rejected {rejected}. Re-run cell 18 to pick them up.")

In [ ]:
# 19. Embedding quality control
#
# A Pillar-0 embedding is L2-normalised by construction, so "norm == 1" proves nothing about
# whether the model actually saw your data. These checks look for the failure modes that DO show
# up as a signature: a collapsed representation (near-zero variance across the dataset), identical
# vectors for different scans (a preprocessing bug feeding the same volume repeatedly), and dead
# dimensions.
import numpy as np
import pandas as pd

for _m, _tab in FEATURES.items():
    cols = FEATURE_COLUMNS[_m]
    X = _tab[cols].to_numpy(dtype=np.float32)
    print(f"\n=== {_m}: {X.shape[0]:,} embeddings x {X.shape[1]} dims ===")

    if X.shape[0] < 10:
        # Cross-scan statistics (per-dimension variance, duplicate detection, PCA) are meaningless
        # on a handful of rows -- per-dim std over one sample is exactly 0, which would look like a
        # dead model rather than a small sample. Report only the per-vector checks.
        norms = np.linalg.norm(X, axis=1)
        print(f"  only {X.shape[0]} embedding(s) -- cross-scan statistics are not meaningful yet.")
        print(f"  L2 norm      : mean {norms.mean():.4f}, min {norms.min():.4f}, max {norms.max():.4f}")
        print(f"  within-vector std: "
              f"{', '.join(f'{v:.5f}' for v in X.std(axis=1)[:5])}"
              f"{' ...' if X.shape[0] > 5 else ''}  "
              f"(expect ~{1/np.sqrt(X.shape[1]):.5f} for a healthy 1/sqrt(dim)-spread vector)")
        print(f"  finite       : {bool(np.isfinite(X).all())}")
        continue

    norms = np.linalg.norm(X, axis=1)
    print(f"  L2 norm      : mean {norms.mean():.4f}, min {norms.min():.4f}, max {norms.max():.4f}")
    print(f"                 (~1.0 is expected -- the model normalises internally)")

    per_dim_std = X.std(axis=0)
    print(f"  per-dim std  : mean {per_dim_std.mean():.5f}, "
          f"min {per_dim_std.min():.6f}, max {per_dim_std.max():.5f}")
    dead = int((per_dim_std < 1e-6).sum())
    print(f"  dead dims    : {dead} / {X.shape[1]}"
          + ("  <-- a lot of these means part of the model never activated" if dead > X.shape[1] * 0.1 else ""))

    # Exact duplicate vectors: different scans should never produce byte-identical embeddings.
    _, uniq_idx = np.unique(np.round(X, 6), axis=0, return_index=True)
    n_dup = X.shape[0] - len(uniq_idx)
    print(f"  duplicate embeddings (to 1e-6): {n_dup}")
    if n_dup:
        print("    -> investigate. Identical embeddings for distinct series usually means the")
        print("       same volume was preprocessed repeatedly (a series-lookup bug), not that")
        print("       two patients' brains are identical.")

    # How much of the variance is real structure vs noise -- a quick, cheap sanity number.
    Xc = X - X.mean(axis=0, keepdims=True)
    n_comp = min(50, min(Xc.shape) - 1)
    if n_comp > 1:
        try:
            s = np.linalg.svd(Xc, compute_uv=False)
            evr = (s ** 2) / (s ** 2).sum()
            print(f"  variance in top 10 PCs: {evr[:10].sum():.1%}, "
                  f"top 50: {evr[:min(50, len(evr))].sum():.1%}")
            print(f"    (a healthy representation spreads variance over many components; >90% in")
            print(f"     the first component or two would suggest collapse)")
        except Exception as e:
            print(f"  (SVD skipped: {e})")

    if "n_slices_expected" in _tab.columns:
        print(f"  source slice counts: median {_tab['n_slices_expected'].median():.0f}, "
              f"range {_tab['n_slices_expected'].min():.0f}-{_tab['n_slices_expected'].max():.0f}")
    if "head_subtype" in _tab.columns:
        print(f"  head subtypes: {_tab['head_subtype'].value_counts().to_dict()}")

In [ ]:
# ==============================================================================================
# 19b. HEAD COVERAGE AND SPATIAL-MAP AUDIT -- both computed WITHOUT re-reading a single scan
# ==============================================================================================
# Two things the paper needs that the extraction log can answer on its own.
#
# 1. HOW MUCH HEAD FITS IN OUR WINDOW. Our grid is 64 slices x 3.0 mm = 192 mm of z and
#    256 px x 1.0 mm = 256 mm in plane. Whether that holds a whole Indian head CT is an
#    empirical question, and it is a LIMITATION THE PAPER MUST STATE -- lateral convexity and
#    vertex are exactly where subdural and epidural collections sit, so a z shortfall does not
#    lower every target equally.
#
#    Measuring it exactly needs the voxels. But every 'ok' record already carries n_slices and
#    the z spacing MEASURED from ImagePositionPatient, so the scanned z extent is free:
#        z_extent_mm = n_slices x sz,   retained_z ~ min(1, 192 / z_extent_mm)
#    Two assumptions, stated rather than hidden: the head fills the scanned z range (true for a
#    head CT protocol, and n_slices here is after the QC drop), and the 192 mm window is centred
#    on the same place the scan is. Both make this an OPTIMISTIC estimate of the exact figure,
#    so treat it as an upper bound on retention and disclose it as an estimate.
#
#    There is also a hard LOWER bound available for free, and it is worth more than it looks.
#    The earlier Pillar-0 run measured retention properly, on the voxels, through a window of
#    128 x 1.25 = 160 mm in z and 256 x 0.5 = 128 mm in plane, and got median 0.960 in z and
#    median 1.000 in plane. Retention rises monotonically with window size for the same centring,
#    and our window is LARGER on both axes (192 mm vs 160 mm; 256 mm vs 128 mm). So our true
#    retention cannot be worse than that. The 68%-lost-more-than-2% line from the previous run
#    describes a window smaller than ours in every dimension.
#
# 2. THE Q3 SPATIAL MAPS. Q3 quanvolution is the best arm on RSNA and reads
#    spatial_maps/<backbone>/<series>.npy, which nothing downstream checks. A relabelled series
#    is never re-extracted, so a missing map would be a permanent hole in the headline
#    experiment. Cell 16d refuses to relabel a series whose map is absent; this counts them.
import os as _os

import numpy as _np

print("=" * 94)
print("HEAD COVERAGE THROUGH OUR WINDOW  (estimated from the log; no scan is re-read)")
print("=" * 94)

_cov_z = TARGET_SHAPE_DHW[0] * TARGET_SPACING_XYZ[2]
_cov_xy = TARGET_SHAPE_DHW[1] * TARGET_SPACING_XYZ[1]
_rows = [r for r in _progress_rows(PROGRESS_LOG_PATH)
         if r.get("status") == "ok" and r.get("preproc_sig") == PREPROC_SIG
         and r.get("n_slices") and r.get("spacing")]
_seen, _ext = set(), []
for _r in _rows:                                   # one row per SERIES, not per (series, model)
    _s = _r.get("series_uid")
    if _s in _seen:
        continue
    _seen.add(_s)
    try:
        _sz = float(_r["spacing"][2])
        _n = int(_r["n_slices"])
    except Exception:
        continue
    if _n > 0 and _sz > 0:
        _ext.append(_n * _sz)

if not _ext:
    print("  no current-signature records with n_slices yet -- run the extraction cell first.")
else:
    _e = _np.asarray(_ext, dtype=float)
    _ret = _np.minimum(1.0, _cov_z / _e)
    _q = _np.percentile(_e, [10, 50, 90])
    print(f"  window            : {_cov_z:.0f} mm in z ({TARGET_SHAPE_DHW[0]} x "
          f"{TARGET_SPACING_XYZ[2]} mm) | {_cov_xy:.0f} mm in plane")
    print(f"  scanned z extent  : median {_q[1]:.0f} mm, 10th {_q[0]:.0f}, 90th {_q[2]:.0f} "
          f"(n={_e.size:,} series)")
    print(f"  estimated z kept  : median {_np.median(_ret):.3f}, 10th pct "
          f"{_np.percentile(_ret, 10):.3f}, min {_ret.min():.3f}")
    _short = int((_e > _cov_z).sum())
    print(f"  longer than the window: {_short:,} / {_e.size:,} series ({100.0 * _short / _e.size:.1f}%)")
    print(f"  DISCLOSE THIS. It is an estimate (head assumed to fill the scanned z range and to")
    print(f"  be centred in it), so it is an UPPER bound on retention. The measured LOWER bound")
    print(f"  from the earlier 160 mm / 128 mm run is median 0.960 in z and 1.000 in plane, and")
    print(f"  our window is larger on both axes -- so the truth sits between the two.")
    if "head_retained" in {k for _r in _rows for k in _r} and any(
            _r.get("head_retained") is not None for _r in _rows):
        _hr = _np.asarray([_r["head_retained"] for _r in _rows
                           if _r.get("head_retained") is not None], dtype=float)
        print(f"  MEASURED on voxels for {_hr.size:,} record(s) extracted since the fix: "
              f"median {_np.median(_hr):.3f}, min {_hr.min():.3f}")
        print("  -> prefer these over the estimate wherever the two disagree.")

print()
print("=" * 94)
print("Q3 SPATIAL MAPS -- the arm the paper leads with cannot run without these")
print("=" * 94)
_missing_total = 0
for _b in MODELS_TO_RUN:
    _sids = sorted({r.get("series_uid") for r in _progress_rows(PROGRESS_LOG_PATH)
                    if r.get("status") == "ok" and r.get("model_name") == _b
                    and r.get("preproc_sig") == PREPROC_SIG})
    _d = _os.path.join(SPATIAL_DIR, _b)
    _have = set(f[:-4] for f in _os.listdir(_d)) if _os.path.isdir(_d) else set()
    _miss = [s for s in _sids if s not in _have]
    _missing_total += len(_miss)
    _shape = _dtype = "-"
    if _have:
        try:
            _a = _np.load(_os.path.join(_d, f"{sorted(_have)[0]}.npy"))
            _shape, _dtype = tuple(_a.shape), _a.dtype
        except Exception:
            pass
    print(f"  {_b:<26} ok={len(_sids):,}  maps={len(_have):,}  missing={len(_miss):,}  "
          f"shape={_shape} {_dtype}")
    for _s in _miss[:3]:
        print(f"      missing: {_s}")
if _missing_total:
    print(f"  !! {_missing_total:,} map(s) missing. Cell 16d will NOT relabel those series, so")
    print("     cell 17 re-extracts them. Re-run cell 17, then this cell, until it reads 0.")
else:
    print("  every current-signature embedding has its spatial map. Q3 can run on this cohort.")
print("=" * 94)


## From report text to training labels

The finding columns in this export are prefixed by body system:

| prefix | n | what it covers |
|---|---|---|
| `b_` | 44 | brain / head — hemorrhage, infarct, edema, mass, hydrocephalus, skull & facial fractures, sinusitis |
| `f_` | 5 | paranasal sinuses |
| `n_` | 5 | neck |
| `c_` | 25 | chest |
| `a_` | 28 | abdomen |
| `s_` | 17 | spine |
| `p_` | 15 | pelvis / extremity |
| `v_` | 6 | vascular |
| `g_` | 10 | global — artifacts, "no X overall" summary flags, workup suggestions |

A head-CT model is evaluated on `b_` and `f_` columns. Training a head model to predict
`a_fatty_liver` is not a hard problem, it is a meaningless one — the finding is not in the pixels.

**Two label bugs this cell exists to avoid.**

*Abbreviations.* The five hemorrhage compartments are named `b_sdh`, `b_sah`, `b_edh`, `b_ich`,
`b_ivh`. Keyword matching on `"subdural"`/`"subarachnoid"`/`"intraventricular"` — which is what the
v6 notebook did — matches **none** of them. Matching is done on whole underscore-separated tokens
here, so `ich` matches `b_ich` and nothing else, and every matched column is printed.

*Negations counted as findings.* `g_no_intracranial_hemorrhage_overall`, `s_no_bony_injury`,
`f_pns_normal`, `v_normal_study` are all columns that are positive when something is **absent**. An
"abnormal = any finding column is 1" rule that includes them labels healthy scans as abnormal. Any
column carrying a `no` / `normal` / `unremarkable` / `negative` token is treated as a
normal-indicator, never as an abnormality signal — and they are used instead as an independent
cross-check on the derived label.

In [ ]:
# 20. Build labels and cohorts from the report findings
import numpy as np
import pandas as pd

REGION_FINDING_PREFIXES = {
    "head":    ("b_", "f_"),
    "chest":   ("c_",),
    "abdomen": ("a_", "p_"),
}

# A column carrying any of these as a whole token is positive when something is ABSENT.
NEGATION_TOKENS = {"no", "normal", "unremarkable", "negative", "nil"}

# Whole-token matching is what makes `b_sdh` work, but "normal" is also a real word inside real
# finding names -- `b_normal_pressure_hydrocephalus` would be classified as a normal-indicator,
# then dropped from the eligible findings, excluded from IS_ABNORMAL, and used as an INVERTED
# cross-check. No such column exists in this export, but the rule is generic. The full
# classification is printed below; if anything in NORMAL_INDICATOR_COLS is actually a finding,
# name it here and re-run.
FORCE_ABNORMAL_SIGNAL_COLS = []          # e.g. ["b_normal_pressure_hydrocephalus"]
FORCE_NORMAL_INDICATOR_COLS = []


def _tokens(col):
    return set(col.lower().replace("-", "_").split("_"))


def is_normal_indicator(col):
    if col in FORCE_ABNORMAL_SIGNAL_COLS:
        return False
    if col in FORCE_NORMAL_INDICATOR_COLS:
        return True
    return bool(_tokens(col) & NEGATION_TOKENS)


# ---- where the labels come from, per dataset -------------------------------------------------
if DATA_SOURCE == "india":
    ACTIVE_PREFIXES = REGION_FINDING_PREFIXES.get(TARGET_REGION, tuple(FINDING_PREFIXES))
    REGION_FINDING_COLS = [c for c in FINDING_COLS_ALL if c.lower().startswith(ACTIVE_PREFIXES)]
    LABEL_KEY = "study_uid"
    # Labels come from the matched REPORT, i.e. they are per-study. Embeddings are per-series.
    # With PROCESS_ALL_QUALIFYING_SERIES = True several series of one study share a label;
    # cross-validation below always groups by patient, so those never straddle a split.
    _src = df[["StudyInstanceUID"] + REGION_FINDING_COLS].copy()
    LABEL_TABLE = _src.groupby("StudyInstanceUID", as_index=False).max(numeric_only=True)
    LABEL_TABLE = LABEL_TABLE.rename(columns={"StudyInstanceUID": LABEL_KEY})
    _incons = (_src.groupby("StudyInstanceUID")[REGION_FINDING_COLS].nunique(dropna=False) > 1)
    _n_incons = int(_incons.any(axis=1).sum())
    if _n_incons:
        print(f"NOTE: {_n_incons} study/studies have rows that disagree on at least one finding "
              f"column. Taking the per-study max (a finding mentioned on any row counts as present).")
    print(f"region {TARGET_REGION!r} -> finding prefixes {ACTIVE_PREFIXES}")
else:
    # RSNA: expert per-slice labels, already aggregated to series level in cell B3 (a series is
    # positive for a subtype if any of its slices is). `rsna_any` is deliberately NOT in the
    # per-finding list -- it is the normal-vs-abnormal cohort below, and evaluating it twice under
    # two names would just put the same number on the leaderboard twice.
    ACTIVE_PREFIXES = ("rsna_",)
    _label_cols = [c for c in manifest.columns
                   if c.startswith("rsna_") and not c.endswith("_n_pos_slices")]
    REGION_FINDING_COLS = [c for c in _label_cols if c != "rsna_any"]
    LABEL_KEY = "series_uid"
    LABEL_TABLE = manifest[["series_uid"] + _label_cols].drop_duplicates("series_uid")
    print("RSNA labels: expert per-slice annotations, max-aggregated to series level")
    print(f"  subtype columns: {REGION_FINDING_COLS}")
    print("  rsna_any is held out of the per-finding sweep and used as the "
          "normal-vs-abnormal cohort instead")

NORMAL_INDICATOR_COLS = [c for c in REGION_FINDING_COLS if is_normal_indicator(c)]
ABNORMAL_SIGNAL_COLS = [c for c in REGION_FINDING_COLS if c not in NORMAL_INDICATOR_COLS]

print(f"  finding columns in scope   : {len(REGION_FINDING_COLS)}")
print(f"  normal-indicator columns   : {len(NORMAL_INDICATOR_COLS)} -> {NORMAL_INDICATOR_COLS}")
if NORMAL_INDICATOR_COLS:
    print("    ^ read this list. Anything there that is really a FINDING (a name that merely")
    print("      contains the word 'normal', e.g. normal-pressure hydrocephalus) must be added to")
    print("      FORCE_ABNORMAL_SIGNAL_COLS above -- otherwise it is excluded from training and")
    print("      used as an inverted cross-check.")
print(f"  abnormality-signal columns : {len(ABNORMAL_SIGNAL_COLS)}")

LABELS = {}
_label_value_cols = [c for c in LABEL_TABLE.columns if c != LABEL_KEY]
for _m, _tab in FEATURES.items():
    lab = _tab.merge(LABEL_TABLE, on=LABEL_KEY, how="left", suffixes=("", "_lbl"))
    n_unlabelled = int(lab[_label_value_cols].isna().all(axis=1).sum())
    for c in _label_value_cols:
        lab[c] = pd.to_numeric(lab[c], errors="coerce").fillna(0).astype(np.int8)
    LABELS[_m] = lab
    print(f"\n[{_m}] labelled {len(lab):,} series on {LABEL_KEY}"
          + (f"; {n_unlabelled:,} had no matching label row (all set to 0 -- exclude them if that "
             f"matters to you)" if n_unlabelled else ""))

_ref = next(iter(LABELS.values())) if LABELS else None
if _ref is None:
    print("\nNo feature table yet -- run cells 17 and 18 first.")
else:
    # --- eligible per-finding targets ---------------------------------------------------------
    _counts = [(c, int(_ref[c].sum())) for c in ABNORMAL_SIGNAL_COLS]
    _counts.sort(key=lambda t: -t[1])
    ELIGIBLE_FINDINGS = [c for c, n in _counts if n >= MIN_POSITIVES]
    if MAX_FINDINGS_TO_EVALUATE is not None:
        ELIGIBLE_FINDINGS = ELIGIBLE_FINDINGS[:MAX_FINDINGS_TO_EVALUATE]
    FINDING_POSITIVES = dict(_counts)

    print(f"\nfindings with >= {MIN_POSITIVES} positives among the extracted scans: "
          f"{len(ELIGIBLE_FINDINGS)} of {len(ABNORMAL_SIGNAL_COLS)}")
    for c, n in _counts[:20]:
        mark = "  " if c in ELIGIBLE_FINDINGS else "x "
        print(f"  {mark}{c:45s} {n:5d}  ({n/len(_ref):.1%})")
    if len(_counts) > 20:
        print(f"  ... {len(_counts) - 20} more")

    # --- cohort: normal vs abnormal -----------------------------------------------------------
    if DATA_SOURCE == "rsna" and "rsna_any" in _ref.columns:
        # Use the LABELLED `any` column rather than re-deriving it from the five compartments.
        # They are nearly the same but not identically so: an annotator can mark hemorrhage
        # present without committing to a compartment, and the labelled column is the ground
        # truth as recorded.
        IS_ABNORMAL = pd.Series((_ref["rsna_any"].to_numpy() > 0).astype(int),
                                 index=_ref["series_uid"], name="cohort_abnormal")
        _derived5 = (_ref[ABNORMAL_SIGNAL_COLS].to_numpy() > 0).any(axis=1)
        _mismatch = int((IS_ABNORMAL.to_numpy().astype(bool) != _derived5).sum())
        print(f"\nCOHORT normal-vs-abnormal (= labelled rsna_any): "
              f"{int(IS_ABNORMAL.sum()):,}/{len(IS_ABNORMAL):,} ({IS_ABNORMAL.mean():.1%}) positive.")
        print(f"  differs from OR(5 compartments) on {_mismatch:,} series "
              f"({_mismatch/max(len(IS_ABNORMAL),1):.2%}).")
    else:
        _abn = _ref[ABNORMAL_SIGNAL_COLS].to_numpy() > 0
        IS_ABNORMAL = pd.Series(_abn.any(axis=1).astype(int), index=_ref["series_uid"],
                                 name="cohort_abnormal")
        print(f"\nCOHORT normal-vs-abnormal: {int(IS_ABNORMAL.sum()):,}/{len(IS_ABNORMAL):,} "
              f"({IS_ABNORMAL.mean():.1%}) abnormal, from {len(ABNORMAL_SIGNAL_COLS)} signal columns.")
    if NORMAL_INDICATOR_COLS:
        _explicit_normal = (_ref[NORMAL_INDICATOR_COLS].to_numpy() > 0).any(axis=1)
        _derived = IS_ABNORMAL.to_numpy().astype(bool)
        _consistent = (_explicit_normal != _derived)
        # The overall rate is biased low by construction: a clean study with a terse report emits
        # neither an explicit normal flag nor any finding, so both sides are False and the
        # opposite-polarity test counts it as a disagreement. Report the informative subset too.
        _informative = _explicit_normal | _derived
        _inf_rate = (_consistent[_informative].mean() if _informative.any() else float("nan"))
        print(f"  cross-check against the explicit normal-indicator columns: consistent for "
              f"{_consistent.mean():.1%} of all scans, and for {_inf_rate:.1%} of the "
              f"{int(_informative.sum()):,} scans where at least one of the two signals fired "
              f"(the second number is the meaningful one -- see the comment in this cell).")
        print(f"  Disagreement is expected in part -- a report can say 'no significant abnormality'")
        print(f"  and still flag something minor. The derived any-positive signal is the training")
        print(f"  label because it is grounded in the per-finding data, not a summary flag.")

    # --- cohort: intracranial hemorrhage ------------------------------------------------------
    # Whole-token matching, so the abbreviations this dataset actually uses are caught.
    HEMORRHAGE_SUBTYPE_TERMS = {
        "SUBDURAL":         {"subdural", "sdh"},
        "EPIDURAL":         {"epidural", "extradural", "edh"},
        "SUBARACHNOID":     {"subarachnoid", "sah"},
        "INTRAVENTRICULAR": {"intraventricular", "ivh"},
        "INTRAPARENCHYMAL": {"intracerebral", "intraparenchymal", "ich", "iph"},
    }
    _GENERIC_HEM_SUBSTRINGS = ("hemorrhag", "haemorrhag", "hematoma", "haematoma", "bleed")

    # ...but a generic bleed word is not enough on its own, because some of these collections are
    # OUTSIDE the skull. In the first overnight run `b_soft_tissue_hematoma` matched on "hematoma"
    # and contributed 192 of the cohort's 304 positives -- so 63% of what was reported as
    # "intracranial hemorrhage" was scalp. Any column naming a soft-tissue/scalp collection is
    # excluded from the generic sweep. Named-compartment columns (sdh/edh/sah/ivh/ich) are never
    # touched by this, and `b_hemorrhagic_contusion` -- genuinely intracranial -- is kept.
    EXTRACRANIAL_HEM_SUBSTRINGS = (
        "soft_tissue", "soft tissue", "scalp", "subgaleal", "subcutaneous",
        "cephalhematoma", "cephalohematoma", "periorbital", "preseptal",
    )

    HEMORRHAGE_SUBTYPE_COLS = {}
    for _sub, _terms in HEMORRHAGE_SUBTYPE_TERMS.items():
        _cols = [c for c in ABNORMAL_SIGNAL_COLS if _tokens(c) & _terms]
        if _cols:
            HEMORRHAGE_SUBTYPE_COLS[_sub] = _cols

    _named = {c for cols in HEMORRHAGE_SUBTYPE_COLS.values() for c in cols}
    _generic_all = [c for c in ABNORMAL_SIGNAL_COLS
                    if c not in _named and any(s in c.lower() for s in _GENERIC_HEM_SUBSTRINGS)]
    _extracranial = [c for c in _generic_all
                     if any(s in c.lower() for s in EXTRACRANIAL_HEM_SUBSTRINGS)]
    _generic = [c for c in _generic_all if c not in _extracranial]
    ANY_HEMORRHAGE_COLS = sorted(_named | set(_generic))

    print(f"\nCOHORT intracranial hemorrhage: {len(ANY_HEMORRHAGE_COLS)} column(s)")
    for _sub, _cols in HEMORRHAGE_SUBTYPE_COLS.items():
        print(f"  {_sub:18s}: {_cols}")
    if _generic:
        print(f"  {'UNSPECIFIED':18s}: {_generic}  (generic bleed term, no named compartment)")
    if _extracranial:
        print(f"  {'EXCLUDED':18s}: {_extracranial}")
        print(f"  {'':18s}  ^ matched a bleed word but names an EXTRACRANIAL collection, so it is")
        print(f"  {'':18s}    not part of an intracranial cohort. Each is still evaluated as its")
        print(f"  {'':18s}    own finding. Edit EXTRACRANIAL_HEM_SUBSTRINGS above to change this.")
    if not ANY_HEMORRHAGE_COLS:
        print("  none matched -- the hemorrhage cohort will be skipped.")
        ANY_HEMORRHAGE = None
    else:
        ANY_HEMORRHAGE = pd.Series(
            (_ref[ANY_HEMORRHAGE_COLS].to_numpy() > 0).any(axis=1).astype(int),
            index=_ref["series_uid"], name="cohort_any_hemorrhage")
        print(f"  -> {int(ANY_HEMORRHAGE.sum()):,}/{len(ANY_HEMORRHAGE):,} "
              f"({ANY_HEMORRHAGE.mean():.1%}) positive")
        _neg_check = [c for c in NORMAL_INDICATOR_COLS
                      if "hemorrhag" in c.lower() or "haemorrhag" in c.lower()]
        if _neg_check:
            _says_none = (_ref[_neg_check].to_numpy() > 0).any(axis=1)
            _conflict = int((_says_none & (ANY_HEMORRHAGE.to_numpy() > 0)).sum())
            print(f"  independent cross-check against {_neg_check}: {_conflict} scan(s) are "
                  f"flagged BOTH 'no intracranial hemorrhage' and a specific bleed. "
                  f"{'Worth eyeballing those reports.' if _conflict else 'Consistent.'}")

    # --- cohort: any skull / facial fracture ---------------------------------------------------
    # RSNA carries no fracture labels; this cohort simply does not exist there.
    FRACTURE_COLS = [c for c in ABNORMAL_SIGNAL_COLS if "fracture" in c.lower()]
    ANY_FRACTURE = None
    if FRACTURE_COLS:
        ANY_FRACTURE = pd.Series(
            (_ref[FRACTURE_COLS].to_numpy() > 0).any(axis=1).astype(int),
            index=_ref["series_uid"], name="cohort_any_fracture")
        print(f"\nCOHORT any fracture: {len(FRACTURE_COLS)} column(s), "
              f"{int(ANY_FRACTURE.sum()):,}/{len(ANY_FRACTURE):,} ({ANY_FRACTURE.mean():.1%}) positive")

    # A cohort with no positives (or no negatives) cannot be evaluated; drop it here rather than
    # letting it become a confusing "skipped" line in the sweep.
    SPECIAL_COHORTS = {}
    for _cname, _cvals in (("normal_vs_abnormal", IS_ABNORMAL),
                            ("any_intracranial_hemorrhage", ANY_HEMORRHAGE),
                            ("any_fracture", ANY_FRACTURE)):
        if _cvals is None:
            continue
        _npos = int(_cvals.sum())
        if _npos < MIN_POSITIVES or (len(_cvals) - _npos) < MIN_POSITIVES:
            print(f"  cohort {_cname!r} dropped: {_npos} positive / "
                  f"{len(_cvals) - _npos} negative, below MIN_POSITIVES={MIN_POSITIVES}")
            continue
        SPECIAL_COHORTS[_cname] = _cvals
    print(f"\nspecial cohorts ready: {list(SPECIAL_COHORTS)}")

In [ ]:
# 20b. Evaluation metrics -- the numbers, and how uncertain each one is
#
# Every metric below is used by BOTH the per-finding sweep (cell 21) and the cross-dataset
# hemorrhage matrix (Part C), so the two are directly comparable.
#
# Four things this does that a bare `roc_auc_score` does not:
#
#   CONFIDENCE INTERVALS BY CLUSTER BOOTSTRAP. Resampling ROWS is wrong here: one patient can
#     contribute several series, so rows are not independent and a row bootstrap reports an
#     interval that is too narrow. Patients are resampled instead, carrying all their rows.
#
#   INTERVALS ON THE OPERATING-POINT METRICS TOO. A sensitivity of 0.83 from 6 positives is not
#     the same claim as 0.83 from 600, and quoting both as "0.83" hides the entire difference.
#     Wilson intervals (not normal-approximation ones, which are badly wrong near 0 and 1).
#
#   CALIBRATION, NOT JUST DISCRIMINATION. AUROC is invariant to any monotone transform of the
#     scores, so a model can rank perfectly and still output probabilities that mean nothing --
#     which matters the moment anyone acts on a threshold. Brier score, calibration slope and
#     intercept, and expected calibration error.
#
#   PAIRED COMPARISONS. "0.84 vs 0.81" is not a result until you know whether the difference
#     survives resampling. The paired bootstrap keeps both models on the same resampled patients,
#     which is far more powerful than comparing two independent intervals.
import numpy as np
from sklearn.metrics import (average_precision_score, brier_score_loss,
                              precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.linear_model import LogisticRegression

N_BOOTSTRAP_SWEEP = 500      # per finding in cell 21 -- keep modest, it runs ~100 times
N_BOOTSTRAP_MAIN = 2000      # for the headline hemorrhage results in Part C


def fast_auroc(y, p):
    """Rank-based AUROC. Same value as sklearn's, ~10x faster, which matters at 2,000 bootstrap
    replicates x many targets. Handles ties by mid-ranks, exactly like the Mann-Whitney statistic."""
    y = np.asarray(y)
    n_pos = int(y.sum())
    n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    order = np.argsort(p, kind="mergesort")
    ranks = np.empty(len(p), dtype=np.float64)
    sp = np.asarray(p)[order]
    i = 0
    while i < len(sp):
        j = i
        while j + 1 < len(sp) and sp[j + 1] == sp[i]:
            j += 1
        ranks[order[i:j + 1]] = 0.5 * (i + j) + 1.0
        i = j + 1
    return float((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def fast_auprc(y, p):
    """Average precision, computed directly from the sorted scores. Same definition as sklearn's
    `average_precision_score` (a step-wise sum, not trapezoidal interpolation), several times
    faster -- which is what makes 2,000 bootstrap replicates practical."""
    y = np.asarray(y).astype(np.int64)
    n_pos = int(y.sum())
    if n_pos == 0:
        return float("nan")
    order = np.argsort(-np.asarray(p, dtype=float), kind="mergesort")
    ys = y[order]
    ps = np.asarray(p, dtype=float)[order]
    tp = np.cumsum(ys)
    fp = np.cumsum(1 - ys)
    # Collapse tied scores to their last index, so all tied items share one precision/recall point.
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp = tp[last], fp[last]
    precision = tp / np.maximum(tp + fp, 1)
    recall = tp / n_pos
    d_recall = np.diff(np.r_[0.0, recall])
    return float(np.sum(precision * d_recall))


def wilson_ci(k, n, alpha=0.05):
    """Wilson score interval for a proportion. Correct near 0 and 1, where the normal
    approximation produces intervals that run outside [0, 1]."""
    if n == 0:
        return (float("nan"), float("nan"))
    # stdlib rather than scipy: identical to scipy.stats.norm.ppf to 12 decimal places, and it
    # removes a dependency the notebook never installs -- which would otherwise be a NameError
    # deep inside the metrics, hours into a run.
    from statistics import NormalDist
    z = NormalDist().inv_cdf(1 - alpha / 2)
    ph = k / n
    d = 1 + z ** 2 / n
    c = ph + z ** 2 / (2 * n)
    half = z * np.sqrt(ph * (1 - ph) / n + z ** 2 / (4 * n ** 2))
    lo, hi = (c - half) / d, (c + half) / d
    # At k == 0 the exact answer is 0, but c and half are algebraically equal and differ by one
    # ulp in IEEE double, leaving ~1e-17 behind. Snap it, so a "0.000" bound really is zero rather
    # than a number that merely prints as one.
    if abs(lo) < 1e-12:
        lo = 0.0
    if abs(hi - 1.0) < 1e-12:
        hi = 1.0
    return (float(max(0.0, lo)), float(min(1.0, hi)))


def _group_index(groups):
    uniq, inv = np.unique(np.asarray(groups), return_inverse=True)
    members = [np.where(inv == i)[0] for i in range(len(uniq))]
    return uniq, members


def cluster_bootstrap(y, p, groups, stat_fns, n_boot=1000, alpha=0.05, seed=0):
    """Resample PATIENTS with replacement, recompute each statistic, return percentile intervals.

    Returns {name: (point, lo, hi)}. A replicate that ends up with only one class present is
    dropped rather than counted as 0.5 -- silently folding degenerate replicates in is how a
    bootstrap interval ends up biased toward chance.
    """
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    point = {name: fn(y, p) for name, fn in stat_fns.items()}
    _, members = _group_index(groups)
    n_groups = len(members)
    if n_groups < 3 or n_boot <= 0:
        return {k: (v, float("nan"), float("nan")) for k, v in point.items()}
    rng = np.random.default_rng(seed)
    acc = {name: [] for name in stat_fns}
    n_degenerate = 0
    for _ in range(n_boot):
        pick = rng.integers(0, n_groups, size=n_groups)
        idx = np.concatenate([members[i] for i in pick])
        yb = y[idx]
        if yb.sum() == 0 or yb.sum() == len(yb):
            n_degenerate += 1
            continue
        pb = p[idx]
        for name, fn in stat_fns.items():
            try:
                acc[name].append(fn(yb, pb))
            except Exception:
                pass
    out = {}
    for name, vals in acc.items():
        v = np.asarray([x for x in vals if np.isfinite(x)], dtype=float)
        if len(v) < max(20, 0.1 * n_boot):
            out[name] = (point[name], float("nan"), float("nan"))
        else:
            out[name] = (point[name],
                          float(np.percentile(v, 100 * alpha / 2)),
                          float(np.percentile(v, 100 * (1 - alpha / 2))))
    out["_n_degenerate_replicates"] = n_degenerate
    return out


def discrimination_metrics(y, p, groups=None, n_boot=1000, seed=0):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    if groups is None:
        groups = np.arange(len(y))
    stats = {"auroc": fast_auroc, "auprc": fast_auprc}
    ci = cluster_bootstrap(y, p, groups, stats, n_boot=n_boot, seed=seed)
    prev = float(y.mean())
    return {
        "n": int(len(y)), "n_pos": int(y.sum()), "prevalence": prev,
        "auroc": ci["auroc"][0], "auroc_lo": ci["auroc"][1], "auroc_hi": ci["auroc"][2],
        "auprc": ci["auprc"][0], "auprc_lo": ci["auprc"][1], "auprc_hi": ci["auprc"][2],
        # AUPRC has to be read against prevalence: a random ranker scores exactly the prevalence,
        # so 0.30 is excellent at 5% prevalence and worthless at 40%.
        "auprc_baseline": prev,
        "auprc_lift": (ci["auprc"][0] / prev) if prev > 0 else float("nan"),
    }


def threshold_metrics(y, p, thr, alpha=0.05):
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p, dtype=float) >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    sens_lo, sens_hi = wilson_ci(tp, tp + fn, alpha)
    spec_lo, spec_hi = wilson_ci(tn, tn + fp, alpha)
    ppv_lo, ppv_hi = wilson_ci(tp, tp + fp, alpha)
    npv_lo, npv_hi = wilson_ci(tn, tn + fn, alpha)
    sens = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    ppv = tp / (tp + fp) if (tp + fp) else float("nan")
    npv = tn / (tn + fn) if (tn + fn) else float("nan")
    f1 = (2 * ppv * sens / (ppv + sens)) if (np.isfinite(ppv) and np.isfinite(sens)
                                              and (ppv + sens) > 0) else float("nan")
    return {
        "threshold": float(thr), "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "sensitivity": sens, "sensitivity_lo": sens_lo, "sensitivity_hi": sens_hi,
        "specificity": spec, "specificity_lo": spec_lo, "specificity_hi": spec_hi,
        "ppv": ppv, "ppv_lo": ppv_lo, "ppv_hi": ppv_hi,
        "npv": npv, "npv_lo": npv_lo, "npv_hi": npv_hi,
        "f1": f1,
        "balanced_accuracy": ((sens + spec) / 2 if np.isfinite(sens) and np.isfinite(spec)
                               else float("nan")),
        "accuracy": (tp + tn) / max(tp + tn + fp + fn, 1),
    }


def calibration_metrics(y, p, n_bins=10):
    """Brier, calibration slope/intercept, and ECE.

    The slope/intercept come from regressing the outcome on the model's own log-odds. Perfect
    calibration is slope 1, intercept 0. Slope < 1 means the scores are too extreme (the usual
    result of training on a class-weighted objective and then reading the probabilities
    literally); intercept != 0 means a systematic shift, which is exactly what happens when a
    model trained at one prevalence is applied at another -- the transfer cells in Part C.
    """
    y = np.asarray(y).astype(int)
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    out = {"brier": float(brier_score_loss(y, p))}
    logit = np.log(p / (1 - p)).reshape(-1, 1)
    # A very large C rather than penalty=None or C=inf. `penalty=None` is deprecated from
    # sklearn 1.8; `C=np.inf` still triggers "Setting penalty=None will ignore the C parameter" on
    # every call, and this function is called once per evaluation. 1e12 is unregularised for any
    # practical purpose and warns about nothing.
    try:
        lr = LogisticRegression(C=1e12, solver="lbfgs", max_iter=1000)
        lr.fit(logit, y)
        out["calibration_slope"] = float(lr.coef_[0][0])
        out["calibration_intercept"] = float(lr.intercept_[0])
    except Exception as e:
        # Recorded, not swallowed: a silent NaN slope looks identical to "we did not measure it".
        out["calibration_slope"] = float("nan")
        out["calibration_intercept"] = float("nan")
        out["calibration_error"] = f"{type(e).__name__}: {e}"
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    ece = 0.0
    bins = []
    for b in range(n_bins):
        m = idx == b
        if not m.any():
            continue
        conf, acc, w = float(p[m].mean()), float(y[m].mean()), int(m.sum())
        ece += (w / len(y)) * abs(acc - conf)
        bins.append({"bin": b, "n": w, "mean_pred": conf, "observed": acc})
    out["ece"] = float(ece)
    out["calibration_bins"] = bins
    return out


def pick_thresholds(y_ref, p_ref, target_sensitivity):
    """Chosen from REFERENCE (cross-validated training-set) predictions only -- never from a test
    set. Three operating points, because a fixed 0.5 on a class-weighted classifier is not a
    neutral choice and reporting only that number is how a usable model gets written off."""
    out = {"fixed_0.5": 0.5}
    try:
        fpr, tpr, thr = roc_curve(y_ref, p_ref)
        thr = np.clip(thr, 0.0, 1.0)                # roc_curve's first threshold is +inf
        out["youden"] = float(thr[int(np.argmax(tpr - fpr))])
        ok = np.where(tpr >= target_sensitivity)[0]
        out["high_sensitivity"] = float(thr[int(ok[0])]) if len(ok) else 0.0
    except Exception:
        out["youden"] = out["high_sensitivity"] = 0.5
    return {k: float(min(max(v, 0.0), 1.0)) for k, v in out.items()}


def paired_auroc_difference(y, p_a, p_b, groups=None, n_boot=2000, seed=0):
    """Bootstrap the DIFFERENCE in AUROC between two models on the same test set.

    Paired: both models are scored on the same resampled patients each replicate, so the shared
    difficulty of those patients cancels. That is much more sensitive than checking whether two
    independent confidence intervals overlap -- two overlapping intervals routinely hide a
    difference that is consistent in every single replicate."""
    y = np.asarray(y).astype(int)
    p_a = np.asarray(p_a, dtype=float); p_b = np.asarray(p_b, dtype=float)
    obs = fast_auroc(y, p_a) - fast_auroc(y, p_b)
    if groups is None:
        groups = np.arange(len(y))
    _, members = _group_index(groups)
    n_groups = len(members)
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        pick = rng.integers(0, n_groups, size=n_groups)
        idx = np.concatenate([members[i] for i in pick])
        yb = y[idx]
        if yb.sum() == 0 or yb.sum() == len(yb):
            continue
        diffs.append(fast_auroc(yb, p_a[idx]) - fast_auroc(yb, p_b[idx]))
    d = np.asarray(diffs, dtype=float)
    d = d[np.isfinite(d)]
    if len(d) < 20:
        return {"auroc_diff": obs, "lo": float("nan"), "hi": float("nan"), "p_two_sided": float("nan")}
    # A bootstrap two-sided "p-value": twice the smaller tail mass on the wrong side of zero.
    p_val = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return {"auroc_diff": float(obs),
            "lo": float(np.percentile(d, 2.5)), "hi": float(np.percentile(d, 97.5)),
            "p_two_sided": float(min(1.0, p_val))}


def full_evaluation(y_test, p_test, groups_test, y_ref, p_ref,
                     target_sensitivity=None, n_boot=None, seed=0, label=""):
    """One test set, one score vector -> every number worth reporting."""
    target_sensitivity = (TARGET_SENSITIVITY if target_sensitivity is None
                          else target_sensitivity)
    n_boot = N_BOOTSTRAP_MAIN if n_boot is None else n_boot
    res = {"label": label}
    res.update(discrimination_metrics(y_test, p_test, groups_test, n_boot=n_boot, seed=seed))
    _cal = calibration_metrics(y_test, p_test)          # computed once, not twice
    res.update({f"cal_{k}": v for k, v in _cal.items() if k != "calibration_bins"})
    res["calibration_bins"] = _cal["calibration_bins"]
    thrs = pick_thresholds(y_ref, p_ref, target_sensitivity)
    res["operating_points"] = {name: threshold_metrics(y_test, p_test, t)
                                for name, t in thrs.items()}
    return res


def _self_test():
    rng = np.random.default_rng(0)
    n = 4000
    y = (rng.random(n) < 0.25).astype(int)
    p = np.clip(0.5 + 0.28 * (y * 2 - 1) + rng.normal(0, 0.18, n), 1e-4, 1 - 1e-4)
    a1 = fast_auroc(y, p); a2 = float(roc_auc_score(y, p))
    assert abs(a1 - a2) < 1e-9, (a1, a2)
    tied = np.round(p, 1)
    assert abs(fast_auroc(y, tied) - float(roc_auc_score(y, tied))) < 1e-9, "tie handling differs"
    assert abs(fast_auprc(y, p) - float(average_precision_score(y, p))) < 1e-9, "auprc differs"
    assert abs(fast_auprc(y, tied) - float(average_precision_score(y, tied))) < 1e-9, "auprc ties"

    # The cluster bootstrap must actually widen when rows within a patient are CORRELATED --
    # which is the entire reason for clustering. Independent rows should give the same width.
    n_pt = 400
    yp_ = (rng.random(n_pt) < 0.25).astype(int)
    y_c = np.repeat(yp_, 5)
    p_c = np.clip(np.repeat(0.5 + 0.25 * (yp_ * 2 - 1) + rng.normal(0, 0.18, n_pt), 5)
                  + rng.normal(0, 0.01, n_pt * 5), 1e-4, 1 - 1e-4)
    grp_c = np.repeat(np.arange(n_pt), 5)
    w_row = np.subtract(*cluster_bootstrap(y_c, p_c, np.arange(len(y_c)),
                                            {"a": fast_auroc}, n_boot=400, seed=3)["a"][::-1][:2])
    w_clu = np.subtract(*cluster_bootstrap(y_c, p_c, grp_c,
                                            {"a": fast_auroc}, n_boot=400, seed=3)["a"][::-1][:2])
    assert w_clu > 1.5 * w_row, (
        f"clustering did not widen the interval on correlated rows: row {w_row:.4f} vs "
        f"cluster {w_clu:.4f} -- the patient grouping is not being applied")
    # Tolerance, not equality. An exact-zero assertion here was a real bug: it depended on two
    # algebraically identical doubles cancelling bit-for-bit, which they do not, and it took the
    # whole metrics cell down with it -- and with it cells 21, 22 and all of Part C.
    lo, hi = wilson_ci(0, 20)
    assert lo < 1e-12 and 0.0 < hi < 0.30, (lo, hi)
    lo1, hi1 = wilson_ci(20, 20)
    assert 0.70 < lo1 < 1.0 and hi1 <= 1.0, (lo1, hi1)
    m_lo, m_hi = wilson_ci(5, 10)
    assert m_lo < 0.5 < m_hi, (m_lo, m_hi)
    d = discrimination_metrics(y, p, groups=np.arange(n), n_boot=200, seed=1)
    assert d["auroc_lo"] < d["auroc"] < d["auroc_hi"]
    same = paired_auroc_difference(y, p, p, np.arange(n), n_boot=200)
    assert abs(same["auroc_diff"]) < 1e-12 and same["p_two_sided"] > 0.9, same
    better = np.clip(p + 0.10 * (y * 2 - 1), 1e-4, 1 - 1e-4)
    diff = paired_auroc_difference(y, better, p, np.arange(n), n_boot=400)
    assert diff["auroc_diff"] > 0 and diff["lo"] > 0 and diff["p_two_sided"] < 0.05, diff
    cal = calibration_metrics(y, np.full(n, y.mean()))
    assert abs(cal["ece"]) < 0.02, cal
    return a1, d, diff, (w_row, w_clu)


_a, _d, _diff, (_wrow, _wclu) = _self_test()
print(f"metrics ready. self-test: fast_auroc == sklearn ({_a:.6f}); "
      f"bootstrap CI [{_d['auroc_lo']:.3f}, {_d['auroc_hi']:.3f}] brackets the point estimate; "
      f"paired test detects a planted +{_diff['auroc_diff']:.3f} AUROC gap "
      f"(95% CI [{_diff['lo']:.3f}, {_diff['hi']:.3f}], p={_diff['p_two_sided']:.4f}) "
      f"and reports no difference when a model is compared with itself. "
      f"On correlated rows (5 series per patient) the patient-clustered interval is "
      f"{_wclu/_wrow:.1f}x wider than a naive row bootstrap -- which is the whole point of "
      f"clustering, and the amount by which a row bootstrap would overstate confidence.")

In [ ]:
# 20c. LEAKAGE AUDIT -- run on the real cohort, before any result is computed
#
# ===============================================================================================
# WHY THIS CELL EXISTS
# ===============================================================================================
# Everything below was previously checked by a test suite against a synthetic fixture, and the
# suite passed while the production run leaked. The reason is worth stating plainly, because it
# is the general lesson:
#
#   The fixture had 4 series per patient and the real run has 1, so the tests exercised
#   grouping in a regime the real data never enters. They proved that GroupShuffleSplit groups.
#   They could not prove that the thing being grouped on was a PERSON -- and it was not. The
#   DICOM PatientID in this export is a registration counter issued at each visit (Spearman 0.78
#   with StudyDate; its per-month minimum marches upward all year; 189 people appear under two
#   or more of them, one registered twice on the same day). So a leakage audit that asserted
#   "no patient_id spans the split" passed, by construction, while the same skull sat in train
#   and in test.
#
# A synthetic fixture cannot catch that. Only the real table can. So these checks run here, on
# the cohort that is about to be used, and the sweep refuses to start if a HARD check fails.
#
# The checks are ordered: identity, then split discipline, then estimator discipline, then
# behavioural controls that would catch a leak nobody anticipated. The behavioural ones matter
# most -- they do not depend on my having thought of the right mechanism.
# ===============================================================================================
if DATA_SOURCE != "india":
    print("DATA_SOURCE is {!r} -- skipping this cell (Part A / India only).".format(DATA_SOURCE))
elif not FEATURES:
    print("no feature table yet -- run cell 18 first.")
else:
    import os
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

    AUDIT = {"hard_fail": [], "soft": [], "info": []}

    def _hard(label, ok, detail=""):
        print(("  PASS  " if ok else "  FAIL  ") + label + (f"   {detail}" if detail else ""))
        if not ok:
            AUDIT["hard_fail"].append(label)
        return ok

    def _soft(label, ok, detail=""):
        print(("  ok    " if ok else "  WARN  ") + label + (f"   {detail}" if detail else ""))
        if not ok:
            AUDIT["soft"].append(label)
        return ok

    # CLASSIFIERS_TO_RUN is defined by cell 21, which runs AFTER this one -- reading it here
    # was an ordering bug that only shows up in a fresh kernel. The audit does not need cell
    # 21's choice anyway: its controls use their own estimator on purpose (see section 4).
    _AUDIT_CLF = (globals().get("CLASSIFIERS_TO_RUN") or ["logreg"])[0]
    _ck = MODELS_TO_RUN[0]
    tab = FEATURES[_ck]
    print("=" * 100)
    print(f"LEAKAGE AUDIT -- {len(tab):,} scans, checkpoint {_ck}")
    print("=" * 100)

    # ---- 1. IDENTITY ---------------------------------------------------------------------------
    print("\n1. IDENTITY -- is the thing we group on a person?")
    _has_pk = "person_key" in tab.columns
    _hard("the feature table carries person_key", _has_pk)
    if not _has_pk:
        # An actionable message, not a diagnosis. The first version said "rebuild the manifest",
        # which was true and useless: cell 9 is cached, and re-running it changed nothing.
        _mp = globals().get("MANIFEST_PATH", "<OUTPUT_DIR>/manifest.csv")
        print()
        print("        The manifest on Drive predates person_key. Cell 9 caches, so re-running")
        print("        it alone will not help -- delete its cache first. Run this in a new cell:")
        print()
        print(f'          !rm -f "{_mp}" "{_mp.replace(".csv", "_meta.json")}"')
        print(f'          !rm -f "{os.path.join(OUTPUT_DIR, "features_" + _ck + ".parquet")}"')
        print()
        print("        then re-run cell 9 (rebuilds the manifest, ~1 min, opens no archive),")
        print("        cell 18 (rebuilds the feature table from the .npy files already on")
        print("        Drive, seconds), and this cell. NO re-extraction is needed -- the")
        print("        embeddings are unaffected, only the metadata beside them changes.")
        print()
    if _has_pk:
        groups = tab["person_key"].astype(str).to_numpy()
        n_person = len(set(groups))
        n_pid = tab["patient_id"].nunique() if "patient_id" in tab.columns else n_person
        AUDIT["info"].append(f"{n_person:,} people / {len(tab):,} scans")
        print(f"        {n_person:,} people across {len(tab):,} scans "
              f"({n_pid - n_person:,} visit-IDs merged into an existing person)")
        _soft("more than one scan per person exists, so grouping does something",
              n_person < len(tab),
              f"{len(tab) - n_person:,} scan(s) share a person with another scan")
        # A person holding a large share of the cohort would distort every fold.
        _big = pd.Series(groups).value_counts()
        _hard("no single person dominates the cohort", _big.iloc[0] <= max(20, 0.01 * len(tab)),
              f"largest person has {int(_big.iloc[0])} scans")
    else:
        groups = tab.get("patient_id", pd.Series(range(len(tab)))).astype(str).to_numpy()

    _hard("no series appears twice", tab["series_uid"].is_unique,
          f"{len(tab) - tab['series_uid'].nunique()} duplicate(s)")
    if "study_uid" in tab.columns and not PROCESS_ALL_QUALIFYING_SERIES:
        _n_multi = int((tab.groupby("study_uid").size() > 1).sum())
        _hard("one series per study, as PROCESS_ALL_QUALIFYING_SERIES=False promises",
              _n_multi == 0, f"{_n_multi} study/studies contribute more than one series")

    # ---- 2. SPLIT DISCIPLINE, on the real groups ------------------------------------------------
    print("\n2. SPLIT DISCIPLINE -- run the real splitter on the real groups")
    _y = None
    for _k in ("any_intracranial_hemorrhage", "normal_vs_abnormal"):
        if _k in SPECIAL_COHORTS:
            _y = tab["series_uid"].map(SPECIAL_COHORTS[_k]).fillna(0).astype(int).to_numpy()
            print(f"        using cohort {_k!r} ({int(_y.sum())} positive) to drive the splits")
            break
    if _y is None or _y.sum() < 5:
        _y = (np.arange(len(tab)) % 7 == 0).astype(int)
        print("        no usable cohort yet -- driving the splits with a synthetic label")

    _gss = GroupShuffleSplit(n_splits=1, test_size=HOLDOUT_FRAC, random_state=RANDOM_SEED)
    _pool, _hold = next(_gss.split(np.zeros((len(tab), 1)), _y, groups))
    _gp, _gh = set(groups[_pool]), set(groups[_hold])
    _hard("no person is in both the pool and the holdout", not (_gp & _gh),
          f"{len(_gp & _gh)} person/people straddle it")
    _hard("no scan is in both", not (set(_pool) & set(_hold)))
    _hard("pool and holdout together cover every scan",
          len(_pool) + len(_hold) == len(tab))

    _sgkf = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    _bad_folds, _empty_folds = 0, 0
    for _tr, _te in _sgkf.split(np.zeros((len(_pool), 1)), _y[_pool], groups[_pool]):
        if set(groups[_pool][_tr]) & set(groups[_pool][_te]):
            _bad_folds += 1
        if _y[_pool][_te].sum() == 0 or _y[_pool][_tr].sum() == 0:
            _empty_folds += 1
    _hard("no person spans train and test in any CV fold", _bad_folds == 0,
          f"{_bad_folds} fold(s) leak")
    _soft("every CV fold has positives on both sides", _empty_folds == 0,
          f"{_empty_folds} fold(s) would be skipped, shrinking the CV denominator silently")

    # ---- 3. ESTIMATOR DISCIPLINE ---------------------------------------------------------------
    print("\n3. ESTIMATOR -- is anything fitted before the split?")
    from sklearn.pipeline import Pipeline
    _mk = globals().get("_classifier")
    if _mk is None:
        _soft("cell 21 has not run in this kernel, so its estimator cannot be inspected", False,
              "structural checks below are skipped; the behavioural ones still run")
    else:
        _clf = _mk(_AUDIT_CLF, RANDOM_SEED)
        _hard("the classifier is a Pipeline, so scaling/PCA refit inside every fold",
              isinstance(_clf, Pipeline), type(_clf).__name__)
        if isinstance(_clf, Pipeline):
            _steps = [n for n, _ in _clf.steps]
            _hard("  and the scaler is a pipeline step, not applied to the full matrix",
                  any("scal" in n for n in _steps), str(_steps))
            # An already-fitted transformer smuggled into the pipeline defeats all of the above.
            _pre = [n for n, st_ in _clf.steps
                    if any(a.endswith("_") and not a.startswith("__") for a in vars(st_))]
            _hard("  no pipeline step arrives pre-fitted", not _pre, str(_pre))

    # ---- 4. BEHAVIOURAL CONTROLS ---------------------------------------------------------------
    #
    # These are the checks that would catch a leak nobody anticipated, because they name no
    # mechanism -- they assert that impossible tasks score at chance. If one of them beats
    # chance, something is wrong even when every structural check above passed.
    #
    # The evaluator here is written from scratch rather than calling cell 21's evaluate_target,
    # and that is deliberate. An audit that runs the code it is auditing can only find bugs in
    # the DATA. A bug in the splitter, the pipeline, or the fold loop would be invisible,
    # because the audit would reproduce it faithfully and report a clean result. Twenty lines of
    # duplication buys independence. If the two ever disagree, that disagreement is itself the
    # finding.
    print("\n4. BEHAVIOURAL CONTROLS -- do impossible tasks score at chance?")
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import roc_auc_score

    _feat = FEATURE_COLUMNS[_ck]
    _X = tab[_feat].to_numpy(dtype=np.float32)
    _rng = np.random.default_rng(RANDOM_SEED)

    def _audit_auroc(X, y, g, folds=3, seed=RANDOM_SEED):
        """Grouped CV AUROC, implemented independently of cell 21. Returns nan if unusable."""
        y = np.asarray(y).astype(int)
        g = np.asarray(g).astype(str)
        if y.sum() < 8 or (y == 0).sum() < 8 or len(set(g)) < folds:
            return float("nan")
        oof = np.full(len(y), np.nan)
        skf = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
        for tr, te in skf.split(X, y, g):
            if y[tr].sum() == 0 or y[te].sum() == 0:
                continue
            pipe = Pipeline([("scale", StandardScaler()),
                              ("clf", LogisticRegression(max_iter=2000, C=1.0,
                                                          class_weight="balanced"))])
            pipe.fit(X[tr], y[tr])
            oof[te] = pipe.predict_proba(X[te])[:, 1]
        m = np.isfinite(oof)
        if m.sum() < 20 or len(np.unique(y[m])) < 2:
            return float("nan")
        return float(roc_auc_score(y[m], oof[m]))

    def _chance_band(n_pos, n_neg, sd=4.0):
        """How far from 0.5 an AUROC can land by luck alone, at this sample size.

        A fixed band is wrong, and wrong in the direction that matters. Under the null the
        Mann-Whitney AUROC has standard error sqrt((n1+n0+1)/(12*n1*n0)) -- 0.07 at 18 positives
        against 162 negatives, so a perfectly clean cohort produces 0.39 or 0.61 routinely. A
        hardcoded [0.42, 0.58] therefore FAILS on clean small data, and a check that cries wolf
        gets overridden, which is how a real failure later gets waved through. At 3,000 scans
        with 200 positives the same formula gives 0.021, so the band tightens to where it can
        actually catch something.
        """
        if n_pos < 2 or n_neg < 2:
            return 0.0, 1.0
        se = float(np.sqrt((n_pos + n_neg + 1.0) / (12.0 * n_pos * n_neg)))
        return max(0.0, 0.5 - sd * se), min(1.0, 0.5 + sd * se)

    # (a) random labels must not be predictable at all.
    #
    # Averaged over several draws: one draw at 4 sigma is a coin-flip away from a spurious
    # failure, and the average of k draws has 1/sqrt(k) the spread, so the band can be that much
    # tighter for the same false-alarm rate.
    _rand_aucs, _n_draw = [], 3
    for _d in range(_n_draw):
        _yr = (_rng.random(len(tab)) < 0.10).astype(int)
        _a = _audit_auroc(_X, _yr, groups, seed=RANDOM_SEED + _d)
        if np.isfinite(_a):
            _rand_aucs.append(_a)
    _auc_rand = float(np.mean(_rand_aucs)) if _rand_aucs else float("nan")
    _npos_r = max(1, int(0.10 * len(tab)))
    _lo_r, _hi_r = _chance_band(_npos_r, len(tab) - _npos_r,
                                sd=4.0 / np.sqrt(max(1, len(_rand_aucs))))
    _hard(f"random labels score at chance ({_lo_r:.2f}-{_hi_r:.2f} at this sample size)",
          np.isfinite(_auc_rand) and _lo_r <= _auc_rand <= _hi_r,
          f"AUROC {_auc_rand:.3f} (mean of {len(_rand_aucs)} draw(s))")

    # (b) THE NEAR-DUPLICATE CONTROL -- and whether it can discriminate at all.
    #
    # The idea: two scans of one skull should land close together in embedding space, so find
    # close pairs and fail if any straddles the split. No identifier is consulted, so a wrong
    # identifier cannot hide the answer -- which is the whole point, since a per-visit PatientID
    # is exactly what got past the previous audit.
    #
    # The first version hardcoded a cosine threshold of 0.97 and, on the real cohort, reported
    # "0 pair(s) checked" and PASSED. That is the vacuous-check failure this file warns about
    # elsewhere and then committed anyway. The measurement that exposes it:
    #
    #     same-person pairs, median cosine   0.273
    #     random pairs, mean cosine          0.235
    #
    # A threshold of 0.97 sits eighteen times the entire same-vs-random gap above the
    # same-person median. It could never fire.
    #
    # The finding underneath is more interesting than the bug. Pillar-0 embeddings barely
    # encode WHO the patient is -- two CTs of one skull are almost as far apart as two random
    # skulls, and an ungrouped identity probe scores 0.543. The model encodes pathology, not
    # identity. So the threshold has to be calibrated from the data, and the check has to be
    # honest about whether it is capable of discriminating at all before it reports anything.
    print("\n        near-duplicate scan detection (identifier-free):")

    # Centre first. Every head CT shares a large common component -- that is why random pairs
    # sit at 0.235 rather than near 0 -- and it swamps whatever identity signal exists.
    # Subtracting the cohort mean before normalising is the standard way to look past it.
    _Xc = _X - _X.mean(axis=0, keepdims=True)
    _Xc = _Xc / np.maximum(np.linalg.norm(_Xc, axis=1, keepdims=True), 1e-9)
    _Xr = _X / np.maximum(np.linalg.norm(_X, axis=1, keepdims=True), 1e-9)

    _by = {}
    for _ix, _g in enumerate(groups):
        _by.setdefault(_g, []).append(_ix)
    _same_pairs = []
    for _g, _ixs in _by.items():
        if len(_ixs) > 1:
            for _a in range(len(_ixs)):
                for _b in range(_a + 1, min(len(_ixs), _a + 4)):
                    _same_pairs.append((_ixs[_a], _ixs[_b]))

    def _separability(M):
        """Can cosine similarity tell a same-person pair from a random pair? Returns AUROC."""
        if len(_same_pairs) < 8:
            return float("nan"), float("nan"), float("nan")
        _s = np.array([float(M[a] @ M[b]) for a, b in _same_pairs])
        _ri = _rng.integers(0, len(M), size=(max(2000, 8 * len(_s)), 2))
        _ri = _ri[_ri[:, 0] != _ri[:, 1]]
        _r = np.array([float(M[int(a)] @ M[int(b)]) for a, b in _ri])
        # AUROC by rank, no sklearn needed and no ties problem at this precision
        _all = np.concatenate([_s, _r])
        _rank = _all.argsort().argsort().astype(float) + 1
        _n1, _n0 = len(_s), len(_r)
        _auc = (_rank[:_n1].sum() - _n1 * (_n1 + 1) / 2) / (_n1 * _n0)
        return float(_auc), _s, _r

    _sep_raw, _s_raw, _r_raw = _separability(_Xr)
    _sep_ctr, _s_ctr, _r_ctr = _separability(_Xc)
    if np.isfinite(_sep_raw):
        print(f"          same-person pairs: n={len(_same_pairs)}  "
              f"median cosine raw {np.median(_s_raw):.3f} / centred {np.median(_s_ctr):.3f}")
        print(f"          random pairs:      "
              f"median cosine raw {np.median(_r_raw):.3f} / centred {np.median(_r_ctr):.3f}")
        print(f"          separability (can cosine tell them apart?): "
              f"raw {_sep_raw:.3f}  centred {_sep_ctr:.3f}")

    _use_ctr = (not np.isfinite(_sep_raw)) or (np.isfinite(_sep_ctr) and _sep_ctr >= _sep_raw)
    _M = _Xc if _use_ctr else _Xr

    # ---- the threshold comes from the SIMILARITY DISTRIBUTION, not from person_key ------------
    #
    # Calibrating on same-person pairs looked reasonable and quietly destroyed the one property
    # that makes this check worth having. With a per-visit identifier every group has exactly
    # one member, so there are no same-person pairs, so there is nothing to calibrate on, so
    # the check reports "cannot discriminate" and waves through precisely the failure it exists
    # to catch. The calibration has to be as identifier-free as the detection.
    #
    # So: a near-duplicate is a pair that is anomalously similar RELATIVE TO THE COHORT'S OWN
    # pairwise distribution. Sample the bulk, take mean and sd, and set the bar high enough
    # that fewer than one pair should clear it by chance across all n(n-1)/2 comparisons. For
    # 3,000 scans that is 4.5 million comparisons, so the bar lands around 5 sd.
    # FINDING THE DUPLICATES WITHOUT KNOWING WHAT A DUPLICATE LOOKS LIKE
    #
    # Two earlier attempts failed in opposite directions, and both are worth recording because
    # each looked correct:
    #
    #   * a fixed cosine bar of 0.97 -- eighteen times the whole same-person-vs-random gap above
    #     the same-person median on the real cohort. It reported "0 pairs, PASS" and could never
    #     have fired.
    #   * a bar at mean + z*sd with z solved from the normal tail -- the similarity distribution
    #     is bounded and right-skewed, so two unrelated pairs cleared it, one straddled the
    #     split, and the audit hard-failed a cohort with no leakage in it.
    #   * a bar extrapolated from the SAMPLE MAXIMUM -- but the sample is drawn from the same
    #     cohort, so the duplicates are inside it, and they drag the maximum up until the bar
    #     sits above them. It stopped crying wolf and stopped detecting anything.
    #
    # What survives all three: duplicates, if present, form a CLUSTER separated from the bulk by
    # a gap, and a gap is a shape rather than a level. So take the top slice of the pairwise
    # distribution, measure the typical spacing between consecutive values in its lower half
    # (which the duplicates cannot contaminate, since they are all above it), and cut at the
    # largest gap that is far wider than that spacing. No assumed distribution, no assumed
    # threshold, and no dependence on any identifier.
    _TOPK = int(min(4000, max(200, len(_M) * 2)))
    _top = np.full(_TOPK, -np.inf)
    _top_ij = [None] * _TOPK
    _BS = 512
    for _a0 in range(0, len(_M), _BS):
        _blk = _M[_a0:_a0 + _BS] @ _M.T
        for _row in range(_blk.shape[0]):
            _ix = _a0 + _row
            _blk[_row, :_ix + 1] = -np.inf              # upper triangle only
            _cand = np.argpartition(_blk[_row], -min(_TOPK, _blk.shape[1] - 1))[
                -min(_TOPK, _blk.shape[1] - 1):]
            for _jx in _cand:
                _v = float(_blk[_row, _jx])
                if _v > _top[-1]:
                    _pos = int(np.searchsorted(-_top, -_v))
                    _top[_pos + 1:] = _top[_pos:-1]
                    _top_ij[_pos + 1:] = _top_ij[_pos:-1]
                    _top[_pos] = _v
                    _top_ij[_pos] = (_ix, int(_jx))
    _valid = int(np.isfinite(_top).sum())
    _top, _top_ij = _top[:_valid], _top_ij[:_valid]

    # A reference sample for the robust location/scale. Drawn at random, so duplicates are a
    # negligible fraction of it -- and the median/MAD are insensitive to the few that land in.
    _rs = _rng.integers(0, len(_M), size=(20000, 2))
    _rs = _rs[_rs[:, 0] != _rs[:, 1]]
    _bulk_ref = np.einsum("ij,ij->i", _M[_rs[:, 0]], _M[_rs[:, 1]])

    _pairs = []
    if _valid < 40:
        _soft("enough pairs to look for a near-duplicate cluster", False, f"{_valid}")
        _thr = float("inf")
    else:
        _d = -np.diff(_top)                              # gaps, descending order so >= 0
        _lower = _d[_valid // 2:]                        # bulk spacing: duplicates are all above
        _typ = float(np.median(_lower)) or 1e-9
        _search = _d[: max(1, _valid // 4)]              # a duplicate cluster lives in the top
        _gi = int(np.argmax(_search))
        _gap = float(_search[_gi])
        _ratio = _gap / _typ
        _GAP_FACTOR = globals().get("NEAR_DUP_GAP_FACTOR", 12.0)
        # A gap alone is not enough -- over tens of thousands of comparisons the top of any
        # distribution is ragged, and a two-pair "cluster" cleared a 12x-spacing test by luck on
        # a fixture containing no duplicates at all. So the cluster must also be separated in
        # ABSOLUTE terms and sit above the bulk.
        #
        # The first attempt at that second condition was "median + 6 * MAD-sigma", and on the
        # real cohort it computed a floor of 1.123 -- above the maximum possible cosine of 1.0.
        # No pair, however identical, could ever have satisfied it. A candidate pair at cosine
        # 0.883 with a gap one thousand times the typical spacing was dismissed by it, and the
        # check reported "no duplicates" for a reason that had nothing to do with the data. The
        # same vacuity failure as the hardcoded 0.97, arrived at by a different route.
        #
        # Two conditions replace it, both bounded by construction:
        #   * the gap must be wide in cosine, not just relative to a spacing that can be
        #     arbitrarily small (a ragged tail has gaps of 0.001; duplicates are separated by
        #     tenths);
        #   * the cluster floor must clear the bulk's own extreme -- the 99.99th percentile of
        #     random pairs -- which is always a reachable number because it IS an observed one.
        # The reference sample is drawn from the same cohort, so any duplicates are INSIDE it
        # and drag its extreme upward until the floor sits above them -- a contamination trap
        # that already defeated two earlier versions of this threshold. Trim the top of the
        # reference before reading its extreme. Duplicates are a tiny fraction of all pairs
        # (90 of 16,110 on the strong fixture, 0.6%), so a 5% trim removes them with room to
        # spare while barely moving a clean distribution's tail.
        _ref_sorted = np.sort(_bulk_ref)
        _ref_clean = _ref_sorted[: max(10, int(0.95 * len(_ref_sorted)))]
        _q9999 = float(_ref_clean[-1])
        _MIN_GAP = globals().get("NEAR_DUP_MIN_GAP", 0.05)
        _floor = _q9999
        _high_enough = float(_top[_gi]) >= _floor
        _wide_enough = _gap >= _MIN_GAP
        if _ratio >= _GAP_FACTOR and not (_high_enough and _wide_enough):
            _why = []
            if not _wide_enough:
                _why.append(f"the gap is only {_gap:.4f} in cosine (needs {_MIN_GAP:.2f})")
            if not _high_enough:
                _why.append(f"it sits at {float(_top[_gi]):.3f}, below the bulk's own "
                            f"99.99th percentile {_floor:.3f}")
            print(f"          a {_gi + 1}-pair gap of {_ratio:.0f}x spacing exists, but "
                  + " and ".join(_why))
            print(f"          -> a ragged tail, not a duplicate cluster")
        if _ratio >= _GAP_FACTOR and _high_enough and _wide_enough:
            _thr = float(_top[_gi])
            _pairs = [(a, b, float(v)) for (a, b), v in zip(_top_ij[: _gi + 1], _top[: _gi + 1])]
            print(f"          a separated cluster of {len(_pairs):,} pair(s) sits above "
                  f"cosine {_thr:.3f}")
            print(f"          (gap {_gap:.4f} = {_ratio:.0f}x the typical spacing "
                  f"{_typ:.5f} -- that separation is what identifies them)")
        else:
            _thr = float("inf")
            print(f"          no separated cluster: largest gap in the top quartile is "
                  f"{_ratio:.1f}x typical spacing (needs {_GAP_FACTOR:.0f}x)")
            print(f"          -> no near-duplicate scans detected in this cohort")

    _hold_set = set(_hold.tolist())
    _straddle = [(a, b, c) for a, b, c in _pairs if (a in _hold_set) != (b in _hold_set)]
    print(f"          {len(_pairs):,} pair(s) above the bar, {len(_straddle):,} straddling "
          f"the pool/holdout boundary")
    _hard("no anomalously-similar scan pair straddles the pool/holdout boundary",
          not _straddle,
          f"{len(_straddle)} pair(s) do -- these are near-certainly the same person under two "
          f"identifiers. Fix the identity; do not raise the bar."
          if _straddle else f"{len(_pairs):,} near-duplicate pair(s) found, all kept on one side")
    AUDIT["info"].append(f"{len(_pairs)} near-dup pairs, {len(_straddle)} straddling")

    # Separability is now a POWER report, not a gate: it says how much this check could have
    # seen, and it is informative either way.
    # HOW MUCH THIS CHECK COULD HAVE SEEN.
    #
    # "No cluster found" means two very different things and they must not print the same way:
    # the cohort has no duplicates, or the embedding cannot express the difference. Reporting
    # only the first is how a check ends up passing on a leak.
    #
    # Separability answers it -- can cosine tell a known same-person pair from a random one? --
    # but it needs same-person pairs to measure, and a per-visit identifier leaves NONE. That
    # is not a reason to stay quiet; it is the loudest case, so it reports low power too.
    _MIN_SEP = globals().get("NEAR_DUP_MIN_SEPARABILITY", 0.80)
    _sep = _sep_ctr if _use_ctr else _sep_raw
    _low_power = (not np.isfinite(_sep)) or _sep < _MIN_SEP
    if _low_power and not _pairs:
        _why = (f"separability {_sep:.3f} < {_MIN_SEP:.2f}" if np.isfinite(_sep)
                else "no same-person pair exists to measure it on -- which is itself what a "
                     "per-visit identifier looks like from here")
        _soft("cosine similarity can tell a same-person pair from a random one", False,
              f"{_why}. The check above therefore found nothing AND could not have found "
              f"much: treat it as unmeasured, not as clean.")
        print("          The same fact bounds the risk: features that cannot identify a person")
        print("          give same-person-across-the-split very little to exploit. Identity")
        print("          grouping is still enforced -- it is just not the dominant threat here.")

    # (c) ...and identity must be unlearnable ACROSS the grouped split, where that is testable.
    _multi = pd.Series(groups).value_counts()
    _multi = _multi[_multi > 1]
    if len(_multi) >= 12:
        _target = set(_multi.index[: max(12, len(_multi) // 2)])
        _yp = np.array([1 if g in _target else 0 for g in groups], dtype=int)
        _auc_pf = _audit_auroc(_X, _yp, groups)
        _lo_p, _hi_p = _chance_band(int(_yp.sum()), int((_yp == 0).sum()))
        _bar = max(0.70, _hi_p)
        _hard("a person's own identity is NOT recoverable across the grouped split",
              (not np.isfinite(_auc_pf)) or _auc_pf <= _bar,
              f"AUROC {_auc_pf:.3f} against a bar of {_bar:.3f}")

        # The vacuity arm: is the control capable of failing at all?
        #
        # This was a HARD check requiring the ungrouped probe to reach 0.65, on the reasoning
        # that a control which passes on features carrying no identity has proved nothing. The
        # reasoning is right; making it a hard failure was not. On this cohort the ungrouped
        # probe scores 0.543, so the audit BLOCKED THE SWEEP on correct data -- the cry-wolf
        # failure this same cell warns about two checks earlier.
        #
        # "The features do not encode identity" is a finding, not a defect. It is reported as
        # what it is: the control cannot be validated, and the same fact bounds the risk.
        _auc_ng = _audit_auroc(_X, _yp, np.arange(len(tab)).astype(str))
        _validated = (not np.isfinite(_auc_ng)) or _auc_ng >= 0.65
        _soft("  the identity control is validated (identity IS learnable without grouping)",
              _validated,
              f"AUROC {_auc_ng:.3f} ungrouped -- below 0.65, so the grouped result above is "
              f"uninformative rather than reassuring. Not a defect: it means the embedding "
              f"carries little patient identity, which also limits what identity leakage "
              f"could inflate.")
        print(f"        identity probe: grouped {_auc_pf:.3f} vs ungrouped {_auc_ng:.3f}"
              + ("  -- the gap IS the protection grouping provides" if _validated else
                 "  -- neither is above chance; identity is not in these features"))
    else:
        _soft("no person has a repeat scan, so identity leakage cannot be tested this way",
              False,
              f"{len(_multi)} repeat-person group(s). If the near-duplicate check above found "
              f"pairs, the identifier is splitting people, not the data lacking repeats.")

    # ---- 5. COHORT QUALITY -- not leakage, but it decides what the number means -----------------
    print("\n5. COHORT QUALITY -- what is actually in the negatives?")
    # With STRICT_COHORT on, the cohort composition checks are HARD. The user asked for a run
    # in which no non-brain scan and no non-CT series can reach the classifier, and an advisory
    # that prints a warning and continues does not deliver that -- the previous run printed
    # exactly this warning and produced a number anyway.
    _grade = _hard if globals().get("STRICT_COHORT", False) else _soft
    if "head_subtype" in tab.columns:
        _sub = tab["head_subtype"].value_counts()
        _nonbrain = int(len(tab) - _sub.get("brain", 0))
        print(f"        head subtypes: {dict(_sub)}")
        _grade("every scan is a brain acquisition", _nonbrain == 0,
              f"{_nonbrain:,} ({_nonbrain/len(tab):.1%}) are dedicated sinus / facial-bone / "
              f"inner-ear scans -- guaranteed hemorrhage-negative and trivially separable, "
              f"worth roughly +0.02 AUROC. Set RESTRICT_TO_ALLOWED_HEAD_SUBTYPES=True in cell 9 "
              f"to measure it.")
    if "region" in tab.columns:
        _wr = tab[tab["region"].astype(str) != TARGET_REGION]
        _grade(f"every scan is region {TARGET_REGION!r}", len(_wr) == 0,
               f"{len(_wr):,} are not: {dict(_wr['region'].value_counts())}" if len(_wr) else "")
    if "study_uid" in tab.columns and "person_key" in tab.columns:
        _np_ = int(tab["person_key"].isna().sum()
                   + (tab["person_key"].astype(str).str.len() == 0).sum())
        _hard("every scan has a person_key to group on", _np_ == 0, f"{_np_:,} missing")
    # Body-part mismatch stays advisory even under STRICT_COHORT. It is a LABEL problem, not a
    # cohort-mixing one, its direction is deflationary, and dropping 14% of the data is a
    # separate decision the user should make deliberately rather than inherit from a flag.
    if "studytype_agrees" in tab.columns:
        _dis = int((~tab["studytype_agrees"].fillna(False)).sum())
        _soft("every label comes from a report about a head study", _dis == 0,
              f"{_dis:,} ({_dis/len(tab):.1%}) are labelled from a report about another body "
              f"part -- forced negatives. Deflationary, but the labels are wrong. "
              f"REQUIRE_STUDYTYPE_AGREEMENT=True drops them.")
    if "match_method" in tab.columns:
        print(f"        report match method: {dict(tab['match_method'].value_counts())}")

    # ---- verdict -------------------------------------------------------------------------------
    print("\n" + "=" * 100)
    if AUDIT["hard_fail"]:
        print(f"LEAKAGE AUDIT FAILED -- {len(AUDIT['hard_fail'])} hard check(s):")
        for _f in AUDIT["hard_fail"]:
            print(f"   - {_f}")
        print("\nThe sweep will refuse to run. Fix these before computing anything: a number")
        print("produced under a failing audit is not a result.")
    else:
        print(f"LEAKAGE AUDIT PASSED -- every hard check, on the real cohort."
              + (f" {len(AUDIT['soft'])} quality warning(s) above." if AUDIT["soft"] else ""))
    print("=" * 100)
    LEAKAGE_AUDIT = AUDIT

In [ ]:
# 21. Classifier sweep on the Pillar-0 embeddings  [RESUMABLE per (cohort, feature set, model)]
#
# Design decisions worth knowing about, because each one changes the numbers:
#
#  PATIENT-LEVEL SPLITS. Every split -- the holdout and every CV fold -- groups by patient_id, so
#    no patient appears on both sides of a split. With PROCESS_ALL_QUALIFYING_SERIES = True one
#    study contributes several series; a random row-level split would put two reconstructions of
#    the same scan in train and test and inflate every metric. Grouping removes that entirely.
#
#  A DEMOGRAPHICS-ONLY BASELINE runs alongside the embeddings for every target. If age and sex
#    alone reach the same AUROC, the embedding is not what is doing the work. This is the single
#    cheapest guard against reporting a result that is really an age-prevalence effect, and it is
#    on by default rather than an optional extra.
#
#  THRESHOLDS COME FROM CROSS-VALIDATION, NEVER FROM THE HOLDOUT. Three operating points are
#    reported side by side: fixed 0.5, Youden (maximises sensitivity + specificity - 1), and the
#    lowest threshold whose CV sensitivity reaches TARGET_SENSITIVITY. A fixed 0.5 on a
#    class-weight-balanced classifier is not a neutral choice, and reporting only that number is
#    how a usable model gets written off as insensitive. Which operating point you would actually
#    deploy at is a clinical decision, not one this notebook should make silently.
import json
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Deliberately NOT a blanket filterwarnings("ignore", UserWarning): sklearn's ConvergenceWarning
# subclasses UserWarning, and silencing it means a LogisticRegression that never converged still
# produces a probability vector, an AUROC, and a leaderboard row that looks like a result. It is
# caught per-fit below and recorded on the record instead.
from sklearn.exceptions import ConvergenceWarning

# Which feature sets and classifiers to sweep.
#
# The default is a LINEAR PROBE, and that is a deliberate choice rather than a shortcut. Fitting a
# linear classifier on frozen features is the standard protocol for evaluating a foundation model:
# it measures how much the representation has already separated, not how much a powerful
# downstream learner can dig back out of it. It also happens to be the only option here that is
# not painfully slow.
#
# Measured on a Colab CPU, ONE fit on 3,200 samples x 1,152 dims:
#     logreg                       0.4 s
#     HistGradientBoosting(300)   23   s
#     RandomForest(300)           11   s
# A full sweep is roughly n_targets x n_feature_sets x (CV_FOLDS + 1) fits. At ~47 targets and 2
# feature sets that is ~560 fits: about 5 minutes with logreg, about 4 hours with HGB. If you do
# want a non-linear model, use the "hgb_pca256" CLASSIFIER, which reduces to 256 dims inside the
# cross-validation loop -- same question asked, a fraction of the cost, and no leakage.
# NOTE ON THE NAME. This list used to read ["pillar0", ...] and that string went straight into
# the results CSV and the leaderboard, so every row of our own SuPreM / MedicalNet results was
# labelled "pillar0". Nothing was computed wrongly -- the key only ever meant "the embedding
# columns of whichever feature table is loaded" -- but a table in a paper that labels 2,304-d
# SwinUNETR features "pillar0" is a table nobody can defend in review. The backbone is already
# recorded per row in the model column; this now says what the feature set IS.
FEATURE_SETS_TO_RUN = ["embeddings", "demographics_only"]
CLASSIFIERS_TO_RUN = ["logreg"]        # also available: logreg_strong_reg, logreg_pca256,
                                        # hgb_pca256, hgb, rf

# Optional label-trust filter. Cell 8 recorded, per series, whether MatchedStudyType agrees with
# what the DICOM says was scanned. Where they disagree, the report attached to that scan may be
# describing a different study -- so the LABEL is suspect, not the image. Off by default so the
# first numbers you see are on all the data; flip it on and re-run to see how much it moves.
REQUIRE_STUDYTYPE_AGREEMENT = True     # drops the ~10% labelled from a report about another body part


def _classifier(name, seed):
    if name == "logreg":
        return Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced",
                                        random_state=seed)),
        ])
    if name == "logreg_pca256":
        # PCA lives INSIDE the pipeline on purpose. Fitting it once on the full matrix and then
        # carving out a holdout is transductive leakage -- the components have already seen the
        # test rows. Here it is refit on the training fold every time, which is the only version
        # of "PCA + classifier" whose holdout number means anything.
        return Pipeline([
            ("scale", StandardScaler()),
            ("pca", PCA(n_components=256, random_state=seed)),
            ("clf", LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced",
                                        random_state=seed)),
        ])
    if name == "hgb_pca256":
        return Pipeline([
            ("pca", PCA(n_components=256, random_state=seed)),
            ("clf", HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08,
                                                    class_weight="balanced", random_state=seed)),
        ])
    if name == "logreg_strong_reg":
        # Worth trying when n_positives is small relative to 1,152 dimensions.
        return Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=3000, C=0.05, class_weight="balanced",
                                        random_state=seed)),
        ])
    if name == "hgb":
        return HistGradientBoostingClassifier(
            max_iter=150, learning_rate=0.08, max_features=0.2,
            class_weight="balanced", random_state=seed)
    if name == "rf":
        return RandomForestClassifier(
            n_estimators=300, n_jobs=-1, class_weight="balanced_subsample", random_state=seed)
    raise ValueError(f"unknown classifier {name!r}")


_AGE_UNIT_YEARS = {"Y": 1.0, "M": 1.0 / 12.0, "W": 1.0 / 52.1775, "D": 1.0 / 365.25}


def parse_dicom_age_years(value):
    """DICOM PatientAge is nnnD | nnnW | nnnM | nnnY -- the unit is not optional.

    Stripping the digits and ignoring the suffix turns "006M" (six months) into a six-year-old and
    "003D" (three days) into a three-year-old, in the same column as adults. Head CT populations
    contain infants, so this is not hypothetical: it flattens the real age signal, which
    UNDERSTATES the demographics-only baseline, which INFLATES lift_over_demographics -- the exact
    number the leaderboard tells you to read first."""
    if value is None:
        return np.nan
    s = str(value).strip().upper()
    m = re.match(r"^0*(\d+)\s*([DWMY])$", s)
    if m:
        return float(m.group(1)) * _AGE_UNIT_YEARS[m.group(2)]
    m = re.match(r"^0*(\d+(?:\.\d+)?)$", s)   # a bare number: assume years
    if m:
        return float(m.group(1))
    return np.nan


def _demographic_matrix(tab):
    raw = tab.get("patient_age", pd.Series([None] * len(tab), index=tab.index))
    age = raw.map(parse_dicom_age_years).astype(float)
    n_missing = int(age.isna().sum())
    if n_missing == len(age):
        print("    WARNING: no parseable PatientAge at all -- the 'demographics' baseline is "
              "sex-only. Interpret lift_over_demographics accordingly.")
    # Fill with 0 plus an explicit missingness indicator rather than the median: a median computed
    # over all rows (holdout included) is a small but real leak, and the indicator lets the model
    # learn that missing-age is its own thing instead of pretending it is an average patient.
    sex = tab.get("patient_sex", pd.Series([""] * len(tab), index=tab.index)).astype(str).str.upper().str[:1]
    return np.column_stack([
        age.fillna(0.0).to_numpy(dtype=np.float32),
        age.isna().to_numpy(dtype=np.float32),
        (sex == "M").to_numpy(dtype=np.float32),
        (sex == "F").to_numpy(dtype=np.float32),
    ])


def build_feature_matrices(tab, feature_cols):
    """Feature SETS are raw matrices only. Any learned transform (PCA, scaling) belongs inside the
    classifier pipeline so it is refit per fold -- see the `logreg_pca256` entry above."""
    out = {"embeddings": tab[feature_cols].to_numpy(dtype=np.float32),
           "demographics_only": _demographic_matrix(tab)}
    unknown = [fs for fs in FEATURE_SETS_TO_RUN if fs not in out]
    if unknown:
        raise ValueError(f"unknown feature set(s) {unknown}. Available: {list(out)}. "
                          f"(For PCA, use the 'logreg_pca256'/'hgb_pca256' CLASSIFIERS instead -- "
                          f"PCA has to be fit inside the cross-validation, not before it.)")
    return {k: out[k] for k in FEATURE_SETS_TO_RUN}


# Threshold and interval machinery all lives in cell 20b, so the per-finding sweep and the
# cross-dataset hemorrhage matrix in Part C report exactly the same quantities computed exactly the
# same way. Nothing is redefined here.
_threshold_metrics = threshold_metrics
_pick_thresholds = pick_thresholds


def _wide_halfwidth(cv_disc, fold_auc):
    """Bootstrap half-width and fold-mean standard error, combined in quadrature."""
    lo, hi = cv_disc.get("auroc_lo"), cv_disc.get("auroc_hi")
    boot_hw = (hi - lo) / 2.0 if (lo is not None and hi is not None
                                   and np.isfinite(lo) and np.isfinite(hi)) else 0.0
    fold_se = (float(np.std(fold_auc, ddof=1)) / np.sqrt(len(fold_auc))
               if len(fold_auc) > 1 else 0.0)
    # The fold sd itself, not just the sd of its mean, is what a re-run with a different fold
    # assignment would move by. Use the larger of (se of the mean) and (sd / 2) so the interval
    # does not collapse when the folds happen to agree by luck at 5 folds.
    fold_term = max(fold_se, (float(np.std(fold_auc, ddof=1)) / 2.0) if len(fold_auc) > 1 else 0.0)
    return float(np.sqrt(boot_hw ** 2 + fold_term ** 2))


def _wide_lo(cv_disc, fold_auc):
    return float(max(0.0, cv_disc["auroc"] - _wide_halfwidth(cv_disc, fold_auc)))


def _wide_hi(cv_disc, fold_auc):
    return float(min(1.0, cv_disc["auroc"] + _wide_halfwidth(cv_disc, fold_auc)))


def evaluate_target(X, y, groups, model_name, seed=None):
    seed = RANDOM_SEED if seed is None else seed
    y = np.asarray(y).astype(int)
    groups = np.asarray(groups)

    if y.sum() < MIN_POSITIVES or (len(y) - y.sum()) < MIN_POSITIVES:
        return {"skipped": f"too few positives ({int(y.sum())}) or negatives ({int(len(y)-y.sum())})"}

    # ---- REPEATED HOLDOUTS, not one ----------------------------------------------------------
    #
    # This used to draw a single GroupShuffleSplit at RANDOM_SEED and report that one holdout's
    # AUROC as the number. With ~600 held-out scans and ~40 positives, the Hanley-McNeil standard
    # error alone is about 0.04 -- so which patients happened to land in the holdout moved the
    # reported figure more than most of the defects this project has chased. Measured on the real
    # cohort, changing only the fold seed moved the cross-validated AUROC across a 0.023 range;
    # the holdout, being one draw rather than a pooled five, is looser still.
    #
    # So the holdout is now drawn N_EVAL_REPEATS times with different seeds and the distribution
    # is reported: the median is the estimate, the spread is the honest uncertainty. The first
    # repeat uses RANDOM_SEED, so the primary split is identical to before and every number
    # remains comparable with earlier runs.
    _n_rep = max(1, int(globals().get("N_EVAL_REPEATS", 1)))
    gss = GroupShuffleSplit(n_splits=1, test_size=HOLDOUT_FRAC, random_state=seed)
    pool_idx, hold_idx = next(gss.split(X, y, groups))
    if y[hold_idx].sum() < 5 or y[pool_idx].sum() < CV_FOLDS:
        return {"skipped": f"holdout has {int(y[hold_idx].sum())} positive(s) / pool has "
                            f"{int(y[pool_idx].sum())} -- not enough to evaluate honestly"}

    Xp, yp, gp = X[pool_idx], y[pool_idx], groups[pool_idx]
    Xh, yh = X[hold_idx], y[hold_idx]

    n_splits = min(CV_FOLDS, int(yp.sum()), int((yp == 0).sum()), len(np.unique(gp)))
    if n_splits < 2:
        return {"skipped": "not enough groups/positives for cross-validation"}

    oof = np.full(len(yp), np.nan)
    fold_auc = []
    n_nonconverged = 0
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, te in sgkf.split(Xp, yp, gp):
        if yp[tr].sum() == 0 or yp[te].sum() == 0:
            continue
        clf = _classifier(model_name, seed)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            clf.fit(Xp[tr], yp[tr])
        if any(issubclass(w.category, ConvergenceWarning) for w in caught):
            n_nonconverged += 1
        p = clf.predict_proba(Xp[te])[:, 1]
        oof[te] = p
        try:
            fold_auc.append(float(roc_auc_score(yp[te], p)))
        except Exception:
            pass

    mask = np.isfinite(oof)
    if mask.sum() < 10 or len(np.unique(yp[mask])) < 2:
        return {"skipped": "cross-validation produced no usable out-of-fold predictions"}

    cv_disc = discrimination_metrics(yp[mask], oof[mask], gp[mask],
                                      n_boot=N_BOOTSTRAP_SWEEP, seed=seed)
    cv_auroc = cv_disc["auroc"]

    # ---- how many rows the CV metric was ACTUALLY computed on --------------------------------
    #
    # A fold that is skipped just above (all-positive or all-negative test side) leaves its rows
    # NaN forever, and `mask` then quietly drops them. Nothing in the record said so: the field
    # beside it, cv_folds_used, held the PLANNED fold count, and the leaderboard's
    # n_rows_evaluated is len(work) -- a third number again. So a CV AUROC could be computed on
    # 81% of the pool at a different prevalence and read as if it covered everything.
    n_cv_rows = int(mask.sum())
    n_folds_ran = len(fold_auc)
    if n_cv_rows < len(yp):
        print(f"      note: CV metric computed on {n_cv_rows:,} of {len(yp):,} pool rows "
              f"({n_folds_ran} of {n_splits} folds produced predictions)")
    cv_ap = cv_disc["auprc"]
    thresholds = _pick_thresholds(yp[mask], oof[mask], TARGET_SENSITIVITY)

    final = _classifier(model_name, seed)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        final.fit(Xp, yp)
    if any(issubclass(w.category, ConvergenceWarning) for w in caught):
        n_nonconverged += 1
    ph = final.predict_proba(Xh)[:, 1]
    gh = groups[hold_idx]
    if len(np.unique(yh)) > 1:
        hold_disc = discrimination_metrics(yh, ph, gh, n_boot=N_BOOTSTRAP_SWEEP, seed=seed + 1)
        hold_cal = calibration_metrics(yh, ph)
    else:
        hold_disc = {"auroc": float("nan"), "auroc_lo": float("nan"), "auroc_hi": float("nan"),
                      "auprc": float("nan"), "auprc_lo": float("nan"), "auprc_hi": float("nan"),
                      "auprc_baseline": float("nan"), "auprc_lift": float("nan")}
        hold_cal = {"brier": float("nan"), "calibration_slope": float("nan"),
                     "calibration_intercept": float("nan"), "ece": float("nan")}

    # ---- the repeats ---------------------------------------------------------------------------
    # Each repeat redraws the holdout, refits on its pool, and scores. Nothing is selected on
    # them -- they are a measurement of how much the single reported holdout could have differed.
    _rep_auc = []
    if _n_rep > 1:
        for _r in range(1, _n_rep):
            _g2 = GroupShuffleSplit(n_splits=1, test_size=HOLDOUT_FRAC, random_state=seed + _r)
            _p2, _h2 = next(_g2.split(X, y, groups))
            if y[_h2].sum() < 3 or y[_p2].sum() < 3:
                continue
            try:
                _m2 = _classifier(model_name, seed)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    _m2.fit(X[_p2], y[_p2])
                _rep_auc.append(float(roc_auc_score(y[_h2], _m2.predict_proba(X[_h2])[:, 1])))
            except Exception:
                pass

    return {
        "n_repeats": int(len(_rep_auc) + 1),
        "holdout_auroc_repeats_median": (float(np.median(_rep_auc + [hold_disc["auroc"]]))
                                          if _rep_auc else float("nan")),
        "holdout_auroc_repeats_lo": (float(np.quantile(_rep_auc + [hold_disc["auroc"]], 0.05))
                                      if _rep_auc else float("nan")),
        "holdout_auroc_repeats_hi": (float(np.quantile(_rep_auc + [hold_disc["auroc"]], 0.95))
                                      if _rep_auc else float("nan")),
        "holdout_auroc_repeats_sd": (float(np.std(_rep_auc + [hold_disc["auroc"]], ddof=1))
                                      if len(_rep_auc) > 1 else float("nan")),
        "n_total": int(len(y)), "n_positive": int(y.sum()),
        "n_pool": int(len(yp)), "n_holdout": int(len(yh)),
        "n_holdout_positive": int(yh.sum()),
        "n_holdout": int(len(yh)),
        # The holdout's OWN prevalence, not the overall one. Accuracy is computed against
        # this in cell 22, and a grouped split does not reproduce the overall rate exactly.
        "holdout_prevalence": float(yh.mean()),
        "n_patients": int(len(np.unique(groups))),
        "cv_folds_used": int(n_splits),
        "cv_folds_ran": int(n_folds_ran),
        "n_cv_rows": int(n_cv_rows),
        "cv_prevalence_evaluated": float(yp[mask].mean()) if n_cv_rows else float("nan"),
        "n_nonconverged_fits": int(n_nonconverged),
        "cv_auroc": cv_auroc, "cv_auroc_lo": cv_disc["auroc_lo"], "cv_auroc_hi": cv_disc["auroc_hi"],
        "cv_ap": cv_ap, "cv_ap_lo": cv_disc["auprc_lo"], "cv_ap_hi": cv_disc["auprc_hi"],
        "cv_auroc_fold_mean": float(np.mean(fold_auc)) if fold_auc else float("nan"),
        # ddof=1. With 5 folds, ddof=0 understates the spread by ~11%, and this number is the
        # honest one -- see cv_auroc_lo_wide below.
        "cv_auroc_fold_std": (float(np.std(fold_auc, ddof=1)) if len(fold_auc) > 1
                               else float("nan")),
        # ---- the interval that should be quoted -------------------------------------------
        #
        # cv_auroc_lo/hi come from a cluster bootstrap that resamples PATIENTS while holding the
        # fold assignment and the fitted models FIXED. It therefore measures evaluation-set
        # sampling variability and nothing else -- not which rows landed in which fold, not the
        # refit variance of a 1,152-dimension probe fitted on ~150 positives.
        #
        # That the omission is material is visible in this notebook's own output: for the
        # hemorrhage cohort it printed "CV AUROC 0.769 [0.734-0.799] (fold sd 0.062)" -- a CI
        # half-width of 0.032 sitting beside a fold-to-fold sd of 0.062, twice as large. Quoting
        # the narrower of two numbers the code computes, when the wider one captures a source of
        # variance the narrower ignores, is how a result gets over-claimed.
        #
        # So: combine them. The bootstrap half-width and the standard error of the fold mean are
        # roughly independent sources, added in quadrature. This is deliberately conservative.
        "cv_auroc_lo_wide": _wide_lo(cv_disc, fold_auc),
        "cv_auroc_hi_wide": _wide_hi(cv_disc, fold_auc),
        "holdout_auroc": hold_disc["auroc"],
        "holdout_auroc_lo": hold_disc["auroc_lo"], "holdout_auroc_hi": hold_disc["auroc_hi"],
        "holdout_ap": hold_disc["auprc"],
        "holdout_ap_lo": hold_disc["auprc_lo"], "holdout_ap_hi": hold_disc["auprc_hi"],
        "holdout_ap_baseline": hold_disc["auprc_baseline"],
        "holdout_ap_lift": hold_disc["auprc_lift"],
        "holdout_brier": hold_cal["brier"],
        "holdout_cal_slope": hold_cal["calibration_slope"],
        "holdout_cal_intercept": hold_cal["calibration_intercept"],
        "holdout_ece": hold_cal["ece"],
        "prevalence": float(y.mean()),
        "operating_points": {
            name: {"cv": _threshold_metrics(yp[mask], oof[mask], thr),
                    "holdout": _threshold_metrics(yh, ph, thr)}
            for name, thr in thresholds.items()
        },
    }


# ---- the sweep -------------------------------------------------------------------------------
# ---- refuse to compute anything under a failed leakage audit --------------------------------
#
# Cell 20c runs the audit on the real cohort. A number produced while a hard check is failing is
# not a result, and the one thing this project has learned the hard way is that a wrong number
# which looks reasonable costs far more than a run that stops.
_audit = globals().get("LEAKAGE_AUDIT")
if _audit is None:
    print("NOTE: cell 20c (leakage audit) has not run in this kernel. Run it before trusting "
          "anything below.")
elif _audit.get("hard_fail"):
    raise RuntimeError(
        "the leakage audit failed "
        f"({len(_audit['hard_fail'])} hard check(s): {_audit['hard_fail']}).\n"
        "Fix them and re-run cell 20c. To override deliberately, set "
        "LEAKAGE_AUDIT['hard_fail'] = [] -- but any number produced that way is not reportable.")


def _cohort_fingerprint():
    """A short hash of what every target actually MEANS, not just how many rows there are.

    Covers the special cohorts' membership vectors (so redefining a cohort changes the key) and
    the eligible finding list with its positive counts (so adding or dropping a finding does
    too). Hashing the membership rather than the column-name lists catches a redefinition that
    happens downstream of cell 20 -- which is exactly how the patch workflow applies one.
    """
    import hashlib
    h = hashlib.sha1()
    for k in sorted(globals().get("SPECIAL_COHORTS", {})):
        v = SPECIAL_COHORTS[k]
        try:
            vals = np.asarray(v.sort_index().to_numpy(), dtype=np.int8).tobytes()
        except Exception:
            vals = repr(v).encode()
        h.update(k.encode()); h.update(vals)
    for c in sorted(globals().get("ELIGIBLE_FINDINGS", [])):
        h.update(c.encode())
    return h.hexdigest()[:12]


def sweep_signature(n_rows):
    """Everything that changes a sweep result, folded into the resume key.

    Without this, the key was (checkpoint, target, feature set, classifier) -- which covers none
    of it. Two concrete consequences, both of them workflows this notebook recommends:
      * run the sweep at 300 embeddings, finish extraction to 3,200, re-run -> every key is
        already present, "0 new results", and the leaderboard silently reports AUROCs computed on
        300 scans.
      * flip REQUIRE_STUDYTYPE_AGREEMENT "to see how much it moves" -> it cannot move.
    """
    return "|".join([
        f"n={int(n_rows)}",
        f"agree={bool(REQUIRE_STUDYTYPE_AGREEMENT)}",
        f"seed={RANDOM_SEED}", f"folds={CV_FOLDS}", f"holdout={HOLDOUT_FRAC}",
        f"minpos={MIN_POSITIVES}", f"targetsens={TARGET_SENSITIVITY}",
        # WHAT the embeddings are, not just how many. Re-extracting all 4,000 series under a new
        # spacing leaves the row count identical, so without this every key is already "done", the
        # sweep prints "0 new result(s)", and the leaderboard reports the OLD embeddings' numbers
        # as the result of the new preprocessing.
        f"preproc={globals().get('PREPROC_SIG', 'unknown')}",
        f"boot={N_BOOTSTRAP_SWEEP}",
        # WHAT THE LABELS MEAN. The signature covered how many rows and how they were
        # preprocessed, and not one thing about the target definition -- while the repo's own
        # patch workflow (build/patch/fix_hemorrhage_and_accuracy.py) exists to REDEFINE a
        # cohort and then says "re-run cell 21 and cell 22".
        #
        # Concretely: that patch narrows any_intracranial_hemorrhage from 304 positives (scalp
        # haematomas included) to 196 (intracranial only). If the row count happens to be
        # unchanged, every resume key is byte-identical, the sweep prints "0 new result(s)",
        # and cell 22 reports the OLD 304-positive AUROC under the NEW cohort's name. The two
        # differ by 0.045. Hashing the cohort membership itself makes that impossible.
        f"cohorts={_cohort_fingerprint()}",
        # WHO IS A GROUP. Splits group on person_key now rather than the per-visit PatientID;
        # results computed under the old grouping are leakage-inflated and must not be resumed
        # onto silently.
        f"groupby={'person' if globals().get('REQUIRE_PERSON_KEY', True) else 'visit'}",
        f"reps={globals().get('N_EVAL_REPEATS', 1)}",
        f"strict={bool(globals().get('STRICT_COHORT', False))}",
        # bumped 6 -> 7 when the feature set was renamed pillar0 -> embeddings, so the
        # renamed rows land under a NEW signature instead of sitting in the same CSV beside
        # identically-valued rows under the old name.
        "sweepv=7",
    ])


def load_done_keys(path):
    """Only genuinely COMPLETED results count as done -- a `skipped` or `error` record is retried,
    exactly like a failed extraction. Treating skips as done meant a finding skipped early (for
    "holdout has 3 positives") was excluded forever, and since the leaderboard filters skipped
    rows out, it vanished rather than showing up as missing."""
    done = set()
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                try:
                    r = json.loads(line)
                except Exception:
                    continue
                if "skipped" in r or "error" in r:
                    continue
                done.add((r.get("model_ckpt"), r.get("target"), r.get("feature_set"),
                          r.get("classifier"), r.get("sweep_sig")))
    return done


# Cell 20 only defines these when it had a feature table to work with.
if "ELIGIBLE_FINDINGS" not in globals() or "SPECIAL_COHORTS" not in globals():
    raise RuntimeError(
        "No targets are defined. Run cell 20 (labels) first -- and if that cell reported no "
        "feature table, run cells 17 and 18 before it."
    )

DONE_SWEEP = load_done_keys(CLASSIFIER_RESULTS_PATH)
SKIP_REASONS = {}
print(f"resuming classifier sweep: {len(DONE_SWEEP)} result(s) already computed")

sweep_t0 = time.time()
n_new = n_skipped = 0
sweep_stopped = None
_out = open(CLASSIFIER_RESULTS_PATH, "a")

try:
    for _ckpt, tab in LABELS.items():
        feature_cols = FEATURE_COLUMNS[_ckpt]
        work = tab
        if REQUIRE_STUDYTYPE_AGREEMENT and "studytype_agrees" in work.columns:
            before = len(work)
            work = work[work["studytype_agrees"].fillna(False)]
            print(f"[{_ckpt}] label-trust filter: {len(work):,}/{before:,} series kept")
        if len(work) < 4 * MIN_POSITIVES:
            print(f"[{_ckpt}] only {len(work)} series -- skipping the sweep.")
            continue

        sweep_sig = sweep_signature(len(work))
        print(f"[{_ckpt}] sweep signature: {sweep_sig}")
        # ---- GROUP BY PERSON, NOT BY VISIT --------------------------------------------------
        #
        # This used to be `patient_id`, straight from the DICOM PatientID tag. In this export
        # PatientID is a registration counter issued at each visit, not a medical record number:
        # its Spearman correlation with StudyDate is 0.78, its per-month minimum marches upward
        # all year, and 189 people appear under two or more of them across 401 studies --
        # including one patient registered twice on the same day.
        #
        # Grouping on it therefore did not do what the comment above claims. Two scans of one
        # person under two PatientIDs were two groups, so the same skull could sit in train and
        # in test, and a leakage audit that checks "no patient_id spans the split" passes while
        # it happens. Measured through this same function at this cohort's size, that was worth
        # +0.01 to +0.03 AUROC.
        #
        # Cell 9 now derives `person_key` (name + sex + birth year, falling back to the
        # PatientID when the name is too thin to identify anyone). If it is missing -- an older
        # manifest -- this refuses rather than silently regrouping on the visit counter, because
        # the failure mode is a number that looks fine and is not.
        if "person_key" in work.columns:
            groups = work["person_key"].astype(str).to_numpy()
            _n_pid = work["patient_id"].nunique() if "patient_id" in work.columns else 0
            _n_per = len(set(groups))
            print(f"[{_ckpt}] grouping splits on person_key: {_n_per:,} people "
                  f"across {len(work):,} scans"
                  + (f" ({_n_pid - _n_per:,} visit-IDs merged into an existing person)"
                     if _n_pid > _n_per else ""))
        elif REQUIRE_PERSON_KEY:
            raise RuntimeError(
                "the feature table has no `person_key` column, so splits would group on the "
                "per-visit PatientID and leak.\nRebuild the manifest (delete "
                f"{MANIFEST_PATH} and re-run cell 9), then re-run cell 18.\n"
                "Set REQUIRE_PERSON_KEY = False in cell 4 to proceed anyway -- but the numbers "
                "will be inflated and should not be reported.")
        else:
            groups = work.get("patient_id", pd.Series(range(len(work)), index=work.index)).astype(str).to_numpy()
            print(f"[{_ckpt}] WARNING: grouping on patient_id (a per-visit counter). "
                  f"Results are leakage-inflated by roughly +0.01 to +0.03 AUROC.")
        matrices = build_feature_matrices(work, feature_cols)

        # Cheap, high-value cohorts first, so even an interrupted sweep leaves useful results.
        # Cohorts are keyed by series_uid, not by position. SPECIAL_COHORTS is built in cell 20
        # from the FIRST model's table; aligning a second model's rows by positional index would
        # silently pair each row with someone else's label.
        targets = []
        for k, v in SPECIAL_COHORTS.items():
            aligned = work["series_uid"].map(v).to_numpy()
            if np.isnan(aligned.astype(float)).any():
                print(f"  cohort {k!r} skipped for {_ckpt}: "
                      f"{int(np.isnan(aligned.astype(float)).sum())} row(s) have no cohort label")
                continue
            targets.append((f"cohort:{k}", aligned.astype(int)))
        targets += [(f"finding:{c}", work[c].to_numpy()) for c in ELIGIBLE_FINDINGS]

        for target_name, y in targets:
            for fs_name, X in matrices.items():
                for clf_name in CLASSIFIERS_TO_RUN:
                    key = (_ckpt, target_name, fs_name, clf_name, sweep_sig)
                    if key in DONE_SWEEP:
                        continue
                    if (MAX_SECONDS_PER_CLASSIFIER_SWEEP is not None
                            and time.time() - sweep_t0 > MAX_SECONDS_PER_CLASSIFIER_SWEEP):
                        sweep_stopped = "time limit reached"
                        raise StopIteration
                    t0 = time.time()
                    try:
                        res = evaluate_target(X, y, groups, clf_name)
                    except Exception as e:
                        res = {"error": f"{type(e).__name__}: {e}"}
                    rec = {"model_ckpt": _ckpt, "target": target_name, "feature_set": fs_name,
                            "classifier": clf_name, "sweep_sig": sweep_sig,
                            "n_rows_evaluated": int(len(work)),
                            "seconds": round(time.time() - t0, 2), **res}
                    _out.write(json.dumps(rec) + "\n")
                    _out.flush()
                    DONE_SWEEP.add(key)
                    if "skipped" in res or "error" in res:
                        n_skipped += 1
                        _why = res.get("skipped") or res.get("error", "")
                        SKIP_REASONS.setdefault(_why[:160], []).append(target_name)
                    else:
                        n_new += 1
                        _warn = ("  [!! {} fit(s) did not converge]".format(res["n_nonconverged_fits"])
                                  if res.get("n_nonconverged_fits") else "")
                        print(f"  {target_name:48s} {fs_name:18s} {clf_name:12s} "
                              f"CV {res['cv_auroc']:.3f} [{res['cv_auroc_lo']:.3f}-{res['cv_auroc_hi']:.3f}]"
                              f"  hold {res['holdout_auroc']:.3f}"
                              f"  AP {res['holdout_ap']:.3f}/{res['holdout_ap_baseline']:.3f}"
                              f"  ({res['n_positive']} pos){_warn}")
except StopIteration:
    pass
except KeyboardInterrupt:
    sweep_stopped = "interrupted by user"
finally:
    _out.close()

print(f"\nsweep: {n_new} new result(s), {n_skipped} skipped/errored, "
      f"{(time.time()-sweep_t0)/60:.1f} min")
if SKIP_REASONS:
    # Silence about skips is how a sweep that evaluated almost nothing gets mistaken for a sweep
    # that found nothing. Say what was skipped and why.
    print("skip reasons (a skip is not a failure -- it means the data cannot support an honest "
          "estimate for that target):")
    for _why, _targets in sorted(SKIP_REASONS.items(), key=lambda kv: -len(kv[1])):
        _uniq = sorted(set(_targets))
        print(f"  [{len(_targets)}x] {_why}")
        print(f"        targets: {', '.join(t.split(':', 1)[-1] for t in _uniq[:6])}"
              + (f" ... +{len(_uniq)-6} more" if len(_uniq) > 6 else ""))
if sweep_stopped:
    print(f"stopped early ({sweep_stopped}) -- re-run this cell to continue where it left off.")

In [ ]:
# 22. Leaderboard -- and the comparison that actually matters
#
# The headline is NOT "Pillar-0 got 0.8x AUROC on finding Y". It is the gap between the Pillar-0
# embeddings and the demographics-only baseline on the same target, same split, same classifier.
# A target where that gap is ~0 is a target where the model contributed nothing, however good the
# absolute number looks.
import json
import os

import numpy as np
import pandas as pd

# Set to one of the signatures printed below to pin the tables to a specific sweep configuration;
# None means "the one cell 21 just ran", falling back to all of them.
LEADERBOARD_SWEEP_SIG = None

rows = []
if os.path.exists(CLASSIFIER_RESULTS_PATH):
    with open(CLASSIFIER_RESULTS_PATH) as f:
        for line in f:
            try:
                r = json.loads(line)
            except Exception:
                continue
            if "skipped" in r or "error" in r:
                continue
            flat = {k: v for k, v in r.items() if k != "operating_points"}
            for op_name, op in r.get("operating_points", {}).items():
                for split in ("cv", "holdout"):
                    for mk, mv in op.get(split, {}).items():
                        flat[f"{op_name}__{split}__{mk}"] = mv
            rows.append(flat)

def print_operating_points(r, indent="  "):
    """Confusion matrix, accuracy and balanced accuracy at each operating point.

    The counts were always computed and stored -- threshold_metrics returns tp/tn/fp/fn -- and
    were simply never printed, so accuracy had to be re-derived from sensitivity, specificity
    and prevalence. That re-derivation is not the same number: it assumes the holdout's
    prevalence exactly, while the real accuracy comes from integer counts. These are the counts.
    """
    _prev = float(r.get("holdout_prevalence", r.get("prevalence", float("nan"))))
    _majority = max(_prev, 1.0 - _prev)
    print(f"{indent}{'operating point':<18s} {'thr':>6s} {'TP':>5s} {'FP':>5s} {'FN':>5s} "
          f"{'TN':>6s} {'sens (95% CI)':>21s} {'spec (95% CI)':>21s} "
          f"{'PPV':>6s} {'NPV':>6s} {'ACC':>6s} {'BAL':>6s} {'vs maj':>7s}")
    for op in ("fixed_0.5", "youden", "high_sensitivity"):
        c = f"{op}__holdout__"
        if c + "sensitivity" not in r:
            continue
        tp, fp = int(r.get(c + "tp", 0)), int(r.get(c + "fp", 0))
        fn, tn = int(r.get(c + "fn", 0)), int(r.get(c + "tn", 0))
        _sens, _spec = float(r[c + "sensitivity"]), float(r[c + "specificity"])
        _n = tp + tn + fp + fn
        # Prefer the stored accuracy, then the integer counts, and only then fall back to
        # deriving it from sensitivity, specificity and prevalence.
        #
        # The order matters. An earlier version divided by max(n, 1) with no fallback, so a
        # record written before the counts were stored -- any result already in the jsonl --
        # printed accuracy as 0.000 next to a perfectly good sensitivity. A zero that looks
        # like a measurement is worse than a blank, so when there is nothing to compute from,
        # the counts print as "-" rather than as zeros.
        if c + "accuracy" in r and np.isfinite(float(r[c + "accuracy"])):
            _acc = float(r[c + "accuracy"])
        elif _n > 0:
            _acc = (tp + tn) / _n
        else:
            _acc = _sens * _prev + _spec * (1.0 - _prev)
        _bal = float(r.get(c + "balanced_accuracy", (_sens + _spec) / 2.0))
        _cnt = (f"{tp:>5d} {fp:>5d} {fn:>5d} {tn:>6d}" if _n > 0
                else f"{'-':>5s} {'-':>5s} {'-':>5s} {'-':>6s}")
        _s = (f"{_sens:.3f} [{r.get(c+'sensitivity_lo', float('nan')):.2f}-"
              f"{r.get(c+'sensitivity_hi', float('nan')):.2f}]")
        _p = (f"{_spec:.3f} [{r.get(c+'specificity_lo', float('nan')):.2f}-"
              f"{r.get(c+'specificity_hi', float('nan')):.2f}]")
        print(f"{indent}{op:<18s} {r[c+'threshold']:>6.3f} {_cnt} "
              f"{_s:>21s} {_p:>21s} {r[c+'ppv']:>6.3f} {r[c+'npv']:>6.3f} "
              f"{_acc:>6.3f} {_bal:>6.3f} {_acc - _majority:>+7.3f}")
    return _majority



if not rows:
    print("No completed classifier results yet -- run cell 21 first.")
else:
    res = pd.DataFrame(rows)
    res = res.drop_duplicates(
        subset=["model_ckpt", "target", "feature_set", "classifier", "sweep_sig"], keep="last")
    # A results file can legitimately hold several sweeps. Select on the SWEEP SIGNATURE, not on
    # row count. Row count conflates two different things: a dataset that grew (an earlier sweep is
    # then genuinely superseded) and a deliberate subset such as REQUIRE_STUDYTYPE_AGREEMENT=True,
    # which produces FEWER rows on purpose. Filtering on the maximum row count silently discarded
    # every result from that A/B -- the exact experiment cell 21's own comments tell you to run --
    # so the user saw unchanged numbers and concluded the filter did nothing.
    if "sweep_sig" in res.columns and res["sweep_sig"].nunique() > 1:
        print(f"this results file contains {res['sweep_sig'].nunique()} distinct sweep "
              f"configuration(s):")
        for _sig, _grp in res.groupby("sweep_sig"):
            _n = int(_grp["n_rows_evaluated"].iloc[0]) if "n_rows_evaluated" in _grp else -1
            print(f"  [{len(_grp):3d} result(s), n={_n:,}]  {_sig}")
        if LEADERBOARD_SWEEP_SIG is not None:
            res = res[res["sweep_sig"] == LEADERBOARD_SWEEP_SIG]
            print(f"showing LEADERBOARD_SWEEP_SIG only ({len(res)} row(s))")
        else:
            _cur = globals().get("sweep_sig")
            if _cur is not None and (res["sweep_sig"] == _cur).any():
                res = res[res["sweep_sig"] == _cur]
                print(f"showing the configuration cell 21 just ran ({len(res)} row(s)). Set "
                      f"LEADERBOARD_SWEEP_SIG to a signature above to look at a different one.")
            else:
                print("no single configuration selected -- showing ALL of them together. Read the "
                      "sweep_sig column before comparing rows.")
    out_csv = os.path.join(OUTPUT_DIR, "classifier_leaderboard.csv")
    res.to_csv(out_csv, index=False)
    print(f"{len(res):,} result row(s) -> {out_csv}\n")

    # --- signal above the demographics floor ---------------------------------------------------
    # Every feature set that is NOT the demographics floor is an image-feature arm. This used to
    # test `fs.startswith("pillar0")`, which silently returned an empty list the moment the
    # feature set was renamed -- and an empty list here does not raise, it just produces a
    # comparison table with no rows in it. Defining the arms by what they are not keeps this
    # working whatever the embedding arm is called next.
    emb_sets = [fs for fs in res["feature_set"].unique() if fs != "demographics_only"]
    if not emb_sets:
        print("  no image-feature rows in this configuration -- nothing to compare against the "
              "demographics floor.")
    base = (res[res["feature_set"] == "demographics_only"]
            .groupby(["model_ckpt", "target", "classifier"])["cv_auroc"].max()
            .rename("baseline_auroc"))
    emb = (res[res["feature_set"].isin(emb_sets)]
           .sort_values("cv_auroc", ascending=False)
           .drop_duplicates(["model_ckpt", "target", "classifier"]))
    # `baseline_auroc` is DERIVED here, never read from the results file. If a stale file written
    # by an older version of this notebook already carries a column of that name, the merge below
    # collides with it and the next line dies on a bare KeyError that says nothing about why.
    # Drop it first so the derived value always wins.
    emb = emb.drop(columns=["baseline_auroc"], errors="ignore")
    comp = emb.merge(base, on=["model_ckpt", "target", "classifier"], how="left")
    comp["lift_over_demographics"] = comp["cv_auroc"] - comp["baseline_auroc"]
    best = (comp.sort_values("cv_auroc", ascending=False)
            .drop_duplicates(["model_ckpt", "target"]))

    # n_rows_evaluated is shown because a sweep can legitimately contain results computed on
    # different numbers of embeddings (you ran it mid-extraction, then again after). The resume
    # key now separates those, so both can coexist -- but you should be able to see which is which.
    show = ["target", "classifier", "feature_set", "n_rows_evaluated", "n_positive", "prevalence",
            "cv_auroc", "cv_auroc_lo", "cv_auroc_hi", "holdout_auroc",
            "holdout_ap", "holdout_ap_baseline",
            "baseline_auroc", "lift_over_demographics"]
    show = [c for c in show if c in best.columns]

    print("=" * 100)
    print("TOP TARGETS BY CROSS-VALIDATED AUROC (best feature set / classifier per target)")
    print("=" * 100)
    top = best.sort_values("cv_auroc", ascending=False).head(30)
    with pd.option_context("display.width", 200, "display.max_columns", 50):
        print(top[show].to_string(index=False, float_format=lambda v: f"{v:.3f}"))

    print()
    print("=" * 100)
    print("RANKED BY LIFT OVER THE DEMOGRAPHICS-ONLY BASELINE  (this is the honest ranking)")
    print("=" * 100)
    lift = best.dropna(subset=["lift_over_demographics"]).sort_values(
        "lift_over_demographics", ascending=False)
    with pd.option_context("display.width", 200, "display.max_columns", 50):
        print(lift[show].head(30).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

    # Two separate ways a result can be empty, and a target can fail either one.
    _flat = lift[lift["lift_over_demographics"] < 0.02]
    if len(_flat):
        print(f"\n{len(_flat)} target(s) sit within 0.02 AUROC of the age/sex baseline -- the")
        print("embedding added nothing over demographics. Report them that way, not as model")
        print("performance:")
        print("  " + ", ".join(t.split(":", 1)[-1] for t in _flat["target"].head(15)))

    _chance = best[best["cv_auroc"] < 0.53]
    if len(_chance):
        print(f"\n{len(_chance)} target(s) are at or near chance in ABSOLUTE terms (CV AUROC < "
              f"0.53), regardless of lift. A positive lift over a below-chance baseline is not a")
        print("result; these are targets the model cannot predict at all:")
        print("  " + ", ".join(t.split(":", 1)[-1] for t in _chance["target"].head(15)))

    # --- operating points for the headline cohorts ---------------------------------------------
    print()
    print("=" * 100)
    print("OPERATING POINTS + CONFUSION MATRIX (holdout; thresholds chosen on CV only)")
    print("=" * 100)
    # globals().get, not a bare read: this is a display-only knob added to cell 4, and a
    # harness that execs cell 22 on its own should get a sensible default rather than a
    # NameError from a cell it is not testing.
    # Cohorts first, then every finding. This used to print for the three special cohorts only,
    # so for 22 of the 25 targets there was an AUROC and no way to see what it meant at a
    # threshold -- no counts, no sensitivity, no accuracy. The numbers were all computed and
    # stored; only the printing stopped short.
    _cohort_targets = ([t for t in best["target"] if t.startswith("cohort:")]
                       + [t for t in best["target"] if not t.startswith("cohort:")][
                           :globals().get("MAX_TARGETS_WITH_OPERATING_POINTS", 12)])
    if not _cohort_targets:
        print("\n(no cohort-level results yet -- cell 20 either found no usable cohort or the "
              "sweep has not reached them)")
    for target in _cohort_targets:
        sub = comp[comp["target"] == target].sort_values("cv_auroc", ascending=False)
        if not len(sub):
            continue
        r = sub.iloc[0]
        print(f"\n{target}   [{r['feature_set']} / {r['classifier']}]  "
              f"n={int(r['n_total'])}, {int(r['n_positive'])} positive "
              f"({r['prevalence']:.1%}), {int(r['n_patients'])} patients")
        print(f"  CV AUROC     {r['cv_auroc']:.3f} "
              f"[{r.get('cv_auroc_lo', float('nan')):.3f}-{r.get('cv_auroc_hi', float('nan')):.3f}] "
              f"(fold sd {r.get('cv_auroc_fold_std', float('nan')):.3f})")
        _rmed, _rlo, _rhi = (r.get("holdout_auroc_repeats_median"),
                              r.get("holdout_auroc_repeats_lo"),
                              r.get("holdout_auroc_repeats_hi"))
        if _rmed is not None and np.isfinite(_rmed):
            print(f"  repeated holdouts ({int(r.get('n_repeats', 1))} draws)   median "
                  f"{_rmed:.3f}  5-95% [{_rlo:.3f}-{_rhi:.3f}]  "
                  f"sd {r.get('holdout_auroc_repeats_sd', float('nan')):.3f}")
            print(f"               ^ the single holdout below is ONE of these draws. This spread "
                  f"is how much it")
            print(f"                 could have differed on nothing but which patients were held "
                  f"out.")
        _wlo, _whi = r.get("cv_auroc_lo_wide"), r.get("cv_auroc_hi_wide")
        if _wlo is not None and np.isfinite(_wlo):
            print(f"               [{_wlo:.3f}-{_whi:.3f}]  <-- QUOTE THIS ONE. The interval "
                  f"above resamples patients while holding")
            print(f"               the fold assignment and the fitted models fixed, so it "
                  f"misses refit and fold variance;")
            print(f"               the fold sd beside it is often twice its half-width. This "
                  f"one includes both.")
        _ncv, _npool = r.get("n_cv_rows"), r.get("n_pool")
        if _ncv is not None and _npool and int(_ncv) < int(_npool):
            print(f"               NOTE: computed on {int(_ncv):,} of {int(_npool):,} pool rows "
                  f"({int(r.get('cv_folds_ran', 0))} of {int(r.get('cv_folds_used', 0))} folds); "
                  f"evaluated prevalence {r.get('cv_prevalence_evaluated', float('nan')):.1%} "
                  f"vs {r.get('prevalence', float('nan')):.1%} overall")
        print(f"  holdout      AUROC {r['holdout_auroc']:.3f} "
              f"[{r.get('holdout_auroc_lo', float('nan')):.3f}-{r.get('holdout_auroc_hi', float('nan')):.3f}]"
              f"   AP {r.get('holdout_ap', float('nan')):.3f} "
              f"[{r.get('holdout_ap_lo', float('nan')):.3f}-{r.get('holdout_ap_hi', float('nan')):.3f}]"
              f"  vs prevalence baseline {r.get('holdout_ap_baseline', float('nan')):.3f}")
        print(f"  calibration  Brier {r.get('holdout_brier', float('nan')):.4f}  "
              f"slope {r.get('holdout_cal_slope', float('nan')):.2f} (1.0 = ideal)  "
              f"intercept {r.get('holdout_cal_intercept', float('nan')):+.2f} (0.0 = ideal)  "
              f"ECE {r.get('holdout_ece', float('nan')):.3f}")
        print(f"  demographics-only baseline AUROC {r.get('baseline_auroc', float('nan')):.3f}")
        # ACCURACY, and the only number that makes accuracy interpretable next to it.
        #
        # At 13.9% hemorrhage prevalence, predicting "no bleed" for everyone scores 85.6%. Printing
        # accuracy on its own invites exactly that number being quoted as model performance, so
        # every accuracy here is shown beside the majority-class rate and the difference. Balanced
        # accuracy -- the mean of sensitivity and specificity -- is the prevalence-free version and
        # is the single figure worth reporting.
        _prev = float(r.get("holdout_prevalence", r.get("prevalence", float("nan"))))
        _majority = max(_prev, 1.0 - _prev)
        print_operating_points(r, indent="    ")
        print(f"    always-predict-the-common-class scores {_majority:.3f} on this holdout. "
              f"An accuracy below that is not a result;")
        print(f"    read the BAL (balanced accuracy) column, or AUROC with its interval, instead.")
        print(f"    (interval widths here are driven by {int(r['n_holdout_positive'])} holdout "
              f"positive(s) -- a sensitivity from a handful of positives is not a stable number)")


    # --- a compact picture ----------------------------------------------------------------------
    try:
        import matplotlib.pyplot as plt

        plot_df = lift.head(20).iloc[::-1]
        if len(plot_df):
            # Deliberately NOT a bar chart. Bars drawn from zero on an axis clipped at 0.4 have a
            # visible length that is not proportional to their value, which exaggerates every
            # difference -- and this figure is the artifact most likely to be shown to someone
            # who was not in the room. A dot plot with a connector encodes the lift as the
            # distance between two marks, which is the quantity being claimed.
            fig, ax = plt.subplots(figsize=(9.5, max(4, 0.34 * len(plot_df))))
            ypos = np.arange(len(plot_df))
            ax.hlines(ypos, plot_df["baseline_auroc"], plot_df["cv_auroc"],
                       color="#9AA6B2", lw=2, zorder=1)
            ax.scatter(plot_df["baseline_auroc"], ypos, s=42, color="#B0B0B0",
                        zorder=2, label="age + sex only")
            ax.scatter(plot_df["cv_auroc"], ypos, s=52, color="#4C78A8",
                        zorder=3, label="Pillar-0 embeddings")
            ax.axvline(0.5, color="#444444", lw=1, ls="--")
            ax.text(0.5, len(plot_df) - 0.4, " chance", fontsize=8, color="#444444", va="top")
            ax.set_yticks(ypos)
            ax.set_yticklabels([t.replace("finding:", "").replace("cohort:", "* ")
                                 for t in plot_df["target"]], fontsize=8)
            ax.set_xlim(0.35, 1.0)
            ax.set_xlabel("cross-validated AUROC (patient-grouped)")
            _n = int(plot_df["n_rows_evaluated"].iloc[0]) if "n_rows_evaluated" in plot_df else 0
            ax.set_title(f"Pillar-0 vs the demographics floor -- top 20 by lift "
                          f"({TARGET_REGION} CT" + (f", n={_n:,}" if _n else "") + ")")
            ax.legend(loc="lower right", fontsize=8, frameon=False)
            ax.spines[["top", "right"]].set_visible(False)
            plt.tight_layout()
            fig_path = os.path.join(OUTPUT_DIR, "leaderboard_lift.png")
            plt.savefig(fig_path, dpi=140)
            plt.show()
            print(f"\nfigure saved: {fig_path}")
    except Exception as e:
        print(f"(plot skipped: {e})")

---

# Part C — Intracranial hemorrhage: RSNA vs India vs both

One binary question — **is there an intracranial hemorrhage on this scan** — asked three ways:

| model | trained on |
|---|---|
| `rsna_only` | RSNA-ICH training split |
| `india_only` | India training split |
| `combined` | both training splits pooled |

and every one of them scored on **the same three test sets**: RSNA held-out, India held-out, and
the two pooled. That is a 3 × 3 matrix in which every column is an identical set of patients, so
the columns are directly comparable — which is the whole reason for doing it this way rather than
giving each model its own split.

## Run Parts A and B first

This section reads the feature tables both halves produce. Extract with `DATA_SOURCE = "india"`,
then again with `DATA_SOURCE = "rsna"`, then come here. Cell C1 checks both exist and says exactly
what is missing if not.

## How the label is defined on each side

| | positive | negative |
|---|---|---|
| **RSNA** | `rsna_any` — expert per-slice annotation, any positive slice makes the series positive | no hemorrhage annotated |
| **India** | any of `b_sdh`, `b_edh`, `b_sah`, `b_ivh`, `b_ich`, `b_hemorrhagic_contusion` from the matched report | none of those reported |

**Read the negatives carefully — they are not the same thing.** An RSNA negative means *no
hemorrhage was annotated*; that scan may well have an infarct, a mass or atrophy, because RSNA
labels nothing except hemorrhage. A India negative means *the report mentioned no hemorrhage*,
and that report also covers everything else the radiologist saw. So "normal vs abnormal" in the
everyday sense is only answerable on the Indian side, and it is reported separately as
`abnormal_any_finding`. The cross-dataset matrix is strictly **hemorrhage vs no hemorrhage**,
because that is the only question both datasets actually answer.

Two further asymmetries worth holding in mind when reading the numbers:

* **Label provenance.** RSNA's labels are expert annotations made specifically for this task.
  India's are extracted from free-text reports. A gap between the two is at least partly a
  measurement of the report extraction, not of Pillar-0.
* **Prevalence.** They differ, and a model trained at one prevalence and applied at another will
  be miscalibrated even when it ranks perfectly. That is why the calibration slope and intercept
  are reported alongside AUROC, and why the transfer cells are the ones to look at them on.

## What else this section reports

* **A covariate-shift diagnostic** — a classifier trained to tell the two datasets apart from the
  embeddings alone. If that reaches AUROC ≈ 1.0, the two live in separate regions of the feature
  space, and that is the explanation for any transfer gap you see. Scanner, protocol, reconstruction
  kernel and slice thickness all differ between them.
* **Per-dataset standardisation** as a cheap domain-adaptation baseline, run as a second pass over
  the whole matrix so you can see whether it closes the gap.
* **Paired bootstrap comparisons.** "0.84 versus 0.81" is not a finding until you know whether it
  survives resampling the patients. Every head-to-head on a shared test set gets a paired interval
  and a bootstrap p-value.

In [ ]:
# C1. Load both datasets' embeddings and build one hemorrhage table  [Part C]
import json
import os

import numpy as np
import pandas as pd

# Part C is dataset-agnostic -- it reads both halves' outputs off disk -- but it does depend on the
# shared metrics from cell 20b. Say so plainly rather than failing on a NameError three cells later.
for _needed in ("full_evaluation", "discrimination_metrics", "paired_auroc_difference",
                 "calibration_metrics", "fast_auroc", "N_BOOTSTRAP_MAIN"):
    if _needed not in globals():
        raise RuntimeError(
            f"'{_needed}' is not defined -- run cell 20b (Evaluation metrics) first. Part C uses "
            f"exactly the same metric code as the per-finding sweep, which is what makes the two "
            f"sets of numbers comparable."
        )

# Both directories come straight from the config cell, so there is exactly one place where these
# paths are defined. Hardcoding them again here is how Part C ends up looking in the wrong folder
# after somebody changes OUTPUT_DIR.
# Fall back to the first backbone this notebook actually defines, never to a Pillar-0
# checkpoint name -- a fallback that names a model the paper does not use is how a stray
# Pillar-0 path reappears in a results folder.
PARTC_MODEL = (MODELS_TO_RUN[0] if MODELS_TO_RUN
               else (BACKBONE_IDS[0] if "BACKBONE_IDS" in globals() else None))
if PARTC_MODEL is None:
    raise RuntimeError("no backbone defined -- run the config and shim cells first")

def _side_status(output_dir, model_name, which):
    """Is this dataset's feature table ready, and if not, how far along is it?

    Returns (path, n_rows_or_None, human status).
    """
    fp = os.path.join(output_dir, f"features_{model_name}.parquet")
    if os.path.exists(fp):
        # Count rows WITHOUT assuming a column name. This probe used to ask for "series_uid",
        # which our RSNA Phase-A notebook does not have -- it writes `series`, exactly the
        # rename _normalise_schema below already knows about. So a perfectly good RSNA table
        # made this return n=None, PARTC_READY went False, and Part C skipped itself for
        # weeks while printing the word "ready" next to the very file it was refusing to
        # open. Read the parquet FOOTER instead: it carries the row count and needs no
        # column at all.
        n = None
        try:
            import pyarrow.parquet as _pq
            n = int(_pq.ParquetFile(fp).metadata.num_rows)
        except Exception:
            for _c in ("series_uid", "series"):
                try:
                    n = len(pd.read_parquet(fp, columns=[_c]))
                    break
                except Exception:
                    continue
            if n is None:
                try:
                    n = len(pd.read_parquet(fp))
                except Exception:
                    n = None
        if n is None:
            # The file is there but unreadable -- that is a real problem, and calling it
            # "ready" is how it stays hidden.
            return fp, None, "PRESENT BUT UNREADABLE -- the parquet exists and cannot be opened"
        return fp, n, f"ready ({n:,} embeddings)"
    # Not ready. Say how far the extraction has got rather than only that it is absent -- when
    # the answer is "RSNA is running in another notebook", the useful number is how many
    # embeddings exist so far.
    emb = os.path.join(output_dir, "embeddings", model_name)
    n_npy = len([f for f in os.listdir(emb) if f.endswith(".npy")]) if os.path.isdir(emb) else 0
    if n_npy:
        return fp, None, (f"extraction IN PROGRESS -- {n_npy:,} embedding(s) written so far, "
                          f"no feature table yet (run cell 18 with DATA_SOURCE='{which}')")
    if os.path.isdir(output_dir):
        return fp, None, "output folder exists but no embeddings yet -- extraction not started"
    return fp, None, "not started (output folder does not exist)"


# ---- Part C needs BOTH datasets, and says so instead of raising --------------------------------
#
# This cell used to raise FileNotFoundError the moment either feature table was absent. Since the
# normal state of an India-only run is "RSNA has not been extracted yet", every single India run
# for weeks ended in a red traceback describing a situation that was entirely expected -- which
# trains you to ignore the last cell, and an ignored error is one you will also ignore when it is
# real.
#
# Cells B1-B4 already had the right pattern: when the run is not theirs, they print one line and
# skip. Part C now does the same, and reports how far the missing side has got, because when the
# answer is "RSNA is extracting in the other notebook" the useful information is the count.
_ip, _in, _istat = _side_status(INDIA_OUTPUT_DIR, PARTC_MODEL, "india")
_rp, _rn, _rstat = _side_status(RSNA_OUTPUT_DIR, PARTC_MODEL, "rsna")
PARTC_READY = (_in is not None) and (_rn is not None)

if not PARTC_READY:
    print("=" * 94)
    print("PART C (cross-dataset 3x3 matrix) -- NOT READY, skipping. This is not an error.")
    print("=" * 94)
    print(f"  india : {_istat}")
    print(f"          {_ip}")
    print(f"  rsna  : {_rstat}")
    print(f"          {_rp}")
    print()
    _missing = [w for w, n in (("india", _in), ("rsna", _rn)) if n is None]
    print(f"  Part C trains on one dataset and tests on the other, so it needs both. Missing: "
          f"{', '.join(_missing)}.")
    print(f"  To produce the missing side: set DATA_SOURCE = '{_missing[0]}' in cell 4, run")
    print(f"  cells 1-18, then come back and run C1-C3 with either setting.")
    print()
    print("  Cells C2 and C3 will skip for the same reason. Everything above this point --")
    print("  the extraction, the leakage audit, the sweep and the leaderboard -- is unaffected.")
    print("=" * 94)
else:
    PARTC_DIR = os.path.join(INDIA_OUTPUT_DIR, "part_c_hemorrhage")
    os.makedirs(PARTC_DIR, exist_ok=True)

    # India hemorrhage columns.
    #
    # This block used to re-implement the selection from scratch, and got it wrong in the exact way
    # cell 20 was patched for. It had the generic-bleed-word half of the rule and NOT the
    # extracranial exclusion, so `b_soft_tissue_hematoma`, `b_subgaleal_hematoma` and
    # `b_periorbital_hematoma` were all counted as intracranial hemorrhage. In the first overnight
    # run `b_soft_tissue_hematoma` alone was 192 of 304 positives -- 63% of the cohort was scalp.
    #
    # That is not a small error here, it invalidates the whole matrix: RSNA has no extracranial
    # label at all, so every India cell would have been scoring "intracranial bleed OR scalp
    # haematoma" against a dataset that cannot express the second half. A scalp haematoma is a big
    # bright rind outside the skull and trivially visible to a CT model, so the India diagonal
    # inflates and RSNA->India transfer deflates -- and the headline conclusion "RSNA does not
    # transfer to India" would have been manufactured by the label bug alone.
    #
    # It also differed from cell 20 in two quieter ways: it omitted the "bleed" substring, and it
    # only looked at b_ columns while cell 20 scans b_ and f_.
    #
    # So it no longer re-implements anything. Cell 20 owns the definition; Part C uses it, and
    # refuses to guess when it is absent.
    NEGATION = {"no", "normal", "unremarkable", "negative", "nil"}


    def _toks(c):
        return set(c.lower().replace("-", "_").split("_"))


    def india_hemorrhage_columns(cols):
        """The SAME columns cell 20 uses for cohort:any_intracranial_hemorrhage.

        Part C is meant to be run in its own session after both extractions, so cell 20 may not
        have executed in this kernel. When it has, ANY_HEMORRHAGE_COLS is authoritative and is used
        verbatim. When it has not, the rule is re-derived here from cell 20's own constants -- which
        are read out of the notebook rather than retyped, so the two cannot drift.
        """
        declared = globals().get("ANY_HEMORRHAGE_COLS")
        if declared:
            chosen = [c for c in declared if c in set(cols)]
            if chosen:
                print(f"[india] hemorrhage columns taken from cell 20's ANY_HEMORRHAGE_COLS "
                      f"({len(chosen)} of {len(declared)} present in this table)")
                return sorted(chosen)

        named_terms = {"subdural", "sdh", "epidural", "extradural", "edh",
                       "subarachnoid", "sah", "intraventricular", "ivh",
                       "intracerebral", "intraparenchymal", "ich", "iph"}
        generic = ("hemorrhag", "haemorrhag", "hematoma", "haematoma", "bleed")
        extracranial = ("soft_tissue", "soft tissue", "scalp", "subgaleal", "subcutaneous",
                        "cephalhematoma", "cephalohematoma", "periorbital", "preseptal")

        out, dropped = [], []
        for c in cols:
            t = _toks(c)
            if t & NEGATION:
                continue
            if t & named_terms:                      # a named compartment is always intracranial
                out.append(c)
                continue
            if any(x in c.lower() for x in generic):
                if any(x in c.lower() for x in extracranial):
                    dropped.append(c)                # outside the skull -- not this cohort
                else:
                    out.append(c)
        print(f"[india] cell 20 has not run in this kernel; hemorrhage columns re-derived from its "
              f"rule: {len(out)} kept")
        if dropped:
            print(f"[india]   EXCLUDED as extracranial (they are NOT intracranial hemorrhage, and "
                  f"RSNA has no equivalent label): {dropped}")
        return sorted(out)


    def _normalise_schema(tab, which):
        """Part C was written against THIS notebook's table layout. Our RSNA Phase-A
        notebook writes the same information under different names -- `series`/`patient`/
        `study` instead of `series_uid`/`patient_id`/`study_uid`, and `f_0000...` instead
        of `emb_0000...` -- so the two are renamed onto one schema here rather than
        duplicating several hundred MB of parquet just to change column headers.

        The feature columns end up identically named and identically ordered on both
        sides, which is what the equality check below actually depends on. Both notebooks
        zero-pad to four digits, so f_0000 -> emb_0000 lines up exactly.
        """
        ren = {"series": "series_uid", "patient": "patient_id", "study": "study_uid"}
        tab = tab.rename(columns={k: v for k, v in ren.items()
                                  if k in tab.columns and v not in tab.columns})
        _f = sorted(c for c in tab.columns if c.startswith("f_"))
        if _f and not any(c.startswith("emb_") for c in tab.columns):
            tab = tab.rename(columns={c: "emb_" + c[2:] for c in _f})
        # Phase A calls the hemorrhage label `any`; Part C looks for `rsna_any` and, when it
        # cannot find it, goes hunting for an rsna_manifest.csv this pipeline never wrote.
        if which == "rsna" and "rsna_any" not in tab.columns and "any" in tab.columns:
            tab["rsna_any"] = tab["any"]
        _need = [c for c in ("series_uid", "study_uid", "patient_id") if c not in tab.columns]
        if _need:
            raise ValueError(
                f"[{which}] feature table is missing {_need}. Part C needs a series id, a "
                f"study id and a patient id to build patient-grouped splits; without the "
                f"patient id every split would leak the same person across train and test.")
        return tab


    def _load_side(output_dir, model_name, which):
        fp, n, _status = _side_status(output_dir, model_name, which)
        tab = _normalise_schema(pd.read_parquet(fp), which)
        # Phase A admits a series on its progress record without checking manifest
        # membership, so features_B1_swinunetr_suprem.parquet carries 68 rows from an
        # earlier cohort with NaN for every label. They would enter the pooled table as
        # unlabelled rows.
        if "rsna_any" in tab.columns:
            _n0 = len(tab)
            tab = tab[tab["rsna_any"].notna()]
            if len(tab) != _n0:
                print(f"[{which}] dropped {_n0 - len(tab):,} row(s) with no label "
                      f"(out-of-cohort leftovers)")
        print(f"[{which}] {len(tab):,} embeddings from {fp}")
        return tab


    def build_hemorrhage_table():
        india = _load_side(INDIA_OUTPUT_DIR, PARTC_MODEL, "india")
        rsna = _load_side(RSNA_OUTPUT_DIR, PARTC_MODEL, "rsna")

        feat_i = sorted(c for c in india.columns if c.startswith("emb_"))
        feat_r = sorted(c for c in rsna.columns if c.startswith("emb_"))
        if feat_i != feat_r:
            raise ValueError(
                f"the two feature tables have different embedding columns "
                f"({len(feat_i)} vs {len(feat_r)}). They must come from the same checkpoint and the "
                f"same preprocessing, or nothing trained on one can be applied to the other."
            )
        feat_cols = feat_i

        # --- India labels: read straight from the matched-scans table -------------------------------
        # WHERE THIS FILE ACTUALLY IS. Cell 7 writes it to OUTPUT_DIR as it stood WHEN CELL 7 RAN,
        # and the config cell further down re-points INDIA_OUTPUT_DIR at our own output folder. So
        # joining INDIA_OUTPUT_DIR here looks in a folder cell 7 never wrote to, and Part C dies on
        # a missing file that exists. Cell 7 exports the path it used as TABLE_CACHE_PATH -- use
        # that, and fall back to searching the folders this notebook knows about.
        india_csv = globals().get("TABLE_CACHE_PATH") or ""
        if not os.path.exists(india_csv):
            _cands = [os.path.join(_d, "matched_table.parquet") for _d in dict.fromkeys(
                [globals().get("OUTPUT_DIR"), INDIA_OUTPUT_DIR,
                 globals().get("INDIA_INPUT_DIR"), os.path.dirname(PROGRESS_LOG_PATH)]) if _d]
            india_csv = next((c for c in _cands if os.path.exists(c)), _cands[0] if _cands else "")
        if not os.path.exists(india_csv):
            raise FileNotFoundError(
                f"{india_csv} is missing -- it is written by cell 7. Run Part A's cell 7 once "
                f"(with DATA_SOURCE = 'india') so the labels are available here."
            )
        itab = pd.read_parquet(india_csv)
        # b_ and f_, matching cell 20's head prefixes. Restricting to b_ made any f_-prefixed
        # hemorrhage column invisible to Part C while cell 20 counted it.
        hem_cols = india_hemorrhage_columns(
            [c for c in itab.columns if c.lower().startswith(("b_", "f_"))])
        finding_cols = [c for c in itab.columns
                        if c.lower().startswith(("b_", "f_")) and not (_toks(c) & NEGATION)]
        print(f"[india] hemorrhage columns: {hem_cols}")
        print(f"[india] all-finding columns for the abnormal cohort: {len(finding_cols)}")
        if not hem_cols:
            raise ValueError("no India hemorrhage column matched -- check the b_ column names.")

        # dict.fromkeys, not hem_cols + finding_cols: hem_cols is a SUBSET of finding_cols, so the
        # concatenation asked pandas for duplicated columns. It happened to work; it was not intended.
        _agg_cols = list(dict.fromkeys(list(hem_cols) + list(finding_cols)))
        per_study = itab.groupby("StudyInstanceUID", as_index=False)[_agg_cols].max()
        per_study["hemorrhage"] = (per_study[hem_cols].to_numpy() > 0).any(axis=1).astype(int)
        per_study["abnormal_any_finding"] = (per_study[finding_cols].to_numpy() > 0).any(axis=1).astype(int)
        india = india.merge(per_study[["StudyInstanceUID", "hemorrhage", "abnormal_any_finding"]],
                            left_on="study_uid", right_on="StudyInstanceUID", how="left")
        n_missing = int(india["hemorrhage"].isna().sum())
        if n_missing:
            print(f"[india] {n_missing:,} embedding(s) had no matching report row and are DROPPED "
                  f"(silently zero-filling them would manufacture negatives)")
            india = india[india["hemorrhage"].notna()]
        india["hemorrhage"] = india["hemorrhage"].astype(int)
        india["abnormal_any_finding"] = india["abnormal_any_finding"].fillna(0).astype(int)

        # --- RSNA labels: already per-series on the manifest ----------------------------------------
        if "rsna_any" not in rsna.columns:
            rman = os.path.join(RSNA_OUTPUT_DIR, "rsna_manifest.csv")
            if not os.path.exists(rman):
                raise FileNotFoundError(f"{rman} is missing -- run Part B's cell B4.")
            rm = pd.read_csv(rman)
            keep = ["series_uid"] + [c for c in rm.columns if c.startswith("rsna_")]
            rsna = rsna.merge(rm[keep].drop_duplicates("series_uid"), on="series_uid", how="left")
        n_missing_r = int(rsna["rsna_any"].isna().sum())
        if n_missing_r:
            print(f"[rsna] {n_missing_r:,} embedding(s) had no label row and are DROPPED")
            rsna = rsna[rsna["rsna_any"].notna()]
        rsna["hemorrhage"] = (rsna["rsna_any"].to_numpy() > 0).astype(int)
        rsna["abnormal_any_finding"] = np.nan     # RSNA labels nothing except hemorrhage

        common = ["series_uid", "study_uid", "patient_id", "hemorrhage", "abnormal_any_finding"]
        india["dataset"] = "india"
        rsna["dataset"] = "rsna"
        both = pd.concat([india[common + feat_cols + ["dataset"]],
                          rsna[common + feat_cols + ["dataset"]]], ignore_index=True)
        # Patient ids are only unique WITHIN a dataset; prefixing prevents a collision from putting two
        # different people's scans into the same cross-validation group.
        both["patient_key"] = both["dataset"] + ":" + both["patient_id"].astype(str)
        return both, feat_cols


    PARTC, PARTC_FEATURES = build_hemorrhage_table()

    print()
    print(f"{'dataset':<10s} {'series':>8s} {'patients':>9s} {'hemorrhage +':>13s} {'prevalence':>11s}")
    for _ds, _g in PARTC.groupby("dataset"):
        print(f"{_ds:<10s} {len(_g):>8,} {_g['patient_key'].nunique():>9,} "
              f"{int(_g['hemorrhage'].sum()):>13,} {_g['hemorrhage'].mean():>10.1%}")
    print(f"{'TOTAL':<10s} {len(PARTC):>8,} {PARTC['patient_key'].nunique():>9,} "
          f"{int(PARTC['hemorrhage'].sum()):>13,} {PARTC['hemorrhage'].mean():>10.1%}")

    _ab = PARTC[PARTC["dataset"] == "india"]["abnormal_any_finding"]
    print(f"\nindia only -- abnormal_any_finding (any b_/f_ report finding): "
          f"{int(_ab.sum()):,}/{len(_ab):,} ({_ab.mean():.1%})")
    print("RSNA has no equivalent: it annotates hemorrhage and nothing else, so an RSNA negative is")
    print("'no hemorrhage', NOT 'normal scan'. The cross-dataset matrix uses hemorrhage only.")

    _d = dict(PARTC.groupby("dataset")["hemorrhage"].mean())
    if len(_d) == 2 and min(_d.values()) > 0:
        print(f"\nPrevalence ratio between the two datasets: "
              f"{max(_d.values())/min(_d.values()):.1f}x. A model fitted at one prevalence and applied "
              f"at the other is miscalibrated by construction even if it ranks perfectly -- which is "
              f"why the transfer rows below report calibration slope and intercept, not just AUROC.")

In [ ]:
# C2. The 3 x 3 matrix: train on RSNA / India / both, test on all three  [Part C, RESUMABLE]
#
# The protocol, stated once so every number below can be read against it:
#
#   1. Split EACH dataset into train/test by PATIENT, once, with a fixed seed. Those two test sets
#      are then frozen and shared by every model. This is what makes the columns comparable -- if
#      each model got its own split, a difference between models could just be a difference in
#      which patients they happened to be tested on.
#   2. Train three models: RSNA train split, India train split, and the two pooled.
#   3. Score all three on all three test sets (RSNA test, India test, pooled test).
#   4. Choose operating-point thresholds from CROSS-VALIDATED predictions on each model's own
#      TRAINING data. No test set is ever consulted to pick a threshold.
#   5. Report AUROC and AUPRC with patient-clustered bootstrap intervals, calibration, and
#      sensitivity/specificity/PPV/NPV with Wilson intervals at all three operating points.
#
# Two passes: raw embeddings, and per-dataset standardised embeddings. The second is the cheapest
# domain-adaptation baseline there is -- centre and scale each dataset by its OWN statistics -- and
# it directly tests whether a transfer gap is a simple distribution offset or something deeper.

# Part C needs both datasets. C1 reports which side is missing and why; this cell skips quietly
# rather than raising a NameError on something C1 never got far enough to define.
if not globals().get("PARTC_READY", False):
    print("PART C step C2: skipped -- see cell C1 for which dataset is missing.")
else:
    import json
    import os
    import time

    import numpy as np
    import pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    PARTC_TARGET = "hemorrhage"
    PARTC_TEST_FRAC = 0.25
    PARTC_CV_FOLDS = 5
    PARTC_SEED = RANDOM_SEED
    PARTC_RESULTS_PATH = os.path.join(PARTC_DIR, "matrix_results.jsonl")
    PARTC_PREDICTIONS_PATH = os.path.join(PARTC_DIR, "test_predictions.parquet")
    PARTC_NORMALISATIONS = ["raw", "per_dataset_zscore"]


    def partc_classifier(seed):
        return Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=5000, C=1.0, class_weight="balanced",
                                        random_state=seed)),
        ])


    def make_frozen_splits(tab, seed=PARTC_SEED):
        """One patient-grouped train/test split per dataset, shared by every model."""
        assign = pd.Series("", index=tab.index, dtype=object)
        for ds, g in tab.groupby("dataset"):
            y = g[PARTC_TARGET].to_numpy()
            grp = g["patient_key"].to_numpy()
            gss = GroupShuffleSplit(n_splits=1, test_size=PARTC_TEST_FRAC, random_state=seed)
            tr, te = next(gss.split(np.zeros(len(g)), y, grp))
            assign.loc[g.index[tr]] = "train"
            assign.loc[g.index[te]] = "test"
        return assign


    def per_dataset_zscore(tab, feat_cols, fit_on):
        """Centre and scale each dataset's embeddings by that dataset's OWN training statistics.

        Fitting on the training rows only, per dataset, is what keeps this honest -- computing the
        mean over all rows including the test set is a small but real leak, and it is the sort that
        quietly improves every transfer number.
        """
        out = tab[feat_cols].to_numpy(dtype=np.float32).copy()
        for ds in tab["dataset"].unique():
            m_all = (tab["dataset"] == ds).to_numpy()
            m_fit = m_all & fit_on
            if m_fit.sum() < 10:
                m_fit = m_all
            mu = out[m_fit].mean(axis=0, keepdims=True)
            sd = out[m_fit].std(axis=0, keepdims=True)
            sd[sd < 1e-8] = 1.0
            out[m_all] = (out[m_all] - mu) / sd
        return out


    def cv_reference_predictions(X, y, groups, seed):
        """Out-of-fold predictions on the TRAINING data, used only to choose thresholds."""
        oof = np.full(len(y), np.nan)
        n_splits = min(PARTC_CV_FOLDS, int(y.sum()), int((y == 0).sum()), len(np.unique(groups)))
        if n_splits < 2:
            return oof
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr, te in sgkf.split(X, y, groups):
            if y[tr].sum() == 0 or y[tr].sum() == len(tr):
                continue
            clf = partc_classifier(seed)
            clf.fit(X[tr], y[tr])
            oof[te] = clf.predict_proba(X[te])[:, 1]
        return oof


    def run_matrix(tab, feat_cols):
        split = make_frozen_splits(tab)
        tab = tab.assign(split=split.values)
        is_train = (tab["split"] == "train").to_numpy()

        print("frozen patient-grouped splits (shared by every model):")
        for ds, g in tab.groupby("dataset"):
            for sp, gg in g.groupby("split"):
                print(f"  {ds:<6s} {sp:<5s}  {len(gg):>7,} series  {gg['patient_key'].nunique():>6,} "
                      f"patients  {int(gg[PARTC_TARGET].sum()):>6,} positive ({gg[PARTC_TARGET].mean():.1%})")
        _ov = set(tab.loc[tab.split == "train", "patient_key"]) & set(tab.loc[tab.split == "test", "patient_key"])
        print(f"  patients appearing in both train and test: {len(_ov)} (must be 0)")
        assert not _ov, "patient leakage between the frozen splits"

        # The resume key has to say WHAT the matrix was computed on, not just which cell it was.
        #
        # There are only 18 cells (2 normalisations x 3 train x 3 test), so a key of (norm, train,
        # test) alone marks the whole matrix done after the very first run. Night 1 with 20 India
        # embeddings writes 18 records; night 3, with 3,205, C1 prints the new counts, C2 skips every
        # cell, and C3 renders the AUROC matrix, the paired tests and all the curves from the 20-scan
        # run under a header that says 3,205. This is the third time this exact trap has appeared in
        # this notebook -- preproc_signature() and sweep_signature() exist for the same reason -- so
        # it gets the same treatment.
        sig = "|".join([
            f"n={len(tab)}",
            "".join(f"{d}={int((tab['dataset'] == d).sum())};" for d in sorted(tab["dataset"].unique())),
            f"pos={int(tab[PARTC_TARGET].sum())}",
            f"target={PARTC_TARGET}", f"seed={PARTC_SEED}",
            f"test={PARTC_TEST_FRAC}", f"folds={PARTC_CV_FOLDS}",
            f"feat={len(feat_cols)}",
            f"preproc={globals().get('PREPROC_SIG', 'unknown')}",
            "matrixv=2",
        ])
        # Exported, not just local: cell C3 selects which run to display by this signature, and a
        # local would leave it reading `None` and silently showing every run mixed together.
        globals()["PARTC_MATRIX_SIG"] = sig
        print(f"\nmatrix signature: {sig}")

        done = set()
        n_other = 0
        if os.path.exists(PARTC_RESULTS_PATH):
            with open(PARTC_RESULTS_PATH) as f:
                for line in f:
                    try:
                        r = json.loads(line)
                    except Exception:
                        continue
                    if r.get("matrix_sig") == sig:
                        done.add((r["normalisation"], r["train_on"], r["test_on"]))
                    else:
                        n_other += 1
        if done:
            print(f"resuming: {len(done)}/18 matrix cell(s) already computed for THIS signature")
        if n_other:
            print(f"({n_other} record(s) from a different dataset size or configuration are kept in "
                  f"the file and will be listed separately by cell C3 -- they are not reused here.)")

        TRAIN_SETS = {"rsna_only": ["rsna"], "india_only": ["india"], "combined": ["rsna", "india"]}
        TEST_SETS = {"rsna": ["rsna"], "india": ["india"], "combined": ["rsna", "india"]}

        pred_rows = []
        out = open(PARTC_RESULTS_PATH, "a")
        try:
            for norm in PARTC_NORMALISATIONS:
                X = (tab[feat_cols].to_numpy(dtype=np.float32) if norm == "raw"
                     else per_dataset_zscore(tab, feat_cols, is_train))
                y = tab[PARTC_TARGET].to_numpy().astype(int)
                groups = tab["patient_key"].to_numpy()
                ds = tab["dataset"].to_numpy()

                for train_name, train_ds in TRAIN_SETS.items():
                    tr_mask = is_train & np.isin(ds, train_ds)
                    if y[tr_mask].sum() < 10 or (y[tr_mask] == 0).sum() < 10:
                        print(f"  [{norm}] {train_name}: too few examples to train, skipped")
                        continue

                    t0 = time.time()
                    oof = cv_reference_predictions(X[tr_mask], y[tr_mask], groups[tr_mask], PARTC_SEED)
                    ref_ok = np.isfinite(oof)
                    model = partc_classifier(PARTC_SEED)
                    model.fit(X[tr_mask], y[tr_mask])
                    fit_s = time.time() - t0

                    for test_name, test_ds in TEST_SETS.items():
                        if (norm, train_name, test_name) in done:
                            continue
                        te_mask = (~is_train) & np.isin(ds, test_ds)
                        if te_mask.sum() == 0 or len(np.unique(y[te_mask])) < 2:
                            continue
                        p = model.predict_proba(X[te_mask])[:, 1]
                        ev = full_evaluation(
                            y[te_mask], p, groups[te_mask],
                            y[tr_mask][ref_ok], oof[ref_ok],
                            n_boot=N_BOOTSTRAP_MAIN, seed=PARTC_SEED,
                            label=f"{norm}|{train_name}->{test_name}")
                        rec = {"normalisation": norm, "train_on": train_name, "test_on": test_name,
                                "matrix_sig": sig,
                                "is_transfer": bool(set(train_ds) != set(test_ds)
                                                    and not set(test_ds) <= set(train_ds)),
                                "n_train": int(tr_mask.sum()),
                                "n_train_pos": int(y[tr_mask].sum()),
                                "train_prevalence": float(y[tr_mask].mean()),
                                "fit_seconds": round(fit_s, 1), **ev}
                        out.write(json.dumps(rec) + "\n")
                        out.flush()
                        done.add((norm, train_name, test_name))
                        print(f"  [{norm:<18s}] train {train_name:<11s} -> test {test_name:<9s} "
                              f"AUROC {ev['auroc']:.3f} [{ev['auroc_lo']:.3f}-{ev['auroc_hi']:.3f}]  "
                              f"AP {ev['auprc']:.3f} (base {ev['auprc_baseline']:.3f})  "
                              f"slope {ev['cal_calibration_slope']:.2f}")
                        if norm == "raw" and test_name in ("rsna", "india"):
                            pred_rows.append(pd.DataFrame({
                                "series_uid": tab.loc[te_mask, "series_uid"].to_numpy(),
                                "patient_key": groups[te_mask],
                                "dataset": ds[te_mask],
                                "y": y[te_mask], "p": p,
                                "train_on": train_name, "test_on": test_name,
                            }))
        finally:
            out.close()

        if pred_rows:
            preds = pd.concat(pred_rows, ignore_index=True)
            if os.path.exists(PARTC_PREDICTIONS_PATH):
                old = pd.read_parquet(PARTC_PREDICTIONS_PATH)
                preds = pd.concat([old, preds], ignore_index=True).drop_duplicates(
                    subset=["series_uid", "train_on", "test_on"], keep="last")
            # Read-modify-write on a FUSE mount, so tmp+rename: this file is the ONLY input to C3's
            # paired comparisons and curves, and losing it to a mid-write disconnect would mean
            # re-running the whole matrix to get it back.
            _tmp = PARTC_PREDICTIONS_PATH + ".tmp.parquet"
            preds.to_parquet(_tmp, index=False)
            os.replace(_tmp, PARTC_PREDICTIONS_PATH)
            print(f"\nper-scan test predictions saved -> {PARTC_PREDICTIONS_PATH} "
                  f"({len(preds):,} rows). These are what the paired comparisons and the ROC/PR "
                  f"curves in cell C3 are computed from.")
        return tab


    print(f"target: {PARTC_TARGET}  |  {len(PARTC_FEATURES)} embedding dims  |  "
          f"{N_BOOTSTRAP_MAIN} bootstrap replicates per cell")
    PARTC = run_matrix(PARTC, PARTC_FEATURES)

In [ ]:
# C3. Read the matrix: tables, paired comparisons, curves, and the shift diagnostic  [Part C]

# Part C needs both datasets. C1 reports which side is missing and why; this cell skips quietly
# rather than raising a NameError on something C1 never got far enough to define.
if not globals().get("PARTC_READY", False):
    print("PART C step C3: skipped -- see cell C1 for which dataset is missing.")
else:
    import json
    import os

    import numpy as np
    import pandas as pd

    rows = []
    if os.path.exists(PARTC_RESULTS_PATH):
        with open(PARTC_RESULTS_PATH) as f:
            for line in f:
                try:
                    rows.append(json.loads(line))
                except Exception:
                    pass

    if not rows:
        print("No matrix results yet -- run cell C2.")
    else:
        flat = []
        for r in rows:
            d = {k: v for k, v in r.items() if k not in ("operating_points", "calibration_bins")}
            for op, m in r.get("operating_points", {}).items():
                for k, v in m.items():
                    d[f"{op}__{k}"] = v
            flat.append(d)
        M = pd.DataFrame(flat)
        if "matrix_sig" in M.columns and M["matrix_sig"].nunique() > 1:
            print(f"this results file holds {M['matrix_sig'].nunique()} matrix run(s) at different "
                  f"dataset sizes/configurations:")
            for _sig, _g in M.groupby("matrix_sig"):
                print(f"  [{len(_g):2d} cell(s)]  {_sig}")
            _cur = globals().get("PARTC_MATRIX_SIG")
            if _cur is not None and (M["matrix_sig"] == _cur).any():
                M = M[M["matrix_sig"] == _cur]
                print(f"showing the run cell C2 just did ({len(M)} cell(s)). Everything else stays in "
                      f"the file.")
            else:
                _newest = M.groupby("matrix_sig")["n"].max().idxmax()
                M = M[M["matrix_sig"] == _newest]
                print(f"cell C2 has not run this session -- showing the largest one ({len(M)} cell(s)).")
        M = M.drop_duplicates(subset=["normalisation", "train_on", "test_on"], keep="last")
        M.to_csv(os.path.join(PARTC_DIR, "matrix_results.csv"), index=False)

        ORDER_TR = ["rsna_only", "india_only", "combined"]
        ORDER_TE = ["rsna", "india", "combined"]

        for norm in M["normalisation"].unique():
            sub = M[M["normalisation"] == norm]
            print("=" * 96)
            print(f"NORMALISATION: {norm}")
            print("=" * 96)
            print("\nAUROC with 95% patient-clustered bootstrap CI   (rows = trained on, cols = tested on)")
            hdr = f"{'':<13s}" + "".join(f"{t:>26s}" for t in ORDER_TE if t in set(sub['test_on']))
            print(hdr)
            for tr in ORDER_TR:
                line = f"{tr:<13s}"
                for te in ORDER_TE:
                    r = sub[(sub.train_on == tr) & (sub.test_on == te)]
                    if len(r):
                        r = r.iloc[-1]
                        mark = "*" if r["is_transfer"] else " "
                        line += f"{mark}{r['auroc']:.3f} [{r['auroc_lo']:.3f}-{r['auroc_hi']:.3f}]".rjust(26)
                    else:
                        line += f"{'-':>26s}"
                print(line)
            print("  * = at least part of this test set comes from a dataset the model never trained")
            print("      on. For the single-dataset columns that is true transfer; for the 'combined'")
            print("      column it is half transfer and half in-domain, which is why that column is")
            print("      the least interesting one to read.")

            print(f"\nAverage precision (prevalence baseline in brackets)")
            for tr in ORDER_TR:
                line = f"{tr:<13s}"
                for te in ORDER_TE:
                    r = sub[(sub.train_on == tr) & (sub.test_on == te)]
                    line += (f"{r.iloc[-1]['auprc']:.3f} ({r.iloc[-1]['auprc_baseline']:.3f})".rjust(26)
                             if len(r) else f"{'-':>26s}")
                print(line)

            print(f"\nCalibration on the test set: slope / intercept / Brier   (ideal 1.00 / 0.00)")
            for tr in ORDER_TR:
                line = f"{tr:<13s}"
                for te in ORDER_TE:
                    r = sub[(sub.train_on == tr) & (sub.test_on == te)]
                    if len(r):
                        r = r.iloc[-1]
                        line += (f"{r['cal_calibration_slope']:.2f}/"
                                 f"{r['cal_calibration_intercept']:+.2f}/"
                                 f"{r['cal_brier']:.3f}").rjust(26)
                    else:
                        line += f"{'-':>26s}"
                print(line)
            print()

        # ---- the operating points that matter clinically ----------------------------------------
        print("=" * 96)
        print("OPERATING POINTS (thresholds chosen on each model's own cross-validated TRAINING")
        print("predictions, never on a test set). Wilson 95% intervals.")
        print("=" * 96)
        for norm in M["normalisation"].unique():
            for _, r in M[M["normalisation"] == norm].iterrows():
                print(f"\n[{norm}] train {r['train_on']} -> test {r['test_on']}   "
                      f"n={int(r['n'])}, {int(r['n_pos'])} positive ({r['prevalence']:.1%})"
                      + ("   [TRANSFER]" if r["is_transfer"] else ""))
                print(f"    {'operating point':<18s} {'thr':>5s} {'sens (95% CI)':>21s} "
                      f"{'spec (95% CI)':>21s} {'PPV':>6s} {'NPV':>6s} {'TP':>5s} {'FN':>5s}")
                for op in ("fixed_0.5", "youden", "high_sensitivity"):
                    c = f"{op}__"
                    if c + "sensitivity" not in r:
                        continue
                    se = f"{r[c+'sensitivity']:.3f} [{r[c+'sensitivity_lo']:.2f}-{r[c+'sensitivity_hi']:.2f}]"
                    sp = f"{r[c+'specificity']:.3f} [{r[c+'specificity_lo']:.2f}-{r[c+'specificity_hi']:.2f}]"
                    print(f"    {op:<18s} {r[c+'threshold']:>5.3f} {se:>21s} {sp:>21s} "
                          f"{r[c+'ppv']:>6.3f} {r[c+'npv']:>6.3f} {int(r[c+'tp']):>5d} {int(r[c+'fn']):>5d}")

        # ---- paired head-to-heads on identical test sets ------------------------------------------
        if os.path.exists(PARTC_PREDICTIONS_PATH):
            preds = pd.read_parquet(PARTC_PREDICTIONS_PATH)
            print()
            print("=" * 96)
            print("PAIRED COMPARISONS -- same test scans, same resampled patients each replicate.")
            print("An interval that excludes 0 is a difference that survived resampling; one that")
            print("straddles 0 is not a finding, however large the point estimate looks.")
            print("=" * 96)
            for test_on, g in preds.groupby("test_on"):
                wide = g.pivot_table(index=["series_uid", "patient_key"], columns="train_on",
                                      values="p", aggfunc="last")
                ytab = g.drop_duplicates("series_uid").set_index("series_uid")["y"]
                wide = wide.dropna()
                if wide.empty:
                    continue
                y = ytab.reindex(wide.index.get_level_values(0)).to_numpy().astype(int)
                grp = wide.index.get_level_values(1).to_numpy()
                names = [c for c in ["combined", "rsna_only", "india_only"] if c in wide.columns]
                print(f"\ntest set: {test_on}   (n={len(y):,}, {int(y.sum()):,} positive, "
                      f"{len(np.unique(grp)):,} patients)")
                for i in range(len(names)):
                    for j in range(i + 1, len(names)):
                        a, b = names[i], names[j]
                        d = paired_auroc_difference(y, wide[a].to_numpy(), wide[b].to_numpy(),
                                                     grp, n_boot=N_BOOTSTRAP_MAIN, seed=PARTC_SEED)
                        verdict = ("difference survives resampling"
                                   if (d["lo"] > 0 or d["hi"] < 0) else "not distinguishable")
                        print(f"  {a:<11s} - {b:<11s}  dAUROC {d['auroc_diff']:+.4f} "
                              f"[{d['lo']:+.4f}, {d['hi']:+.4f}]  p={d['p_two_sided']:.4f}   {verdict}")

        # ---- how far apart are the two datasets in embedding space? -------------------------------
        print()
        print("=" * 96)
        print("COVARIATE SHIFT: can a classifier tell the two datasets apart from the embeddings alone?")
        print("=" * 96)
        from sklearn.model_selection import GroupShuffleSplit as _GSS
        _X = PARTC[PARTC_FEATURES].to_numpy(dtype=np.float32)
        _yd = (PARTC["dataset"] == "rsna").astype(int).to_numpy()
        _g = PARTC["patient_key"].to_numpy()
        _tr, _te = next(_GSS(n_splits=1, test_size=0.3, random_state=PARTC_SEED).split(_X, _yd, _g))
        _m = partc_classifier(PARTC_SEED).fit(_X[_tr], _yd[_tr])
        _p = _m.predict_proba(_X[_te])[:, 1]
        _shift = discrimination_metrics(_yd[_te], _p, _g[_te], n_boot=N_BOOTSTRAP_MAIN, seed=PARTC_SEED)
        print(f"  dataset-discrimination AUROC: {_shift['auroc']:.4f} "
              f"[{_shift['auroc_lo']:.4f}-{_shift['auroc_hi']:.4f}]")
        if _shift["auroc"] > 0.98:
            print("  -> The two datasets are essentially perfectly separable in embedding space. Any")
            print("     transfer gap above has a straightforward explanation: a model fitted on one")
            print("     is being applied to a region of the space it never saw. Different scanners,")
            print("     protocols, reconstruction kernels and slice thicknesses all contribute.")
            print("     Compare the two normalisation passes above to see how much of that a simple")
            print("     per-dataset recentering recovers.")
        elif _shift["auroc"] > 0.9:
            print("  -> Substantially separable. Expect transfer to cost real accuracy.")
        else:
            print("  -> Largely overlapping, which is a good sign for transfer.")

        # ---- curves -------------------------------------------------------------------------------
        try:
            import matplotlib.pyplot as plt
            from sklearn.metrics import precision_recall_curve, roc_curve

            if os.path.exists(PARTC_PREDICTIONS_PATH):
                preds = pd.read_parquet(PARTC_PREDICTIONS_PATH)
                tests = [t for t in ["rsna", "india"] if t in set(preds["test_on"])]
                if tests:
                    fig, axes = plt.subplots(2, len(tests), figsize=(6.2 * len(tests), 10),
                                              squeeze=False)
                    colours = {"rsna_only": "#4C78A8", "india_only": "#E45756", "combined": "#54A24B"}
                    for ci, test_on in enumerate(tests):
                        g = preds[preds["test_on"] == test_on]
                        ax_roc, ax_pr = axes[0][ci], axes[1][ci]
                        prev = None
                        for train_on, gg in g.groupby("train_on"):
                            gg = gg.drop_duplicates("series_uid")
                            y, p = gg["y"].to_numpy(), gg["p"].to_numpy()
                            prev = y.mean()
                            fpr, tpr, _ = roc_curve(y, p)
                            ax_roc.plot(fpr, tpr, lw=1.8, color=colours.get(train_on),
                                        label=f"{train_on}  AUROC {fast_auroc(y, p):.3f}")
                            pr, rc, _ = precision_recall_curve(y, p)
                            ax_pr.plot(rc, pr, lw=1.8, color=colours.get(train_on),
                                       label=f"{train_on}  AP {fast_auprc(y, p):.3f}")
                        ax_roc.plot([0, 1], [0, 1], ls="--", lw=1, color="#888888")
                        ax_roc.set_title(f"ROC -- tested on {test_on}")
                        ax_roc.set_xlabel("1 - specificity"); ax_roc.set_ylabel("sensitivity")
                        ax_roc.legend(loc="lower right", fontsize=8, frameon=False)
                        if prev is not None:
                            ax_pr.axhline(prev, ls="--", lw=1, color="#888888",
                                          label=f"prevalence {prev:.3f}")
                        ax_pr.set_title(f"Precision-recall -- tested on {test_on}")
                        ax_pr.set_xlabel("recall (sensitivity)"); ax_pr.set_ylabel("precision (PPV)")
                        ax_pr.legend(loc="upper right", fontsize=8, frameon=False)
                        for a in (ax_roc, ax_pr):
                            a.set_xlim(-0.02, 1.02); a.set_ylim(-0.02, 1.02)
                            a.spines[["top", "right"]].set_visible(False)
                    fig.suptitle("Intracranial hemorrhage from Pillar-0 embeddings "
                                  "(raw features, linear probe)", y=0.995)
                    plt.tight_layout()
                    fp = os.path.join(PARTC_DIR, "roc_pr_curves.png")
                    plt.savefig(fp, dpi=140); plt.show()
                    print(f"\ncurves saved: {fp}")

                    # calibration, which is where transfer usually breaks first
                    fig2, ax2 = plt.subplots(1, len(tests), figsize=(5.6 * len(tests), 5),
                                              squeeze=False)
                    for ci, test_on in enumerate(tests):
                        a = ax2[0][ci]
                        g = preds[preds["test_on"] == test_on]
                        for train_on, gg in g.groupby("train_on"):
                            gg = gg.drop_duplicates("series_uid")
                            b = calibration_metrics(gg["y"].to_numpy(), gg["p"].to_numpy())
                            bins = pd.DataFrame(b["calibration_bins"])
                            if len(bins):
                                a.plot(bins["mean_pred"], bins["observed"], marker="o", ms=4, lw=1.5,
                                       color=colours.get(train_on),
                                       label=f"{train_on}  slope {b['calibration_slope']:.2f}, "
                                             f"ECE {b['ece']:.3f}")
                        a.plot([0, 1], [0, 1], ls="--", lw=1, color="#888888")
                        a.set_title(f"Calibration -- tested on {test_on}")
                        a.set_xlabel("mean predicted probability"); a.set_ylabel("observed frequency")
                        a.legend(loc="upper left", fontsize=8, frameon=False)
                        a.spines[["top", "right"]].set_visible(False)
                    plt.tight_layout()
                    fp2 = os.path.join(PARTC_DIR, "calibration.png")
                    plt.savefig(fp2, dpi=140); plt.show()
                    print(f"calibration curves saved: {fp2}")
        except Exception as e:
            print(f"(plots skipped: {type(e).__name__}: {e})")

        print()
        print("=" * 96)
        print("HOW TO READ THIS")
        print("=" * 96)
        print("* Diagonal cells are internal validation. They are the optimistic numbers.")
        print("* Starred cells are true transfer -- no scan from that dataset was in training. The gap")
        print("  between a starred cell and the diagonal cell in the same COLUMN is the cost of the")
        print("  domain shift, measured on identical patients.")
        print("* `combined` rows are not transfer for either column, so they should beat both transfer")
        print("  cells. Whether they beat the matching single-dataset diagonal is the real question:")
        print("  more data helps, a different distribution hurts, and the paired tests say which won.")
        print("* Read AP against its prevalence baseline, never on its own.")
        print("* A calibration slope far below 1 on a transfer cell means the scores are too extreme")
        print("  for that population -- the ranking may be fine while the probabilities are not, so")
        print("  any fixed threshold carried across datasets will not behave as expected.")

## Notes, provenance and honest limitations

### Where each preprocessing number comes from

| value | source | how it was checked |
|---|---|---|
| 256 × 256 × 128 (H, W, D) head input | arXiv 2511.17803, Table 7 + Appendix A.5 | quoted verbatim: "CTs were cropped or padded to 256 × 256 × 128" |
| 11-channel multi-windowing | same Appendix A.5 | `rve.get_available_windows("CT")` returns exactly 11; cell 2c pins the rve commit and asserts it |
| window centers/widths | `vision_engine/utils/windowing_utils.py` | `rve.apply_windowing` is called directly, never reimplemented |
| 0.5 × 0.5 × 1.25 mm spacing | `rave/configs/ct_brain.yaml` | read from the shipped config |
| linear interpolation, `SetDefaultPixelValue(0)` | `vision_engine/processing/ct_processor.py::_resample_volume` | matched exactly |
| pad value = `float(volume.min())` | `ct_processor.py::_center_crop_pad_3d` | matched exactly |
| center crop `start = (current-target)//2` | `ct_processor.py::_crop_pad_volume` | matched exactly — a true center crop, not stride sampling |
| chest 256×256×256 @ 1.25 mm | Table 7 **and** `ct_chest.yaml` | two independent sources agree |
| abdomen 384×384×384 @ 1.25 mm | Table 7 **and** `ct_abdomen.yaml` | two independent sources agree |
| `extract_vision_feats(image={key: tensor})` | `modeling_clip.py`, `multimodal_atlas.py` | `forward` does `list(x.items())[0]`; a bare tensor cannot work |
| modality key | the loaded model itself | cell 15 reads its `ModuleDict` keys rather than trusting a string |

Arithmetic cross-check on Table 7, which is how you tell a correctly-read table from a
pattern-matched one: head 32·32·32 = 32,768 ("32k"); chest 32·32·64 = 65,536 ("64k"); abdomen
64·64·64 = 262,144 ("256k"). All three land exactly on the paper's stated token counts.

### Known limitations — stated, not hidden

* **Z padding follows the paper, not RVE.** RVE's `_crop_pad_volume` crops Z to 128 but never pads
  it. The paper says "cropped or padded". This notebook pads. If a series resamples to fewer than
  128 slices, the difference between the two behaviours is real: padded air slices at both ends.
  You can see how often it matters — the progress log records each series' native slice count and
  measured spacing, so `n_slices × sz / 1.25 < 128` identifies exactly which volumes were padded.
* **Multi-frame z positions are approximated.** For a single-file multi-frame series a uniform
  slice spacing is assumed; `PerFrameFunctionalGroupsSequence` is not read. Those series are
  flagged in the progress log with a `note`.
* **Non-axial series are excluded, not handled.** Sagittal/coronal reformats are dropped for head,
  because the 128-slice center crop along Z assumes an axial stack. A study whose only remaining
  series is a coronal MPR contributes nothing.
* **One primary series per study by default.** The choice — brain sub-anatomy, native over MPR,
  soft kernel over bone, then most slices — is a defensible heuristic, not a validated one. The
  honest way to check it is `PROCESS_ALL_QUALIFYING_SERIES = True` and comparing per-series
  results; the manifest keeps `priority_rank` so you can slice the comparison afterwards.
* **Labels are report-derived and study-level.** They inherit whatever the report extraction got
  wrong, and they describe the study, not the specific series. `studytype_agrees` records where the
  report's own study type contradicts what the DICOM says was scanned — those labels are the ones
  most likely to be describing a different scan.
* **The same series appearing in two day-zips** (a re-export) would be extracted once and keyed by
  series UID, so the second copy is simply skipped. That is the right behaviour here, but it means
  a genuine re-acquisition sharing a UID would be silently ignored.
* **AMP is on by default.** Cell 16b measures the fp32 agreement; if it reports below ~0.999,
  turn it off. Every progress record carries `amp`, so a mixed run can be audited or partially
  redone.
* **Nothing here is a diagnostic device.** These are retrospective research embeddings and linear
  probes on report-derived labels from a single hospital and, essentially, a single scanner. Every
  number is an internal estimate. External validation is a separate exercise.

## The two datasets, named accurately

**"india"** — the matched hospital scans. Earlier drafts of this notebook called this source
`nellore`; that was wrong and came from a filename, not from the data. By `InstitutionName` the
collection is:

| site | series | share |
|---|---:|---:|
| Government General Hospital **Kadapa** | 28,027 | 66% |
| (blank) | 12,087 | 29% |
| RIMS **Ongole** | 648 | 1.5% |
| GGH **Nellore** (three spellings) | ~957 | 2.3% |

So it is multi-site but heavily dominated by Kadapa, on essentially one scanner (Siemens SOMATOM
go.Now, 41,589 of 42,193 series). An internal-validation number here is close to single-centre
however large the row count looks — which is exactly why Part C's RSNA transfer column is the
interesting one. `DATA_SOURCE = "nellore"` is still accepted as an alias so a half-finished run
does not break, but it prints a note and continues as `"india"`. The manifest now carries
`hospital` and `institution` per series, so a site can be held out or used as a covariate.

**"rsna"** — RSNA 2019 Intracranial Hemorrhage Detection.

## Data layout

All locations are set in `config.yaml` (see the repository README); nothing in this notebook names
a folder. The keys this notebook reads:

```
external_matched_csv   matched-scans table: DICOM metadata + report findings     <- CSV_PATH
external_ct_root       archive tree laid out as <external_ct_root>/<MM>/<DD>.zip  <- CT_ROOT
pillar0_rsna_archive_dir   RSNA split archive (parts/part_00000 ...) and
                           rsna-intracranial-hemorrhage-detection/stage_2_train.csv
```

Some calendar months and individual days are absent from the archive tree, so a manifest count
below the full-year figure is expected rather than a bug; cell 10's coverage report is the
authority.

## Running it overnight

Cell 1 is the only cell that needs you present: it mounts Drive (one click) and reads `HF_TOKEN`
from Colab Secrets. `notebook_login()` is deliberately not used — it renders a widget and waits,
which turns an unattended "Run all" into a notebook that did nothing by morning.

Set up once: key icon in the Colab sidebar → Add new secret → name it exactly `HF_TOKEN`, paste a
token from huggingface.co/settings/tokens, and enable **Notebook access**. That account also needs
access granted at huggingface.co/YalaLab/Pillar0-HeadCT, which is a separate thing from having a
valid token — cell 6 checks both.

**Cell 6 is the preflight**, and it is the reason you can walk away. It verifies, in about two
minutes: the GPU, the token, gated repo access, the model weights actually downloading, every data
path (including opening one real day-zip), that `OUTPUT_DIR` can be written *and read back*, local
disk, and — after a real near-miss — that the run is not still capped at a 20-series trial. With
`UNATTENDED_RUN = True` any failure stops the notebook there rather than at 3am.

Two habits worth keeping, because both have already caused a wasted evening in this project:

* **Anything the notebook checks, it also pins.** An earlier version verified `transformers` was v4
  and then raised when Colab supplied v5 — because the install line never pinned it. A check whose
  precondition nothing establishes is not a safety net, it is a trap that fires on somebody else's
  release schedule. `unattended_test` now reads cell 2's pin list and cell 2b's requirement table
  and fails if they disagree.
* **No shell magics.** Every code cell is plain Python, so the syntax check covers 100% of the
  notebook rather than skipping the `!pip` lines. pip is called through `sys.executable -m pip`
  with its exit status checked, so a failed install stops the run instead of printing into the void.

### What has actually been tested

Twelve suites, all green, run against synthetic data built to reproduce the real failure modes
rather than to pass:

| suite | what it proves |
|---|---|
| `mpf_test` | the multi-part reader is byte-exact against the original tar and zip, including reads that straddle part boundaries, past-EOF, `SEEK_END`, and zero-length parts |
| `corrupt_test` | four induced corruptions are each caught: flipped bytes inside a part, a truncated middle part, a missing part number, and two same-sized parts swapped — with the bad part correctly named |
| `pixelfix_test` | the 12-bit correction reproduces ground-truth HU exactly (air −1000, brain +35) on every synthetic study, and without it the same study reads air at +3096 |
| `fixes_test` | tilted-gantry ordering, a negative-z slice normal, the 20-of-100-slices refusal, pax long names, base-256 tar sizes, and the stale-signature exclusion in the feature table |
| `rsna_resume_test` | a partial header index does not freeze the label cache, and a chunk whose reads mostly failed is not recorded as done |
| `e2e` / `e2e_wide` | full India path on synthetic day-zips, then again with the selection widened — the case that used to silently do nothing |
| `hostile` | a day-zip built to break things: garbage files, truncated members, mixed slice shapes in reverse z order, duplicate SOPs, blank tags, multi-frame, a missing series, a missing month, and two series whose UIDs are prefixes of one another |
| `e2e_rsna` (tar and zip) | full RSNA path, with series labels checked against labels recomputed independently from the DICOMs — 30/30 series on all six subtypes |
| `sweep_test` | the probe finds a planted signal, reports chance for planted noise, and the demographics baseline wins on a planted age confounder |
| `unattended_test` | nothing blocks on a human; every version cell 2b checks is a version cell 2 pins; cells 1–6 run against a mocked Colab; and the failure paths (no token, no GPU, missing data, a trial cap left on) each stop the run with an actionable message |
| `xref_test` | every "see cell N" resolves to a cell that exists, and no constant is read before some cell defines it |
| `partc_test` | the 3 × 3 matrix on data with a known answer: internal validation recovers the signal, transfer degrades under a planted 60° rotation, combined beats transfer in both columns, the shift diagnostic reads 1.0000, and a pure translation moves calibration without moving AUROC |

### If something goes wrong

| symptom | where to look |
|---|---|
| "series not present in this day-zip" for many series | cell 10 — is that month actually uploaded? Then cell 12's `match_mode` for that zip |
| "zip index verification failed" | a real UID collision or a corrupt member; the note carries the header's actual UID |
| large numbers of "undecodable" instances | cell 5's codec self-test — a plugin is not working in this runtime |
| every pair failing with the same message | fix it and just re-run cell 17; failures are never sticky |
| embeddings all identical | cell 19's duplicate check; almost always a series-lookup bug, not the model |
| AUROC ≈ demographics baseline | that is the result. Report it that way |

---

## Part B — RSNA-ICH: where each fact comes from

| claim | source |
|---|---|
| `stage_2_train` / `stage_2_test` folder names (not `_images`) | confirmed against your own Drive copy of `rsna-intracranial-hemorrhage-detection/`, which contains `stage_2_test` and `stage_2_sample_submission.csv` — solution repos disagree on this, so it was checked rather than assumed |
| 752,803 train / 121,232 test images | 674,258 + 78,545 (stage-1 train + test) = 752,803, and 752,803 + 121,232 = 874,035, which matches the image count in the RSNA dataset paper exactly. Two independent chains, same number. |
| labels are per-slice, six rows per image, `ID_<9 hex>_<subtype>` | the label CSV's own format; `any` last, subtypes alphabetical |
| four duplicated images in `stage_1_train.csv` | 4,045,572 rows vs 674,258 × 6 = 4,045,548 → 24 extra rows → 4 images. Cell B3 deduplicates and reports the count it actually saw. |
| 20–60 axial slices, 3–5 mm | RSNA dataset paper (*Radiology: AI*, 2020) |
| the 12-bit `RescaleIntercept` correction | Jeremy Howard's fastai cleaning kernel, copied character-for-character into several medal-winning solutions. **Not** in `fastai.medical.imaging` — its `scaled_px` is a naive `px * slope + intercept`; the fix has always been kernel-side. |
| `ID_6431af929.dcm` has corrupt PixelData | independently confirmed by three solution repos, all of which simply drop it. So does cell B4. |
| slice ordering by the slice normal | general CT practice, and the right thing here specifically because at least one medal-winning solution ships an explicit gantry-tilt correction stage — meaning tilted studies are present, and tilted studies are exactly where sorting on raw `ImagePositionPatient[2]` misorders slices. For an ordinary axial study the two methods are identical. |

### RSNA-specific limitations

* **The correction's recoverable range is `[-1000, +3095]`, by construction.** It asserts that
  stored values are signed HU wrapped into unsigned 12 bits, so anything originally below −1000
  comes back as a large positive number instead. That is inherent to the accepted fix, not
  something introduced here. Volume assembly records each series' 1st/50th/99th HU percentiles and
  flags any series whose air is not near −1000, so if it ever bites you the progress log says so.
* **Most RSNA volumes are padded in z, not cropped.** A 20-slice 5 mm series resamples to 80
  slices at 1.25 mm and a 40-slice 3 mm series to 96 — both under the 128 Pillar-0 expects. The
  India data (median 185 slices at 2 mm) is the opposite case. Cell B4 prints the estimated
  resampled depth distribution before you spend GPU hours. If RSNA results come out systematically
  lower than India's, this is the first thing to look at, not the model.
* **Series-level labels make this an easier task than the competition's.** A series is positive if
  any slice is. No number this notebook prints is comparable to a competition log-loss, and it
  should not be presented as one. Published *exam-level* AUROC for "any" on RSNA is around 0.96,
  which is the right order of magnitude to sanity-check against.
* **Epidural is rare** — roughly 0.4% of slices. With `RSNA_MAX_SERIES` set, check the printed
  per-subtype counts: if epidural lands under `MIN_POSITIVES` it is skipped, and a wide confidence
  interval on it means very little either way.
* **The subsample is proportional, deliberately.** Oversampling the rare compartments would give a
  prettier epidural AUROC at a prevalence matching no real population, and would quietly invalidate
  every average-precision and PPV number in the sweep.
* **`any` is used as labelled, not recomputed** from the five compartments. Cell B3 reports how
  often the two disagree; an annotator can record hemorrhage without committing to a compartment.
* **Patient-level disjointness of the original train/test split is asserted by the dataset paper,
  not verified here** — irrelevant for this notebook, which only uses `stage_2_train` and makes its
  own patient-grouped splits, but worth knowing if you ever bring the test set in.
* **The header scan is the expensive step.** ~752,803 members × ~16 KB. It is threaded, chunked,
  and cached to Drive as parquet; interrupt it freely. If you already have one of the published
  "RSNA train metadata" CSVs, you can skip it by building `RSNA_INDEX` from that instead — the only
  columns cell B4 needs are `member, sop, study_uid, series_uid, patient_id, z, iop, px_x, px_y,
  rows, cols, needs_pixel_fix`.

### Verifying the archive (cell B1b)

There is no checksum file next to the parts, so integrity is established from the archive itself:
part geometry (contiguous numbering, only the last part short), structural parse of the central
directory, and CRC32 verification of a sample of members **stratified across every part**. Stratified
rather than uniform on purpose — uniform sampling only gives a probability of catching a bad part
(200 of 874,035 members leaves an ~11% chance of missing a wholly corrupt 2 GB part out of 91),
while one sample per part-sized region makes it a guarantee at the same read cost. When failures do
occur the cell prints how many landed in each part, which distinguishes one bad upload from a flaky
connection.

The four failure modes were each induced deliberately and confirmed caught: flipped bytes inside a
part, a truncated middle part, a missing part number, and two same-sized parts swapped.

### If the archive will not open

| symptom | what it means |
|---|---|
| "these part(s) are not the same size as part 0 … but are not last" | an upload is still running, or a part is truncated. Wait or re-upload; reading past a short part corrupts every member after it. |
| "the archive is gzip-compressed" | no random access is possible. Re-upload as an uncompressed tar or a zip — DICOM pixel data barely compresses, so you lose almost nothing. |
| "could not identify the archive" | `part_00000` is not the first part. Check `RSNA_PARTS_PATTERN` and the sort order of the names. |
| "no .dcm members matched RSNA_SPLITS" | the folder inside the archive is named differently — the cell prints a sample of what IS there. |

---

## Part C — the hemorrhage matrix, and how to read it honestly

**The task is hemorrhage vs no hemorrhage, and only that.** Subtypes are deliberately out of scope
here: epidural is ~0.4% of RSNA slices, and the Indian side has 145 SDH / 133 SAH / 144 ICH, so a
per-subtype cross-dataset comparison would be reporting the width of its own confidence intervals.
The binary question is the one both datasets can actually answer at usable sample size.

**The negatives are not the same thing on the two sides, and this matters.** An RSNA negative means
*no hemorrhage was annotated* — that scan may still have an infarct, a mass or marked atrophy,
because RSNA annotates hemorrhage and nothing else. A India negative means *the report mentioned
no hemorrhage*, from a report that covers everything the radiologist saw. So:

* the cross-dataset matrix is **hemorrhage vs no hemorrhage**, which is well defined on both;
* **normal vs abnormal** in the everyday sense is only answerable on the Indian side, and is
  reported separately as `abnormal_any_finding`.

Treating the two as the same task would be the single easiest way to produce a number that looks
like a result and is not one.

### What each metric is there for

| metric | what it tells you that AUROC does not |
|---|---|
| AUROC + patient-clustered bootstrap CI | how much of the ranking is real vs sampling noise. Rows are clustered by patient, because one patient contributing several series makes a row-level bootstrap too narrow — measurably so: on correlated rows the clustered interval comes out about 2× wider, and cell 20b's self-test asserts exactly that. |
| Average precision, always shown with its prevalence baseline | at 5% prevalence a random ranker scores 0.05; at 40% it scores 0.40. AP quoted alone is uninterpretable. |
| Sensitivity / specificity / PPV / NPV with **Wilson** intervals | a sensitivity of 0.83 from 6 positives and one from 600 are not the same claim. Wilson rather than normal-approximation, which misbehaves near 0 and 1. |
| Calibration slope, intercept, Brier, ECE | AUROC is invariant to any monotone transform of the scores, so a model can rank perfectly and still emit probabilities that mean nothing. This is where transfer breaks first — a pure shift in the feature distribution leaves ranking untouched and moves the intercept a long way, which is verified directly in the test suite. |
| Paired bootstrap difference | "0.84 vs 0.81" is not a finding until you know whether it survives resampling the same patients. Overlapping individual intervals routinely hide a difference that is consistent in every replicate. |
| Dataset-discrimination AUROC | if a classifier can tell the two datasets apart from the embeddings alone at ~1.0, that IS the explanation for a transfer gap, and no amount of tuning the probe will fix it. |

### Reading the 3 × 3 matrix

* Diagonal cells are internal validation — the optimistic numbers.
* Starred cells are true transfer. The gap between a starred cell and the diagonal cell **in the
  same column** is the cost of the domain shift, measured on identical patients.
* `combined` should beat both transfer cells. Whether it beats each dataset's own diagonal is the
  genuinely interesting question — more data helps, a different distribution hurts, and the paired
  tests are what say which won.
* The second normalisation pass (per-dataset z-scoring) separates *kinds* of shift: an affine
  offset it removes, a rotation of the signal direction it cannot. If z-scoring closes the gap, the
  problem was intensity/scaling; if it does not, the two populations encode the finding differently
  and pooling the training data is the real fix.

### Part C limitations

* **Label provenance differs.** RSNA's labels are expert annotations made for this task; India's
  come out of free-text reports. A gap between the two is partly a measurement of the report
  extraction, not of Pillar-0.
* **A linear probe on frozen features is a lower bound**, not the best achievable. It measures how
  much the representation has already separated — which is the right question for a foundation
  model, and not the same question as "how well could this be made to work".
* **Everything is retrospective and internal.** One Indian hospital and essentially one scanner on
  one side; a competition dataset on the other. Neither is a prospective evaluation, and nothing
  here is a diagnostic device.
* **Series-level RSNA labels make the task easier than the competition's per-slice one.** No number
  here is comparable to a competition log-loss. The right sanity anchor is published *exam-level*
  AUROC for "any hemorrhage" on RSNA, around 0.96.

---

### What to run next

0. Leave `UNATTENDED_RUN = True`. Run cells 1–6 and read the preflight verdict. Cell 1 is the only
   one that needs you present — it mounts Drive and reads `HF_TOKEN` from Colab Secrets. If
   preflight passes, nothing after it will ask you for anything.
1. `TRIAL_RUN_MAX_SERIES = 20`, run cells 7–19. That exercises the full path — zip, decode,
   preprocess, model, save, feature table, QC — on real data in a few minutes.
2. Look at cell 17's time split and try the other `ZIP_ACCESS_MODE` for one zip. Pick with numbers.
3. `TRIAL_RUN_MAX_SERIES = None`, `MAX_SECONDS_PER_RUN` set to fit your session, and let it run.
   Re-run the cell each session until cell 17 reports nothing pending.
4. Run cells 20–22. Read the *lift over demographics* table before the AUROC table.
5. Then the interesting question: flip `REQUIRE_STUDYTYPE_AGREEMENT` to `True` and re-run the
   sweep. The resume key includes that flag, so both versions coexist in the results file and the
   leaderboard shows them side by side. If the numbers move materially, label noise is a bigger
   lever on your results than the model is.

**For RSNA** (`DATA_SOURCE = "rsna"`, which also switches `OUTPUT_DIR`, so the two datasets never
share a checkpoint file):

1. Run cells 1–6, then **B1** and **B1b**. B1b verifies the split archive: part geometry, the
   central directory, and CRC32 on a sample stratified across every part. Stop here if it reports
   anything — everything after depends on the archive being intact.
2. **B2** is the long one: one header pass over ~752,803 members. Chunked and resumable; run it
   until it reports all chunks complete. Set `MAX_SECONDS_PER_RUN` to bound a session.
3. **B3 / B4** are fast. Read B4's printed prevalence table and the resampled-depth estimate before
   committing GPU time.
4. Then cells 11–22 exactly as for India.
5. The comparison worth making at the end: `cohort:any_intracranial_hemorrhage` on RSNA versus the
   same cohort on India, same checkpoint, same preprocessing, same probe. Expert per-slice labels
   on one side, report-derived labels on the other — the gap between them is a measurement of your
   report extraction, not of Pillar-0.

---
## 23. crop vs fit_head, measured

Run this only once both output dirs have a `features_pillar0_headct.parquet`.
It compares the two arms on the series they share, so a bigger cohort cannot masquerade
as a preprocessing win.

---

In [ ]:
# 23. BACKBONE HEAD-TO-HEAD on India: B1 vs B2 vs B3 under ONE protocol
# ---------------------------------------------------------------------------------------------
# This cell used to compare only Pillar-0's two preprocessing arms (crop vs fit_head). It is now
# the general version, and it exists to answer a question the paper cannot dodge.
#
# WHAT IT IS FOR. B1 / B2 / B3 is one of the paper's own axes: Table 1 already compares the
# three backbones on RSNA, and the external-validation section needs the same comparison on the
# Indian cohort. This cell provides it, on the series all three share, with one protocol.
#
# WHY IT IS NOT THE SWEEP. The sweep above removes a 20% holdout and cross-validates the
# remaining pool, so its CV column is computed on ~860 series with ~75 positives and reported
# per backbone separately. This cell pools out-of-fold predictions over ALL 1,200 shared series
# and scores once, with a patient-clustered PAIRED bootstrap between backbones -- pairing is
# what makes a difference between two backbones measurable, since both see identical folds.
#
# The two are not interchangeable: pooled-OOF and cross-validated-after-holdout differ
# systematically when positives are scarce (94 here). Quote one or the other in the paper, and
# say which. Do not put a number from each into the same table.
# ---------------------------------------------------------------------------------------------
import os
import re as _re

import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

# Our backbones, named by backbone id, plus Pillar-0's two arms when they are on Drive.
# Our three frozen backbones, and nothing else. Earlier versions of this cell also loaded the
# Pillar-0 feature tables from the superseded runs; Pillar-0 is not part of this paper and a
# baseline the paper never claims does not belong in its results.
ARMS = {b: os.path.join(OUTPUT_DIR, f"features_{b}.parquet") for b in MODELS_TO_RUN}
# Differences are reported against this backbone. B1 is the natural reference: it is the widest
# representation and the one the RSNA finalist adapter was trained on.
REFERENCE_ARM = MODELS_TO_RUN[0] if MODELS_TO_RUN else None
N_BOOT, SEED, FOLDS = 500, 12345, 5

ARMS = {k: v for k, v in ARMS.items() if os.path.exists(v)}
if REFERENCE_ARM not in ARMS:
    REFERENCE_ARM = next(iter(ARMS), None)

if len(ARMS) < 2:
    print(f"skipping: need at least two backbone feature tables to compare, found {len(ARMS)}. "
          f"Run cell 18 first.")
else:
    frames = {}
    for name, path in ARMS.items():
        d = pd.read_parquet(path)
        idcol = next((c for c in ("series_uid", "series") if c in d.columns), d.columns[0])
        frames[name] = d.set_index(idcol)
        print(f"  {name:<26} {d.shape[0]:,} rows x {d.shape[1]:,} cols")

    shared = sorted(set.intersection(*(set(d.index) for d in frames.values())))
    print(f"\nshared series across {len(ARMS)} arm(s): {len(shared):,}")
    if len(shared) < 100:
        raise SystemExit("too few shared series to compare")

    try:
        cohorts = dict(SPECIAL_COHORTS)
    except NameError:
        raise SystemExit("run cell 20 first -- SPECIAL_COHORTS is not defined")

    ref_tab = frames[REFERENCE_ARM].loc[shared]
    gcol = next((c for c in ("person_key", "patient_id", "patient", "PatientID")
                 if c in ref_tab.columns), None)
    if gcol is None:
        raise SystemExit("no person/patient column on the feature table to group on")
    groups = ref_tab[gcol].astype(str).to_numpy()
    print(f"grouping on {gcol}: {len(np.unique(groups)):,} people across {len(shared):,} scans")

    _FEATCOL = _re.compile(r"^(emb_|f_|dim_)\d+$")

    def featmat(name, d):
        """ONLY the embedding columns.

        The previous version fell back to 'any float column', which would quietly pull age or a
        derived score into the matrix and inflate the arm that happened to carry one. Both
        notebooks zero-pad their feature columns, so an exact pattern is available -- use it,
        and refuse rather than guess when it matches nothing."""
        d = d.loc[shared]
        cols = [c for c in d.columns if _FEATCOL.match(str(c))]
        if not cols:
            raise SystemExit(f"{name}: no columns matching emb_/f_/dim_ + digits; refusing to "
                             f"guess which of {list(d.columns)[:8]}... are features")
        cols = sorted(cols, key=lambda c: int(_re.sub(r"\D", "", c)))
        return d[cols].to_numpy(dtype=np.float32), len(cols)

    def cv_scores(X, y, groups):
        """Pooled out-of-fold probabilities under folds every arm sees identically."""
        oof = np.zeros(len(y), float)
        skf = StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
        for tr, te in skf.split(X, y, groups):
            pipe = Pipeline([("sc", StandardScaler()),
                             ("lr", LogisticRegression(max_iter=4000, class_weight="balanced"))])
            pipe.fit(X[tr], y[tr])
            oof[te] = pipe.predict_proba(X[te])[:, 1]
        return oof

    def paired_boot(y, pa, pb, groups, n=N_BOOT):
        """Patient-clustered paired bootstrap of AUROC(b) - AUROC(a)."""
        rng = np.random.default_rng(SEED)
        uniq = np.unique(groups)
        idx_by_g = {g: np.flatnonzero(groups == g) for g in uniq}
        out = []
        for _ in range(n):
            pick = rng.choice(uniq, size=len(uniq), replace=True)
            ii = np.concatenate([idx_by_g[g] for g in pick])
            if len(np.unique(y[ii])) < 2:
                continue
            out.append(roc_auc_score(y[ii], pb[ii]) - roc_auc_score(y[ii], pa[ii]))
        out = np.array(out)
        return out.mean(), np.percentile(out, 2.5), np.percentile(out, 97.5)

    X, NDIM = {}, {}
    for k, v in frames.items():
        X[k], NDIM[k] = featmat(k, v)
    print("feature widths: " + " | ".join(f"{k} {NDIM[k]:,}d" for k in X))

    rows = []
    for cname, cvals in cohorts.items():
        y = cvals.reindex(shared).to_numpy()
        if pd.isna(y).any() or len(np.unique(y)) < 2 or y.sum() < 10:
            print(f"  skip {cname}: {int(np.nansum(y))} positive on the shared set")
            continue
        y = y.astype(int)
        p = {k: cv_scores(X[k], y, groups) for k in ARMS}
        for k in ARMS:
            d, lo, hi = paired_boot(y, p[REFERENCE_ARM], p[k], groups)
            rows.append(dict(
                cohort=cname, n=len(y), pos=int(y.sum()), arm=k, dims=NDIM[k],
                auroc=round(roc_auc_score(y, p[k]), 3),
                ap=round(average_precision_score(y, p[k]), 3),
                delta_vs_ref=round(d, 3), lo=round(lo, 3), hi=round(hi, 3),
                verdict=("--" if k == REFERENCE_ARM else
                         "BETTER than ref" if lo > 0 else
                         "worse than ref" if hi < 0 else "no difference")))

    res = pd.DataFrame(rows)
    print("\n" + "=" * 104)
    print(f"BACKBONE HEAD-TO-HEAD on India -- same series, same patients, same folds, same "
          f"classifier (reference: {REFERENCE_ARM})")
    print("=" * 104)
    for cname in res["cohort"].unique():
        sub = res[res["cohort"] == cname].sort_values("auroc", ascending=False)
        print(f"\n{cname}  (n={sub['n'].iloc[0]:,}, {sub['pos'].iloc[0]} positive)")
        print(sub[["arm", "dims", "auroc", "ap", "delta_vs_ref", "lo", "hi",
                   "verdict"]].to_string(index=False))
    print("\n  Every AUROC here is a POOLED out-of-fold score on all shared series, so these are")
    print("  comparable with each other and NOT with the sweep's cross-validated column, which")
    print("  is computed on the 80% pool after a holdout is removed. Quote one or the other.")
    print("  [lo, hi] straddling 0 means indistinguishable on this sample -- a real answer.")
    out = os.path.join(OUTPUT_DIR, "india_backbone_head_to_head.csv")
    res.to_csv(out, index=False)
    print(f"\n  -> {out}")
